# EquiNexus external warm-start postprocessing sweeps

This CPU-only notebook reuses validation probability caches from the completed ChroTel and Flat U-Net Kaggle runs. It searches foreground and boundary thresholds, then instance and seed area cutoffs, using only the frozen observation-grouped competition validation split and organizer-aligned scorer. It does not retrain models, use validation labels as training targets, or read public MAGFiLO test annotations. The held-out observations remain excluded from model fitting.

Frozen split SHA-256: `30cb2ec52437daf97e769a3a61bf83350062e6cf4f5d00af1f8399649065e48e`. The source probability maps remain mounted read-only from the two private experiment kernels; only sweep results are written to `/kaggle/working/`.


In [ ]:
import base64, hashlib, json, os, sys
from pathlib import Path
REPOSITORY_PATH = Path('/kaggle/working/equinexus')
REPOSITORY_PATH.mkdir(parents=True, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = str(REPOSITORY_PATH / '.cache' / 'pip')
os.environ['TMP'] = str(REPOSITORY_PATH / '.tmp')
os.environ['TEMP'] = str(REPOSITORY_PATH / '.tmp')
Path(os.environ['TMP']).mkdir(parents=True, exist_ok=True)
SOURCE_FILES = json.loads("{\"src/filament_baseline/__init__.py\": \"IiIiU2NvcmVyLWFsaWduZWQgdmFsaWRhdGlvbiBiYXNlbGluZSBmb3IgRXF1aU5leHVzLiIiIgoKX192ZXJzaW9uX18gPSAiMC4xLjAiCg==\", \"src/filament_baseline/classical.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmRhdGEgaW1wb3J0IElNQUdFX1NJWkUKZnJvbSAucHJlZGljdG9yIGltcG9ydCBQcmVkaWN0ZWRJbnN0YW5jZSwgZW5jb2RlX2Nyb3BwZWRfYmluYXJ5X21hc2sKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDbGFzc2ljYWxDb25maWc6CiAgICB6X3RocmVzaG9sZDogZmxvYXQgPSAyLjU1CiAgICBicmlkZ2Vfc2l6ZTogaW50ID0gOQogICAgbWluX2luc3RhbmNlX2FyZWE6IGludCA9IDE2NQogICAgbGltYl9mcmFjdGlvbjogZmxvYXQgPSAwLjk0NQoKCmRlZiBfZGlza19nZW9tZXRyeShncmF5OiBucC5uZGFycmF5KSAtPiB0dXBsZVtpbnQsIGludCwgaW50XToKICAgIGhlaWdodCwgd2lkdGggPSBncmF5LnNoYXBlCiAgICByZWR1Y2VkID0gY3YyLnJlc2l6ZShncmF5LCAobWF4KDEsIHdpZHRoIC8vIDQpLCBtYXgoMSwgaGVpZ2h0IC8vIDQpKSwgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfQVJFQSkKICAgIF8sIGJyaWdodCA9IGN2Mi50aHJlc2hvbGQocmVkdWNlZCwgMCwgMjU1LCBjdjIuVEhSRVNIX0JJTkFSWSB8IGN2Mi5USFJFU0hfT1RTVSkKICAgIGNvbnRvdXJzLCBfID0gY3YyLmZpbmRDb250b3VycyhicmlnaHQsIGN2Mi5SRVRSX0VYVEVSTkFMLCBjdjIuQ0hBSU5fQVBQUk9YX1NJTVBMRSkKICAgIGlmIGNvbnRvdXJzOgogICAgICAgIChjZW50ZXJfeCwgY2VudGVyX3kpLCByYWRpdXMgPSBjdjIubWluRW5jbG9zaW5nQ2lyY2xlKG1heChjb250b3Vycywga2V5PWN2Mi5jb250b3VyQXJlYSkpCiAgICAgICAgY2VudGVyX3gsIGNlbnRlcl95LCByYWRpdXMgPSBpbnQocm91bmQoY2VudGVyX3ggKiA0KSksIGludChyb3VuZChjZW50ZXJfeSAqIDQpKSwgaW50KHJvdW5kKHJhZGl1cyAqIDQpKQogICAgICAgIGlmIDY1MCA8PSByYWRpdXMgPD0gMTAyMDoKICAgICAgICAgICAgcmV0dXJuIGNlbnRlcl94LCBjZW50ZXJfeSwgcmFkaXVzCiAgICByZXR1cm4gd2lkdGggLy8gMiwgaGVpZ2h0IC8vIDIsIGludCgwLjQzICogbWluKGhlaWdodCwgd2lkdGgpKQoKCmRlZiBjb21wdXRlX3NhbGllbmNlX3ptYXAoZ3JheV9pbWFnZTogbnAubmRhcnJheSwgbGltYl9mcmFjdGlvbjogZmxvYXQgPSAwLjk0NSkgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICAiIiJFc3RpbWF0ZSBkaXNrLW5vcm1hbGl6ZWQgZGFyayBmaWxhbWVudCByZXNwb25zZSBhbmQgYSBzYWZlLWRpc2sgbWFzay4iIiIKICAgIGlmIGdyYXlfaW1hZ2UubmRpbSAhPSAyIG9yIGdyYXlfaW1hZ2UuZHR5cGUgIT0gbnAudWludDg6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgYW4gOC1iaXQgZ3JheXNjYWxlIGltYWdlIikKICAgIGlmIGdyYXlfaW1hZ2Uuc2hhcGUgIT0gKElNQUdFX1NJWkUsIElNQUdFX1NJWkUpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCB7SU1BR0VfU0laRX14e0lNQUdFX1NJWkV9LCBnb3Qge2dyYXlfaW1hZ2Uuc2hhcGV9IikKCiAgICByZWR1Y2VkID0gY3YyLnJlc2l6ZShncmF5X2ltYWdlLCAoMTAyNCwgMTAyNCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX0FSRUEpLmFzdHlwZShucC5mbG9hdDMyKQogICAgY2VudGVyX3gsIGNlbnRlcl95LCByYWRpdXMgPSBfZGlza19nZW9tZXRyeShyZWR1Y2VkLmFzdHlwZShucC51aW50OCkpCiAgICB5eSwgeHggPSBucC5vZ3JpZFs6MTAyNCwgOjEwMjRdCiAgICBzYWZlX2Rpc2sgPSAoeHggLSBjZW50ZXJfeCkgKiogMiArICh5eSAtIGNlbnRlcl95KSAqKiAyIDw9IChsaW1iX2ZyYWN0aW9uICogcmFkaXVzKSAqKiAyCgogICAgYnJvYWQgPSBjdjIuR2F1c3NpYW5CbHVyKHJlZHVjZWQsICgwLCAwKSwgc2lnbWFYPTI4LjApCiAgICBuYXJyb3cgPSBjdjIuR2F1c3NpYW5CbHVyKHJlZHVjZWQsICgwLCAwKSwgc2lnbWFYPTE0LjApCiAgICBsb2NhbF9iYWNrZ3JvdW5kID0gMC42NSAqIGJyb2FkICsgMC4zNSAqIG5hcnJvdwogICAgYWJzb3JwdGlvbiA9IG5wLmNsaXAoCiAgICAgICAgKGxvY2FsX2JhY2tncm91bmQgLSByZWR1Y2VkKSAvIG5wLm1heGltdW0obG9jYWxfYmFja2dyb3VuZCwgMjUuMCksIDAuMCwgMS4wCiAgICApCgogICAgcmVkdWNlZF91OCA9IG5wLmNsaXAocmVkdWNlZCwgMCwgMjU1KS5hc3R5cGUobnAudWludDgpCiAgICByZXNwb25zZXMgPSBbXQogICAgZm9yIHNpemUgaW4gKDksIDE5LCAzMyk6CiAgICAgICAgZWxlbWVudCA9IGN2Mi5nZXRTdHJ1Y3R1cmluZ0VsZW1lbnQoY3YyLk1PUlBIX0VMTElQU0UsIChzaXplLCBzaXplKSkKICAgICAgICByZXNwb25zZSA9IGN2Mi5tb3JwaG9sb2d5RXgocmVkdWNlZF91OCwgY3YyLk1PUlBIX0JMQUNLSEFULCBlbGVtZW50KQogICAgICAgIHJlc3BvbnNlcy5hcHBlbmQocmVzcG9uc2UuYXN0eXBlKG5wLmZsb2F0MzIpIC8gMjU1LjApCiAgICBzYWxpZW5jZSA9ICgKICAgICAgICAwLjUwICogYWJzb3JwdGlvbgogICAgICAgICsgMC4xNSAqIHJlc3BvbnNlc1swXQogICAgICAgICsgMC4yMCAqIHJlc3BvbnNlc1sxXQogICAgICAgICsgMC4xNSAqIHJlc3BvbnNlc1syXQogICAgKQogICAgZGlza192YWx1ZXMgPSBzYWxpZW5jZVtzYWZlX2Rpc2tdCiAgICBtZWRpYW4gPSBmbG9hdChucC5tZWRpYW4oZGlza192YWx1ZXMpKQogICAgcm9idXN0X3NjYWxlID0gMS40ODI2ICogZmxvYXQobnAubWVkaWFuKG5wLmFicyhkaXNrX3ZhbHVlcyAtIG1lZGlhbikpKSArIDFlLTUKICAgIHpfbWFwID0gKHNhbGllbmNlIC0gbWVkaWFuKSAvIHJvYnVzdF9zY2FsZQogICAgel9tYXBbfnNhZmVfZGlza10gPSAtbnAuaW5mCiAgICByZXR1cm4gel9tYXAuYXN0eXBlKG5wLmZsb2F0MzIpLCBzYWZlX2Rpc2sKCgpkZWYgcHJlZGljdGlvbnNfZnJvbV96bWFwKAogICAgel9tYXA6IG5wLm5kYXJyYXksCiAgICBzYWZlX2Rpc2s6IG5wLm5kYXJyYXksCiAgICBmaWxlX25hbWU6IHN0ciwKICAgIGNvbmZpZzogQ2xhc3NpY2FsQ29uZmlnIHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV06CiAgICBjb25maWcgPSBjb25maWcgb3IgQ2xhc3NpY2FsQ29uZmlnKCkKICAgIGlmIHpfbWFwLnNoYXBlICE9ICgxMDI0LCAxMDI0KSBvciBzYWZlX2Rpc2suc2hhcGUgIT0gel9tYXAuc2hhcGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgYSAxMDI0eDEwMjQgc2FsaWVuY2UgbWFwIGFuZCBkaXNrIG1hc2siKQogICAgaWYgY29uZmlnLmJyaWRnZV9zaXplIDwgMSBvciBjb25maWcuYnJpZGdlX3NpemUgJSAyID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYnJpZGdlX3NpemUgbXVzdCBiZSBhIHBvc2l0aXZlIG9kZCBudW1iZXIiKQoKICAgIHdlYWsgPSAoKHpfbWFwID49IGNvbmZpZy56X3RocmVzaG9sZCkgJiBzYWZlX2Rpc2spLmFzdHlwZShucC51aW50OCkKICAgIHN0cm9uZyA9ICh6X21hcCA+PSBjb25maWcuel90aHJlc2hvbGQgKyAwLjU1KSAmIHNhZmVfZGlzawogICAgaWYgY29uZmlnLmJyaWRnZV9zaXplID4gMToKICAgICAgICBicmlkZ2UgPSBjdjIuZ2V0U3RydWN0dXJpbmdFbGVtZW50KAogICAgICAgICAgICBjdjIuTU9SUEhfRUxMSVBTRSwgKGNvbmZpZy5icmlkZ2Vfc2l6ZSwgY29uZmlnLmJyaWRnZV9zaXplKQogICAgICAgICkKICAgICAgICB3ZWFrID0gY3YyLm1vcnBob2xvZ3lFeCh3ZWFrLCBjdjIuTU9SUEhfQ0xPU0UsIGJyaWRnZSkKICAgICAgICB3ZWFrW35zYWZlX2Rpc2tdID0gMAoKICAgIGNvbXBvbmVudF9jb3VudCwgbGFiZWxzLCBzdGF0cywgXyA9IGN2Mi5jb25uZWN0ZWRDb21wb25lbnRzV2l0aFN0YXRzKAogICAgICAgIHdlYWssIGNvbm5lY3Rpdml0eT04CiAgICApCiAgICBtaW5fcmVkdWNlZF9hcmVhID0gbWF4KDIwLCBjb25maWcubWluX2luc3RhbmNlX2FyZWEgLy8gNCkKICAgIGltYWdlX3N0ZW0gPSBQYXRoKGZpbGVfbmFtZSkuc3RlbQogICAgcHJlZGljdGlvbnMgPSBbXQogICAgZm9yIGNvbXBvbmVudF9pZCBpbiByYW5nZSgxLCBjb21wb25lbnRfY291bnQpOgogICAgICAgIHJlZHVjZWRfYXJlYSA9IGludChzdGF0c1tjb21wb25lbnRfaWQsIGN2Mi5DQ19TVEFUX0FSRUFdKQogICAgICAgIGlmIHJlZHVjZWRfYXJlYSA8IG1pbl9yZWR1Y2VkX2FyZWE6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgY29tcG9uZW50ID0gbGFiZWxzID09IGNvbXBvbmVudF9pZAogICAgICAgIGlmIGludChucC5jb3VudF9ub256ZXJvKHN0cm9uZ1tjb21wb25lbnRdKSkgPCA0OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNvbnRvdXJzLCBfID0gY3YyLmZpbmRDb250b3VycygKICAgICAgICAgICAgY29tcG9uZW50LmFzdHlwZShucC51aW50OCksIGN2Mi5SRVRSX0VYVEVSTkFMLCBjdjIuQ0hBSU5fQVBQUk9YX05PTkUKICAgICAgICApCiAgICAgICAgaWYgY29udG91cnM6CiAgICAgICAgICAgIHBlcmltZXRlciA9IG1heChmbG9hdChjdjIuYXJjTGVuZ3RoKGNvbnRvdXJzWzBdLCBUcnVlKSksIDEuMCkKICAgICAgICAgICAgY29tcGFjdG5lc3MgPSA0LjAgKiBucC5waSAqIHJlZHVjZWRfYXJlYSAvIChwZXJpbWV0ZXIgKiBwZXJpbWV0ZXIpCiAgICAgICAgICAgIGlmIGNvbXBhY3RuZXNzID4gMC43OCBhbmQgcmVkdWNlZF9hcmVhIDwgMTgwIGFuZCBmbG9hdCh6X21hcFtjb21wb25lbnRdLm1heCgpKSA+IDYuNToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHgwID0gaW50KHN0YXRzW2NvbXBvbmVudF9pZCwgY3YyLkNDX1NUQVRfTEVGVF0pCiAgICAgICAgeTAgPSBpbnQoc3RhdHNbY29tcG9uZW50X2lkLCBjdjIuQ0NfU1RBVF9UT1BdKQogICAgICAgIGJveF93aWR0aCA9IGludChzdGF0c1tjb21wb25lbnRfaWQsIGN2Mi5DQ19TVEFUX1dJRFRIXSkKICAgICAgICBib3hfaGVpZ2h0ID0gaW50KHN0YXRzW2NvbXBvbmVudF9pZCwgY3YyLkNDX1NUQVRfSEVJR0hUXSkKICAgICAgICBzbWFsbF9jcm9wID0gY29tcG9uZW50W3kwIDogeTAgKyBib3hfaGVpZ2h0LCB4MCA6IHgwICsgYm94X3dpZHRoXS5hc3R5cGUobnAudWludDgpCiAgICAgICAgZnVsbF9jcm9wID0gY3YyLnJlc2l6ZSgKICAgICAgICAgICAgc21hbGxfY3JvcCwKICAgICAgICAgICAgKGJveF93aWR0aCAqIDIsIGJveF9oZWlnaHQgKiAyKSwKICAgICAgICAgICAgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfTElORUFSLAogICAgICAgICkgPiAwCiAgICAgICAgY3JvcF95LCBjcm9wX3ggPSBucC5ub256ZXJvKGZ1bGxfY3JvcCkKICAgICAgICBmdWxsX3ggPSB4MCAqIDIgKyBpbnQoY3JvcF94Lm1pbigpKQogICAgICAgIGZ1bGxfeSA9IHkwICogMiArIGludChjcm9wX3kubWluKCkpCiAgICAgICAgdGlnaHRfY3JvcCA9IGZ1bGxfY3JvcFsKICAgICAgICAgICAgY3JvcF95Lm1pbigpIDogY3JvcF95Lm1heCgpICsgMSwKICAgICAgICAgICAgY3JvcF94Lm1pbigpIDogY3JvcF94Lm1heCgpICsgMSwKICAgICAgICBdCiAgICAgICAgYXJlYSA9IGludChucC5jb3VudF9ub256ZXJvKHRpZ2h0X2Nyb3ApKQogICAgICAgIGlmIGFyZWEgPCBjb25maWcubWluX2luc3RhbmNlX2FyZWE6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcHJlZGljdGlvbnMuYXBwZW5kKAogICAgICAgICAgICBQcmVkaWN0ZWRJbnN0YW5jZSgKICAgICAgICAgICAgICAgIGZpbGFtZW50X2lkPWYie2ltYWdlX3N0ZW19X3tsZW4ocHJlZGljdGlvbnMpICsgMX0iLAogICAgICAgICAgICAgICAgaW1hZ2Vfc3RlbT1pbWFnZV9zdGVtLAogICAgICAgICAgICAgICAgYXJlYT1hcmVhLAogICAgICAgICAgICAgICAgcmxlPWVuY29kZV9jcm9wcGVkX2JpbmFyeV9tYXNrKAogICAgICAgICAgICAgICAgICAgIHRpZ2h0X2Nyb3AsCiAgICAgICAgICAgICAgICAgICAgZnVsbF94LAogICAgICAgICAgICAgICAgICAgIGZ1bGxfeSwKICAgICAgICAgICAgICAgICAgICAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSksCiAgICAgICAgICAgICAgICApLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgcmV0dXJuIHByZWRpY3Rpb25zCgoKZGVmIHByZWRpY3RfY2xhc3NpY2FsKAogICAgZ3JheV9pbWFnZTogbnAubmRhcnJheSwKICAgIGZpbGVfbmFtZTogc3RyLAogICAgY29uZmlnOiBDbGFzc2ljYWxDb25maWcgfCBOb25lID0gTm9uZSwKKSAtPiBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXToKICAgIGNvbmZpZyA9IGNvbmZpZyBvciBDbGFzc2ljYWxDb25maWcoKQogICAgel9tYXAsIHNhZmVfZGlzayA9IGNvbXB1dGVfc2FsaWVuY2Vfem1hcChncmF5X2ltYWdlLCBjb25maWcubGltYl9mcmFjdGlvbikKICAgIHJldHVybiBwcmVkaWN0aW9uc19mcm9tX3ptYXAoel9tYXAsIHNhZmVfZGlzaywgZmlsZV9uYW1lLCBjb25maWcpCg==\", \"src/filament_baseline/classical_runner.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmNsYXNzaWNhbCBpbXBvcnQgQ2xhc3NpY2FsQ29uZmlnLCBjb21wdXRlX3NhbGllbmNlX3ptYXAsIHByZWRpY3Rpb25zX2Zyb21fem1hcApmcm9tIC5kYXRhIGltcG9ydCBpbnZlbnRvcnksIGxvYWRfZGF0YXNldApmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IHJlYWRfZ3JheXNjYWxlCmZyb20gLnJ1bm5lciBpbXBvcnQgKAogICAgX2dpdF9zdGF0dXMsCiAgICBfcmVzb2x2ZV9kYXRhX3Jvb3QsCiAgICBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzLAogICAgX3dyaXRlX292ZXJsYXlzLAogICAgX3dyaXRlX3ByZWRpY3Rpb25zLAopCmZyb20gLnJ1bnRpbWUgaW1wb3J0IGhhcmR3YXJlX2luZm8sIHBhY2thZ2VfdmVyc2lvbnMKZnJvbSAuc2NvcmVyIGltcG9ydCBzY29yZV92YWxpZGF0aW9uCmZyb20gLnNwbGl0IGltcG9ydCBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQKZnJvbSAudmlzdWFscyBpbXBvcnQgd3JpdGVfY29tcGFyaXNvbl9vdmVybGF5cwoKCmRlZiBydW5fY2xhc3NpY2FsX3ZhbGlkYXRpb24oCiAgICAqLAogICAgcmVwb19yb290OiBzdHIgfCBQYXRoLAogICAgZGF0YV9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSwKICAgIHJ1bl9uYW1lOiBzdHIsCiAgICBvYnNlcnZhdGlvbl9saW1pdDogaW50IHwgTm9uZSA9IE5vbmUsCiAgICBzaW5nbGVfY29uZmlnOiBDbGFzc2ljYWxDb25maWcgfCBOb25lID0gTm9uZSwKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHJlcG9fcm9vdCA9IFBhdGgocmVwb19yb290KS5yZXNvbHZlKCkKICAgIGRhdGFzZXQgPSBsb2FkX2RhdGFzZXQoX3Jlc29sdmVfZGF0YV9yb290KGRhdGFfcm9vdCwgcmVwb19yb290KSkKICAgIGNvbmZpZyA9IGpzb24ubG9hZHMoKHJlcG9fcm9vdCAvICJjb25maWdzIiAvICJ2YWxpZGF0aW9uLWJhc2VsaW5lLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGNvbmZpZ19zaGEyNTYgPSBoYXNobGliLnNoYTI1NigKICAgICAgICAocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfYnl0ZXMoKQogICAgKS5oZXhkaWdlc3QoKQogICAgc3BsaXQgPSBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICByZXBvX3Jvb3QgLyAiZGF0YSIgLyAic3BsaXRzIiwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdChjb25maWdbInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICApCiAgICBpZiBzcGxpdC5zaGEyNTYgIT0gY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBjbGFzc2ljYWwgZXhwZXJpbWVudCByZXF1aXJlcyB0aGUgZnJvemVuIHNwbGl0IikKICAgIGFsbF92YWxpZGF0aW9uX25hbWVzID0gbGlzdChzcGxpdC52YWxpZGF0aW9uX25hbWVzKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGFsbF92YWxpZGF0aW9uX25hbWVzCiAgICBpZiBvYnNlcnZhdGlvbl9saW1pdCBpcyBub3QgTm9uZToKICAgICAgICBpZiBvYnNlcnZhdGlvbl9saW1pdCA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm9ic2VydmF0aW9uX2xpbWl0IG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIGlmIG9ic2VydmF0aW9uX2xpbWl0IDwgbGVuKGFsbF92YWxpZGF0aW9uX25hbWVzKToKICAgICAgICAgICAgaWYgb2JzZXJ2YXRpb25fbGltaXQgPT0gMToKICAgICAgICAgICAgICAgIHZhbGlkYXRpb25fbmFtZXMgPSBbYWxsX3ZhbGlkYXRpb25fbmFtZXNbbGVuKGFsbF92YWxpZGF0aW9uX25hbWVzKSAvLyAyXV0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGluZGljZXMgPSBzb3J0ZWQoCiAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICByb3VuZCgKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGluZGV4ICogKGxlbihhbGxfdmFsaWRhdGlvbl9uYW1lcykgLSAxKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgLyAob2JzZXJ2YXRpb25fbGltaXQgLSAxKQogICAgICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpbmRleCBpbiByYW5nZShvYnNlcnZhdGlvbl9saW1pdCkKICAgICAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICB2YWxpZGF0aW9uX25hbWVzID0gW2FsbF92YWxpZGF0aW9uX25hbWVzW2luZGV4XSBmb3IgaW5kZXggaW4gaW5kaWNlc10KICAgIHJ1bl9yb290ID0gcmVwb19yb290IC8gImFydGlmYWN0cyIgLyAiY2xhc3NpY2FsLWV4cGVyaW1lbnRzIiAvIHJ1bl9uYW1lCiAgICBydW5fcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICByZXBvc2l0b3J5X2luZm8gPSBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3QpCiAgICBjYWNoZV9kaXIgPSBydW5fcm9vdCAvICJzYWxpZW5jZS16bWFwcyIKICAgIGNhY2hlX2Rpci5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGZvciBpbmRleCwgZmlsZV9uYW1lIGluIGVudW1lcmF0ZSh2YWxpZGF0aW9uX25hbWVzLCBzdGFydD0xKToKICAgICAgICBjYWNoZV9wYXRoID0gY2FjaGVfZGlyIC8gZiJ7UGF0aChmaWxlX25hbWUpLnN0ZW19Lm5weiIKICAgICAgICBpZiBjYWNoZV9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpbWFnZSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyIC8gZmlsZV9uYW1lKQogICAgICAgIHpfbWFwLCBkaXNrID0gY29tcHV0ZV9zYWxpZW5jZV96bWFwKGltYWdlKQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoCiAgICAgICAgICAgIGNhY2hlX3BhdGgsCiAgICAgICAgICAgIHpfbWFwPXpfbWFwLmFzdHlwZShucC5mbG9hdDE2KSwKICAgICAgICAgICAgZGlzaz1kaXNrLmFzdHlwZShucC51aW50OCksCiAgICAgICAgKQogICAgICAgIGlmIGluZGV4ICUgMjAgPT0gMCBvciBpbmRleCA9PSBsZW4odmFsaWRhdGlvbl9uYW1lcyk6CiAgICAgICAgICAgIHByaW50KGYiQ2FjaGVkIHtpbmRleH0ve2xlbih2YWxpZGF0aW9uX25hbWVzKX0gY2xhc3NpY2FsIHNhbGllbmNlIG1hcHMiLCBmbHVzaD1UcnVlKQogICAgcHJlcHJvY2Vzc2luZ19zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKCiAgICBjYW5kaWRhdGVzID0gKAogICAgICAgIFtzaW5nbGVfY29uZmlnXQogICAgICAgIGlmIHNpbmdsZV9jb25maWcgaXMgbm90IE5vbmUKICAgICAgICBlbHNlIFsKICAgICAgICAgICAgQ2xhc3NpY2FsQ29uZmlnKAogICAgICAgICAgICAgICAgel90aHJlc2hvbGQ9el90aHJlc2hvbGQsCiAgICAgICAgICAgICAgICBicmlkZ2Vfc2l6ZT1icmlkZ2Vfc2l6ZSwKICAgICAgICAgICAgICAgIG1pbl9pbnN0YW5jZV9hcmVhPW1pbl9hcmVhLAogICAgICAgICAgICApCiAgICAgICAgICAgIGZvciB6X3RocmVzaG9sZCwgYnJpZGdlX3NpemUsIG1pbl9hcmVhIGluIGl0ZXJ0b29scy5wcm9kdWN0KAogICAgICAgICAgICAgICAgKDIuOSwgMy4yKSwgKDUsIDksIDEzKSwgKDI1MCwgNTAwKQogICAgICAgICAgICApCiAgICAgICAgXQogICAgKQogICAgc3dlZXBfcmVzdWx0cyA9IFtdCiAgICBiZXN0X3BxID0gLTEuMAogICAgc2NvcmluZ19zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIGluZGV4LCBjYW5kaWRhdGUgaW4gZW51bWVyYXRlKGNhbmRpZGF0ZXMsIHN0YXJ0PTEpOgogICAgICAgIHByZWRpY3Rpb25zID0ge30KICAgICAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgICAgIGNhY2hlX3BhdGggPSBjYWNoZV9kaXIgLyBmIntQYXRoKGZpbGVfbmFtZSkuc3RlbX0ubnB6IgogICAgICAgICAgICB3aXRoIG5wLmxvYWQoY2FjaGVfcGF0aCkgYXMgY2FjaGVkOgogICAgICAgICAgICAgICAgcHJlZGljdGlvbnNbZmlsZV9uYW1lXSA9IHByZWRpY3Rpb25zX2Zyb21fem1hcCgKICAgICAgICAgICAgICAgICAgICBjYWNoZWRbInpfbWFwIl0uYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICAgICAgICAgIGNhY2hlZFsiZGlzayJdLmFzdHlwZShib29sKSwKICAgICAgICAgICAgICAgICAgICBmaWxlX25hbWUsCiAgICAgICAgICAgICAgICAgICAgY2FuZGlkYXRlLAogICAgICAgICAgICAgICAgKQogICAgICAgIG1ldHJpY3MsIHJvd3MgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgICAgICBkYXRhc2V0LAogICAgICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdChjb25maWdbImlvdV90aHJlc2hvbGQiXSksCiAgICAgICAgICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ9ZmxvYXQoY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0pLAogICAgICAgICkKICAgICAgICBnbG9iYWxfbWV0cmljcyA9IG1ldHJpY3NbImdsb2JhbCJdCiAgICAgICAgcm93ID0gewogICAgICAgICAgICAiY29uZmlnIjogY2FuZGlkYXRlLl9fZGljdF9fLAogICAgICAgICAgICAicHEiOiBnbG9iYWxfbWV0cmljc1sicHEiXSwKICAgICAgICAgICAgInRwIjogZ2xvYmFsX21ldHJpY3NbInRwIl0sCiAgICAgICAgICAgICJmcCI6IGdsb2JhbF9tZXRyaWNzWyJmcCJdLAogICAgICAgICAgICAiZm4iOiBnbG9iYWxfbWV0cmljc1siZm4iXSwKICAgICAgICAgICAgInByZWRpY3Rpb25zX2Fjcm9zc192aWV3cyI6IGdsb2JhbF9tZXRyaWNzWwogICAgICAgICAgICAgICAgInByZWRpY3Rpb25fY291bnRfYWNyb3NzX2Fubm90YXRvcl92aWV3cyIKICAgICAgICAgICAgXSwKICAgICAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLAogICAgICAgIH0KICAgICAgICBzd2VlcF9yZXN1bHRzLmFwcGVuZChyb3cpCiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiW3tpbmRleH0ve2xlbihjYW5kaWRhdGVzKX1dIHo9e2NhbmRpZGF0ZS56X3RocmVzaG9sZDouMmZ9ICIKICAgICAgICAgICAgZiJicmlkZ2U9e2NhbmRpZGF0ZS5icmlkZ2Vfc2l6ZX0gYXJlYT17Y2FuZGlkYXRlLm1pbl9pbnN0YW5jZV9hcmVhfSAiCiAgICAgICAgICAgIGYiUFE9e3Jvd1sncHEnXTouNmZ9IFRQPXtyb3dbJ3RwJ119IEZQPXtyb3dbJ2ZwJ119IEZOPXtyb3dbJ2ZuJ119IiwKICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICApCiAgICAgICAgaWYgcm93WyJwcSJdID4gYmVzdF9wcToKICAgICAgICAgICAgYmVzdF9wcSA9IHJvd1sicHEiXQogICAgICAgICAgICBiZXN0X3Jvb3QgPSBydW5fcm9vdCAvICgKICAgICAgICAgICAgICAgICJiZXN0LXNjcmVlbiIgaWYgb2JzZXJ2YXRpb25fbGltaXQgaXMgbm90IE5vbmUgZWxzZSAiYmVzdCIKICAgICAgICAgICAgKQogICAgICAgICAgICBiZXN0X3Jvb3QubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgKGJlc3Rfcm9vdCAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgICAgICB7InNwbGl0X3NoYTI1NiI6IHNwbGl0LnNoYTI1NiwgImNvbmZpZyI6IGNhbmRpZGF0ZS5fX2RpY3RfXywgIm1ldHJpY3MiOiBtZXRyaWNzfSwKICAgICAgICAgICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICAgICAgICAgICAgICBzb3J0X2tleXM9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICsgIlxuIiwKICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgICAgICkKICAgICAgICAgICAgX3dyaXRlX3ByZWRpY3Rpb25zKGJlc3Rfcm9vdCAvICJwcmVkaWN0aW9ucy5jc3YiLCBwcmVkaWN0aW9ucykKICAgICAgICAgICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcyhiZXN0X3Jvb3QgLyAicGVyX29ic2VydmF0aW9uLmNzdiIsIHJvd3MsIHByZWRpY3Rpb25zKQogICAgICAgICAgICBfd3JpdGVfb3ZlcmxheXMoZGF0YXNldCwgcHJlZGljdGlvbnMsIGJlc3Rfcm9vdCAvICJvdmVybGF5cyIsIDEyKQogICAgICAgICAgICB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzKAogICAgICAgICAgICAgICAgZGF0YXNldCwgcHJlZGljdGlvbnMsIGJlc3Rfcm9vdCAvICJjb21wYXJpc29uLW92ZXJsYXlzIiwgMTIKICAgICAgICAgICAgKQogICAgc2NvcmluZ19zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHNjb3Jpbmdfc3RhcnRlZAoKICAgIHN3ZWVwX3Jlc3VsdHMuc29ydChrZXk9bGFtYmRhIGl0ZW06IGl0ZW1bInBxIl0sIHJldmVyc2U9VHJ1ZSkKICAgIHN3ZWVwX3N0ZW0gPSAic2NyZWVuLXN3ZWVwIiBpZiBvYnNlcnZhdGlvbl9saW1pdCBpcyBub3QgTm9uZSBlbHNlICJzd2VlcCIKICAgIChydW5fcm9vdCAvIGYie3N3ZWVwX3N0ZW19Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMoc3dlZXBfcmVzdWx0cywgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHdpdGggKHJ1bl9yb290IC8gZiJ7c3dlZXBfc3RlbX0uY3N2Iikub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICBmaWVsZHMgPSAoCiAgICAgICAgICAgICpDbGFzc2ljYWxDb25maWcoKS5fX2RpY3RfXy5rZXlzKCksCiAgICAgICAgICAgICJwcSIsCiAgICAgICAgICAgICJ0cCIsCiAgICAgICAgICAgICJmcCIsCiAgICAgICAgICAgICJmbiIsCiAgICAgICAgKQogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1maWVsZHMsIGxpbmV0ZXJtaW5hdG9yPSJcbiIpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIHN3ZWVwX3Jlc3VsdHM6CiAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAqKnJvd1siY29uZmlnIl0sCiAgICAgICAgICAgICAgICAgICAgKip7CiAgICAgICAgICAgICAgICAgICAgICAgIGZpZWxkOiByb3dbZmllbGRdCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBmaWVsZCBpbiBmaWVsZHNbbGVuKHJvd1siY29uZmlnIl0pIDpdCiAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQoKICAgIHJlc3VsdDogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInN0YXR1cyI6ICJjb21wbGV0ZSIgaWYgb2JzZXJ2YXRpb25fbGltaXQgaXMgTm9uZSBlbHNlICJ2YWxpZGF0aW9uX3N1YnNldCIsCiAgICAgICAgInJ1bl9uYW1lIjogcnVuX25hbWUsCiAgICAgICAgInJlcG9zaXRvcnkiOiByZXBvc2l0b3J5X2luZm8sCiAgICAgICAgImNvbmZpZ3VyYXRpb25fc2hhMjU2IjogY29uZmlnX3NoYTI1NiwKICAgICAgICAiZGF0YXNldF9pbnZlbnRvcnkiOiBpbnZlbnRvcnkoZGF0YXNldCksCiAgICAgICAgInNwbGl0X3NoYTI1NiI6IHNwbGl0LnNoYTI1NiwKICAgICAgICAidmFsaWRhdGlvbl9vYnNlcnZhdGlvbnMiOiBsZW4odmFsaWRhdGlvbl9uYW1lcyksCiAgICAgICAgImZ1bGxfaG9sZG91dF9vYnNlcnZhdGlvbnMiOiBsZW4oYWxsX3ZhbGlkYXRpb25fbmFtZXMpLAogICAgICAgICJtZXRob2QiOiAiaW5kZXBlbmRlbnQgaW1wbGVtZW50YXRpb24gb2YgcHVibGljIHJhZGlhbC1ub3JtYWxpemF0aW9uLCBtdWx0aXNjYWxlIGJsYWNrLWhhdCwgcm9idXN0IHotc2NvcmUsIGh5c3RlcmVzaXMsIGFuZCBjb21wb25lbnQgZmlsdGVyaW5nIGFwcHJvYWNoIiwKICAgICAgICAicmVmZXJlbmNlIjogImh0dHBzOi8vd3d3LmthZ2dsZS5jb20vY29kZS9yb2JpbmFtaXJiYWhhci9zb2xhci1maWxhbWVudC1wYW5vcHRpYy1tb3JwaG9sb2d5LXNlZ21lbnRhdGlvbiIsCiAgICAgICAgImJlc3QiOiBzd2VlcF9yZXN1bHRzWzBdLAogICAgICAgICJwcmVwcm9jZXNzaW5nX3NlY29uZHMiOiBwcmVwcm9jZXNzaW5nX3NlY29uZHMsCiAgICAgICAgInN3ZWVwX2FuZF9zY29yaW5nX3NlY29uZHMiOiBzY29yaW5nX3NlY29uZHMsCiAgICAgICAgInJ1bnRpbWUiOiB7KipoYXJkd2FyZV9pbmZvKCksICJwYWNrYWdlX3ZlcnNpb25zIjogcGFja2FnZV92ZXJzaW9ucygpfSwKICAgICAgICAiYXJ0aWZhY3RzIjogewogICAgICAgICAgICAic3dlZXBfY3N2IjogZiJ7c3dlZXBfc3RlbX0uY3N2IiwKICAgICAgICAgICAgInN3ZWVwX2pzb24iOiBmIntzd2VlcF9zdGVtfS5qc29uIiwKICAgICAgICAgICAgImJlc3RfZGlyIjogImJlc3Qtc2NyZWVuIiBpZiBvYnNlcnZhdGlvbl9saW1pdCBpcyBub3QgTm9uZSBlbHNlICJiZXN0IiwKICAgICAgICB9LAogICAgfQogICAgKHJ1bl9yb290IC8gIm1ldHJpY3MuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcmludCgKICAgICAgICBmIkJlc3QgY2xhc3NpY2FsIFBRIG9uIHtsZW4odmFsaWRhdGlvbl9uYW1lcyl9IG9ic2VydmF0aW9uczoge3N3ZWVwX3Jlc3VsdHNbMF1bJ3BxJ106LjZmfSAiCiAgICAgICAgZiJ3aXRoIHtzd2VlcF9yZXN1bHRzWzBdWydjb25maWcnXX07IG91dHB1dHMge3J1bl9yb290fSIsCiAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICkKICAgIHJldHVybiByZXN1bHQK\", \"src/filament_baseline/data.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKQU5OT1RBVElPTl9GSUxFTkFNRSA9ICJNQUdGaUxPXzEuMF9Bbm5vdGF0aW9uc19rYWdnbGUyMDI2X3RyYWluLmpzb24iCklNQUdFX1NJWkUgPSAyMDQ4CgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgSW1hZ2VWaWV3OgogICAgaW1hZ2VfaWQ6IHN0cgogICAgZmlsZV9uYW1lOiBzdHIKICAgIHdpZHRoOiBpbnQKICAgIGhlaWdodDogaW50CiAgICB5ZWFyOiBpbnQKICAgIHNpdGU6IHN0cgoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIENvbXBldGl0aW9uRGF0YXNldDoKICAgIHJvb3Q6IFBhdGgKICAgIHRyYWluX2ltYWdlX2RpcjogUGF0aAogICAgdGVzdF9pbWFnZV9kaXI6IFBhdGgKICAgIGFubm90YXRpb25fcGF0aDogUGF0aAogICAgaW1hZ2Vfdmlld3M6IHR1cGxlW0ltYWdlVmlldywgLi4uXQogICAgYW5ub3RhdGlvbnNfYnlfaW1hZ2VfaWQ6IGRpY3Rbc3RyLCB0dXBsZVtkaWN0W3N0ciwgQW55XSwgLi4uXV0KICAgIHZpZXdzX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgdHVwbGVbSW1hZ2VWaWV3LCAuLi5dXQogICAgY2F0ZWdvcmllczogdHVwbGVbZGljdFtzdHIsIEFueV0sIC4uLl0KICAgIGltYWdlX3Jlc2l6ZV90bzogdHVwbGVbaW50LCBpbnRdIHwgTm9uZSA9IE5vbmUKCiAgICBAcHJvcGVydHkKICAgIGRlZiBvYnNlcnZhdGlvbl9uYW1lcyhzZWxmKSAtPiB0dXBsZVtzdHIsIC4uLl06CiAgICAgICAgcmV0dXJuIHR1cGxlKHNvcnRlZChzZWxmLnZpZXdzX2J5X29ic2VydmF0aW9uKSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiB0ZXN0X2ltYWdlX2NvdW50KHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc3VtKHBhdGguaXNfZmlsZSgpIGZvciBwYXRoIGluIHNlbGYudGVzdF9pbWFnZV9kaXIuaXRlcmRpcigpKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGFubm90YXRpb25fY291bnQoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzdW0obGVuKGl0ZW1zKSBmb3IgaXRlbXMgaW4gc2VsZi5hbm5vdGF0aW9uc19ieV9pbWFnZV9pZC52YWx1ZXMoKSkKCgpkZWYgX2xvb2tzX2xpa2VfZGF0YXNldChyb290OiBQYXRoKSAtPiBib29sOgogICAgcmV0dXJuICgKICAgICAgICAocm9vdCAvICJ0cmFpbiIgLyBBTk5PVEFUSU9OX0ZJTEVOQU1FKS5pc19maWxlKCkKICAgICAgICBhbmQgKHJvb3QgLyAidHJhaW4iIC8gInRyYWluX2ltYWdlcyIpLmlzX2RpcigpCiAgICAgICAgYW5kIChyb290IC8gInRlc3QiIC8gInRlc3RfaW1hZ2VzIikuaXNfZGlyKCkKICAgICkKCgpkZWYgZmluZF9kYXRhc2V0X3Jvb3Qoc2VhcmNoX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IFBhdGg6CiAgICAiIiJGaW5kIHRoZSBjb21wZXRpdGlvbiBpbnB1dCBmb2xkZXIgd2l0aG91dCByZWx5aW5nIG9uIGEgZml4ZWQgS2FnZ2xlIG1vdW50IG5hbWUuIiIiCiAgICBzZWFyY2hfcm9vdCA9IFBhdGgoc2VhcmNoX3Jvb3QpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBzZWFyY2hfcm9vdC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkRhdGEgc2VhcmNoIHBhdGggZG9lcyBub3QgZXhpc3Q6IHtzZWFyY2hfcm9vdH0iKQogICAgaWYgX2xvb2tzX2xpa2VfZGF0YXNldChzZWFyY2hfcm9vdCk6CiAgICAgICAgcmV0dXJuIHNlYXJjaF9yb290CgogICAgbWF0Y2hlcyA9IHNvcnRlZCgKICAgICAgICB7cGF0aC5wYXJlbnQucGFyZW50LnJlc29sdmUoKSBmb3IgcGF0aCBpbiBzZWFyY2hfcm9vdC5yZ2xvYihBTk5PVEFUSU9OX0ZJTEVOQU1FKX0KICAgICAgICBpZiBzZWFyY2hfcm9vdC5pc19kaXIoKQogICAgICAgIGVsc2Ugc2V0KCkKICAgICkKICAgIG1hdGNoZXMgPSBbcGF0aCBmb3IgcGF0aCBpbiBtYXRjaGVzIGlmIF9sb29rc19saWtlX2RhdGFzZXQocGF0aCldCiAgICBpZiBsZW4obWF0Y2hlcykgPT0gMToKICAgICAgICByZXR1cm4gbWF0Y2hlc1swXQogICAgaWYgbm90IG1hdGNoZXM6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiQ291bGQgbm90IGZpbmQge0FOTk9UQVRJT05fRklMRU5BTUV9IHdpdGggdHJhaW4vdHJhaW5faW1hZ2VzIGFuZCAiCiAgICAgICAgICAgIGYidGVzdC90ZXN0X2ltYWdlcyBiZWxvdyB7c2VhcmNoX3Jvb3R9IgogICAgICAgICkKICAgIHJlbmRlcmVkID0gIlxuIi5qb2luKHN0cihwYXRoKSBmb3IgcGF0aCBpbiBtYXRjaGVzKQogICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICJNb3JlIHRoYW4gb25lIGNvbXBldGl0aW9uIGRhdGFzZXQgd2FzIGZvdW5kOyBwYXNzIGFuIGV4cGxpY2l0IGRhdGFzZXQgcm9vdDpcbiIKICAgICAgICBmIntyZW5kZXJlZH0iCiAgICApCgoKZGVmIF95ZWFyX2FuZF9zaXRlKGZpbGVfbmFtZTogc3RyKSAtPiB0dXBsZVtpbnQsIHN0cl06CiAgICBzdGVtID0gUGF0aChmaWxlX25hbWUpLnN0ZW0KICAgIGlmIGxlbihzdGVtKSA8IDQgb3Igbm90IHN0ZW1bOjRdLmlzZGlnaXQoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2Fubm90IGRlcml2ZSBjYXB0dXJlIHllYXIgZnJvbSBpbWFnZSBmaWxlbmFtZToge2ZpbGVfbmFtZX0iKQogICAgeWVhciA9IGludChzdGVtWzo0XSkKICAgIHNpdGVfbWF0Y2ggPSByZS5mdWxsbWF0Y2gociJcZHsxNH0oW0EtWmEtel17Mn0pIiwgc3RlbSkKICAgIHNpdGUgPSBzaXRlX21hdGNoLmdyb3VwKDEpIGlmIHNpdGVfbWF0Y2ggZWxzZSAidW5rbm93biIKICAgIHJldHVybiB5ZWFyLCBzaXRlCgoKZGVmIGxvYWRfZGF0YXNldChkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IENvbXBldGl0aW9uRGF0YXNldDoKICAgIHJvb3QgPSBmaW5kX2RhdGFzZXRfcm9vdChkYXRhX3Jvb3QpCiAgICBhbm5vdGF0aW9uX3BhdGggPSByb290IC8gInRyYWluIiAvIEFOTk9UQVRJT05fRklMRU5BTUUKICAgIHRyYWluX2ltYWdlX2RpciA9IHJvb3QgLyAidHJhaW4iIC8gInRyYWluX2ltYWdlcyIKICAgIHRlc3RfaW1hZ2VfZGlyID0gcm9vdCAvICJ0ZXN0IiAvICJ0ZXN0X2ltYWdlcyIKICAgIHBheWxvYWQgPSBqc29uLmxvYWRzKGFubm90YXRpb25fcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCgogICAgaW1hZ2Vfdmlld3M6IGxpc3RbSW1hZ2VWaWV3XSA9IFtdCiAgICBpbWFnZV9pZHM6IHNldFtzdHJdID0gc2V0KCkKICAgIHZpZXdzX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgbGlzdFtJbWFnZVZpZXddXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3IgcmVjb3JkIGluIHBheWxvYWQuZ2V0KCJpbWFnZXMiLCBbXSk6CiAgICAgICAgaW1hZ2VfaWQgPSBzdHIocmVjb3JkWyJpZCJdKQogICAgICAgIGZpbGVfbmFtZSA9IHN0cihyZWNvcmRbImZpbGVfbmFtZSJdKQogICAgICAgIGlmIFBhdGgoZmlsZV9uYW1lKS5uYW1lICE9IGZpbGVfbmFtZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIGEgYmFzZW5hbWUgaW4gQ09DTyBmaWxlX25hbWUsIGdvdDoge2ZpbGVfbmFtZX0iKQogICAgICAgIGlmIGltYWdlX2lkIGluIGltYWdlX2lkczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkR1cGxpY2F0ZSBDT0NPIGltYWdlIGlkOiB7aW1hZ2VfaWR9IikKICAgICAgICBpbWFnZV9pZHMuYWRkKGltYWdlX2lkKQogICAgICAgIHllYXIsIHNpdGUgPSBfeWVhcl9hbmRfc2l0ZShmaWxlX25hbWUpCiAgICAgICAgdmlldyA9IEltYWdlVmlldygKICAgICAgICAgICAgaW1hZ2VfaWQ9aW1hZ2VfaWQsCiAgICAgICAgICAgIGZpbGVfbmFtZT1maWxlX25hbWUsCiAgICAgICAgICAgIHdpZHRoPWludChyZWNvcmRbIndpZHRoIl0pLAogICAgICAgICAgICBoZWlnaHQ9aW50KHJlY29yZFsiaGVpZ2h0Il0pLAogICAgICAgICAgICB5ZWFyPXllYXIsCiAgICAgICAgICAgIHNpdGU9c2l0ZSwKICAgICAgICApCiAgICAgICAgaWYgdmlldy53aWR0aCAhPSBJTUFHRV9TSVpFIG9yIHZpZXcuaGVpZ2h0ICE9IElNQUdFX1NJWkU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmIkV4cGVjdGVkIHtJTUFHRV9TSVpFfXh7SU1BR0VfU0laRX0sIGdvdCB7dmlldy53aWR0aH14e3ZpZXcuaGVpZ2h0fSAiCiAgICAgICAgICAgICAgICBmImZvciB7ZmlsZV9uYW1lfSIKICAgICAgICAgICAgKQogICAgICAgIGltYWdlX3ZpZXdzLmFwcGVuZCh2aWV3KQogICAgICAgIHZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV0uYXBwZW5kKHZpZXcpCgogICAgYW5ub3RhdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W2RpY3Rbc3RyLCBBbnldXV0gPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIGFubm90YXRpb24gaW4gcGF5bG9hZC5nZXQoImFubm90YXRpb25zIiwgW10pOgogICAgICAgIGltYWdlX2lkID0gc3RyKGFubm90YXRpb25bImltYWdlX2lkIl0pCiAgICAgICAgaWYgaW1hZ2VfaWQgbm90IGluIGltYWdlX2lkczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkFubm90YXRpb24gcmVmZXJlbmNlcyB1bmtub3duIGltYWdlIGlkOiB7aW1hZ2VfaWR9IikKICAgICAgICBpZiBpbnQoYW5ub3RhdGlvbi5nZXQoImlzY3Jvd2QiLCAwKSkgIT0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGhlIGNvbXBldGl0aW9uIHRyYWluaW5nIGFubm90YXRpb25zIHVuZXhwZWN0ZWRseSBjb250YWluIGNyb3dkIG1hc2tzIikKICAgICAgICBhbm5vdGF0aW9uc1tpbWFnZV9pZF0uYXBwZW5kKGFubm90YXRpb24pCgogICAgZm9yIGltYWdlX2lkIGluIGltYWdlX2lkczoKICAgICAgICBhbm5vdGF0aW9uc1tpbWFnZV9pZF0uc29ydChrZXk9bGFtYmRhIGl0ZW06IHN0cihpdGVtWyJpZCJdKSkKICAgICAgICBpZiBub3QgYW5ub3RhdGlvbnNbaW1hZ2VfaWRdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQW5ub3RhdG9yIHZpZXcge2ltYWdlX2lkfSBoYXMgbm8gZmlsYW1lbnQgYW5ub3RhdGlvbnMiKQoKICAgIGZvciBmaWxlX25hbWUsIHZpZXdzIGluIHZpZXdzX2J5X29ic2VydmF0aW9uLml0ZW1zKCk6CiAgICAgICAgZXhwZWN0ZWRfcGF0aCA9IHRyYWluX2ltYWdlX2RpciAvIGZpbGVfbmFtZQogICAgICAgIGlmIG5vdCBleHBlY3RlZF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDT0NPIGltYWdlIGZpbGUgaXMgbWlzc2luZzoge2V4cGVjdGVkX3BhdGh9IikKICAgICAgICB2aWV3cy5zb3J0KGtleT1sYW1iZGEgdmlldzogdmlldy5pbWFnZV9pZCkKCiAgICB0ZXN0X2ZpbGVzID0gW3BhdGggZm9yIHBhdGggaW4gdGVzdF9pbWFnZV9kaXIuaXRlcmRpcigpIGlmIHBhdGguaXNfZmlsZSgpXQogICAgaWYgbm90IHRlc3RfZmlsZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk5vIHRlc3QgaW1hZ2VzIGZvdW5kIGluIHt0ZXN0X2ltYWdlX2Rpcn0iKQoKICAgIHJldHVybiBDb21wZXRpdGlvbkRhdGFzZXQoCiAgICAgICAgcm9vdD1yb290LAogICAgICAgIHRyYWluX2ltYWdlX2Rpcj10cmFpbl9pbWFnZV9kaXIsCiAgICAgICAgdGVzdF9pbWFnZV9kaXI9dGVzdF9pbWFnZV9kaXIsCiAgICAgICAgYW5ub3RhdGlvbl9wYXRoPWFubm90YXRpb25fcGF0aCwKICAgICAgICBpbWFnZV92aWV3cz10dXBsZShzb3J0ZWQoaW1hZ2Vfdmlld3MsIGtleT1sYW1iZGEgdmlldzogKHZpZXcuZmlsZV9uYW1lLCB2aWV3LmltYWdlX2lkKSkpLAogICAgICAgIGFubm90YXRpb25zX2J5X2ltYWdlX2lkPXsKICAgICAgICAgICAgaW1hZ2VfaWQ6IHR1cGxlKGl0ZW1zKSBmb3IgaW1hZ2VfaWQsIGl0ZW1zIGluIGFubm90YXRpb25zLml0ZW1zKCkKICAgICAgICB9LAogICAgICAgIHZpZXdzX2J5X29ic2VydmF0aW9uPXsKICAgICAgICAgICAgZmlsZV9uYW1lOiB0dXBsZSh2aWV3cykKICAgICAgICAgICAgZm9yIGZpbGVfbmFtZSwgdmlld3MgaW4gc29ydGVkKHZpZXdzX2J5X29ic2VydmF0aW9uLml0ZW1zKCkpCiAgICAgICAgfSwKICAgICAgICBjYXRlZ29yaWVzPXR1cGxlKHBheWxvYWQuZ2V0KCJjYXRlZ29yaWVzIiwgW10pKSwKICAgICkKCgpkZWYgaW52ZW50b3J5KGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCkgLT4gZGljdFtzdHIsIGludF06CiAgICByZXR1cm4gewogICAgICAgICJvYnNlcnZhdGlvbl9jb3VudCI6IGxlbihkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uKSwKICAgICAgICAiYW5ub3RhdG9yX3ZpZXdfY291bnQiOiBsZW4oZGF0YXNldC5pbWFnZV92aWV3cyksCiAgICAgICAgImFubm90YXRpb25fY291bnQiOiBkYXRhc2V0LmFubm90YXRpb25fY291bnQsCiAgICAgICAgInRyYWluX2ltYWdlX2ZpbGVfY291bnQiOiBzdW0oCiAgICAgICAgICAgIHBhdGguaXNfZmlsZSgpIGZvciBwYXRoIGluIGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyLml0ZXJkaXIoKQogICAgICAgICksCiAgICAgICAgInRlc3RfaW1hZ2VfZmlsZV9jb3VudCI6IGRhdGFzZXQudGVzdF9pbWFnZV9jb3VudCwKICAgIH0K\", \"src/filament_baseline/detector_fusion.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gcHljb2NvdG9vbHMgaW1wb3J0IG1hc2sgYXMgbWFza191dGlscwpmcm9tIHNjaXB5IGltcG9ydCBuZGltYWdlCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgRGV0ZWN0b3JGdXNpb25Db25maWc6CiAgICAiIiJVc2UgZGV0ZWN0b3IgaW5zdGFuY2VzIHRvIGdyb3VwIGZvcmVncm91bmQgcGl4ZWxzIGZyb20gYSBmdWxsLXJlc29sdXRpb24gc2VnbWVudGVyLiIiIgoKICAgIGZvcmVncm91bmRfdGhyZXNob2xkOiBmbG9hdCA9IDAuNQogICAgZGV0ZWN0aW9uX3RocmVzaG9sZDogZmxvYXQgPSAwLjMKICAgIGdyb3dfcmFkaXVzOiBpbnQgPSA0CiAgICBtaW5pbXVtX2NsYWltZWRfYXJlYTogaW50ID0gMjAwCiAgICBjbG9zaW5nX3JhZGl1czogaW50ID0gMwogICAgc29sYXJfcmFkaXVzOiBpbnQgPSA5NTAKICAgIGtlZXBfdW5jbGFpbWVkX21pbmltdW1fYXJlYTogaW50ID0gMAoKCmRlZiBfZGVjb2RlX2RldGVjdGlvbihjb3VudHM6IHN0ciB8IGJ5dGVzLCBzaGFwZTogdHVwbGVbaW50LCBpbnRdKSAtPiBucC5uZGFycmF5OgogICAgZW5jb2RlZF9jb3VudHMgPSBjb3VudHMgaWYgaXNpbnN0YW5jZShjb3VudHMsIGJ5dGVzKSBlbHNlIGNvdW50cy5lbmNvZGUoImFzY2lpIikKICAgIGRlY29kZWQgPSBtYXNrX3V0aWxzLmRlY29kZSgKICAgICAgICB7InNpemUiOiBsaXN0KHNoYXBlKSwgImNvdW50cyI6IGVuY29kZWRfY291bnRzfQogICAgKQogICAgaWYgZGVjb2RlZC5uZGltID09IDM6CiAgICAgICAgZGVjb2RlZCA9IGRlY29kZWRbOiwgOiwgMF0KICAgIHJldHVybiBkZWNvZGVkLmFzdHlwZShib29sKQoKCmRlZiBfZGlza19tYXNrKHNoYXBlOiB0dXBsZVtpbnQsIGludF0sIHJhZGl1czogaW50KSAtPiBucC5uZGFycmF5OgogICAgaGVpZ2h0LCB3aWR0aCA9IHNoYXBlCiAgICBkaXNrID0gbnAuemVyb3MoKGhlaWdodCwgd2lkdGgpLCBkdHlwZT1ucC51aW50OCkKICAgIGNlbnRlciA9ICgod2lkdGggLSAxKSAvLyAyLCAoaGVpZ2h0IC0gMSkgLy8gMikKICAgIGN2Mi5jaXJjbGUoZGlzaywgY2VudGVyLCByYWRpdXMsIGNvbG9yPTEsIHRoaWNrbmVzcz0tMSkKICAgIHJldHVybiBkaXNrLmFzdHlwZShib29sKQoKCmRlZiBzb2xhcl9kaXNrX21hc2tfZnJvbV9nZW9tZXRyeSgKICAgIHNoYXBlOiB0dXBsZVtpbnQsIGludF0sCiAgICBnZW9tZXRyeTogZGljdFtzdHIsIGZsb2F0XSwKICAgICosCiAgICBzaHJpbms6IGZsb2F0ID0gMS4wLAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJCdWlsZCBhbiBpbWFnZS1kZXJpdmVkIHNvbGFyIGRpc2sgbWFzayBmcm9tIGl0cyBmaXR0ZWQgY2lyY2xlLiIiIgogICAgaWYgbm90IDAuOCA8PSBzaHJpbmsgPD0gMS4wNToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzaHJpbmsgbXVzdCBiZSBpbiBbMC44LCAxLjA1XSIpCiAgICBoZWlnaHQsIHdpZHRoID0gc2hhcGUKICAgIGN4LCBjeSA9IGZsb2F0KGdlb21ldHJ5WyJjeCJdKSwgZmxvYXQoZ2VvbWV0cnlbImN5Il0pCiAgICByYWRpdXMgPSBmbG9hdChnZW9tZXRyeVsicmFkaXVzIl0pICogc2hyaW5rCiAgICBpZiByYWRpdXMgPD0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzb2xhciBkaXNrIHJhZGl1cyBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHl5LCB4eCA9IG5wLm9ncmlkWzpoZWlnaHQsIDp3aWR0aF0KICAgIHJldHVybiAoeHggLSBjeCkgKiogMiArICh5eSAtIGN5KSAqKiAyIDw9IHJhZGl1cyoqMgoKCmRlZiBfZ3Jvd19tYXNrKG1hc2s6IG5wLm5kYXJyYXksIHJhZGl1czogaW50KSAtPiBucC5uZGFycmF5OgogICAgaWYgcmFkaXVzIDw9IDAgb3Igbm90IG5wLmFueShtYXNrKToKICAgICAgICByZXR1cm4gbWFzawogICAgeXMsIHhzID0gbnAubm9uemVybyhtYXNrKQogICAgeTAsIHkxID0gbWF4KGludCh5cy5taW4oKSkgLSByYWRpdXMsIDApLCBtaW4oaW50KHlzLm1heCgpKSArIHJhZGl1cyArIDEsIG1hc2suc2hhcGVbMF0pCiAgICB4MCwgeDEgPSBtYXgoaW50KHhzLm1pbigpKSAtIHJhZGl1cywgMCksIG1pbihpbnQoeHMubWF4KCkpICsgcmFkaXVzICsgMSwgbWFzay5zaGFwZVsxXSkKICAgIGtlcm5lbCA9IGN2Mi5nZXRTdHJ1Y3R1cmluZ0VsZW1lbnQoCiAgICAgICAgY3YyLk1PUlBIX0VMTElQU0UsICgyICogcmFkaXVzICsgMSwgMiAqIHJhZGl1cyArIDEpCiAgICApCiAgICBjcm9wID0gbWFza1t5MDp5MSwgeDA6eDFdLmFzdHlwZShucC51aW50OCkKICAgIGdyb3duID0gY3YyLmRpbGF0ZShjcm9wLCBrZXJuZWwsIGl0ZXJhdGlvbnM9MSkuYXN0eXBlKGJvb2wpCiAgICBvdXRwdXQgPSBucC56ZXJvcyhtYXNrLnNoYXBlLCBkdHlwZT1ib29sKQogICAgb3V0cHV0W3kwOnkxLCB4MDp4MV0gPSBncm93bgogICAgcmV0dXJuIG91dHB1dAoKCmRlZiBmdXNlX2RldGVjdG9yX2luc3RhbmNlcygKICAgIGZvcmVncm91bmRfcHJvYmFiaWxpdHk6IG5wLm5kYXJyYXksCiAgICBkZXRlY3Rpb25zOiBsaXN0W2RpY3Rbc3RyLCBBbnldXSwKICAgIGNvbmZpZzogRGV0ZWN0b3JGdXNpb25Db25maWcgfCBOb25lID0gTm9uZSwKICAgICosCiAgICBkaXNrX21hc2s6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSwKKSAtPiBucC5uZGFycmF5OgogICAgIiIiQXNzaWduIFUtTmV0IGZvcmVncm91bmQgcGl4ZWxzIHRvIGNvbmZpZGVudCBkZXRlY3RvciBtYXNrcywgb25lIGluc3RhbmNlIGVhY2guCgogICAgVGhlIGRldGVjdG9yIHN1cHBsaWVzIGluc3RhbmNlIGdyb3VwaW5nIGFuZCBjb25maWRlbmNlOyB0aGUgZm9yZWdyb3VuZCBtYXAKICAgIHN1cHBsaWVzIGZ1bGwtcmVzb2x1dGlvbiBib3VuZGFyaWVzLiBPdmVybGFwcGluZyBkZXRlY3Rpb25zIGNsYWltIHBpeGVscyBpbgogICAgZGVzY2VuZGluZyBjb25maWRlbmNlIG9yZGVyLCBhbmQgZWFjaCBwaXhlbCBjYW4gYmUgYXNzaWduZWQgb25seSBvbmNlLgogICAgIiIiCiAgICBjb25maWcgPSBjb25maWcgb3IgRGV0ZWN0b3JGdXNpb25Db25maWcoKQogICAgaWYgZm9yZWdyb3VuZF9wcm9iYWJpbGl0eS5uZGltICE9IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZm9yZWdyb3VuZF9wcm9iYWJpbGl0eSBtdXN0IGJlIGEgdHdvLWRpbWVuc2lvbmFsIG1hcCIpCiAgICBpZiBub3QgMC4wIDw9IGNvbmZpZy5mb3JlZ3JvdW5kX3RocmVzaG9sZCA8PSAxLjA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZm9yZWdyb3VuZF90aHJlc2hvbGQgbXVzdCBiZSBiZXR3ZWVuIDAgYW5kIDEiKQogICAgaWYgbm90IDAuMCA8PSBjb25maWcuZGV0ZWN0aW9uX3RocmVzaG9sZCA8PSAxLjA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGV0ZWN0aW9uX3RocmVzaG9sZCBtdXN0IGJlIGJldHdlZW4gMCBhbmQgMSIpCiAgICBpZiBtaW4oCiAgICAgICAgY29uZmlnLmdyb3dfcmFkaXVzLAogICAgICAgIGNvbmZpZy5taW5pbXVtX2NsYWltZWRfYXJlYSwKICAgICAgICBjb25maWcuY2xvc2luZ19yYWRpdXMsCiAgICAgICAgY29uZmlnLmtlZXBfdW5jbGFpbWVkX21pbmltdW1fYXJlYSwKICAgICkgPCAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZ1c2lvbiBzaXplcyBtdXN0IGJlIG5vbm5lZ2F0aXZlIikKCiAgICBzaGFwZSA9IGZvcmVncm91bmRfcHJvYmFiaWxpdHkuc2hhcGUKICAgIGlmIGRpc2tfbWFzayBpcyBOb25lOgogICAgICAgIGRpc2sgPSBfZGlza19tYXNrKHNoYXBlLCBjb25maWcuc29sYXJfcmFkaXVzKQogICAgZWxzZToKICAgICAgICBpZiBkaXNrX21hc2suc2hhcGUgIT0gc2hhcGU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImRpc2tfbWFzayBtdXN0IGhhdmUgdGhlIHNhbWUgc2hhcGUgYXMgZm9yZWdyb3VuZF9wcm9iYWJpbGl0eSIpCiAgICAgICAgZGlzayA9IG5wLmFzYXJyYXkoZGlza19tYXNrLCBkdHlwZT1ib29sKQogICAgYmluYXJ5ID0gKGZvcmVncm91bmRfcHJvYmFiaWxpdHkgPj0gY29uZmlnLmZvcmVncm91bmRfdGhyZXNob2xkKSAmIGRpc2sKICAgIGlmIGNvbmZpZy5jbG9zaW5nX3JhZGl1cyA+IDA6CiAgICAgICAgcmFkaXVzID0gY29uZmlnLmNsb3NpbmdfcmFkaXVzCiAgICAgICAga2VybmVsID0gY3YyLmdldFN0cnVjdHVyaW5nRWxlbWVudCgKICAgICAgICAgICAgY3YyLk1PUlBIX0VMTElQU0UsICgyICogcmFkaXVzICsgMSwgMiAqIHJhZGl1cyArIDEpCiAgICAgICAgKQogICAgICAgIGJpbmFyeSA9IGN2Mi5tb3JwaG9sb2d5RXgoYmluYXJ5LmFzdHlwZShucC51aW50OCksIGN2Mi5NT1JQSF9DTE9TRSwga2VybmVsKS5hc3R5cGUoYm9vbCkKICAgICAgICBiaW5hcnkgJj0gZGlzawogICAgYmluYXJ5ID0gbmRpbWFnZS5iaW5hcnlfZmlsbF9ob2xlcyhiaW5hcnkpICYgZGlzawoKICAgIGxhYmVscyA9IG5wLnplcm9zKHNoYXBlLCBkdHlwZT1ucC5pbnQzMikKICAgIGNsYWltZWQgPSBucC56ZXJvcyhzaGFwZSwgZHR5cGU9Ym9vbCkKICAgIG5leHRfbGFiZWwgPSAxCiAgICBlbGlnaWJsZSA9IFsKICAgICAgICBpdGVtIGZvciBpdGVtIGluIGRldGVjdGlvbnMgaWYgZmxvYXQoaXRlbVsic2NvcmUiXSkgPj0gY29uZmlnLmRldGVjdGlvbl90aHJlc2hvbGQKICAgIF0KICAgIGVsaWdpYmxlLnNvcnQoa2V5PWxhbWJkYSBpdGVtOiAtZmxvYXQoaXRlbVsic2NvcmUiXSkpCiAgICBmb3IgZGV0ZWN0aW9uIGluIGVsaWdpYmxlOgogICAgICAgIHJlZ2lvbiA9IF9kZWNvZGVfZGV0ZWN0aW9uKGRldGVjdGlvblsiY291bnRzIl0sIHNoYXBlKQogICAgICAgIHJlZ2lvbiA9IF9ncm93X21hc2socmVnaW9uLCBjb25maWcuZ3Jvd19yYWRpdXMpCiAgICAgICAgcGl4ZWxzID0gYmluYXJ5ICYgcmVnaW9uICYgfmNsYWltZWQKICAgICAgICBhcmVhID0gaW50KHBpeGVscy5zdW0oKSkKICAgICAgICBpZiBhcmVhIDwgY29uZmlnLm1pbmltdW1fY2xhaW1lZF9hcmVhOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGxhYmVsc1twaXhlbHNdID0gbmV4dF9sYWJlbAogICAgICAgIGNsYWltZWRbcGl4ZWxzXSA9IFRydWUKICAgICAgICBuZXh0X2xhYmVsICs9IDEKCiAgICBpZiBjb25maWcua2VlcF91bmNsYWltZWRfbWluaW11bV9hcmVhID4gMDoKICAgICAgICBjb21wb25lbnRzLCBjb21wb25lbnRfY291bnQgPSBuZGltYWdlLmxhYmVsKGJpbmFyeSAmIH5jbGFpbWVkKQogICAgICAgIHNpemVzID0gbnAuYmluY291bnQoY29tcG9uZW50cy5yYXZlbCgpLCBtaW5sZW5ndGg9Y29tcG9uZW50X2NvdW50ICsgMSkKICAgICAgICBmb3IgY29tcG9uZW50X2lkIGluIHJhbmdlKDEsIGNvbXBvbmVudF9jb3VudCArIDEpOgogICAgICAgICAgICBpZiBzaXplc1tjb21wb25lbnRfaWRdIDwgY29uZmlnLmtlZXBfdW5jbGFpbWVkX21pbmltdW1fYXJlYToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGxhYmVsc1tjb21wb25lbnRzID09IGNvbXBvbmVudF9pZF0gPSBuZXh0X2xhYmVsCiAgICAgICAgICAgIG5leHRfbGFiZWwgKz0gMQogICAgcmV0dXJuIGxhYmVscwo=\", \"src/filament_baseline/ensemble_submission.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgZ2MKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAoKZnJvbSAuZGF0YSBpbXBvcnQgSU1BR0VfU0laRSwgaW52ZW50b3J5LCBsb2FkX2RhdGFzZXQKZnJvbSAuZmVhdHVyZXMgaW1wb3J0IElucHV0TW9kZQpmcm9tIC5pbmZlcmVuY2UgaW1wb3J0IEluc3RhbmNlQ29uZmlnLCBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzLCBwcmVkaWN0X3Byb2JhYmlsaXRpZXMKZnJvbSAubGVhcm5pbmcgaW1wb3J0IEZpbGFtZW50UGF0Y2hEYXRhc2V0LCBUYXJnZXRNb2RlCmZyb20gLm1vZGVsX3J1bm5lciBpbXBvcnQgX2lzX2thZ2dsZV9ydW50aW1lLCBfbG9hZF9tb2RlbCwgX3dpdGhpbgpmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IHJlYWRfZ3JheXNjYWxlCmZyb20gLnJ1bm5lciBpbXBvcnQgX2dpdF9zdGF0dXMsIF9yZXNvbHZlX2RhdGFfcm9vdCwgX3dyaXRlX3ByZWRpY3Rpb25zCmZyb20gLnJ1bnRpbWUgaW1wb3J0IGhhcmR3YXJlX2luZm8sIHBhY2thZ2VfdmVyc2lvbnMKZnJvbSAuc3VibWlzc2lvbiBpbXBvcnQgX2NoZWNrX3N1Ym1pc3Npb24KZnJvbSAudHJhaW5pbmcgaW1wb3J0IHRyYWluX21vZGVsCgoKZGVmIF9maWxlX3NoYTI1NihwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBzdHJlYW06CiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIF9zYXZlX3Byb2JhYmlsaXR5X2NhY2hlKAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICBmb3JlZ3JvdW5kOiBucC5uZGFycmF5LAogICAgYm91bmRhcnk6IG5wLm5kYXJyYXksCikgLT4gTm9uZToKICAgIG91dHB1dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBucC5zYXZlel9jb21wcmVzc2VkKAogICAgICAgIG91dHB1dF9wYXRoLAogICAgICAgIGZvcmVncm91bmQ9Zm9yZWdyb3VuZC5hc3R5cGUobnAuZmxvYXQxNiksCiAgICAgICAgYm91bmRhcnk9Ym91bmRhcnkuYXN0eXBlKG5wLmZsb2F0MTYpLAogICAgKQoKCmRlZiBfZml0X2FuZF9jYWNoZV9tZW1iZXIoCiAgICAqLAogICAgZGF0YXNldDogQW55LAogICAgdHJhaW5fbmFtZXM6IGxpc3Rbc3RyXSwKICAgIHRlc3RfbmFtZXM6IGxpc3Rbc3RyXSwKICAgIG1lbWJlcjogZGljdFtzdHIsIEFueV0sCiAgICBtZW1iZXJfcm9vdDogUGF0aCwKICAgIHNlZWQ6IGludCwKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHRhcmdldF9tb2RlOiBUYXJnZXRNb2RlID0gbWVtYmVyWyJ0YXJnZXRfbW9kZSJdCiAgICBwYXRjaF9zaXplID0gaW50KG1lbWJlclsicGF0Y2hfc2l6ZSJdKQogICAgYmF0Y2hfc2l6ZSA9IGludChtZW1iZXJbImJhdGNoX3NpemUiXSkKICAgIHN0ZXBzID0gaW50KG1lbWJlclsic3RlcHMiXSkKICAgIGJhc2VfY2hhbm5lbHMgPSBpbnQobWVtYmVyLmdldCgiYmFzZV9jaGFubmVscyIsIDE2KSkKICAgIHR0YSA9IGludChtZW1iZXJbInR0YSJdKQogICAgaW5wdXRfbW9kZTogSW5wdXRNb2RlID0gbWVtYmVyLmdldCgiaW5wdXRfbW9kZSIsICJyYXciKQogICAgaWYgdGFyZ2V0X21vZGUgbm90IGluICgiYW5ub3RhdG9yIiwgImNvbnNlbnN1cyIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnN1cHBvcnRlZCB0YXJnZXQgbW9kZSBmb3IgZW5zZW1ibGUgbWVtYmVyOiB7dGFyZ2V0X21vZGV9IikKICAgIGlmIHR0YSBub3QgaW4gKDEsIDIsIDQsIDgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnN1cHBvcnRlZCBUVEEgY291bnQgZm9yIGVuc2VtYmxlIG1lbWJlcjoge3R0YX0iKQoKICAgIG1lbWJlcl9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9RmFsc2UpCiAgICB0cmFpbl9kYXRhc2V0ID0gRmlsYW1lbnRQYXRjaERhdGFzZXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICB0cmFpbl9uYW1lcywKICAgICAgICB0YXJnZXRfbW9kZT10YXJnZXRfbW9kZSwKICAgICAgICBwYXRjaF9zaXplPXBhdGNoX3NpemUsCiAgICAgICAgYXVnbWVudD1UcnVlLAogICAgICAgIHNlZWQ9c2VlZCwKICAgICAgICBpbnB1dF9tb2RlPWlucHV0X21vZGUsCiAgICApCiAgICBjaGVja3BvaW50X3BhdGggPSBtZW1iZXJfcm9vdCAvICJmaW5hbF9tb2RlbC5wdCIKICAgIHRyYWluaW5nID0gdHJhaW5fbW9kZWwoCiAgICAgICAgdHJhaW5fZGF0YXNldD10cmFpbl9kYXRhc2V0LAogICAgICAgIHZhbGlkYXRpb25fZGF0YXNldD1Ob25lLAogICAgICAgIG91dHB1dF9wYXRoPWNoZWNrcG9pbnRfcGF0aCwKICAgICAgICBzdGVwcz1zdGVwcywKICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsCiAgICAgICAgYmFzZV9jaGFubmVscz1iYXNlX2NoYW5uZWxzLAogICAgICAgIHNlZWQ9c2VlZCwKICAgICAgICB2YWxpZGF0aW9uX2ludGVydmFsPW1heCgxMDAsIG1pbig1MDAsIHN0ZXBzIC8vIDUpKSwKICAgICkKCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGE6MCIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgbW9kZWwgPSBfbG9hZF9tb2RlbChjaGVja3BvaW50X3BhdGgsIGRldmljZSkKICAgIGNhY2hlX2RpciA9IG1lbWJlcl9yb290IC8gInByb2JhYmlsaXRpZXMiCiAgICBpbmZlcmVuY2Vfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKGRldmljZSkKICAgIGZvciBpbmRleCwgZmlsZV9uYW1lIGluIGVudW1lcmF0ZSh0ZXN0X25hbWVzLCBzdGFydD0xKToKICAgICAgICBpbWFnZSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXQudGVzdF9pbWFnZV9kaXIgLyBmaWxlX25hbWUpCiAgICAgICAgZm9yZWdyb3VuZCwgYm91bmRhcnkgPSBwcmVkaWN0X3Byb2JhYmlsaXRpZXMoCiAgICAgICAgICAgIG1vZGVsLAogICAgICAgICAgICBpbWFnZSwKICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgcGF0Y2hfc2l6ZT1wYXRjaF9zaXplLAogICAgICAgICAgICBzdHJpZGU9bWF4KHBhdGNoX3NpemUgLy8gMiwgaW50KHBhdGNoX3NpemUgKiAwLjc1KSksCiAgICAgICAgICAgIHR0YT10dGEsCiAgICAgICAgKQogICAgICAgIF9zYXZlX3Byb2JhYmlsaXR5X2NhY2hlKAogICAgICAgICAgICBjYWNoZV9kaXIgLyBmIntQYXRoKGZpbGVfbmFtZSkuc3RlbX0ubnB6IiwgZm9yZWdyb3VuZCwgYm91bmRhcnkKICAgICAgICApCiAgICAgICAgaWYgaW5kZXggJSAxMCA9PSAwIG9yIGluZGV4ID09IGxlbih0ZXN0X25hbWVzKToKICAgICAgICAgICAgcHJpbnQoCiAgICAgICAgICAgICAgICBmInttZW1iZXJbJ25hbWUnXX06IGNhY2hlZCBwcm9iYWJpbGl0aWVzIGZvciB7aW5kZXh9L3tsZW4odGVzdF9uYW1lcyl9IHRlc3QgaW1hZ2VzIiwKICAgICAgICAgICAgICAgIGZsdXNoPVRydWUsCiAgICAgICAgICAgICkKICAgIGluZmVyZW5jZV9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGluZmVyZW5jZV9zdGFydGVkCiAgICBpbmZlcmVuY2VfcGVha19tZW1vcnkgPSAoCiAgICAgICAgaW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoZGV2aWNlKSkgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGVsc2UgMAogICAgKQogICAgZGVsIG1vZGVsLCB0cmFpbl9kYXRhc2V0CiAgICBnYy5jb2xsZWN0KCkKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCgogICAgbWVtYmVyX3Jlc3VsdCA9IHsKICAgICAgICAibmFtZSI6IG1lbWJlclsibmFtZSJdLAogICAgICAgICJjb25maWd1cmF0aW9uIjogewogICAgICAgICAgICAidGFyZ2V0X21vZGUiOiB0YXJnZXRfbW9kZSwKICAgICAgICAgICAgInN0ZXBzIjogc3RlcHMsCiAgICAgICAgICAgICJiYXRjaF9zaXplIjogYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgInBhdGNoX3NpemUiOiBwYXRjaF9zaXplLAogICAgICAgICAgICAiYmFzZV9jaGFubmVscyI6IGJhc2VfY2hhbm5lbHMsCiAgICAgICAgICAgICJ0dGEiOiB0dGEsCiAgICAgICAgICAgICJpbnB1dF9tb2RlIjogaW5wdXRfbW9kZSwKICAgICAgICAgICAgInNlZWQiOiBzZWVkLAogICAgICAgICAgICAiZXh0ZXJuYWxfc3VwZXJ2aXNlZF93ZWlnaHRzIjogRmFsc2UsCiAgICAgICAgfSwKICAgICAgICAidHJhaW5pbmciOiB0cmFpbmluZywKICAgICAgICAiaW5mZXJlbmNlX3NlY29uZHMiOiBpbmZlcmVuY2Vfc2Vjb25kcywKICAgICAgICAiaW5mZXJlbmNlX3BlYWtfZ3B1X21lbW9yeV9ieXRlcyI6IGluZmVyZW5jZV9wZWFrX21lbW9yeSwKICAgICAgICAiY2hlY2twb2ludCI6IHN0cihjaGVja3BvaW50X3BhdGgubmFtZSksCiAgICAgICAgImNoZWNrcG9pbnRfc2hhMjU2IjogX2ZpbGVfc2hhMjU2KGNoZWNrcG9pbnRfcGF0aCksCiAgICAgICAgInByb2JhYmlsaXR5X2NhY2hlIjogc3RyKGNhY2hlX2Rpci5uYW1lKSwKICAgIH0KICAgIChtZW1iZXJfcm9vdCAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMobWVtYmVyX3Jlc3VsdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsCiAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwKICAgICkKICAgIHJldHVybiBtZW1iZXJfcmVzdWx0CgoKZGVmIHJ1bl9maW5hbF9lbnNlbWJsZV9zdWJtaXNzaW9uKAogICAgKiwKICAgIHJlcG9fcm9vdDogc3RyIHwgUGF0aCwKICAgIGRhdGFfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUsCiAgICBydW5fbmFtZTogc3RyLAogICAgZmlyc3RfbWVtYmVyOiBkaWN0W3N0ciwgQW55XSwKICAgIHNlY29uZF9tZW1iZXI6IGRpY3Rbc3RyLCBBbnldLAogICAgZmlyc3Rfd2VpZ2h0OiBmbG9hdCwKICAgIGluc3RhbmNlX2NvbmZpZzogSW5zdGFuY2VDb25maWcsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBpZiBub3QgcmUuZnVsbG1hdGNoKHIiW0EtWmEtejAtOV1bQS1aYS16MC05Ll8tXXswLDYzfSIsIHJ1bl9uYW1lKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJydW5fbmFtZSBtdXN0IHVzZSBvbmx5IGxldHRlcnMsIGRpZ2l0cywgZG90LCB1bmRlcnNjb3JlLCBvciBkYXNoIikKICAgIGlmIG5vdCAwLjAgPD0gZmlyc3Rfd2VpZ2h0IDw9IDEuMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJmaXJzdF93ZWlnaHQgbXVzdCBiZSBiZXR3ZWVuIHplcm8gYW5kIG9uZSIpCiAgICBpZiBmaXJzdF9tZW1iZXIuZ2V0KCJuYW1lIikgPT0gc2Vjb25kX21lbWJlci5nZXQoIm5hbWUiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFbnNlbWJsZSBtZW1iZXIgbmFtZXMgbXVzdCBkaWZmZXIiKQoKICAgIHJlcG9fcm9vdCA9IFBhdGgocmVwb19yb290KS5yZXNvbHZlKCkKICAgIGthZ2dsZV9ydW50aW1lID0gX2lzX2thZ2dsZV9ydW50aW1lKCkKICAgIGRhdGFzZXRfcm9vdCA9IF9yZXNvbHZlX2RhdGFfcm9vdChkYXRhX3Jvb3QsIHJlcG9fcm9vdCkKICAgIGlmIG5vdCBrYWdnbGVfcnVudGltZSBhbmQgbm90IF93aXRoaW4oZGF0YXNldF9yb290LCByZXBvX3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJMb2NhbCBjb21wZXRpdGlvbiBkYXRhIG11c3Qgc3RheSBpbiB0aGUgcmVwb3NpdG9yeToge2RhdGFzZXRfcm9vdH0iKQogICAgcnVuX3Jvb3QgPSAoCiAgICAgICAgUGF0aCgiL2thZ2dsZS93b3JraW5nIikgLyAiZXF1aW5leHVzIiAvIHJ1bl9uYW1lCiAgICAgICAgaWYga2FnZ2xlX3J1bnRpbWUKICAgICAgICBlbHNlIHJlcG9fcm9vdCAvICJhcnRpZmFjdHMiIC8gInN1Ym1pc3Npb25zIiAvIHJ1bl9uYW1lCiAgICApCiAgICBpZiBub3Qga2FnZ2xlX3J1bnRpbWUgYW5kIG5vdCBfd2l0aGluKHJ1bl9yb290LCByZXBvX3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJMb2NhbCBvdXRwdXRzIG11c3Qgc3RheSBpbiB0aGUgcmVwb3NpdG9yeToge3J1bl9yb290fSIpCiAgICBpZiBydW5fcm9vdC5leGlzdHMoKSBhbmQgYW55KHJ1bl9yb290Lml0ZXJkaXIoKSk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKGYiU3VibWlzc2lvbiBvdXRwdXQgYWxyZWFkeSBleGlzdHM6IHtydW5fcm9vdH0iKQogICAgcnVuX3Jvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGNvbmZpZ19wYXRoID0gcmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIKICAgIGNvbmZpZyA9IGpzb24ubG9hZHMoY29uZmlnX3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgY29uZmlnX3NoYTI1NiA9IF9maWxlX3NoYTI1Nihjb25maWdfcGF0aCkKICAgIGRhdGFzZXQgPSBsb2FkX2RhdGFzZXQoZGF0YXNldF9yb290KQogICAgZGF0YV9pbnZlbnRvcnkgPSBpbnZlbnRvcnkoZGF0YXNldCkKICAgIGV4cGVjdGVkID0gewogICAgICAgICJvYnNlcnZhdGlvbl9jb3VudCI6IDcwNywKICAgICAgICAiYW5ub3RhdG9yX3ZpZXdfY291bnQiOiAxXzE1NCwKICAgICAgICAiYW5ub3RhdGlvbl9jb3VudCI6IDhfMTk5LAogICAgICAgICJ0ZXN0X2ltYWdlX2ZpbGVfY291bnQiOiAxODAsCiAgICB9CiAgICBmb3Iga2V5LCB2YWx1ZSBpbiBleHBlY3RlZC5pdGVtcygpOgogICAgICAgIGlmIGRhdGFfaW52ZW50b3J5W2tleV0gIT0gdmFsdWU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCB7dmFsdWV9IHtrZXl9LCBmb3VuZCB7ZGF0YV9pbnZlbnRvcnlba2V5XX0iKQoKICAgIHRyYWluX25hbWVzID0gbGlzdChkYXRhc2V0Lm9ic2VydmF0aW9uX25hbWVzKQogICAgdGVzdF9uYW1lcyA9IHNvcnRlZChwYXRoLm5hbWUgZm9yIHBhdGggaW4gZGF0YXNldC50ZXN0X2ltYWdlX2Rpci5pdGVyZGlyKCkgaWYgcGF0aC5pc19maWxlKCkpCiAgICBpZiBsZW4odGVzdF9uYW1lcykgIT0gZXhwZWN0ZWRbInRlc3RfaW1hZ2VfZmlsZV9jb3VudCJdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgaW1hZ2UgaW52ZW50b3J5IGRpZmZlcnMgZnJvbSB0aGUgb2ZmaWNpYWwgMTgwLWltYWdlIHNldCIpCiAgICBpZiBsZW4oe1BhdGgobmFtZSkuc3RlbSBmb3IgbmFtZSBpbiB0ZXN0X25hbWVzfSkgIT0gbGVuKHRlc3RfbmFtZXMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgaW1hZ2Ugc3RlbXMgbXVzdCBiZSB1bmlxdWUgZm9yIHN0YWJsZSBmaWxhbWVudCBJRHMgYW5kIGNhY2hlcyIpCiAgICBtZW1iZXJzX3Jvb3QgPSBydW5fcm9vdCAvICJtZW1iZXJzIgogICAgbWVtYmVyc19yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHN0YXJ0X3RpbWUgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICByZXBvc2l0b3J5ID0gX2dpdF9zdGF0dXMocmVwb19yb290KQogICAgbWVtYmVyX3Jlc3VsdHMgPSBbXQogICAgZm9yIGluZGV4LCBtZW1iZXIgaW4gZW51bWVyYXRlKChmaXJzdF9tZW1iZXIsIHNlY29uZF9tZW1iZXIpLCBzdGFydD0xKToKICAgICAgICBuYW1lID0gbWVtYmVyLmdldCgibmFtZSIpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UobmFtZSwgc3RyKSBvciBub3QgcmUuZnVsbG1hdGNoKHIiW0EtWmEtejAtOV1bQS1aYS16MC05Ll8tXXswLDYzfSIsIG5hbWUpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFYWNoIGVuc2VtYmxlIG1lbWJlciBuZWVkcyBhIHNhZmUsIHVuaXF1ZSBuYW1lIikKICAgICAgICBwcmludChmIlRyYWluaW5nIGZpbmFsIGVuc2VtYmxlIG1lbWJlciB7aW5kZXh9LzI6IHtuYW1lfSIsIGZsdXNoPVRydWUpCiAgICAgICAgbWVtYmVyX3Jlc3VsdHMuYXBwZW5kKAogICAgICAgICAgICBfZml0X2FuZF9jYWNoZV9tZW1iZXIoCiAgICAgICAgICAgICAgICBkYXRhc2V0PWRhdGFzZXQsCiAgICAgICAgICAgICAgICB0cmFpbl9uYW1lcz10cmFpbl9uYW1lcywKICAgICAgICAgICAgICAgIHRlc3RfbmFtZXM9dGVzdF9uYW1lcywKICAgICAgICAgICAgICAgIG1lbWJlcj1tZW1iZXIsCiAgICAgICAgICAgICAgICBtZW1iZXJfcm9vdD1tZW1iZXJzX3Jvb3QgLyBuYW1lLAogICAgICAgICAgICAgICAgc2VlZD1pbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgICAgICApCiAgICAgICAgKQoKICAgIGZpcnN0X2NhY2hlID0gbWVtYmVyc19yb290IC8gZmlyc3RfbWVtYmVyWyJuYW1lIl0gLyAicHJvYmFiaWxpdGllcyIKICAgIHNlY29uZF9jYWNoZSA9IG1lbWJlcnNfcm9vdCAvIHNlY29uZF9tZW1iZXJbIm5hbWUiXSAvICJwcm9iYWJpbGl0aWVzIgogICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBsaXN0XSA9IHt9CiAgICBjb3ZlcmFnZV9yb3dzID0gW10KICAgIHBvc3Rwcm9jZXNzX3N0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBmb3IgaW5kZXgsIGZpbGVfbmFtZSBpbiBlbnVtZXJhdGUodGVzdF9uYW1lcywgc3RhcnQ9MSk6CiAgICAgICAgc3RlbSA9IFBhdGgoZmlsZV9uYW1lKS5zdGVtCiAgICAgICAgd2l0aCBucC5sb2FkKGZpcnN0X2NhY2hlIC8gZiJ7c3RlbX0ubnB6IikgYXMgZmlyc3QsIG5wLmxvYWQoCiAgICAgICAgICAgIHNlY29uZF9jYWNoZSAvIGYie3N0ZW19Lm5weiIKICAgICAgICApIGFzIHNlY29uZDoKICAgICAgICAgICAgZmlyc3RfZm9yZWdyb3VuZCA9IGZpcnN0WyJmb3JlZ3JvdW5kIl0uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgICAgIGZpcnN0X2JvdW5kYXJ5ID0gZmlyc3RbImJvdW5kYXJ5Il0uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgICAgIHNlY29uZF9mb3JlZ3JvdW5kID0gc2Vjb25kWyJmb3JlZ3JvdW5kIl0uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgICAgIHNlY29uZF9ib3VuZGFyeSA9IHNlY29uZFsiYm91bmRhcnkiXS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICBleHBlY3RlZF9zaGFwZSA9IChJTUFHRV9TSVpFLCBJTUFHRV9TSVpFKQogICAgICAgIGlmIG5vdCAoCiAgICAgICAgICAgIGZpcnN0X2ZvcmVncm91bmQuc2hhcGUKICAgICAgICAgICAgPT0gZmlyc3RfYm91bmRhcnkuc2hhcGUKICAgICAgICAgICAgPT0gc2Vjb25kX2ZvcmVncm91bmQuc2hhcGUKICAgICAgICAgICAgPT0gc2Vjb25kX2JvdW5kYXJ5LnNoYXBlCiAgICAgICAgICAgID09IGV4cGVjdGVkX3NoYXBlCiAgICAgICAgKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgZW5zZW1ibGUgcHJvYmFiaWxpdHkgZGltZW5zaW9ucyBmb3Ige2ZpbGVfbmFtZX0iKQogICAgICAgIGZvcmVncm91bmQgPSBmaXJzdF93ZWlnaHQgKiBmaXJzdF9mb3JlZ3JvdW5kICsgKDEuMCAtIGZpcnN0X3dlaWdodCkgKiBzZWNvbmRfZm9yZWdyb3VuZAogICAgICAgIGJvdW5kYXJ5ID0gZmlyc3Rfd2VpZ2h0ICogZmlyc3RfYm91bmRhcnkgKyAoMS4wIC0gZmlyc3Rfd2VpZ2h0KSAqIHNlY29uZF9ib3VuZGFyeQogICAgICAgIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0gPSBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzKAogICAgICAgICAgICBmb3JlZ3JvdW5kLCBib3VuZGFyeSwgZmlsZV9uYW1lLCBpbnN0YW5jZV9jb25maWcKICAgICAgICApCiAgICAgICAgY292ZXJhZ2Vfcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsiZmlsZV9uYW1lIjogZmlsZV9uYW1lLCAiaW5zdGFuY2VfY291bnQiOiBsZW4ocHJlZGljdGlvbnNbZmlsZV9uYW1lXSl9CiAgICAgICAgKQogICAgICAgIGlmIGluZGV4ICUgMTAgPT0gMCBvciBpbmRleCA9PSBsZW4odGVzdF9uYW1lcyk6CiAgICAgICAgICAgIHByaW50KGYiUG9zdHByb2Nlc3NlZCB7aW5kZXh9L3tsZW4odGVzdF9uYW1lcyl9IHRlc3QgaW1hZ2VzIiwgZmx1c2g9VHJ1ZSkKICAgIHBvc3Rwcm9jZXNzX3NlY29uZHMgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gcG9zdHByb2Nlc3Nfc3RhcnRlZAoKICAgIHN1Ym1pc3Npb25fcGF0aCA9IHJ1bl9yb290IC8gInN1Ym1pc3Npb24uY3N2IgogICAgX3dyaXRlX3ByZWRpY3Rpb25zKHN1Ym1pc3Npb25fcGF0aCwgcHJlZGljdGlvbnMpCiAgICBzdWJtaXNzaW9uX3ZhbGlkYXRpb24gPSBfY2hlY2tfc3VibWlzc2lvbigKICAgICAgICBzdWJtaXNzaW9uX3BhdGgsIGRhdGFzZXQucm9vdCwgdGVzdF9uYW1lcywgcHJlZGljdGlvbnMKICAgICkKICAgIGNvdmVyYWdlX3BhdGggPSBydW5fcm9vdCAvICJ0ZXN0X2NvdmVyYWdlLmNzdiIKICAgIHdpdGggY292ZXJhZ2VfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKAogICAgICAgICAgICBzdHJlYW0sIGZpZWxkbmFtZXM9KCJmaWxlX25hbWUiLCAiaW5zdGFuY2VfY291bnQiKSwgbGluZXRlcm1pbmF0b3I9IlxuIgogICAgICAgICkKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIHdyaXRlci53cml0ZXJvd3MoY292ZXJhZ2Vfcm93cykKCiAgICBpZiBvcy5lbnZpcm9uLmdldCgiRVFVSU5FWFVTX1NPVVJDRV9DT01NSVQiKToKICAgICAgICByZXBvc2l0b3J5WyJjb21taXQiXSA9IG9zLmVudmlyb25bIkVRVUlORVhVU19TT1VSQ0VfQ09NTUlUIl0KICAgICAgICByZXBvc2l0b3J5WyJ3b3JrdHJlZV9kaXJ0eSJdID0gVHJ1ZQogICAgcmVwb3NpdG9yeVsic291cmNlX2J1bmRsZV9zaGEyNTYiXSA9IG9zLmVudmlyb24uZ2V0KAogICAgICAgICJFUVVJTkVYVVNfU09VUkNFX0JVTkRMRV9TSEEyNTYiCiAgICApCiAgICByZXN1bHQgPSB7CiAgICAgICAgInN0YXR1cyI6ICJjb21wbGV0ZSIsCiAgICAgICAgImNyZWF0ZWRfdXRjIjogdGltZS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU06JVNaIiwgdGltZS5nbXRpbWUoKSksCiAgICAgICAgInJ1bl9uYW1lIjogcnVuX25hbWUsCiAgICAgICAgInJlcG9zaXRvcnkiOiByZXBvc2l0b3J5LAogICAgICAgICJjb25maWd1cmF0aW9uX3NoYTI1NiI6IGNvbmZpZ19zaGEyNTYsCiAgICAgICAgImthZ2dsZV9ub3RlYm9va192ZXJzaW9uIjogb3MuZW52aXJvbi5nZXQoIkVRVUlORVhVU19OT1RFQk9PS19WRVJTSU9OIiksCiAgICAgICAgImRhdGFfYm91bmRhcnkiOiB7CiAgICAgICAgICAgICJ0cmFpbmluZ19sYWJlbHMiOiAiY29tcGV0aXRpb24tcHJvdmlkZWQgdHJhaW4gYW5ub3RhdGlvbnMgb25seSIsCiAgICAgICAgICAgICJpbmZlcmVuY2VfaW5wdXRzIjogImF0dGFjaGVkIGNvbXBldGl0aW9uIEgtYWxwaGEgdGVzdCBpbWFnZXMgb25seSIsCiAgICAgICAgICAgICJleHRlcm5hbF9ncm91bmRfdHJ1dGgiOiBGYWxzZSwKICAgICAgICB9LAogICAgICAgICJkYXRhc2V0X2ludmVudG9yeSI6IGRhdGFfaW52ZW50b3J5LAogICAgICAgICJ2YWxpZGF0aW9uX3JlZmVyZW5jZSI6IHsKICAgICAgICAgICAgInNwbGl0X3NoYTI1NiI6IGNvbmZpZ1siZXhwZWN0ZWRfc3BsaXRfc2hhMjU2Il0sCiAgICAgICAgICAgICJzZWxlY3Rpb25fbm90ZSI6ICJUaGUgZnJvemVuIG9ic2VydmF0aW9uLWdyb3VwZWQgaG9sZG91dCBzZWxlY3RlZCB0aGUgZW5zZW1ibGUgYW5kIHBvc3Rwcm9jZXNzaW5nIGJlZm9yZSBmaW5hbCByZWZpdHRpbmcgb24gYWxsIG9mZmljaWFsIHRyYWluaW5nIG9ic2VydmF0aW9ucy4iLAogICAgICAgIH0sCiAgICAgICAgImVuc2VtYmxlIjogewogICAgICAgICAgICAiZmlyc3RfbWVtYmVyIjogZmlyc3RfbWVtYmVyLAogICAgICAgICAgICAic2Vjb25kX21lbWJlciI6IHNlY29uZF9tZW1iZXIsCiAgICAgICAgICAgICJmaXJzdF9tZW1iZXJfd2VpZ2h0IjogZmlyc3Rfd2VpZ2h0LAogICAgICAgICAgICAic2Vjb25kX21lbWJlcl93ZWlnaHQiOiAxLjAgLSBmaXJzdF93ZWlnaHQsCiAgICAgICAgICAgICJpbnN0YW5jZV9jb25maWciOiBpbnN0YW5jZV9jb25maWcuX19kaWN0X18sCiAgICAgICAgICAgICJleHRlcm5hbF9zdXBlcnZpc2VkX3dlaWdodHMiOiBGYWxzZSwKICAgICAgICB9LAogICAgICAgICJtZW1iZXJzIjogbWVtYmVyX3Jlc3VsdHMsCiAgICAgICAgInRlc3RfdmFsaWRhdGlvbiI6IHN1Ym1pc3Npb25fdmFsaWRhdGlvbiwKICAgICAgICAicnVudGltZSI6IHsKICAgICAgICAgICAgKipoYXJkd2FyZV9pbmZvKCksCiAgICAgICAgICAgICJwb3N0cHJvY2Vzc19zZWNvbmRzIjogcG9zdHByb2Nlc3Nfc2Vjb25kcywKICAgICAgICAgICAgImVuZF90b19lbmRfc2Vjb25kcyI6IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydF90aW1lLAogICAgICAgICAgICAicGFja2FnZV92ZXJzaW9ucyI6IHBhY2thZ2VfdmVyc2lvbnMoKSwKICAgICAgICAgICAgIm1heF9tZW1iZXJfdHJhaW5pbmdfZ3B1X21lbW9yeV9ieXRlcyI6IG1heCgKICAgICAgICAgICAgICAgIGl0ZW1bInRyYWluaW5nIl1bInBlYWtfZ3B1X21lbW9yeV9ieXRlcyJdIGZvciBpdGVtIGluIG1lbWJlcl9yZXN1bHRzCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJtYXhfbWVtYmVyX2luZmVyZW5jZV9ncHVfbWVtb3J5X2J5dGVzIjogbWF4KAogICAgICAgICAgICAgICAgaXRlbVsiaW5mZXJlbmNlX3BlYWtfZ3B1X21lbW9yeV9ieXRlcyJdIGZvciBpdGVtIGluIG1lbWJlcl9yZXN1bHRzCiAgICAgICAgICAgICksCiAgICAgICAgfSwKICAgICAgICAiYXJ0aWZhY3RzIjogewogICAgICAgICAgICAic3VibWlzc2lvbiI6IHN1Ym1pc3Npb25fcGF0aC5uYW1lLAogICAgICAgICAgICAiY292ZXJhZ2UiOiBjb3ZlcmFnZV9wYXRoLm5hbWUsCiAgICAgICAgICAgICJtZW1iZXJzX3Jvb3QiOiBzdHIobWVtYmVyc19yb290Lm5hbWUpLAogICAgICAgIH0sCiAgICB9CiAgICAocnVuX3Jvb3QgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHByaW50KAogICAgICAgIGYiVmFsaWRhdGVkIGVuc2VtYmxlIHN1Ym1pc3Npb24gd2l0aCB7c3VibWlzc2lvbl92YWxpZGF0aW9uWydwcmVkaWN0ZWRfaW5zdGFuY2VzJ119IGluc3RhbmNlcyAiCiAgICAgICAgZiJmb3Ige3N1Ym1pc3Npb25fdmFsaWRhdGlvblsndGVzdF9pbWFnZXMnXX0gaW1hZ2VzOiB7c3VibWlzc2lvbl9wYXRofSIsCiAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICkKICAgIHJldHVybiByZXN1bHQK\", \"src/filament_baseline/external_data.py\": \"...\", \"src/filament_baseline/features.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBmdW5jdG9vbHMgaW1wb3J0IGxydV9jYWNoZQpmcm9tIHR5cGluZyBpbXBvcnQgTGl0ZXJhbAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKCgpJbnB1dE1vZGUgPSBMaXRlcmFsWyJyYXciLCAicmF3X2xvY2FsX2NvbnRyYXN0IiwgInJhd19sb2NhbF9jb250cmFzdF9yYWRpYWwiXQpJTlBVVF9NT0RFUzogdHVwbGVbSW5wdXRNb2RlLCAuLi5dID0gKAogICAgInJhdyIsCiAgICAicmF3X2xvY2FsX2NvbnRyYXN0IiwKICAgICJyYXdfbG9jYWxfY29udHJhc3RfcmFkaWFsIiwKKQpMT0NBTF9DT05UUkFTVF9TSUdNQSA9IDguMApSQURJQUxfUFJPRklMRV9CSU5TID0gMjU2ClJBRElBTF9QUk9GSUxFX1NBTVBMRV9TVFJJREUgPSA0CgoKZGVmIGlucHV0X2NoYW5uZWxfY291bnQoaW5wdXRfbW9kZTogSW5wdXRNb2RlKSAtPiBpbnQ6CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInJhdyI6IDEsCiAgICAgICAgICAgICJyYXdfbG9jYWxfY29udHJhc3QiOiAyLAogICAgICAgICAgICAicmF3X2xvY2FsX2NvbnRyYXN0X3JhZGlhbCI6IDMsCiAgICAgICAgfVtpbnB1dF9tb2RlXQogICAgZXhjZXB0IEtleUVycm9yIGFzIGV4YzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgaW5wdXQgbW9kZToge2lucHV0X21vZGV9IikgZnJvbSBleGMKCgpkZWYgaW5wdXRfbW9kZV9mb3JfY2hhbm5lbF9jb3VudChjaGFubmVsX2NvdW50OiBpbnQpIC0+IElucHV0TW9kZToKICAgIHRyeToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAxOiAicmF3IiwKICAgICAgICAgICAgMjogInJhd19sb2NhbF9jb250cmFzdCIsCiAgICAgICAgICAgIDM6ICJyYXdfbG9jYWxfY29udHJhc3RfcmFkaWFsIiwKICAgICAgICB9W2NoYW5uZWxfY291bnRdCiAgICBleGNlcHQgS2V5RXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnN1cHBvcnRlZCBtb2RlbCBpbnB1dCBjaGFubmVsIGNvdW50OiB7Y2hhbm5lbF9jb3VudH0iKSBmcm9tIGV4YwoKCkBscnVfY2FjaGUobWF4c2l6ZT00KQpkZWYgX3NvbGFyX2dlb21ldHJ5KAogICAgaGVpZ2h0OiBpbnQsIHdpZHRoOiBpbnQsIHNvbGFyX3JhZGl1czogaW50CikgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICB5eSwgeHggPSBucC5vZ3JpZFs6aGVpZ2h0LCA6d2lkdGhdCiAgICBjZW50ZXJfeSA9IGZsb2F0KChoZWlnaHQgLSAxKSAvLyAyKQogICAgY2VudGVyX3ggPSBmbG9hdCgod2lkdGggLSAxKSAvLyAyKQogICAgcmFkaXVzX3NxdWFyZWQgPSAoeXkgLSBjZW50ZXJfeSkgKiogMiArICh4eCAtIGNlbnRlcl94KSAqKiAyCiAgICBkaXNrID0gcmFkaXVzX3NxdWFyZWQgPD0gc29sYXJfcmFkaXVzKioyCiAgICByYWRpdXMgPSBucC5zcXJ0KHJhZGl1c19zcXVhcmVkKQogICAgcmFkaWFsX2JpbiA9IG5wLm1pbmltdW0oCiAgICAgICAgKHJhZGl1cyAqIChSQURJQUxfUFJPRklMRV9CSU5TIC8gbWF4KHNvbGFyX3JhZGl1cywgMSkpKS5hc3R5cGUobnAuaW50MTYpLAogICAgICAgIFJBRElBTF9QUk9GSUxFX0JJTlMgLSAxLAogICAgKQogICAgZGlzay5zZXRmbGFncyh3cml0ZT1GYWxzZSkKICAgIHJhZGlhbF9iaW4uc2V0ZmxhZ3Mod3JpdGU9RmFsc2UpCiAgICByZXR1cm4gZGlzaywgcmFkaWFsX2JpbgoKCmRlZiBfcmFkaWFsX2NvbnRyYXN0KAogICAgc2NhbGVkOiBucC5uZGFycmF5LCBkaXNrOiBucC5uZGFycmF5LCByYWRpYWxfYmluOiBucC5uZGFycmF5CikgLT4gbnAubmRhcnJheToKICAgICIiIk5vcm1hbGl6ZSBkYXJrIHN0cnVjdHVyZSBhZ2FpbnN0IHRoZSBtZWRpYW4gaW50ZW5zaXR5IGF0IGVhY2ggc29sYXIgcmFkaXVzLiIiIgogICAgc2FtcGxlZF9zY2FsZWQgPSBzY2FsZWRbOjpSQURJQUxfUFJPRklMRV9TQU1QTEVfU1RSSURFLCA6OlJBRElBTF9QUk9GSUxFX1NBTVBMRV9TVFJJREVdCiAgICBzYW1wbGVkX2Rpc2sgPSBkaXNrWzo6UkFESUFMX1BST0ZJTEVfU0FNUExFX1NUUklERSwgOjpSQURJQUxfUFJPRklMRV9TQU1QTEVfU1RSSURFXQogICAgc2FtcGxlZF9iaW5zID0gcmFkaWFsX2JpblsKICAgICAgICA6OlJBRElBTF9QUk9GSUxFX1NBTVBMRV9TVFJJREUsIDo6UkFESUFMX1BST0ZJTEVfU0FNUExFX1NUUklERQogICAgXQogICAgaW50ZW5zaXR5ID0gbnAucmludChzYW1wbGVkX3NjYWxlZCAqIDI1NS4wKS5jbGlwKDAsIDI1NSkuYXN0eXBlKG5wLnVpbnQ4KQogICAgcGFja2VkX2JpbnMgPSAoCiAgICAgICAgc2FtcGxlZF9iaW5zW3NhbXBsZWRfZGlza10uYXN0eXBlKG5wLmludDMyKSAqIDI1NgogICAgICAgICsgaW50ZW5zaXR5W3NhbXBsZWRfZGlza10uYXN0eXBlKG5wLmludDMyKQogICAgKQogICAgaGlzdG9ncmFtID0gbnAuYmluY291bnQocGFja2VkX2JpbnMsIG1pbmxlbmd0aD1SQURJQUxfUFJPRklMRV9CSU5TICogMjU2KS5yZXNoYXBlKAogICAgICAgIFJBRElBTF9QUk9GSUxFX0JJTlMsIDI1NgogICAgKQogICAgY291bnRzID0gaGlzdG9ncmFtLnN1bShheGlzPTEpCiAgICBjdW11bGF0aXZlID0gaGlzdG9ncmFtLmN1bXN1bShheGlzPTEpCiAgICBtZWRpYW5fYmlucyA9IG5wLmFyZ21heChjdW11bGF0aXZlID49ICgoY291bnRzICsgMSkgLy8gMilbOiwgTm9uZV0sIGF4aXM9MSkKICAgIHBvcHVsYXRlZCA9IGNvdW50cyA+IDAKICAgIHByb2ZpbGUgPSBucC5pbnRlcnAoCiAgICAgICAgbnAuYXJhbmdlKFJBRElBTF9QUk9GSUxFX0JJTlMpLAogICAgICAgIG5wLmZsYXRub256ZXJvKHBvcHVsYXRlZCksCiAgICAgICAgbWVkaWFuX2JpbnNbcG9wdWxhdGVkXS5hc3R5cGUobnAuZmxvYXQzMikgLyAyNTUuMCwKICAgICkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBwcm9maWxlID0gY3YyLkdhdXNzaWFuQmx1cihwcm9maWxlW05vbmUsIDpdLCAoMCwgMCksIHNpZ21hWD0yLjApWzBdCgogICAgcmFkaWFsX2NvbnRyYXN0ID0gcHJvZmlsZVtyYWRpYWxfYmluXSAtIHNjYWxlZAogICAgZGlza192YWx1ZXMgPSByYWRpYWxfY29udHJhc3RbZGlza10KICAgIG1lYW4gPSBmbG9hdChucC5tZWFuKGRpc2tfdmFsdWVzKSkKICAgIHN0ZCA9IG1heChmbG9hdChucC5zdGQoZGlza192YWx1ZXMpKSwgMWUtMykKICAgIG5vcm1hbGl6ZWQgPSBucC5jbGlwKChyYWRpYWxfY29udHJhc3QgLSBtZWFuKSAvIHN0ZCwgLTUuMCwgNS4wKQogICAgbm9ybWFsaXplZFt+ZGlza10gPSAwLjAKICAgIHJldHVybiBub3JtYWxpemVkLmFzdHlwZShucC5mbG9hdDMyLCBjb3B5PUZhbHNlKQoKCmRlZiBwcmVwYXJlX2lucHV0X2NoYW5uZWxzKAogICAgZ3JheV9pbWFnZTogbnAubmRhcnJheSwKICAgIGlucHV0X21vZGU6IElucHV0TW9kZSA9ICJyYXciLAogICAgKiwKICAgIHNvbGFyX3JhZGl1czogaW50ID0gOTUwLAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJCdWlsZCBub3JtYWxpemVkIGNoYW5uZWxzIGZyb20gb25lIDgtYml0IEgtYWxwaGEgb2JzZXJ2YXRpb24uIiIiCiAgICBpZiBncmF5X2ltYWdlLm5kaW0gIT0gMiBvciBncmF5X2ltYWdlLmR0eXBlICE9IG5wLnVpbnQ4OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIGFuIDgtYml0IGdyYXlzY2FsZSBpbWFnZSIpCiAgICBpbnB1dF9jaGFubmVsX2NvdW50KGlucHV0X21vZGUpCgogICAgaGVpZ2h0LCB3aWR0aCA9IGdyYXlfaW1hZ2Uuc2hhcGUKICAgIGRpc2ssIHJhZGlhbF9iaW4gPSBfc29sYXJfZ2VvbWV0cnkoaGVpZ2h0LCB3aWR0aCwgc29sYXJfcmFkaXVzKQogICAgaWYgbm90IG5wLmFueShkaXNrKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTb2xhciBkaXNrIG1hc2sgaXMgZW1wdHkgZm9yIHRoZSBzdXBwbGllZCBpbWFnZSBhbmQgcmFkaXVzIikKCiAgICBzY2FsZWQgPSBncmF5X2ltYWdlLmFzdHlwZShucC5mbG9hdDMyKSAvIDI1NS4wCiAgICBkaXNrX3ZhbHVlcyA9IHNjYWxlZFtkaXNrXQogICAgaW1hZ2VfbWVhbiA9IGZsb2F0KG5wLm1lYW4oZGlza192YWx1ZXMpKQogICAgaW1hZ2Vfc3RkID0gbWF4KGZsb2F0KG5wLnN0ZChkaXNrX3ZhbHVlcykpLCAxZS0zKQogICAgcmF3X25vcm1hbGl6ZWQgPSBucC5jbGlwKChzY2FsZWQgLSBpbWFnZV9tZWFuKSAvIGltYWdlX3N0ZCwgLTUuMCwgNS4wKQogICAgaWYgaW5wdXRfbW9kZSA9PSAicmF3IjoKICAgICAgICByZXR1cm4gcmF3X25vcm1hbGl6ZWRbTm9uZV0uYXN0eXBlKG5wLmZsb2F0MzIsIGNvcHk9RmFsc2UpCgogICAgYmx1cnJlZCA9IGN2Mi5HYXVzc2lhbkJsdXIoCiAgICAgICAgc2NhbGVkLCAoMCwgMCksIHNpZ21hWD1MT0NBTF9DT05UUkFTVF9TSUdNQSwgc2lnbWFZPUxPQ0FMX0NPTlRSQVNUX1NJR01BCiAgICApCiAgICBsb2NhbF9jb250cmFzdCA9IGJsdXJyZWQgLSBzY2FsZWQKICAgIGNvbnRyYXN0X3ZhbHVlcyA9IGxvY2FsX2NvbnRyYXN0W2Rpc2tdCiAgICBjb250cmFzdF9tZWFuID0gZmxvYXQobnAubWVhbihjb250cmFzdF92YWx1ZXMpKQogICAgY29udHJhc3Rfc3RkID0gbWF4KGZsb2F0KG5wLnN0ZChjb250cmFzdF92YWx1ZXMpKSwgMWUtMykKICAgIGNvbnRyYXN0X25vcm1hbGl6ZWQgPSBucC5jbGlwKAogICAgICAgIChsb2NhbF9jb250cmFzdCAtIGNvbnRyYXN0X21lYW4pIC8gY29udHJhc3Rfc3RkLCAtNS4wLCA1LjAKICAgICkKICAgIGNoYW5uZWxzID0gW3Jhd19ub3JtYWxpemVkLCBjb250cmFzdF9ub3JtYWxpemVkXQogICAgaWYgaW5wdXRfbW9kZSA9PSAicmF3X2xvY2FsX2NvbnRyYXN0X3JhZGlhbCI6CiAgICAgICAgY2hhbm5lbHMuYXBwZW5kKF9yYWRpYWxfY29udHJhc3Qoc2NhbGVkLCBkaXNrLCByYWRpYWxfYmluKSkKICAgIHJldHVybiBucC5zdGFjayhjaGFubmVscykuYXN0eXBlKG5wLmZsb2F0MzIsIGNvcHk9RmFsc2UpCg==\", \"src/filament_baseline/gpu_smoke.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCgpkZWYgcnVuX2dwdV9zbW9rZShvdXRwdXRfcGF0aDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGltcG9ydCB0b3JjaAoKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgIlB5VG9yY2ggY2Fubm90IHNlZSBhIENVREEgR1BVLiBDb25maXJtIHRoZSBDVURBLWVuYWJsZWQgd2hlZWwgYW5kIE5WSURJQSBkcml2ZXIuIgogICAgICAgICkKICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhOjAiKQogICAgbW9kZWwgPSB0b3JjaC5ubi5Db252MmQoMSwgOCwga2VybmVsX3NpemU9MywgcGFkZGluZz0xKS50byhkZXZpY2UpCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTFlLTMpCiAgICBpbWFnZSA9IHRvcmNoLnJhbmRuKDEsIDEsIDI1NiwgMjU2LCBkZXZpY2U9ZGV2aWNlKQogICAgdGFyZ2V0ID0gdG9yY2gucmFuZG4oMSwgOCwgMjU2LCAyNTYsIGRldmljZT1kZXZpY2UpCgogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9ImN1ZGEiLCBkdHlwZT10b3JjaC5mbG9hdDE2KToKICAgICAgICBwcmVkaWN0aW9uID0gbW9kZWwoaW1hZ2UpCiAgICAgICAgbG9zcyA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwubXNlX2xvc3MocHJlZGljdGlvbiwgdGFyZ2V0KQogICAgbG9zcy5iYWNrd2FyZCgpCiAgICBvcHRpbWl6ZXIuc3RlcCgpCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZAoKICAgIHByb3BzID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic3RhdHVzIjogInBhc3NlZCIsCiAgICAgICAgInRvcmNoX3ZlcnNpb24iOiB0b3JjaC5fX3ZlcnNpb25fXywKICAgICAgICAidG9yY2hfY3VkYV92ZXJzaW9uIjogdG9yY2gudmVyc2lvbi5jdWRhLAogICAgICAgICJkZXZpY2VfbmFtZSI6IHByb3BzLm5hbWUsCiAgICAgICAgImRldmljZV9tZW1vcnlfYnl0ZXMiOiBpbnQocHJvcHMudG90YWxfbWVtb3J5KSwKICAgICAgICAiY29tcHV0ZV9jYXBhYmlsaXR5IjogZiJ7cHJvcHMubWFqb3J9Lntwcm9wcy5taW5vcn0iLAogICAgICAgICJsb3NzIjogZmxvYXQobG9zcy5kZXRhY2goKS5jcHUoKSksCiAgICAgICAgImVsYXBzZWRfc2Vjb25kcyI6IGVsYXBzZWQsCiAgICAgICAgInBlYWtfbWVtb3J5X2J5dGVzIjogaW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoMCkpLAogICAgICAgICJzaGFwZSI6IFsxLCAxLCAyNTYsIDI1Nl0sCiAgICAgICAgImFtcCI6ICJmbG9hdDE2IiwKICAgIH0KICAgIGlmIG91dHB1dF9wYXRoIGlzIG5vdCBOb25lOgogICAgICAgIG91dHB1dF9wYXRoID0gUGF0aChvdXRwdXRfcGF0aCkKICAgICAgICBvdXRwdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIG91dHB1dF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIpKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgaW1wb3J0IGFyZ3BhcnNlCgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlZlcmlmeSBsb2NhbCBDVURBIGZvcndhcmQvYmFja3dhcmQgc3VwcG9ydC4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1vdXRwdXQiLAogICAgICAgIGRlZmF1bHQ9ImFydGlmYWN0cy9sb2NhbC1ncHUtc21va2UvZ3B1X3Ntb2tlLmpzb24iLAogICAgICAgIGhlbHA9Ik91dHB1dCBKU09OIHBhdGg7IGtlZXAgaXQgaW5zaWRlIHRoZSByZXBvc2l0b3J5LiIsCiAgICApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmV0dXJuIDAgaWYgcnVuX2dwdV9zbW9rZShhcmdzLm91dHB1dClbInN0YXR1cyJdID09ICJwYXNzZWQiIGVsc2UgMQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK\", \"src/filament_baseline/inference.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gc2NpcHkgaW1wb3J0IG5kaW1hZ2UgYXMgbmRpCmZyb20gc2tpbWFnZS5zZWdtZW50YXRpb24gaW1wb3J0IHdhdGVyc2hlZAoKZnJvbSAuZGF0YSBpbXBvcnQgSU1BR0VfU0laRQpmcm9tIC5mZWF0dXJlcyBpbXBvcnQgaW5wdXRfbW9kZV9mb3JfY2hhbm5lbF9jb3VudCwgcHJlcGFyZV9pbnB1dF9jaGFubmVscwpmcm9tIC5tb2RlbCBpbXBvcnQgRmlsYW1lbnRVTmV0CmZyb20gLnByZWRpY3RvciBpbXBvcnQgUHJlZGljdGVkSW5zdGFuY2UsIGVuY29kZV9sYWJlbF9jb21wb25lbnRzCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgSW5zdGFuY2VDb25maWc6CiAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZDogZmxvYXQgPSAwLjM1CiAgICBib3VuZGFyeV90aHJlc2hvbGQ6IGZsb2F0ID0gMC41CiAgICBib3VuZGFyeV93ZWlnaHQ6IGZsb2F0ID0gMS41CiAgICBzZWVkX3RocmVzaG9sZDogZmxvYXQgPSAwLjQ1CiAgICBzZWVkX21pbl9hcmVhOiBpbnQgPSA4CiAgICBtaW5faW5zdGFuY2VfYXJlYTogaW50ID0gNTAKICAgIHNvbGFyX3JhZGl1czogaW50ID0gOTUwCgoKZGVmIF90aWxlX3N0YXJ0cyhsZW5ndGg6IGludCwgdGlsZV9zaXplOiBpbnQsIHN0cmlkZTogaW50KSAtPiBsaXN0W2ludF06CiAgICBpZiB0aWxlX3NpemUgPiBsZW5ndGggb3Igc3RyaWRlIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0aWxlIHNpemUgbXVzdCBmaXQgdGhlIGltYWdlIGFuZCBzdHJpZGUgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBzdGFydHMgPSBsaXN0KHJhbmdlKDAsIGxlbmd0aCAtIHRpbGVfc2l6ZSArIDEsIHN0cmlkZSkpCiAgICBmaW5hbCA9IGxlbmd0aCAtIHRpbGVfc2l6ZQogICAgaWYgbm90IHN0YXJ0cyBvciBzdGFydHNbLTFdICE9IGZpbmFsOgogICAgICAgIHN0YXJ0cy5hcHBlbmQoZmluYWwpCiAgICByZXR1cm4gc3RhcnRzCgoKZGVmIF90cmFuc2Zvcm0odGVuc29yOiB0b3JjaC5UZW5zb3IsIG9wZXJhdGlvbjogaW50KSAtPiB0b3JjaC5UZW5zb3I6CiAgICBpZiBvcGVyYXRpb24gPT0gMToKICAgICAgICByZXR1cm4gdG9yY2guZmxpcCh0ZW5zb3IsIGRpbXM9KC0xLCkpCiAgICBpZiBvcGVyYXRpb24gPT0gMjoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAodGVuc29yLCAyLCBkaW1zPSgtMiwgLTEpKQogICAgaWYgb3BlcmF0aW9uID09IDM6CiAgICAgICAgcmV0dXJuIHRvcmNoLmZsaXAodGVuc29yLCBkaW1zPSgtMiwpKQogICAgaWYgb3BlcmF0aW9uID09IDQ6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJvdDkwKHRlbnNvciwgMSwgZGltcz0oLTIsIC0xKSkKICAgIGlmIG9wZXJhdGlvbiA9PSA1OgogICAgICAgIHJldHVybiB0b3JjaC5yb3Q5MCh0ZW5zb3IsIDMsIGRpbXM9KC0yLCAtMSkpCiAgICBpZiBvcGVyYXRpb24gPT0gNjoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAodG9yY2guZmxpcCh0ZW5zb3IsIGRpbXM9KC0xLCkpLCAxLCBkaW1zPSgtMiwgLTEpKQogICAgaWYgb3BlcmF0aW9uID09IDc6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJvdDkwKHRvcmNoLmZsaXAodGVuc29yLCBkaW1zPSgtMSwpKSwgMywgZGltcz0oLTIsIC0xKSkKICAgIHJldHVybiB0ZW5zb3IKCgpkZWYgX2ludmVyc2VfdHJhbnNmb3JtKHRlbnNvcjogdG9yY2guVGVuc29yLCBvcGVyYXRpb246IGludCkgLT4gdG9yY2guVGVuc29yOgogICAgaWYgb3BlcmF0aW9uID09IDQ6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJvdDkwKHRlbnNvciwgMywgZGltcz0oLTIsIC0xKSkKICAgIGlmIG9wZXJhdGlvbiA9PSA1OgogICAgICAgIHJldHVybiB0b3JjaC5yb3Q5MCh0ZW5zb3IsIDEsIGRpbXM9KC0yLCAtMSkpCiAgICByZXR1cm4gX3RyYW5zZm9ybSh0ZW5zb3IsIG9wZXJhdGlvbikKCgpAdG9yY2guaW5mZXJlbmNlX21vZGUoKQpkZWYgcHJlZGljdF9wcm9iYWJpbGl0aWVzKAogICAgbW9kZWw6IEZpbGFtZW50VU5ldCwKICAgIGdyYXlfaW1hZ2U6IG5wLm5kYXJyYXksCiAgICAqLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UgfCBOb25lID0gTm9uZSwKICAgIHBhdGNoX3NpemU6IGludCA9IDUxMiwKICAgIHN0cmlkZTogaW50ID0gMzg0LAogICAgdHRhOiBpbnQgPSAxLAopIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgaWYgZ3JheV9pbWFnZS5uZGltICE9IDIgb3IgZ3JheV9pbWFnZS5kdHlwZSAhPSBucC51aW50ODoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCBhbiA4LWJpdCBncmF5c2NhbGUgaW1hZ2UiKQogICAgaWYgZ3JheV9pbWFnZS5zaGFwZSAhPSAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIHtJTUFHRV9TSVpFfXh7SU1BR0VfU0laRX0sIGdvdCB7Z3JheV9pbWFnZS5zaGFwZX0iKQogICAgaWYgcGF0Y2hfc2l6ZSAlIDMyIG9yIHBhdGNoX3NpemUgPiBJTUFHRV9TSVpFOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBhdGNoX3NpemUgbXVzdCBiZSBkaXZpc2libGUgYnkgMzIgYW5kIG5vIGxhcmdlciB0aGFuIHRoZSBpbWFnZSIpCiAgICBpZiB0dGEgbm90IGluICgxLCAyLCA0LCA4KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0dGEgbXVzdCBiZSAxLCAyLCA0LCBvciA4IikKICAgIGRldmljZSA9IGRldmljZSBvciBuZXh0KG1vZGVsLnBhcmFtZXRlcnMoKSkuZGV2aWNlCiAgICBtb2RlbC5ldmFsKCkKCiAgICBpbnB1dF9tb2RlID0gaW5wdXRfbW9kZV9mb3JfY2hhbm5lbF9jb3VudChtb2RlbC5pbl9jaGFubmVscykKICAgIG5vcm1hbGl6ZWRfY2hhbm5lbHMgPSBwcmVwYXJlX2lucHV0X2NoYW5uZWxzKAogICAgICAgIGdyYXlfaW1hZ2UsIGlucHV0X21vZGUsIHNvbGFyX3JhZGl1cz05NTAKICAgICkKCiAgICBvbmVfZGltZW5zaW9uYWwgPSBucC5oYW5uaW5nKHBhdGNoX3NpemUpLmFzdHlwZShucC5mbG9hdDMyKQogICAgb25lX2RpbWVuc2lvbmFsID0gbnAubWF4aW11bShvbmVfZGltZW5zaW9uYWwsIDAuMDUpCiAgICB3ZWlnaHQgPSBucC5vdXRlcihvbmVfZGltZW5zaW9uYWwsIG9uZV9kaW1lbnNpb25hbCkKICAgIGFjY3VtdWxhdGlvbiA9IG5wLnplcm9zKCgyLCBJTUFHRV9TSVpFLCBJTUFHRV9TSVpFKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgIHdlaWdodF9zdW0gPSBucC56ZXJvcygoSU1BR0VfU0laRSwgSU1BR0VfU0laRSksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBvcGVyYXRpb25zID0gKDAsIDEsIDIsIDMsIDQsIDUsIDYsIDcpWzp0dGFdCiAgICBncHUgPSBkZXZpY2UudHlwZSA9PSAiY3VkYSIKCiAgICBmb3IgeTAgaW4gX3RpbGVfc3RhcnRzKElNQUdFX1NJWkUsIHBhdGNoX3NpemUsIHN0cmlkZSk6CiAgICAgICAgZm9yIHgwIGluIF90aWxlX3N0YXJ0cyhJTUFHRV9TSVpFLCBwYXRjaF9zaXplLCBzdHJpZGUpOgogICAgICAgICAgICBjcm9wID0gbm9ybWFsaXplZF9jaGFubmVsc1sKICAgICAgICAgICAgICAgIDosIHkwIDogeTAgKyBwYXRjaF9zaXplLCB4MCA6IHgwICsgcGF0Y2hfc2l6ZQogICAgICAgICAgICBdLmNvcHkoKQogICAgICAgICAgICB0ZW5zb3IgPSB0b3JjaC5mcm9tX251bXB5KGNyb3BbTm9uZV0pLnRvKGRldmljZT1kZXZpY2UpCiAgICAgICAgICAgIGlmIGdwdToKICAgICAgICAgICAgICAgIHRlbnNvciA9IHRlbnNvci5jb250aWd1b3VzKG1lbW9yeV9mb3JtYXQ9dG9yY2guY2hhbm5lbHNfbGFzdCkKICAgICAgICAgICAgYmF0Y2ggPSB0b3JjaC5jYXQoW190cmFuc2Zvcm0odGVuc29yLCBvcCkgZm9yIG9wIGluIG9wZXJhdGlvbnNdLCBkaW09MCkKICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgKICAgICAgICAgICAgICAgIGRldmljZV90eXBlPWRldmljZS50eXBlLAogICAgICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICAgICAgICAgIGVuYWJsZWQ9Z3B1LAogICAgICAgICAgICApOgogICAgICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoYmF0Y2gpCiAgICAgICAgICAgIHByb2JhYmlsaXRpZXMgPSBsb2dpdHMuZmxvYXQoKS5zaWdtb2lkKCkuY3B1KCkKICAgICAgICAgICAgYXZlcmFnZWQgPSB0b3JjaC56ZXJvcygoMiwgcGF0Y2hfc2l6ZSwgcGF0Y2hfc2l6ZSksIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgICAgIGZvciBpbmRleCwgb3BlcmF0aW9uIGluIGVudW1lcmF0ZShvcGVyYXRpb25zKToKICAgICAgICAgICAgICAgIGF2ZXJhZ2VkICs9IF9pbnZlcnNlX3RyYW5zZm9ybShwcm9iYWJpbGl0aWVzW2luZGV4XSwgb3BlcmF0aW9uKQogICAgICAgICAgICBhdmVyYWdlZCAvPSBsZW4ob3BlcmF0aW9ucykKICAgICAgICAgICAgcmVnaW9uID0gbnAuc19beTAgOiB5MCArIHBhdGNoX3NpemUsIHgwIDogeDAgKyBwYXRjaF9zaXplXQogICAgICAgICAgICBhY2N1bXVsYXRpb25bOiwgcmVnaW9uWzBdLCByZWdpb25bMV1dICs9IGF2ZXJhZ2VkLm51bXB5KCkgKiB3ZWlnaHRbTm9uZV0KICAgICAgICAgICAgd2VpZ2h0X3N1bVtyZWdpb25dICs9IHdlaWdodAoKICAgIG91dHB1dCA9IGFjY3VtdWxhdGlvbiAvIG5wLm1heGltdW0od2VpZ2h0X3N1bVtOb25lXSwgMWUtNikKICAgIHJldHVybiBvdXRwdXRbMF0sIG91dHB1dFsxXQoKCmRlZiBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzKAogICAgZm9yZWdyb3VuZF9wcm9iYWJpbGl0eTogbnAubmRhcnJheSwKICAgIGJvdW5kYXJ5X3Byb2JhYmlsaXR5OiBucC5uZGFycmF5LAogICAgZmlsZV9uYW1lOiBzdHIsCiAgICBjb25maWc6IEluc3RhbmNlQ29uZmlnIHwgTm9uZSA9IE5vbmUsCiAgICAqLAogICAgZGlza19tYXNrOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV06CiAgICBjb25maWcgPSBjb25maWcgb3IgSW5zdGFuY2VDb25maWcoKQogICAgaWYgZm9yZWdyb3VuZF9wcm9iYWJpbGl0eS5zaGFwZSAhPSBib3VuZGFyeV9wcm9iYWJpbGl0eS5zaGFwZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGb3JlZ3JvdW5kIGFuZCBib3VuZGFyeSBwcm9iYWJpbGl0eSBtYXBzIG11c3QgaGF2ZSBlcXVhbCBzaGFwZXMiKQogICAgaGVpZ2h0LCB3aWR0aCA9IGZvcmVncm91bmRfcHJvYmFiaWxpdHkuc2hhcGUKICAgIGlmIChoZWlnaHQsIHdpZHRoKSAhPSAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIHtJTUFHRV9TSVpFfXh7SU1BR0VfU0laRX0gcHJvYmFiaWxpdHkgbWFwcyIpCiAgICBpZiBkaXNrX21hc2sgaXMgTm9uZToKICAgICAgICB5eSwgeHggPSBucC5vZ3JpZFs6aGVpZ2h0LCA6d2lkdGhdCiAgICAgICAgZGlzayA9ICh5eSAtIDEwMjMuMCkgKiogMiArICh4eCAtIDEwMjMuMCkgKiogMiA8PSBjb25maWcuc29sYXJfcmFkaXVzKioyCiAgICBlbHNlOgogICAgICAgIGlmIGRpc2tfbWFzay5zaGFwZSAhPSAoaGVpZ2h0LCB3aWR0aCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImRpc2tfbWFzayBtdXN0IGhhdmUgdGhlIHNhbWUgc2hhcGUgYXMgdGhlIHByb2JhYmlsaXR5IG1hcHMiKQogICAgICAgIGRpc2sgPSBucC5hc2FycmF5KGRpc2tfbWFzaywgZHR5cGU9Ym9vbCkKICAgIGZvcmVncm91bmQgPSAoZm9yZWdyb3VuZF9wcm9iYWJpbGl0eSA+PSBjb25maWcuZm9yZWdyb3VuZF90aHJlc2hvbGQpICYgZGlzawogICAgaWYgbm90IG5wLmFueShmb3JlZ3JvdW5kKToKICAgICAgICByZXR1cm4gW10KCiAgICBzZWVkX21hc2sgPSAoCiAgICAgICAgZm9yZWdyb3VuZAogICAgICAgICYgKGZvcmVncm91bmRfcHJvYmFiaWxpdHkgPj0gY29uZmlnLnNlZWRfdGhyZXNob2xkKQogICAgICAgICYgKGJvdW5kYXJ5X3Byb2JhYmlsaXR5IDwgY29uZmlnLmJvdW5kYXJ5X3RocmVzaG9sZCkKICAgICkKICAgIG1hcmtlcnMsIG1hcmtlcl9jb3VudCA9IG5kaS5sYWJlbChzZWVkX21hc2ssIHN0cnVjdHVyZT1ucC5vbmVzKCgzLCAzKSwgZHR5cGU9bnAudWludDgpKQogICAgaWYgbWFya2VyX2NvdW50OgogICAgICAgIHNpemVzID0gbnAuYmluY291bnQobWFya2Vycy5yYXZlbCgpLCBtaW5sZW5ndGg9bWFya2VyX2NvdW50ICsgMSkKICAgICAgICBrZWVwID0gc2l6ZXMgPj0gY29uZmlnLnNlZWRfbWluX2FyZWEKICAgICAgICBrZWVwWzBdID0gRmFsc2UKICAgICAgICBzZWVkX21hc2sgJj0ga2VlcFttYXJrZXJzXQogICAgICAgIG1hcmtlcnMsIG1hcmtlcl9jb3VudCA9IG5kaS5sYWJlbCgKICAgICAgICAgICAgc2VlZF9tYXNrLCBzdHJ1Y3R1cmU9bnAub25lcygoMywgMyksIGR0eXBlPW5wLnVpbnQ4KQogICAgICAgICkKICAgIGlmIG1hcmtlcl9jb3VudCA9PSAwOgogICAgICAgIG1hcmtlcnMsIG1hcmtlcl9jb3VudCA9IG5kaS5sYWJlbCgKICAgICAgICAgICAgZm9yZWdyb3VuZCwgc3RydWN0dXJlPW5wLm9uZXMoKDMsIDMpLCBkdHlwZT1ucC51aW50OCkKICAgICAgICApCiAgICBlbGV2YXRpb24gPSAtZm9yZWdyb3VuZF9wcm9iYWJpbGl0eS5hc3R5cGUobnAuZmxvYXQzMikgKyAoCiAgICAgICAgY29uZmlnLmJvdW5kYXJ5X3dlaWdodCAqIGJvdW5kYXJ5X3Byb2JhYmlsaXR5LmFzdHlwZShucC5mbG9hdDMyKQogICAgKQogICAgbGFiZWxzID0gd2F0ZXJzaGVkKAogICAgICAgIGVsZXZhdGlvbiwKICAgICAgICBtYXJrZXJzLmFzdHlwZShucC5pbnQzMiksCiAgICAgICAgbWFzaz1mb3JlZ3JvdW5kLAogICAgICAgIGNvbm5lY3Rpdml0eT1ucC5vbmVzKCgzLCAzKSwgZHR5cGU9bnAudWludDgpLAogICAgICAgIHdhdGVyc2hlZF9saW5lPVRydWUsCiAgICApLmFzdHlwZShucC5pbnQzMiwgY29weT1GYWxzZSkKICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KGxhYmVscy5yYXZlbCgpKQogICAgYWNjZXB0ZWQgPSBbbGFiZWwgZm9yIGxhYmVsIGluIHJhbmdlKDEsIGxlbihjb3VudHMpKSBpZiBjb3VudHNbbGFiZWxdID49IGNvbmZpZy5taW5faW5zdGFuY2VfYXJlYV0KICAgIGlmIG5vdCBhY2NlcHRlZDoKICAgICAgICByZXR1cm4gW10KCiAgICBmaXJzdF9ieV9sYWJlbCA9IG5wLmZ1bGwobGVuKGNvdW50cyksIGxhYmVscy5zaXplLCBkdHlwZT1ucC5pbnQ2NCkKICAgIG5vbnplcm9fcG9zaXRpb25zID0gbnAuZmxhdG5vbnplcm8obGFiZWxzLnJhdmVsKCkpCiAgICBucC5taW5pbXVtLmF0KAogICAgICAgIGZpcnN0X2J5X2xhYmVsLAogICAgICAgIGxhYmVscy5yYXZlbCgpW25vbnplcm9fcG9zaXRpb25zXSwKICAgICAgICBub256ZXJvX3Bvc2l0aW9ucywKICAgICkKICAgIGZpcnN0X3Bvc2l0aW9ucyA9IHtsYWJlbDogaW50KGZpcnN0X2J5X2xhYmVsW2xhYmVsXSkgZm9yIGxhYmVsIGluIGFjY2VwdGVkfQogICAgYWNjZXB0ZWQuc29ydChrZXk9Zmlyc3RfcG9zaXRpb25zLl9fZ2V0aXRlbV9fKQogICAgZW5jb2RlZCA9IGVuY29kZV9sYWJlbF9jb21wb25lbnRzKGxhYmVscywgYWNjZXB0ZWQpCiAgICBpbWFnZV9zdGVtID0gUGF0aChmaWxlX25hbWUpLnN0ZW0KICAgIHJldHVybiBbCiAgICAgICAgUHJlZGljdGVkSW5zdGFuY2UoCiAgICAgICAgICAgIGZpbGFtZW50X2lkPWYie2ltYWdlX3N0ZW19X3tzZXF1ZW5jZX0iLAogICAgICAgICAgICBpbWFnZV9zdGVtPWltYWdlX3N0ZW0sCiAgICAgICAgICAgIGFyZWE9aW50KGNvdW50c1tsYWJlbF0pLAogICAgICAgICAgICBybGU9ZW5jb2RlZFtsYWJlbF0sCiAgICAgICAgKQogICAgICAgIGZvciBzZXF1ZW5jZSwgbGFiZWwgaW4gZW51bWVyYXRlKGFjY2VwdGVkLCBzdGFydD0xKQogICAgXQo=\", \"src/filament_baseline/learning.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBPcmRlcmVkRGljdApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IExpdGVyYWwKCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKCmZyb20gLmRhdGEgaW1wb3J0IENvbXBldGl0aW9uRGF0YXNldCwgSW1hZ2VWaWV3CmZyb20gLmZlYXR1cmVzIGltcG9ydCBJbnB1dE1vZGUsIGlucHV0X2NoYW5uZWxfY291bnQsIHByZXBhcmVfaW5wdXRfY2hhbm5lbHMKClRhcmdldE1vZGUgPSBMaXRlcmFsWyJhbm5vdGF0b3IiLCAiY29uc2Vuc3VzIl0KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBJbnN0YW5jZVBvbHlnb25zOgogICAgcG9seWdvbnM6IHR1cGxlW25wLm5kYXJyYXksIC4uLl0KICAgIGJvdW5kczogdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdCwgZmxvYXRdCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVmlld1RhcmdldHM6CiAgICB2aWV3OiBJbWFnZVZpZXcKICAgIGluc3RhbmNlczogdHVwbGVbSW5zdGFuY2VQb2x5Z29ucywgLi4uXQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFBhdGNoUmVjb3JkOgogICAgZmlsZV9uYW1lOiBzdHIKICAgIHZpZXdzOiB0dXBsZVtWaWV3VGFyZ2V0cywgLi4uXQoKCmRlZiBfdmlld190YXJnZXRzKGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwgdmlldzogSW1hZ2VWaWV3KSAtPiBWaWV3VGFyZ2V0czoKICAgIGluc3RhbmNlcyA9IFtdCiAgICBmb3IgYW5ub3RhdGlvbiBpbiBkYXRhc2V0LmFubm90YXRpb25zX2J5X2ltYWdlX2lkW3ZpZXcuaW1hZ2VfaWRdOgogICAgICAgIHNlZ21lbnRhdGlvbiA9IGFubm90YXRpb25bInNlZ21lbnRhdGlvbiJdCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2Uoc2VnbWVudGF0aW9uLCBsaXN0KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVHJhaW5pbmcgcG9seWdvbnMgbXVzdCBmb2xsb3cgdGhlIGNvbXBldGl0aW9uIENPQ08gcG9seWdvbiBmb3JtYXQiKQogICAgICAgIHBvbHlnb25zID0gdHVwbGUoCiAgICAgICAgICAgIG5wLmFzYXJyYXkocG9seWdvbiwgZHR5cGU9bnAuZmxvYXQzMikucmVzaGFwZSgtMSwgMikKICAgICAgICAgICAgZm9yIHBvbHlnb24gaW4gc2VnbWVudGF0aW9uCiAgICAgICAgICAgIGlmIGxlbihwb2x5Z29uKSA+PSA2CiAgICAgICAgKQogICAgICAgIGlmIG5vdCBwb2x5Z29uczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwb2ludHMgPSBucC5jb25jYXRlbmF0ZShwb2x5Z29ucywgYXhpcz0wKQogICAgICAgIHhfbWluLCB5X21pbiA9IG5wLm1pbihwb2ludHMsIGF4aXM9MCkKICAgICAgICB4X21heCwgeV9tYXggPSBucC5tYXgocG9pbnRzLCBheGlzPTApCiAgICAgICAgaW5zdGFuY2VzLmFwcGVuZCgKICAgICAgICAgICAgSW5zdGFuY2VQb2x5Z29ucygKICAgICAgICAgICAgICAgIHBvbHlnb25zLAogICAgICAgICAgICAgICAgKGZsb2F0KHhfbWluKSwgZmxvYXQoeV9taW4pLCBmbG9hdCh4X21heCksIGZsb2F0KHlfbWF4KSksCiAgICAgICAgICAgICkKICAgICAgICApCiAgICByZXR1cm4gVmlld1RhcmdldHModmlldywgdHVwbGUoaW5zdGFuY2VzKSkKCgpkZWYgcmVjb3Jkc19mb3Jfb2JzZXJ2YXRpb25zKAogICAgZGF0YXNldDogQ29tcGV0aXRpb25EYXRhc2V0LAogICAgb2JzZXJ2YXRpb25zOiB0dXBsZVtzdHIsIC4uLl0gfCBsaXN0W3N0cl0sCiAgICB0YXJnZXRfbW9kZTogVGFyZ2V0TW9kZSwKKSAtPiB0dXBsZVtQYXRjaFJlY29yZCwgLi4uXToKICAgIHJlY29yZHMgPSBbXQogICAgZm9yIGZpbGVfbmFtZSBpbiBzb3J0ZWQob2JzZXJ2YXRpb25zKToKICAgICAgICB2aWV3cyA9IHR1cGxlKAogICAgICAgICAgICBfdmlld190YXJnZXRzKGRhdGFzZXQsIHZpZXcpCiAgICAgICAgICAgIGZvciB2aWV3IGluIGRhdGFzZXQudmlld3NfYnlfb2JzZXJ2YXRpb25bZmlsZV9uYW1lXQogICAgICAgICkKICAgICAgICBpZiB0YXJnZXRfbW9kZSA9PSAiYW5ub3RhdG9yIjoKICAgICAgICAgICAgcmVjb3Jkcy5leHRlbmQoUGF0Y2hSZWNvcmQoZmlsZV9uYW1lLCAodmlldywpKSBmb3IgdmlldyBpbiB2aWV3cykKICAgICAgICBlbGlmIHRhcmdldF9tb2RlID09ICJjb25zZW5zdXMiOgogICAgICAgICAgICByZWNvcmRzLmFwcGVuZChQYXRjaFJlY29yZChmaWxlX25hbWUsIHZpZXdzKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgdGFyZ2V0IG1vZGU6IHt0YXJnZXRfbW9kZX0iKQogICAgcmV0dXJuIHR1cGxlKHJlY29yZHMpCgoKZGVmIF9pbnN0YW5jZV9tYXAodmlldzogVmlld1RhcmdldHMsIHgwOiBpbnQsIHkwOiBpbnQsIHNpemU6IGludCkgLT4gbnAubmRhcnJheToKICAgIGxhYmVscyA9IG5wLnplcm9zKChzaXplLCBzaXplKSwgZHR5cGU9bnAudWludDE2KQogICAgeDEsIHkxID0geDAgKyBzaXplLCB5MCArIHNpemUKICAgIGZvciBsYWJlbCwgaW5zdGFuY2UgaW4gZW51bWVyYXRlKHZpZXcuaW5zdGFuY2VzLCBzdGFydD0xKToKICAgICAgICBsZWZ0LCB0b3AsIHJpZ2h0LCBib3R0b20gPSBpbnN0YW5jZS5ib3VuZHMKICAgICAgICBpZiByaWdodCA8IHgwIG9yIGJvdHRvbSA8IHkwIG9yIGxlZnQgPj0geDEgb3IgdG9wID49IHkxOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGxvY2FsX3BvbHlnb25zID0gW10KICAgICAgICBmb3IgcG9seWdvbiBpbiBpbnN0YW5jZS5wb2x5Z29uczoKICAgICAgICAgICAgcG9pbnRzID0gbnAucmludChwb2x5Z29uIC0gbnAuYXJyYXkoW3gwLCB5MF0sIGR0eXBlPW5wLmZsb2F0MzIpKQogICAgICAgICAgICBwb2ludHMgPSBucC5jbGlwKHBvaW50cywgLXNpemUsIDIgKiBzaXplKS5hc3R5cGUobnAuaW50MzIpCiAgICAgICAgICAgIGlmIGxlbihwb2ludHMpID49IDM6CiAgICAgICAgICAgICAgICBsb2NhbF9wb2x5Z29ucy5hcHBlbmQocG9pbnRzKQogICAgICAgIGlmIGxvY2FsX3BvbHlnb25zOgogICAgICAgICAgICBjdjIuZmlsbFBvbHkobGFiZWxzLCBsb2NhbF9wb2x5Z29ucywgY29sb3I9bGFiZWwsIGxpbmVUeXBlPWN2Mi5MSU5FXzgpCiAgICByZXR1cm4gbGFiZWxzCgoKZGVmIF9mb3JlZ3JvdW5kX2FuZF9ib3VuZGFyeShsYWJlbHM6IG5wLm5kYXJyYXkpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgZm9yZWdyb3VuZCA9IChsYWJlbHMgPiAwKS5hc3R5cGUobnAuZmxvYXQzMikKICAgIGJvdW5kYXJ5ID0gbnAuemVyb3MobGFiZWxzLnNoYXBlLCBkdHlwZT1ib29sKQogICAgaG9yaXpvbnRhbCA9IGxhYmVsc1s6LCAxOl0gIT0gbGFiZWxzWzosIDotMV0KICAgIHZlcnRpY2FsID0gbGFiZWxzWzE6LCA6XSAhPSBsYWJlbHNbOi0xLCA6XQogICAgYm91bmRhcnlbOiwgMTpdIHw9IGhvcml6b250YWwgJiAoKGxhYmVsc1s6LCAxOl0gPiAwKSB8IChsYWJlbHNbOiwgOi0xXSA+IDApKQogICAgYm91bmRhcnlbOiwgOi0xXSB8PSBob3Jpem9udGFsICYgKChsYWJlbHNbOiwgMTpdID4gMCkgfCAobGFiZWxzWzosIDotMV0gPiAwKSkKICAgIGJvdW5kYXJ5WzE6LCA6XSB8PSB2ZXJ0aWNhbCAmICgobGFiZWxzWzE6LCA6XSA+IDApIHwgKGxhYmVsc1s6LTEsIDpdID4gMCkpCiAgICBib3VuZGFyeVs6LTEsIDpdIHw9IHZlcnRpY2FsICYgKChsYWJlbHNbMTosIDpdID4gMCkgfCAobGFiZWxzWzotMSwgOl0gPiAwKSkKICAgIHJldHVybiBmb3JlZ3JvdW5kLCBib3VuZGFyeS5hc3R5cGUobnAuZmxvYXQzMikKCgpjbGFzcyBGaWxhbWVudFBhdGNoRGF0YXNldChEYXRhc2V0W3R1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yXV0pOgogICAgIiIiT24tdGhlLWZseSBmdWxsLXJlc29sdXRpb24gY3JvcHMgZnJvbSBvZmZpY2lhbCBhbm5vdGF0aW9uIHZpZXdzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwKICAgICAgICBvYnNlcnZhdGlvbnM6IHR1cGxlW3N0ciwgLi4uXSB8IGxpc3Rbc3RyXSwKICAgICAgICAqLAogICAgICAgIHRhcmdldF9tb2RlOiBUYXJnZXRNb2RlID0gImNvbnNlbnN1cyIsCiAgICAgICAgcGF0Y2hfc2l6ZTogaW50ID0gNTEyLAogICAgICAgIHBvc2l0aXZlX2ZyYWN0aW9uOiBmbG9hdCA9IDAuNywKICAgICAgICBhdWdtZW50OiBib29sID0gRmFsc2UsCiAgICAgICAgc2VlZDogaW50ID0gNDIsCiAgICAgICAgaW1hZ2VfY2FjaGVfc2l6ZTogaW50ID0gOCwKICAgICAgICBpbnB1dF9tb2RlOiBJbnB1dE1vZGUgPSAicmF3IiwKICAgICkgLT4gTm9uZToKICAgICAgICBpZiBwYXRjaF9zaXplIDwgMTI4IG9yIHBhdGNoX3NpemUgJSAzMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicGF0Y2hfc2l6ZSBtdXN0IGJlIGF0IGxlYXN0IDEyOCBhbmQgZGl2aXNpYmxlIGJ5IDMyIikKICAgICAgICBpZiBub3QgMC4wIDw9IHBvc2l0aXZlX2ZyYWN0aW9uIDw9IDEuMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicG9zaXRpdmVfZnJhY3Rpb24gbXVzdCBiZSBiZXR3ZWVuIDAgYW5kIDEiKQogICAgICAgIHNlbGYucmVjb3JkcyA9IHJlY29yZHNfZm9yX29ic2VydmF0aW9ucyhkYXRhc2V0LCBvYnNlcnZhdGlvbnMsIHRhcmdldF9tb2RlKQogICAgICAgIHNlbGYudGFyZ2V0X21vZGUgPSB0YXJnZXRfbW9kZQogICAgICAgIHNlbGYuaW1hZ2VfZGlyID0gZGF0YXNldC50cmFpbl9pbWFnZV9kaXIKICAgICAgICBzZWxmLmltYWdlX3Jlc2l6ZV90byA9IGRhdGFzZXQuaW1hZ2VfcmVzaXplX3RvCiAgICAgICAgc2VsZi5wYXRjaF9zaXplID0gcGF0Y2hfc2l6ZQogICAgICAgIHNlbGYucG9zaXRpdmVfZnJhY3Rpb24gPSBwb3NpdGl2ZV9mcmFjdGlvbgogICAgICAgIHNlbGYuYXVnbWVudCA9IGF1Z21lbnQKICAgICAgICBzZWxmLnNlZWQgPSBzZWVkCiAgICAgICAgc2VsZi5pbnB1dF9tb2RlID0gaW5wdXRfbW9kZQogICAgICAgIHNlbGYuaW5fY2hhbm5lbHMgPSBpbnB1dF9jaGFubmVsX2NvdW50KGlucHV0X21vZGUpCiAgICAgICAgc2VsZi5lcG9jaCA9IDAKICAgICAgICBzZWxmLmltYWdlX2NhY2hlX3NpemUgPSBpbWFnZV9jYWNoZV9zaXplCiAgICAgICAgc2VsZi5faW1hZ2VfY2FjaGU6IE9yZGVyZWREaWN0WwogICAgICAgICAgICBzdHIsIHR1cGxlW25wLm5kYXJyYXksIGZsb2F0LCBmbG9hdCwgbnAubmRhcnJheSB8IE5vbmVdCiAgICAgICAgXSA9IE9yZGVyZWREaWN0KCkKICAgICAgICBzZWxmLl9kaXNrID0gbnAuemVyb3MoKDIwNDgsIDIwNDgpLCBkdHlwZT1ucC51aW50OCkKICAgICAgICBjdjIuY2lyY2xlKHNlbGYuX2Rpc2ssICgxMDIzLCAxMDIzKSwgOTUwLCBjb2xvcj0xLCB0aGlja25lc3M9LTEpCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5yZWNvcmRzKQoKICAgIGRlZiBzZXRfZXBvY2goc2VsZiwgZXBvY2g6IGludCkgLT4gTm9uZToKICAgICAgICBzZWxmLmVwb2NoID0gZXBvY2gKCiAgICBkZWYgX2ltYWdlX2FuZF9zdGF0cygKICAgICAgICBzZWxmLCBmaWxlX25hbWU6IHN0cgogICAgKSAtPiB0dXBsZVtucC5uZGFycmF5LCBmbG9hdCwgZmxvYXQsIG5wLm5kYXJyYXkgfCBOb25lXToKICAgICAgICBjYWNoZWQgPSBzZWxmLl9pbWFnZV9jYWNoZS5nZXQoZmlsZV9uYW1lKQogICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5faW1hZ2VfY2FjaGUubW92ZV90b19lbmQoZmlsZV9uYW1lKQogICAgICAgICAgICByZXR1cm4gY2FjaGVkCiAgICAgICAgaW1hZ2UgPSBjdjIuaW1yZWFkKHN0cihzZWxmLmltYWdlX2RpciAvIGZpbGVfbmFtZSksIGN2Mi5JTVJFQURfR1JBWVNDQUxFKQogICAgICAgIGlmIGltYWdlIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQ291bGQgbm90IHJlYWQgYSB0cmFpbmluZyBpbWFnZToge2ZpbGVfbmFtZX0iKQogICAgICAgIGlmIGltYWdlLnNoYXBlICE9ICgyMDQ4LCAyMDQ4KSBhbmQgc2VsZi5pbWFnZV9yZXNpemVfdG8gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGhlaWdodCwgd2lkdGggPSBzZWxmLmltYWdlX3Jlc2l6ZV90bwogICAgICAgICAgICBpbWFnZSA9IGN2Mi5yZXNpemUoaW1hZ2UsICh3aWR0aCwgaGVpZ2h0KSwgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfQVJFQSkKICAgICAgICBpZiBpbWFnZS5zaGFwZSAhPSAoMjA0OCwgMjA0OCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQ291bGQgbm90IHJlYWQgYSAyMDQ4eDIwNDggdHJhaW5pbmcgaW1hZ2U6IHtmaWxlX25hbWV9IikKICAgICAgICBzY2FsZWQgPSBpbWFnZS5hc3R5cGUobnAuZmxvYXQzMikgLyAyNTUuMAogICAgICAgIGRpc2tfdmFsdWVzID0gc2NhbGVkW3NlbGYuX2Rpc2sgPiAwXQogICAgICAgIG1lYW4gPSBmbG9hdChucC5tZWFuKGRpc2tfdmFsdWVzKSkKICAgICAgICBzdGQgPSBmbG9hdChucC5zdGQoZGlza192YWx1ZXMpKQogICAgICAgIGF1eGlsaWFyeV9jaGFubmVscyA9ICgKICAgICAgICAgICAgcHJlcGFyZV9pbnB1dF9jaGFubmVscyhpbWFnZSwgc2VsZi5pbnB1dF9tb2RlKVsxOl0KICAgICAgICAgICAgaWYgc2VsZi5pbl9jaGFubmVscyA+IDEKICAgICAgICAgICAgZWxzZSBOb25lCiAgICAgICAgKQogICAgICAgIGNhY2hlZCA9IChpbWFnZSwgbWVhbiwgbWF4KHN0ZCwgMWUtMyksIGF1eGlsaWFyeV9jaGFubmVscykKICAgICAgICBzZWxmLl9pbWFnZV9jYWNoZVtmaWxlX25hbWVdID0gY2FjaGVkCiAgICAgICAgd2hpbGUgbGVuKHNlbGYuX2ltYWdlX2NhY2hlKSA+IHNlbGYuaW1hZ2VfY2FjaGVfc2l6ZToKICAgICAgICAgICAgc2VsZi5faW1hZ2VfY2FjaGUucG9waXRlbShsYXN0PUZhbHNlKQogICAgICAgIHJldHVybiBjYWNoZWQKCiAgICBkZWYgX2Nyb3Bfb3JpZ2luKHNlbGYsIHJlY29yZDogUGF0Y2hSZWNvcmQsIHJuZzogcmFuZG9tLlJhbmRvbSkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIGFubm90YXRpb25zID0gWwogICAgICAgICAgICBpbnN0YW5jZQogICAgICAgICAgICBmb3IgdmlldyBpbiByZWNvcmQudmlld3MKICAgICAgICAgICAgZm9yIGluc3RhbmNlIGluIHZpZXcuaW5zdGFuY2VzCiAgICAgICAgXQogICAgICAgIGlmIGFubm90YXRpb25zIGFuZCBybmcucmFuZG9tKCkgPCBzZWxmLnBvc2l0aXZlX2ZyYWN0aW9uOgogICAgICAgICAgICBpbnN0YW5jZSA9IHJuZy5jaG9pY2UoYW5ub3RhdGlvbnMpCiAgICAgICAgICAgIHBvbHlnb24gPSBybmcuY2hvaWNlKGluc3RhbmNlLnBvbHlnb25zKQogICAgICAgICAgICBjZW50ZXJfeCwgY2VudGVyX3kgPSBwb2x5Z29uW3JuZy5yYW5kcmFuZ2UobGVuKHBvbHlnb24pKV0KICAgICAgICBlbHNlOgogICAgICAgICAgICBjZW50ZXJfeCA9IHJuZy51bmlmb3JtKDAsIDIwNDcpCiAgICAgICAgICAgIGNlbnRlcl95ID0gcm5nLnVuaWZvcm0oMCwgMjA0NykKICAgICAgICBoYWxmID0gc2VsZi5wYXRjaF9zaXplIC8vIDIKICAgICAgICB4MCA9IG1pbihtYXgoaW50KGNlbnRlcl94KSAtIGhhbGYsIDApLCAyMDQ4IC0gc2VsZi5wYXRjaF9zaXplKQogICAgICAgIHkwID0gbWluKG1heChpbnQoY2VudGVyX3kpIC0gaGFsZiwgMCksIDIwNDggLSBzZWxmLnBhdGNoX3NpemUpCiAgICAgICAgcmV0dXJuIHgwLCB5MAoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleDogaW50KSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgcmVjb3JkID0gc2VsZi5yZWNvcmRzW2luZGV4XQogICAgICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VsZi5zZWVkICsgc2VsZi5lcG9jaCAqIDFfMDAwXzAwMyArIGluZGV4ICogOTcpCiAgICAgICAgaW1hZ2UsIG1lYW4sIHN0ZCwgYXV4aWxpYXJ5X2NoYW5uZWxzID0gc2VsZi5faW1hZ2VfYW5kX3N0YXRzKHJlY29yZC5maWxlX25hbWUpCiAgICAgICAgeDAsIHkwID0gc2VsZi5fY3JvcF9vcmlnaW4ocmVjb3JkLCBybmcpCiAgICAgICAgeDEsIHkxID0geDAgKyBzZWxmLnBhdGNoX3NpemUsIHkwICsgc2VsZi5wYXRjaF9zaXplCiAgICAgICAgaW1hZ2VfY3JvcCA9IGltYWdlW3kwOnkxLCB4MDp4MV0uYXN0eXBlKG5wLmZsb2F0MzIpIC8gMjU1LjAKICAgICAgICBhdXhpbGlhcnlfY3JvcCA9ICgKICAgICAgICAgICAgYXV4aWxpYXJ5X2NoYW5uZWxzWzosIHkwOnkxLCB4MDp4MV0uY29weSgpCiAgICAgICAgICAgIGlmIGF1eGlsaWFyeV9jaGFubmVscyBpcyBub3QgTm9uZQogICAgICAgICAgICBlbHNlIE5vbmUKICAgICAgICApCgogICAgICAgIHBlcl92aWV3X2ZvcmVncm91bmQgPSBbXQogICAgICAgIHBlcl92aWV3X2JvdW5kYXJ5ID0gW10KICAgICAgICBmb3IgdmlldyBpbiByZWNvcmQudmlld3M6CiAgICAgICAgICAgIGxhYmVscyA9IF9pbnN0YW5jZV9tYXAodmlldywgeDAsIHkwLCBzZWxmLnBhdGNoX3NpemUpCiAgICAgICAgICAgIGZvcmVncm91bmQsIGJvdW5kYXJ5ID0gX2ZvcmVncm91bmRfYW5kX2JvdW5kYXJ5KGxhYmVscykKICAgICAgICAgICAgcGVyX3ZpZXdfZm9yZWdyb3VuZC5hcHBlbmQoZm9yZWdyb3VuZCkKICAgICAgICAgICAgcGVyX3ZpZXdfYm91bmRhcnkuYXBwZW5kKGJvdW5kYXJ5KQogICAgICAgIHRhcmdldF9mb3JlZ3JvdW5kID0gbnAubWVhbihwZXJfdmlld19mb3JlZ3JvdW5kLCBheGlzPTAsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgdGFyZ2V0X2JvdW5kYXJ5ID0gbnAubWVhbihwZXJfdmlld19ib3VuZGFyeSwgYXhpcz0wLCBkdHlwZT1ucC5mbG9hdDMyKQoKICAgICAgICBpZiBzZWxmLmF1Z21lbnQ6CiAgICAgICAgICAgIHR1cm5zID0gcm5nLnJhbmRyYW5nZSg0KQogICAgICAgICAgICBpbWFnZV9jcm9wID0gbnAucm90OTAoaW1hZ2VfY3JvcCwgdHVybnMpLmNvcHkoKQogICAgICAgICAgICBpZiBhdXhpbGlhcnlfY3JvcCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGF1eGlsaWFyeV9jcm9wID0gbnAucm90OTAoYXV4aWxpYXJ5X2Nyb3AsIHR1cm5zLCBheGVzPSgtMiwgLTEpKS5jb3B5KCkKICAgICAgICAgICAgdGFyZ2V0X2ZvcmVncm91bmQgPSBucC5yb3Q5MCh0YXJnZXRfZm9yZWdyb3VuZCwgdHVybnMpLmNvcHkoKQogICAgICAgICAgICB0YXJnZXRfYm91bmRhcnkgPSBucC5yb3Q5MCh0YXJnZXRfYm91bmRhcnksIHR1cm5zKS5jb3B5KCkKICAgICAgICAgICAgaWYgcm5nLnJhbmRvbSgpIDwgMC41OgogICAgICAgICAgICAgICAgaW1hZ2VfY3JvcCA9IG5wLmZsaXBscihpbWFnZV9jcm9wKS5jb3B5KCkKICAgICAgICAgICAgICAgIGlmIGF1eGlsaWFyeV9jcm9wIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGF1eGlsaWFyeV9jcm9wID0gbnAuZmxpcChhdXhpbGlhcnlfY3JvcCwgYXhpcz0tMSkuY29weSgpCiAgICAgICAgICAgICAgICB0YXJnZXRfZm9yZWdyb3VuZCA9IG5wLmZsaXBscih0YXJnZXRfZm9yZWdyb3VuZCkuY29weSgpCiAgICAgICAgICAgICAgICB0YXJnZXRfYm91bmRhcnkgPSBucC5mbGlwbHIodGFyZ2V0X2JvdW5kYXJ5KS5jb3B5KCkKICAgICAgICAgICAgaWYgcm5nLnJhbmRvbSgpIDwgMC41OgogICAgICAgICAgICAgICAgaW1hZ2VfY3JvcCA9IG5wLmZsaXB1ZChpbWFnZV9jcm9wKS5jb3B5KCkKICAgICAgICAgICAgICAgIGlmIGF1eGlsaWFyeV9jcm9wIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGF1eGlsaWFyeV9jcm9wID0gbnAuZmxpcChhdXhpbGlhcnlfY3JvcCwgYXhpcz0tMikuY29weSgpCiAgICAgICAgICAgICAgICB0YXJnZXRfZm9yZWdyb3VuZCA9IG5wLmZsaXB1ZCh0YXJnZXRfZm9yZWdyb3VuZCkuY29weSgpCiAgICAgICAgICAgICAgICB0YXJnZXRfYm91bmRhcnkgPSBucC5mbGlwdWQodGFyZ2V0X2JvdW5kYXJ5KS5jb3B5KCkKICAgICAgICAgICAgaW1hZ2VfY3JvcCA9IG5wLnBvd2VyKAogICAgICAgICAgICAgICAgbnAuY2xpcChpbWFnZV9jcm9wLCAwLjAsIDEuMCksIHJuZy51bmlmb3JtKDAuOCwgMS4yKQogICAgICAgICAgICApCiAgICAgICAgICAgIGltYWdlX2Nyb3AgPSBucC5jbGlwKAogICAgICAgICAgICAgICAgaW1hZ2VfY3JvcCAqIHJuZy51bmlmb3JtKDAuOSwgMS4xKSArIHJuZy51bmlmb3JtKC0wLjA0LCAwLjA0KSwKICAgICAgICAgICAgICAgIDAuMCwKICAgICAgICAgICAgICAgIDEuMCwKICAgICAgICAgICAgKQoKICAgICAgICBub3JtYWxpemVkID0gbnAuY2xpcCgoaW1hZ2VfY3JvcCAtIG1lYW4pIC8gc3RkLCAtNS4wLCA1LjApCiAgICAgICAgaW5wdXRfY2hhbm5lbHMgPSBbbm9ybWFsaXplZC5hc3R5cGUobnAuZmxvYXQzMildCiAgICAgICAgaWYgYXV4aWxpYXJ5X2Nyb3AgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlucHV0X2NoYW5uZWxzLmV4dGVuZCgKICAgICAgICAgICAgICAgIGNoYW5uZWwuYXN0eXBlKG5wLmZsb2F0MzIpIGZvciBjaGFubmVsIGluIGF1eGlsaWFyeV9jcm9wCiAgICAgICAgICAgICkKICAgICAgICBpbWFnZV90ZW5zb3IgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKGlucHV0X2NoYW5uZWxzKSkKICAgICAgICB0YXJnZXRfdGVuc29yID0gdG9yY2guZnJvbV9udW1weSgKICAgICAgICAgICAgbnAuc3RhY2soKHRhcmdldF9mb3JlZ3JvdW5kLCB0YXJnZXRfYm91bmRhcnkpKS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICApCiAgICAgICAgcmV0dXJuIGltYWdlX3RlbnNvciwgdGFyZ2V0X3RlbnNvcgoKCmRlZiB0cmFpbmluZ19sb3NzKGxvZ2l0czogdG9yY2guVGVuc29yLCB0YXJnZXQ6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgZm9yZWdyb3VuZF90YXJnZXQgPSB0YXJnZXRbOiwgMDoxXQogICAgYm91bmRhcnlfdGFyZ2V0ID0gdGFyZ2V0WzosIDE6Ml0KICAgIGZvcmVncm91bmRfYmNlID0gdG9yY2gubm4uZnVuY3Rpb25hbC5iaW5hcnlfY3Jvc3NfZW50cm9weV93aXRoX2xvZ2l0cygKICAgICAgICBsb2dpdHNbOiwgMDoxXSwgZm9yZWdyb3VuZF90YXJnZXQKICAgICkKICAgIGZvcmVncm91bmRfcHJvYmFiaWxpdHkgPSBsb2dpdHNbOiwgMDoxXS5zaWdtb2lkKCkKICAgIGludGVyc2VjdGlvbiA9IChmb3JlZ3JvdW5kX3Byb2JhYmlsaXR5ICogZm9yZWdyb3VuZF90YXJnZXQpLnN1bShkaW09KDEsIDIsIDMpKQogICAgZGVub21pbmF0b3IgPSBmb3JlZ3JvdW5kX3Byb2JhYmlsaXR5LnN1bShkaW09KDEsIDIsIDMpKSArIGZvcmVncm91bmRfdGFyZ2V0LnN1bSgKICAgICAgICBkaW09KDEsIDIsIDMpCiAgICApCiAgICBkaWNlX2xvc3MgPSAxLjAgLSAoKDIuMCAqIGludGVyc2VjdGlvbiArIDEuMCkgLyAoZGVub21pbmF0b3IgKyAxLjApKS5tZWFuKCkKICAgIGJvdW5kYXJ5X2JjZSA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMoCiAgICAgICAgbG9naXRzWzosIDE6Ml0sIGJvdW5kYXJ5X3RhcmdldCwgcG9zX3dlaWdodD10b3JjaC50ZW5zb3IoCiAgICAgICAgICAgIDUuMCwgZHR5cGU9bG9naXRzLmR0eXBlLCBkZXZpY2U9bG9naXRzLmRldmljZQogICAgICAgICkKICAgICkKICAgIHJldHVybiAwLjQ1ICogZm9yZWdyb3VuZF9iY2UgKyAwLjQ1ICogZGljZV9sb3NzICsgMC4xMCAqIGJvdW5kYXJ5X2JjZQo=\", \"src/filament_baseline/model.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gdG9yY2gubm4gaW1wb3J0IGZ1bmN0aW9uYWwgYXMgRgoKCkRJTk9fVjJfUkVQT1NJVE9SWSA9ICgKICAgICJmYWNlYm9va3Jlc2VhcmNoL2Rpbm92Mjo3NzY0ZWEwZjkxMmU1M2M5MmU4MmViNzhhMmExNjMxZTkyNzI1ZmM4IgopCgoKZGVmIF9ncm91cF9jb3VudChjaGFubmVsczogaW50KSAtPiBpbnQ6CiAgICBmb3IgZ3JvdXBzIGluICg4LCA0LCAyKToKICAgICAgICBpZiBjaGFubmVscyAlIGdyb3VwcyA9PSAwOgogICAgICAgICAgICByZXR1cm4gZ3JvdXBzCiAgICByZXR1cm4gMQoKCmNsYXNzIENvbnZCbG9jayhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzOiBpbnQsIG91dF9jaGFubmVsczogaW50KSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQ29udjJkKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDMsIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIG5uLkdyb3VwTm9ybShfZ3JvdXBfY291bnQob3V0X2NoYW5uZWxzKSwgb3V0X2NoYW5uZWxzKSwKICAgICAgICAgICAgbm4uU2lMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Db252MmQob3V0X2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDMsIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIG5uLkdyb3VwTm9ybShfZ3JvdXBfY291bnQob3V0X2NoYW5uZWxzKSwgb3V0X2NoYW5uZWxzKSwKICAgICAgICAgICAgbm4uU2lMVShpbnBsYWNlPVRydWUpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB2YWx1ZTogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcmV0dXJuIHNlbGYubGF5ZXJzKHZhbHVlKQoKCmNsYXNzIEZpbGFtZW50VU5ldChubi5Nb2R1bGUpOgogICAgIiIiU21hbGwgZnJvbS1zY3JhdGNoIFUtTmV0IHdpdGggZm9yZWdyb3VuZCBhbmQgaW5zdGFuY2UtYm91bmRhcnkgaGVhZHMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzOiBpbnQgPSAxLCBiYXNlX2NoYW5uZWxzOiBpbnQgPSAxNikgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmluX2NoYW5uZWxzID0gaW5fY2hhbm5lbHMKICAgICAgICBzZWxmLmJhc2VfY2hhbm5lbHMgPSBiYXNlX2NoYW5uZWxzCiAgICAgICAgd2lkdGhzID0gW2Jhc2VfY2hhbm5lbHMgKiAoMioqaW5kZXgpIGZvciBpbmRleCBpbiByYW5nZSg1KV0KICAgICAgICBzZWxmLmVuY29kZXJzID0gbm4uTW9kdWxlTGlzdCgpCiAgICAgICAgcHJldmlvdXMgPSBpbl9jaGFubmVscwogICAgICAgIGZvciB3aWR0aCBpbiB3aWR0aHNbOi0xXToKICAgICAgICAgICAgc2VsZi5lbmNvZGVycy5hcHBlbmQoQ29udkJsb2NrKHByZXZpb3VzLCB3aWR0aCkpCiAgICAgICAgICAgIHByZXZpb3VzID0gd2lkdGgKICAgICAgICBzZWxmLnBvb2wgPSBubi5NYXhQb29sMmQoMikKICAgICAgICBzZWxmLmJvdHRsZW5lY2sgPSBDb252QmxvY2sod2lkdGhzWy0yXSwgd2lkdGhzWy0xXSkKICAgICAgICBzZWxmLnVwY29udnMgPSBubi5Nb2R1bGVMaXN0KCkKICAgICAgICBzZWxmLmRlY29kZXJzID0gbm4uTW9kdWxlTGlzdCgpCiAgICAgICAgY3VycmVudCA9IHdpZHRoc1stMV0KICAgICAgICBmb3Igd2lkdGggaW4gcmV2ZXJzZWQod2lkdGhzWzotMV0pOgogICAgICAgICAgICBzZWxmLnVwY29udnMuYXBwZW5kKG5uLkNvbnZUcmFuc3Bvc2UyZChjdXJyZW50LCB3aWR0aCwga2VybmVsX3NpemU9Miwgc3RyaWRlPTIpKQogICAgICAgICAgICBzZWxmLmRlY29kZXJzLmFwcGVuZChDb252QmxvY2sod2lkdGggKiAyLCB3aWR0aCkpCiAgICAgICAgICAgIGN1cnJlbnQgPSB3aWR0aAogICAgICAgIHNlbGYuaGVhZCA9IG5uLkNvbnYyZCh3aWR0aHNbMF0sIDIsIGtlcm5lbF9zaXplPTEpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2U6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHNraXBzID0gW10KICAgICAgICB2YWx1ZSA9IGltYWdlCiAgICAgICAgZm9yIGVuY29kZXIgaW4gc2VsZi5lbmNvZGVyczoKICAgICAgICAgICAgdmFsdWUgPSBlbmNvZGVyKHZhbHVlKQogICAgICAgICAgICBza2lwcy5hcHBlbmQodmFsdWUpCiAgICAgICAgICAgIHZhbHVlID0gc2VsZi5wb29sKHZhbHVlKQogICAgICAgIHZhbHVlID0gc2VsZi5ib3R0bGVuZWNrKHZhbHVlKQogICAgICAgIGZvciB1cGNvbnYsIGRlY29kZXIsIHNraXAgaW4gemlwKAogICAgICAgICAgICBzZWxmLnVwY29udnMsIHNlbGYuZGVjb2RlcnMsIHJldmVyc2VkKHNraXBzKSwgc3RyaWN0PVRydWUKICAgICAgICApOgogICAgICAgICAgICB2YWx1ZSA9IHVwY29udih2YWx1ZSkKICAgICAgICAgICAgaWYgdmFsdWUuc2hhcGVbLTI6XSAhPSBza2lwLnNoYXBlWy0yOl06CiAgICAgICAgICAgICAgICB2YWx1ZSA9IG5uLmZ1bmN0aW9uYWwuaW50ZXJwb2xhdGUoCiAgICAgICAgICAgICAgICAgICAgdmFsdWUsIHNpemU9c2tpcC5zaGFwZVstMjpdLCBtb2RlPSJiaWxpbmVhciIsIGFsaWduX2Nvcm5lcnM9RmFsc2UKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgdmFsdWUgPSBkZWNvZGVyKHRvcmNoLmNhdCgoc2tpcCwgdmFsdWUpLCBkaW09MSkpCiAgICAgICAgcmV0dXJuIHNlbGYuaGVhZCh2YWx1ZSkKCgpjbGFzcyBEaW5vVjJGaWxhbWVudFVOZXQobm4uTW9kdWxlKToKICAgICIiIkZyb3plbiBzZWxmLXN1cGVydmlzZWQgRElOT3YyIGZlYXR1cmVzIHdpdGggYSB0cmFpbmFibGUgZGVuc2UgZGVjb2Rlci4iIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbl9jaGFubmVsczogaW50ID0gMiwKICAgICAgICBiYXNlX2NoYW5uZWxzOiBpbnQgPSAxNiwKICAgICAgICAqLAogICAgICAgIHByZXRyYWluZWRfYmFja2JvbmU6IGJvb2wgPSBGYWxzZSwKICAgICAgICBodWJfcmVwb3NpdG9yeTogc3RyID0gRElOT19WMl9SRVBPU0lUT1JZLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGluX2NoYW5uZWxzIDwgMSBvciBiYXNlX2NoYW5uZWxzIDwgNDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW5fY2hhbm5lbHMgbXVzdCBiZSBwb3NpdGl2ZSBhbmQgYmFzZV9jaGFubmVscyBhdCBsZWFzdCA0IikKICAgICAgICBzZWxmLmluX2NoYW5uZWxzID0gaW5fY2hhbm5lbHMKICAgICAgICBzZWxmLmJhc2VfY2hhbm5lbHMgPSBiYXNlX2NoYW5uZWxzCiAgICAgICAgc2VsZi5odWJfcmVwb3NpdG9yeSA9IGh1Yl9yZXBvc2l0b3J5CiAgICAgICAgc2VsZi5iYWNrYm9uZSA9IHRvcmNoLmh1Yi5sb2FkKAogICAgICAgICAgICBodWJfcmVwb3NpdG9yeSwKICAgICAgICAgICAgImRpbm92Ml92aXRzMTQiLAogICAgICAgICAgICBwcmV0cmFpbmVkPXByZXRyYWluZWRfYmFja2JvbmUsCiAgICAgICAgICAgIHRydXN0X3JlcG89VHJ1ZSwKICAgICAgICApCiAgICAgICAgc2VsZi5iYWNrYm9uZS5ldmFsKCkKICAgICAgICBmb3IgcGFyYW1ldGVyIGluIHNlbGYuYmFja2JvbmUucGFyYW1ldGVycygpOgogICAgICAgICAgICBwYXJhbWV0ZXIucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCgogICAgICAgIHdpZHRocyA9IChiYXNlX2NoYW5uZWxzLCBiYXNlX2NoYW5uZWxzICogMiwgYmFzZV9jaGFubmVscyAqIDQsIGJhc2VfY2hhbm5lbHMgKiA4KQogICAgICAgIHNlbGYuZW5jb2RlcjEgPSBDb252QmxvY2soaW5fY2hhbm5lbHMsIHdpZHRoc1swXSkKICAgICAgICBzZWxmLmVuY29kZXIyID0gQ29udkJsb2NrKHdpZHRoc1swXSwgd2lkdGhzWzFdKQogICAgICAgIHNlbGYuZW5jb2RlcjMgPSBDb252QmxvY2sod2lkdGhzWzFdLCB3aWR0aHNbMl0pCiAgICAgICAgc2VsZi5lbmNvZGVyNCA9IENvbnZCbG9jayh3aWR0aHNbMl0sIHdpZHRoc1szXSkKICAgICAgICBzZWxmLnBvb2wgPSBubi5NYXhQb29sMmQoMikKCiAgICAgICAgZmVhdHVyZV93aWR0aCA9IGJhc2VfY2hhbm5lbHMgKiA0CiAgICAgICAgc2VsZi5mZWF0dXJlX3Byb2plY3Rpb25zID0gbm4uTW9kdWxlTGlzdCgKICAgICAgICAgICAgbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgICAgIG5uLkNvbnYyZCgzODQsIGZlYXR1cmVfd2lkdGgsIGtlcm5lbF9zaXplPTEsIGJpYXM9RmFsc2UpLAogICAgICAgICAgICAgICAgbm4uR3JvdXBOb3JtKF9ncm91cF9jb3VudChmZWF0dXJlX3dpZHRoKSwgZmVhdHVyZV93aWR0aCksCiAgICAgICAgICAgICAgICBubi5TaUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoNCkKICAgICAgICApCiAgICAgICAgc2VsZi5mdXNpb24gPSBDb252QmxvY2sod2lkdGhzWzNdICsgZmVhdHVyZV93aWR0aCAqIDQsIHdpZHRoc1szXSkKICAgICAgICBzZWxmLnVwMyA9IG5uLkNvbnZUcmFuc3Bvc2UyZCh3aWR0aHNbM10sIHdpZHRoc1syXSwga2VybmVsX3NpemU9Miwgc3RyaWRlPTIpCiAgICAgICAgc2VsZi5kZWNvZGVyMyA9IENvbnZCbG9jayh3aWR0aHNbMl0gKiAyLCB3aWR0aHNbMl0pCiAgICAgICAgc2VsZi51cDIgPSBubi5Db252VHJhbnNwb3NlMmQod2lkdGhzWzJdLCB3aWR0aHNbMV0sIGtlcm5lbF9zaXplPTIsIHN0cmlkZT0yKQogICAgICAgIHNlbGYuZGVjb2RlcjIgPSBDb252QmxvY2sod2lkdGhzWzFdICogMiwgd2lkdGhzWzFdKQogICAgICAgIHNlbGYudXAxID0gbm4uQ29udlRyYW5zcG9zZTJkKHdpZHRoc1sxXSwgd2lkdGhzWzBdLCBrZXJuZWxfc2l6ZT0yLCBzdHJpZGU9MikKICAgICAgICBzZWxmLmRlY29kZXIxID0gQ29udkJsb2NrKHdpZHRoc1swXSAqIDIsIHdpZHRoc1swXSkKICAgICAgICBzZWxmLmhlYWQgPSBubi5Db252MmQod2lkdGhzWzBdLCAyLCBrZXJuZWxfc2l6ZT0xKQoKICAgIGRlZiB0cmFpbihzZWxmLCBtb2RlOiBib29sID0gVHJ1ZSkgLT4gRGlub1YyRmlsYW1lbnRVTmV0OgogICAgICAgIHN1cGVyKCkudHJhaW4obW9kZSkKICAgICAgICBzZWxmLmJhY2tib25lLmV2YWwoKQogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2U6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGlmIGltYWdlLnNoYXBlWzFdICE9IHNlbGYuaW5fY2hhbm5lbHM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmIkV4cGVjdGVkIHtzZWxmLmluX2NoYW5uZWxzfSBpbnB1dCBjaGFubmVscywgcmVjZWl2ZWQge2ltYWdlLnNoYXBlWzFdfSIKICAgICAgICAgICAgKQogICAgICAgIGhlaWdodCwgd2lkdGggPSBpbWFnZS5zaGFwZVstMjpdCiAgICAgICAgaWYgaGVpZ2h0ICUgMTQgb3Igd2lkdGggJSAxNDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRElOT3YyIGlucHV0IGhlaWdodCBhbmQgd2lkdGggbXVzdCBiZSBkaXZpc2libGUgYnkgMTQiKQoKICAgICAgICByYXcgPSBpbWFnZVs6LCAwOjFdLmZsb2F0KCkuY2xhbXAoLTMuMCwgMy4wKQogICAgICAgIGdyYXkgPSAocmF3ICsgMy4wKSAvIDYuMAogICAgICAgIHJnYiA9IGdyYXkuZXhwYW5kKC0xLCAzLCAtMSwgLTEpCiAgICAgICAgbWVhbiA9IHJnYi5uZXdfdGVuc29yKCgwLjQ4NSwgMC40NTYsIDAuNDA2KSkudmlldygxLCAzLCAxLCAxKQogICAgICAgIHN0ZCA9IHJnYi5uZXdfdGVuc29yKCgwLjIyOSwgMC4yMjQsIDAuMjI1KSkudmlldygxLCAzLCAxLCAxKQogICAgICAgIGJhY2tib25lX2lucHV0ID0gKHJnYiAtIG1lYW4pIC8gc3RkCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIGZlYXR1cmVzID0gc2VsZi5iYWNrYm9uZS5nZXRfaW50ZXJtZWRpYXRlX2xheWVycygKICAgICAgICAgICAgICAgIGJhY2tib25lX2lucHV0LAogICAgICAgICAgICAgICAgbj0oMiwgNSwgOCwgMTEpLAogICAgICAgICAgICAgICAgcmVzaGFwZT1UcnVlLAogICAgICAgICAgICAgICAgbm9ybT1UcnVlLAogICAgICAgICAgICApCgogICAgICAgIHNraXAxID0gc2VsZi5lbmNvZGVyMShpbWFnZSkKICAgICAgICBza2lwMiA9IHNlbGYuZW5jb2RlcjIoc2VsZi5wb29sKHNraXAxKSkKICAgICAgICBza2lwMyA9IHNlbGYuZW5jb2RlcjMoc2VsZi5wb29sKHNraXAyKSkKICAgICAgICBkZWVwID0gc2VsZi5lbmNvZGVyNChzZWxmLnBvb2woc2tpcDMpKQogICAgICAgIHByb2plY3RlZCA9IFtsYXllcihmZWF0dXJlKSBmb3IgbGF5ZXIsIGZlYXR1cmUgaW4gemlwKAogICAgICAgICAgICBzZWxmLmZlYXR1cmVfcHJvamVjdGlvbnMsIGZlYXR1cmVzLCBzdHJpY3Q9VHJ1ZQogICAgICAgICldCiAgICAgICAgcHJvamVjdGVkID0gWwogICAgICAgICAgICBGLmludGVycG9sYXRlKGZlYXR1cmUsIHNpemU9ZGVlcC5zaGFwZVstMjpdLCBtb2RlPSJiaWxpbmVhciIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpCiAgICAgICAgICAgIGZvciBmZWF0dXJlIGluIHByb2plY3RlZAogICAgICAgIF0KICAgICAgICB2YWx1ZSA9IHNlbGYuZnVzaW9uKHRvcmNoLmNhdCgoZGVlcCwgKnByb2plY3RlZCksIGRpbT0xKSkKICAgICAgICB2YWx1ZSA9IHNlbGYudXAzKHZhbHVlKQogICAgICAgIHZhbHVlID0gRi5pbnRlcnBvbGF0ZSh2YWx1ZSwgc2l6ZT1za2lwMy5zaGFwZVstMjpdLCBtb2RlPSJiaWxpbmVhciIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpCiAgICAgICAgdmFsdWUgPSBzZWxmLmRlY29kZXIzKHRvcmNoLmNhdCgoc2tpcDMsIHZhbHVlKSwgZGltPTEpKQogICAgICAgIHZhbHVlID0gc2VsZi51cDIodmFsdWUpCiAgICAgICAgdmFsdWUgPSBGLmludGVycG9sYXRlKHZhbHVlLCBzaXplPXNraXAyLnNoYXBlWy0yOl0sIG1vZGU9ImJpbGluZWFyIiwgYWxpZ25fY29ybmVycz1GYWxzZSkKICAgICAgICB2YWx1ZSA9IHNlbGYuZGVjb2RlcjIodG9yY2guY2F0KChza2lwMiwgdmFsdWUpLCBkaW09MSkpCiAgICAgICAgdmFsdWUgPSBzZWxmLnVwMSh2YWx1ZSkKICAgICAgICB2YWx1ZSA9IEYuaW50ZXJwb2xhdGUodmFsdWUsIHNpemU9c2tpcDEuc2hhcGVbLTI6XSwgbW9kZT0iYmlsaW5lYXIiLCBhbGlnbl9jb3JuZXJzPUZhbHNlKQogICAgICAgIHZhbHVlID0gc2VsZi5kZWNvZGVyMSh0b3JjaC5jYXQoKHNraXAxLCB2YWx1ZSksIGRpbT0xKSkKICAgICAgICByZXR1cm4gc2VsZi5oZWFkKHZhbHVlKQoKCmRlZiBidWlsZF9maWxhbWVudF9tb2RlbCgKICAgICosCiAgICBtb2RlbF90eXBlOiBzdHIgPSAidW5ldCIsCiAgICBpbl9jaGFubmVsczogaW50ID0gMSwKICAgIGJhc2VfY2hhbm5lbHM6IGludCA9IDE2LAogICAgcHJldHJhaW5lZF9iYWNrYm9uZTogYm9vbCA9IEZhbHNlLAogICAgaHViX3JlcG9zaXRvcnk6IHN0ciA9IERJTk9fVjJfUkVQT1NJVE9SWSwKKSAtPiBubi5Nb2R1bGU6CiAgICBpZiBtb2RlbF90eXBlID09ICJ1bmV0IjoKICAgICAgICByZXR1cm4gRmlsYW1lbnRVTmV0KGluX2NoYW5uZWxzPWluX2NoYW5uZWxzLCBiYXNlX2NoYW5uZWxzPWJhc2VfY2hhbm5lbHMpCiAgICBpZiBtb2RlbF90eXBlID09ICJkaW5vdjJfdml0czE0IjoKICAgICAgICByZXR1cm4gRGlub1YyRmlsYW1lbnRVTmV0KAogICAgICAgICAgICBpbl9jaGFubmVscz1pbl9jaGFubmVscywKICAgICAgICAgICAgYmFzZV9jaGFubmVscz1iYXNlX2NoYW5uZWxzLAogICAgICAgICAgICBwcmV0cmFpbmVkX2JhY2tib25lPXByZXRyYWluZWRfYmFja2JvbmUsCiAgICAgICAgICAgIGh1Yl9yZXBvc2l0b3J5PWh1Yl9yZXBvc2l0b3J5LAogICAgICAgICkKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnN1cHBvcnRlZCBtb2RlbCB0eXBlOiB7bW9kZWxfdHlwZX0iKQo=\", \"src/filament_baseline/model_runner.py\": \"...\", \"src/filament_baseline/oof.py\": \"...\", \"src/filament_baseline/predictor.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucApmcm9tIHB5Y29jb3Rvb2xzIGltcG9ydCBtYXNrIGFzIG1hc2tfdXRpbHMKCmZyb20gLmRhdGEgaW1wb3J0IElNQUdFX1NJWkUKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBCYXNlbGluZUNvbmZpZzoKICAgIGJsdXJfa2VybmVsOiBpbnQgPSAzMQogICAgc29sYXJfZGlza19yYWRpdXNfZnJhY3Rpb246IGZsb2F0ID0gMC40NAogICAgY2xvc2luZ19rZXJuZWw6IGludCA9IDMKICAgIGNvbXBvbmVudF9jb25uZWN0aXZpdHk6IGludCA9IDgKICAgIG1pbl9jb21wb25lbnRfYXJlYTogaW50ID0gNTAKICAgIG1heF9jb21wb25lbnRfYXJlYTogaW50ID0gNTBfMDAwCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUHJlZGljdGVkSW5zdGFuY2U6CiAgICBmaWxhbWVudF9pZDogc3RyCiAgICBpbWFnZV9zdGVtOiBzdHIKICAgIGFyZWE6IGludAogICAgcmxlOiBkaWN0W3N0ciwgQW55XQoKCmRlZiBlbmNvZGVfYmluYXJ5X21hc2soYmluYXJ5X21hc2s6IG5wLm5kYXJyYXkpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgaWYgYmluYXJ5X21hc2submRpbSAhPSAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBhIDJEIG1hc2ssIGdvdCBzaGFwZSB7YmluYXJ5X21hc2suc2hhcGV9IikKICAgIGNvbnRpZ3VvdXMgPSBucC5hc2ZvcnRyYW5hcnJheSgoYmluYXJ5X21hc2sgPiAwKS5hc3R5cGUobnAudWludDgpKQogICAgZW5jb2RlZCA9IG1hc2tfdXRpbHMuZW5jb2RlKGNvbnRpZ3VvdXMpCiAgICBjb3VudHMgPSBlbmNvZGVkWyJjb3VudHMiXQogICAgaWYgaXNpbnN0YW5jZShjb3VudHMsIGJ5dGVzKToKICAgICAgICBjb3VudHMgPSBjb3VudHMuZGVjb2RlKCJhc2NpaSIpCiAgICByZXR1cm4geyJzaXplIjogW2ludChiaW5hcnlfbWFzay5zaGFwZVswXSksIGludChiaW5hcnlfbWFzay5zaGFwZVsxXSldLCAiY291bnRzIjogY291bnRzfQoKCmRlZiBlbmNvZGVfY3JvcHBlZF9iaW5hcnlfbWFzaygKICAgIGJpbmFyeV9jcm9wOiBucC5uZGFycmF5LAogICAgeF9vZmZzZXQ6IGludCwKICAgIHlfb2Zmc2V0OiBpbnQsCiAgICBmdWxsX3NoYXBlOiB0dXBsZVtpbnQsIGludF0sCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJFbmNvZGUgYSBzbWFsbCBtYXNrIGNyb3AgaW50byBhIGZ1bGwtaW1hZ2UgQ09DTyBSTEUgd2l0aG91dCBhIGZ1bGwgY2FudmFzLiIiIgogICAgaWYgYmluYXJ5X2Nyb3AubmRpbSAhPSAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIGEgMkQgYmluYXJ5IGNyb3AiKQogICAgaGVpZ2h0LCB3aWR0aCA9IGZ1bGxfc2hhcGUKICAgIGNyb3BfaGVpZ2h0LCBjcm9wX3dpZHRoID0gYmluYXJ5X2Nyb3Auc2hhcGUKICAgIGlmICgKICAgICAgICBoZWlnaHQgPCAxCiAgICAgICAgb3Igd2lkdGggPCAxCiAgICAgICAgb3IgeF9vZmZzZXQgPCAwCiAgICAgICAgb3IgeV9vZmZzZXQgPCAwCiAgICAgICAgb3IgeF9vZmZzZXQgKyBjcm9wX3dpZHRoID4gd2lkdGgKICAgICAgICBvciB5X29mZnNldCArIGNyb3BfaGVpZ2h0ID4gaGVpZ2h0CiAgICApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkJpbmFyeSBjcm9wIGZhbGxzIG91dHNpZGUgdGhlIGZ1bGwgaW1hZ2UgYm91bmRzIikKICAgIGNyb3AgPSBucC5hc2FycmF5KGJpbmFyeV9jcm9wLCBkdHlwZT1ib29sKQogICAgaWYgbm90IG5wLmFueShjcm9wKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5ub3QgZW5jb2RlIGFuIGVtcHR5IGNvbXBvbmVudCBjcm9wIikKCiAgICBydW5fdmFsdWVzOiBsaXN0W2ludF0gPSBbXQogICAgY3VycmVudF92YWx1ZSA9IDAKICAgIGN1cnJlbnRfbGVuZ3RoID0gMAoKICAgIGRlZiBhcHBlbmRfcnVuKHZhbHVlOiBpbnQsIGxlbmd0aDogaW50KSAtPiBOb25lOgogICAgICAgIG5vbmxvY2FsIGN1cnJlbnRfdmFsdWUsIGN1cnJlbnRfbGVuZ3RoCiAgICAgICAgaWYgbGVuZ3RoIDw9IDA6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHZhbHVlID0gaW50KHZhbHVlKQogICAgICAgIGlmIHZhbHVlID09IGN1cnJlbnRfdmFsdWU6CiAgICAgICAgICAgIGN1cnJlbnRfbGVuZ3RoICs9IGxlbmd0aAogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJ1bl92YWx1ZXMuYXBwZW5kKGN1cnJlbnRfbGVuZ3RoKQogICAgICAgICAgICBjdXJyZW50X3ZhbHVlID0gdmFsdWUKICAgICAgICAgICAgY3VycmVudF9sZW5ndGggPSBsZW5ndGgKCiAgICBhcHBlbmRfcnVuKDAsIHhfb2Zmc2V0ICogaGVpZ2h0KQogICAgZm9yIGNvbHVtbl9pbmRleCBpbiByYW5nZShjcm9wX3dpZHRoKToKICAgICAgICBjb2x1bW4gPSBjcm9wWzosIGNvbHVtbl9pbmRleF0KICAgICAgICBhcHBlbmRfcnVuKDAsIHlfb2Zmc2V0KQogICAgICAgIHN0YXJ0cyA9IG5wLnJfWzAsIG5wLmZsYXRub256ZXJvKGNvbHVtblsxOl0gIT0gY29sdW1uWzotMV0pICsgMV0KICAgICAgICBlbmRzID0gbnAucl9bc3RhcnRzWzE6XSwgY3JvcF9oZWlnaHRdCiAgICAgICAgZm9yIHN0YXJ0LCBlbmQgaW4gemlwKHN0YXJ0cywgZW5kcywgc3RyaWN0PVRydWUpOgogICAgICAgICAgICBhcHBlbmRfcnVuKGludChjb2x1bW5bc3RhcnRdKSwgaW50KGVuZCAtIHN0YXJ0KSkKICAgICAgICBhcHBlbmRfcnVuKDAsIGhlaWdodCAtIHlfb2Zmc2V0IC0gY3JvcF9oZWlnaHQpCiAgICBhcHBlbmRfcnVuKDAsICh3aWR0aCAtIHhfb2Zmc2V0IC0gY3JvcF93aWR0aCkgKiBoZWlnaHQpCiAgICBydW5fdmFsdWVzLmFwcGVuZChjdXJyZW50X2xlbmd0aCkKICAgIGVuY29kZWQgPSBtYXNrX3V0aWxzLmZyUHlPYmplY3RzKAogICAgICAgIHsic2l6ZSI6IFtoZWlnaHQsIHdpZHRoXSwgImNvdW50cyI6IHJ1bl92YWx1ZXN9LCBoZWlnaHQsIHdpZHRoCiAgICApCiAgICByZXR1cm4gewogICAgICAgICJzaXplIjogbGlzdChlbmNvZGVkWyJzaXplIl0pLAogICAgICAgICJjb3VudHMiOiBlbmNvZGVkWyJjb3VudHMiXS5kZWNvZGUoImFzY2lpIiksCiAgICB9CgoKZGVmIGRlY29kZV9ybGUocmxlOiBkaWN0W3N0ciwgQW55XSkgLT4gbnAubmRhcnJheToKICAgIGVuY29kZWQgPSBkaWN0KHJsZSkKICAgIGlmIGlzaW5zdGFuY2UoZW5jb2RlZFsiY291bnRzIl0sIHN0cik6CiAgICAgICAgZW5jb2RlZFsiY291bnRzIl0gPSBlbmNvZGVkWyJjb3VudHMiXS5lbmNvZGUoImFzY2lpIikKICAgIHJldHVybiBtYXNrX3V0aWxzLmRlY29kZShlbmNvZGVkKS5hc3R5cGUobnAudWludDgsIGNvcHk9RmFsc2UpCgoKZGVmIGVuY29kZV9sYWJlbF9jb21wb25lbnRzKAogICAgbGFiZWxzOiBucC5uZGFycmF5LAogICAgY29tcG9uZW50X2xhYmVsczogbGlzdFtpbnRdLAopIC0+IGRpY3RbaW50LCBkaWN0W3N0ciwgQW55XV06CiAgICAiIiJFbmNvZGUgc2VsZWN0ZWQgY29ubmVjdGVkIGNvbXBvbmVudHMgZnJvbSBvbmUgbGFiZWwgaW1hZ2UgaW4gYSBzaW5nbGUgc2Nhbi4iIiIKICAgIGhlaWdodCwgd2lkdGggPSBsYWJlbHMuc2hhcGUKICAgIGZsYXRfbGFiZWxzID0gbGFiZWxzLnJlc2hhcGUoLTEsIG9yZGVyPSJGIikKICAgIGZvcmVncm91bmRfcG9zaXRpb25zID0gbnAuZmxhdG5vbnplcm8oZmxhdF9sYWJlbHMpCiAgICBpZiBub3QgY29tcG9uZW50X2xhYmVscyBvciBmb3JlZ3JvdW5kX3Bvc2l0aW9ucy5zaXplID09IDA6CiAgICAgICAgcmV0dXJuIHt9CiAgICBmb3JlZ3JvdW5kX2xhYmVscyA9IGZsYXRfbGFiZWxzW2ZvcmVncm91bmRfcG9zaXRpb25zXQogICAgb3JkZXIgPSBucC5hcmdzb3J0KGZvcmVncm91bmRfbGFiZWxzLCBraW5kPSJzdGFibGUiKQogICAgc29ydGVkX3Bvc2l0aW9ucyA9IGZvcmVncm91bmRfcG9zaXRpb25zW29yZGVyXQogICAgc29ydGVkX2xhYmVscyA9IGZvcmVncm91bmRfbGFiZWxzW29yZGVyXQogICAgdW5pcXVlX2xhYmVscywgZ3JvdXBfc3RhcnRzID0gbnAudW5pcXVlKHNvcnRlZF9sYWJlbHMsIHJldHVybl9pbmRleD1UcnVlKQogICAgZ3JvdXBfZW5kcyA9IG5wLnJfW2dyb3VwX3N0YXJ0c1sxOl0sIGxlbihzb3J0ZWRfbGFiZWxzKV0KICAgIHJlc3VsdDogZGljdFtpbnQsIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICB0b3RhbF9waXhlbHMgPSBoZWlnaHQgKiB3aWR0aAoKICAgIGZvciBsYWJlbCBpbiBjb21wb25lbnRfbGFiZWxzOgogICAgICAgIGdyb3VwX2luZGV4ID0gaW50KG5wLnNlYXJjaHNvcnRlZCh1bmlxdWVfbGFiZWxzLCBsYWJlbCkpCiAgICAgICAgaWYgZ3JvdXBfaW5kZXggPj0gbGVuKHVuaXF1ZV9sYWJlbHMpIG9yIGludCh1bmlxdWVfbGFiZWxzW2dyb3VwX2luZGV4XSkgIT0gbGFiZWw6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcG9zaXRpb25zID0gc29ydGVkX3Bvc2l0aW9uc1tncm91cF9zdGFydHNbZ3JvdXBfaW5kZXhdIDogZ3JvdXBfZW5kc1tncm91cF9pbmRleF1dCiAgICAgICAgYm91bmRhcmllcyA9IG5wLmZsYXRub256ZXJvKG5wLmRpZmYocG9zaXRpb25zKSA+IDEpICsgMQogICAgICAgICMgVGhlIGV4Y2x1c2l2ZSBlbmRzIGFyZSBvbmUgcGFzdCB0aGUgbGFzdCBmb3JlZ3JvdW5kIHBpeGVsIGluIGVhY2ggcnVuLgogICAgICAgIHN0YXJ0c19hdCA9IG5wLnJfWzAsIGJvdW5kYXJpZXNdCiAgICAgICAgZW5kc19hdCA9IG5wLnJfW2JvdW5kYXJpZXMsIGxlbihwb3NpdGlvbnMpXQogICAgICAgIGNvdW50czogbGlzdFtpbnRdID0gW10KICAgICAgICBjdXJzb3IgPSAwCiAgICAgICAgZm9yIGJlZ2luLCBlbmQgaW4gemlwKHN0YXJ0c19hdCwgZW5kc19hdCk6CiAgICAgICAgICAgIGZvcmVncm91bmRfc3RhcnQgPSBpbnQocG9zaXRpb25zW2JlZ2luXSkKICAgICAgICAgICAgZm9yZWdyb3VuZF9lbmQgPSBpbnQocG9zaXRpb25zW2VuZCAtIDFdKSArIDEKICAgICAgICAgICAgY291bnRzLmFwcGVuZChmb3JlZ3JvdW5kX3N0YXJ0IC0gY3Vyc29yKQogICAgICAgICAgICBjb3VudHMuYXBwZW5kKGZvcmVncm91bmRfZW5kIC0gZm9yZWdyb3VuZF9zdGFydCkKICAgICAgICAgICAgY3Vyc29yID0gZm9yZWdyb3VuZF9lbmQKICAgICAgICB0YWlsID0gdG90YWxfcGl4ZWxzIC0gY3Vyc29yCiAgICAgICAgaWYgdGFpbDoKICAgICAgICAgICAgY291bnRzLmFwcGVuZCh0YWlsKQogICAgICAgIHVuY29tcHJlc3NlZCA9IHsic2l6ZSI6IFtoZWlnaHQsIHdpZHRoXSwgImNvdW50cyI6IGNvdW50c30KICAgICAgICBlbmNvZGVkID0gbWFza191dGlscy5mclB5T2JqZWN0cyh1bmNvbXByZXNzZWQsIGhlaWdodCwgd2lkdGgpCiAgICAgICAgaWYgaXNpbnN0YW5jZShlbmNvZGVkLCBsaXN0KToKICAgICAgICAgICAgZW5jb2RlZCA9IGVuY29kZWRbMF0KICAgICAgICByZXN1bHRbbGFiZWxdID0gX2NvY29fZW5jb2RlZF9ybGUoZW5jb2RlZCkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgX2NvY29fZW5jb2RlZF9ybGUoZW5jb2RlZDogZGljdFtzdHIsIEFueV0pIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgY291bnRzID0gZW5jb2RlZFsiY291bnRzIl0KICAgIGlmIGlzaW5zdGFuY2UoY291bnRzLCBieXRlcyk6CiAgICAgICAgY291bnRzID0gY291bnRzLmRlY29kZSgiYXNjaWkiKQogICAgcmV0dXJuIHsic2l6ZSI6IFtpbnQoZW5jb2RlZFsic2l6ZSJdWzBdKSwgaW50KGVuY29kZWRbInNpemUiXVsxXSldLCAiY291bnRzIjogY291bnRzfQoKCmRlZiBzb2xhcl9kaXNrX21hc2soaGVpZ2h0OiBpbnQsIHdpZHRoOiBpbnQsIHJhZGl1c19mcmFjdGlvbjogZmxvYXQpIC0+IG5wLm5kYXJyYXk6CiAgICByYWRpdXMgPSBpbnQocm91bmQocmFkaXVzX2ZyYWN0aW9uICogbWluKGhlaWdodCwgd2lkdGgpKSkKICAgIGNlbnRlciA9ICh3aWR0aCAvLyAyLCBoZWlnaHQgLy8gMikKICAgIGRpc2sgPSBucC56ZXJvcygoaGVpZ2h0LCB3aWR0aCksIGR0eXBlPW5wLnVpbnQ4KQogICAgY3YyLmNpcmNsZShkaXNrLCBjZW50ZXIsIHJhZGl1cywgY29sb3I9MSwgdGhpY2tuZXNzPS0xLCBsaW5lVHlwZT1jdjIuTElORV84KQogICAgcmV0dXJuIGRpc2suYXN0eXBlKGJvb2wpCgoKZGVmIHByZWRpY3RfaW1hZ2UoCiAgICBncmF5X2ltYWdlOiBucC5uZGFycmF5LAogICAgZmlsZV9uYW1lOiBzdHIsCiAgICBjb25maWc6IEJhc2VsaW5lQ29uZmlnIHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV06CiAgICBjb25maWcgPSBjb25maWcgb3IgQmFzZWxpbmVDb25maWcoKQogICAgaWYgZ3JheV9pbWFnZS5uZGltICE9IDIgb3IgZ3JheV9pbWFnZS5kdHlwZSAhPSBucC51aW50ODoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCBhbiA4LWJpdCBncmF5c2NhbGUgaW1hZ2UiKQogICAgaGVpZ2h0LCB3aWR0aCA9IGdyYXlfaW1hZ2Uuc2hhcGUKICAgIGlmIGhlaWdodCAhPSBJTUFHRV9TSVpFIG9yIHdpZHRoICE9IElNQUdFX1NJWkU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIGEge0lNQUdFX1NJWkV9eHtJTUFHRV9TSVpFfSBpbWFnZSwgZ290IHt3aWR0aH14e2hlaWdodH0iKQogICAgaWYgY29uZmlnLmJsdXJfa2VybmVsIDwgMSBvciBjb25maWcuYmx1cl9rZXJuZWwgJSAyID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYmx1cl9rZXJuZWwgbXVzdCBiZSBhIHBvc2l0aXZlIG9kZCBpbnRlZ2VyIikKICAgIGlmIGNvbmZpZy5jbG9zaW5nX2tlcm5lbCA8IDEgb3IgY29uZmlnLmNsb3Npbmdfa2VybmVsICUgMiA9PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNsb3Npbmdfa2VybmVsIG11c3QgYmUgYSBwb3NpdGl2ZSBvZGQgaW50ZWdlciIpCgogICAgYmx1cnJlZCA9IGN2Mi5HYXVzc2lhbkJsdXIoZ3JheV9pbWFnZSwgKGNvbmZpZy5ibHVyX2tlcm5lbCwgY29uZmlnLmJsdXJfa2VybmVsKSwgMCkKICAgIHJlc3BvbnNlID0gY3YyLnN1YnRyYWN0KGJsdXJyZWQsIGdyYXlfaW1hZ2UpCiAgICBkaXNrID0gc29sYXJfZGlza19tYXNrKGhlaWdodCwgd2lkdGgsIGNvbmZpZy5zb2xhcl9kaXNrX3JhZGl1c19mcmFjdGlvbikKICAgIGRpc2tfdmFsdWVzID0gcmVzcG9uc2VbZGlza10KICAgIGlmIGRpc2tfdmFsdWVzLnNpemUgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTb2xhciBkaXNrIHJlZ2lvbiBpcyBlbXB0eSIpCiAgICB0aHJlc2hvbGQsIF8gPSBjdjIudGhyZXNob2xkKAogICAgICAgIGRpc2tfdmFsdWVzLnJlc2hhcGUoLTEsIDEpLAogICAgICAgIDAsCiAgICAgICAgMjU1LAogICAgICAgIGN2Mi5USFJFU0hfQklOQVJZIHwgY3YyLlRIUkVTSF9PVFNVLAogICAgKQogICAgYmluYXJ5ID0gKChyZXNwb25zZSA+IHRocmVzaG9sZCkgJiBkaXNrKS5hc3R5cGUobnAudWludDgpCiAgICBrZXJuZWwgPSBucC5vbmVzKChjb25maWcuY2xvc2luZ19rZXJuZWwsIGNvbmZpZy5jbG9zaW5nX2tlcm5lbCksIGR0eXBlPW5wLnVpbnQ4KQogICAgY2xvc2VkID0gY3YyLm1vcnBob2xvZ3lFeChiaW5hcnksIGN2Mi5NT1JQSF9DTE9TRSwga2VybmVsKQogICAgY2xvc2VkW35kaXNrXSA9IDAKCiAgICBjb3VudCwgbGFiZWxzLCBzdGF0cywgXyA9IGN2Mi5jb25uZWN0ZWRDb21wb25lbnRzV2l0aFN0YXRzKAogICAgICAgIGNsb3NlZCwgY29ubmVjdGl2aXR5PWNvbmZpZy5jb21wb25lbnRfY29ubmVjdGl2aXR5CiAgICApCiAgICBhY2NlcHRlZF9sYWJlbHMgPSBbCiAgICAgICAgbGFiZWwKICAgICAgICBmb3IgbGFiZWwgaW4gcmFuZ2UoMSwgY291bnQpCiAgICAgICAgaWYgY29uZmlnLm1pbl9jb21wb25lbnRfYXJlYQogICAgICAgIDw9IGludChzdGF0c1tsYWJlbCwgY3YyLkNDX1NUQVRfQVJFQV0pCiAgICAgICAgPD0gY29uZmlnLm1heF9jb21wb25lbnRfYXJlYQogICAgXQogICAgZW5jb2RlZF9jb21wb25lbnRzID0gZW5jb2RlX2xhYmVsX2NvbXBvbmVudHMobGFiZWxzLCBhY2NlcHRlZF9sYWJlbHMpCiAgICBpbWFnZV9zdGVtID0gUGF0aChmaWxlX25hbWUpLnN0ZW0KICAgIHByZWRpY3Rpb25zOiBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXSA9IFtdCiAgICBmb3Igc2VxdWVuY2UsIGxhYmVsIGluIGVudW1lcmF0ZShhY2NlcHRlZF9sYWJlbHMsIHN0YXJ0PTEpOgogICAgICAgIGFyZWEgPSBpbnQoc3RhdHNbbGFiZWwsIGN2Mi5DQ19TVEFUX0FSRUFdKQogICAgICAgIHByZWRpY3Rpb25zLmFwcGVuZCgKICAgICAgICAgICAgUHJlZGljdGVkSW5zdGFuY2UoCiAgICAgICAgICAgICAgICBmaWxhbWVudF9pZD1mIntpbWFnZV9zdGVtfV97c2VxdWVuY2V9IiwKICAgICAgICAgICAgICAgIGltYWdlX3N0ZW09aW1hZ2Vfc3RlbSwKICAgICAgICAgICAgICAgIGFyZWE9YXJlYSwKICAgICAgICAgICAgICAgIHJsZT1lbmNvZGVkX2NvbXBvbmVudHNbbGFiZWxdLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgcmV0dXJuIHByZWRpY3Rpb25zCgoKZGVmIHJlYWRfZ3JheXNjYWxlKHBhdGg6IHN0ciB8IFBhdGgpIC0+IG5wLm5kYXJyYXk6CiAgICBpbWFnZSA9IGN2Mi5pbXJlYWQoc3RyKHBhdGgpLCBjdjIuSU1SRUFEX0dSQVlTQ0FMRSkKICAgIGlmIGltYWdlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDb3VsZCBub3QgcmVhZCBpbWFnZToge3BhdGh9IikKICAgIGlmIGltYWdlLnNoYXBlICE9IChJTUFHRV9TSVpFLCBJTUFHRV9TSVpFKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgYSAyMDQ4eDIwNDggaW1hZ2UgYXQge3BhdGh9LCBnb3Qge2ltYWdlLnNoYXBlfSIpCiAgICByZXR1cm4gaW1hZ2UKCgpkZWYgd3JpdGVfcHJlZGljdGlvbl9vdmVybGF5KAogICAgZ3JheV9pbWFnZTogbnAubmRhcnJheSwKICAgIHByZWRpY3Rpb25zOiBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXSwKICAgIG91dHB1dF9wYXRoOiBzdHIgfCBQYXRoLAopIC0+IE5vbmU6CiAgICBjYW52YXMgPSBjdjIuY3Z0Q29sb3IoZ3JheV9pbWFnZSwgY3YyLkNPTE9SX0dSQVkyQkdSKQogICAgY29sb3JfbGF5ZXIgPSBjYW52YXMuY29weSgpCiAgICBmb3IgaW5kZXgsIHByZWRpY3Rpb24gaW4gZW51bWVyYXRlKHByZWRpY3Rpb25zKToKICAgICAgICBodWUgPSAoaW5kZXggKiAzNyArIDExKSAlIDE4MAogICAgICAgIGhzdiA9IG5wLnVpbnQ4KFtbW2h1ZSwgMjIwLCAyNTVdXV0pCiAgICAgICAgY29sb3IgPSB0dXBsZShpbnQodmFsdWUpIGZvciB2YWx1ZSBpbiBjdjIuY3Z0Q29sb3IoaHN2LCBjdjIuQ09MT1JfSFNWMkJHUilbMCwgMF0pCiAgICAgICAgbWFzayA9IGRlY29kZV9ybGUocHJlZGljdGlvbi5ybGUpCiAgICAgICAgY29sb3JfbGF5ZXJbbWFzayA+IDBdID0gY29sb3IKICAgIGNhbnZhcyA9IGN2Mi5hZGRXZWlnaHRlZChjYW52YXMsIDAuNjIsIGNvbG9yX2xheWVyLCAwLjM4LCAwKQogICAgY3YyLnB1dFRleHQoCiAgICAgICAgY2FudmFzLAogICAgICAgIGYiUHJlZGljdGVkIGluc3RhbmNlczoge2xlbihwcmVkaWN0aW9ucyl9IiwKICAgICAgICAoMzIsIDU2KSwKICAgICAgICBjdjIuRk9OVF9IRVJTSEVZX1NJTVBMRVgsCiAgICAgICAgMS4wLAogICAgICAgICgyNTUsIDI1NSwgMjU1KSwKICAgICAgICAyLAogICAgICAgIGN2Mi5MSU5FX0FBLAogICAgKQogICAgb3V0cHV0X3BhdGggPSBQYXRoKG91dHB1dF9wYXRoKQogICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIG5vdCBjdjIuaW13cml0ZShzdHIob3V0cHV0X3BhdGgpLCBjYW52YXMpOgogICAgICAgIHJhaXNlIE9TRXJyb3IoZiJDb3VsZCBub3Qgd3JpdGUgcHJlZGljdGlvbiBvdmVybGF5OiB7b3V0cHV0X3BhdGh9IikK\", \"src/filament_baseline/reinference.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmZyb20gLmRhdGEgaW1wb3J0IGludmVudG9yeSwgbG9hZF9kYXRhc2V0CmZyb20gLmluZmVyZW5jZSBpbXBvcnQgSW5zdGFuY2VDb25maWcsIGluc3RhbmNlc19mcm9tX3Byb2JhYmlsaXRpZXMsIHByZWRpY3RfcHJvYmFiaWxpdGllcwpmcm9tIC5tb2RlbF9ydW5uZXIgaW1wb3J0IF9naXRfc3RhdHVzLCBfaXNfa2FnZ2xlX3J1bnRpbWUsIF9sb2FkX21vZGVsLCBfcmVzb2x2ZV9kYXRhX3Jvb3QsIF93aXRoaW4KZnJvbSAucHJlZGljdG9yIGltcG9ydCBQcmVkaWN0ZWRJbnN0YW5jZSwgcmVhZF9ncmF5c2NhbGUKZnJvbSAucnVubmVyIGltcG9ydCBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzLCBfd3JpdGVfb3ZlcmxheXMsIF93cml0ZV9wcmVkaWN0aW9ucwpmcm9tIC5ydW50aW1lIGltcG9ydCBoYXJkd2FyZV9pbmZvLCBwYWNrYWdlX3ZlcnNpb25zCmZyb20gLnNjb3JlciBpbXBvcnQgc2NvcmVfdmFsaWRhdGlvbgpmcm9tIC5zcGxpdCBpbXBvcnQgY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0CmZyb20gLnZpc3VhbHMgaW1wb3J0IHdyaXRlX2NvbXBhcmlzb25fb3ZlcmxheXMKCgpkZWYgcmVpbmZlcl92YWxpZGF0aW9uKAogICAgKiwKICAgIHJlcG9fcm9vdDogc3RyIHwgUGF0aCwKICAgIGRhdGFfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUsCiAgICBtb2RlbF9ydW5fcm9vdDogc3RyIHwgUGF0aCwKICAgIHR0YTogaW50LAogICAgb3ZlcmxheV9jb3VudDogaW50ID0gMTIsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJSZWNvbXB1dGUgaG9sZG91dCBwcm9iYWJpbGl0eSBtYXBzIGZyb20gYSBzYXZlZCBjaGVja3BvaW50IHdpdGggYW5vdGhlciBUVEEgY291bnQuIiIiCiAgICByZXBvX3Jvb3QgPSBQYXRoKHJlcG9fcm9vdCkucmVzb2x2ZSgpCiAgICBtb2RlbF9ydW5fcm9vdCA9IFBhdGgobW9kZWxfcnVuX3Jvb3QpLnJlc29sdmUoKQogICAga2FnZ2xlX3J1bnRpbWUgPSBfaXNfa2FnZ2xlX3J1bnRpbWUoKQogICAgaWYgbm90IGthZ2dsZV9ydW50aW1lIGFuZCBub3QgX3dpdGhpbihtb2RlbF9ydW5fcm9vdCwgcmVwb19yb290KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgbW9kZWwgYXJ0aWZhY3RzIG11c3Qgc3RheSBpbnNpZGUgdGhlIHJlcG9zaXRvcnk6IHttb2RlbF9ydW5fcm9vdH0iKQogICAgaWYgdHRhIG5vdCBpbiAoMSwgMiwgNCwgOCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHRhIG11c3QgYmUgMSwgMiwgNCwgb3IgOCIpCgogICAgZGF0YXNldF9yb290ID0gX3Jlc29sdmVfZGF0YV9yb290KGRhdGFfcm9vdCwgcmVwb19yb290KQogICAgaWYgbm90IGthZ2dsZV9ydW50aW1lIGFuZCBub3QgX3dpdGhpbihkYXRhc2V0X3Jvb3QsIHJlcG9fcm9vdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkxvY2FsIGNvbXBldGl0aW9uIGRhdGEgbXVzdCBzdGF5IGluc2lkZSB0aGUgcmVwb3NpdG9yeToge2RhdGFzZXRfcm9vdH0iKQogICAgbW9kZWxfbWV0cmljc19wYXRoID0gbW9kZWxfcnVuX3Jvb3QgLyAibWV0cmljcy5qc29uIgogICAgY2hlY2twb2ludF9wYXRoID0gbW9kZWxfcnVuX3Jvb3QgLyAiYmVzdF9tb2RlbC5wdCIKICAgIGlmIG5vdCBtb2RlbF9tZXRyaWNzX3BhdGguaXNfZmlsZSgpIG9yIG5vdCBjaGVja3BvaW50X3BhdGguaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKCJtb2RlbF9ydW5fcm9vdCBtdXN0IGNvbnRhaW4gbWV0cmljcy5qc29uIGFuZCBiZXN0X21vZGVsLnB0IikKICAgIG1vZGVsX21ldHJpY3MgPSBqc29uLmxvYWRzKG1vZGVsX21ldHJpY3NfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBtb2RlbF9jb25maWd1cmF0aW9uID0gbW9kZWxfbWV0cmljcy5nZXQoIm1ldGhvZCIsIHt9KQogICAgcGF0Y2hfc2l6ZSA9IGludChtb2RlbF9jb25maWd1cmF0aW9uWyJwYXRjaF9zaXplIl0pCiAgICBkYXRhc2V0ID0gbG9hZF9kYXRhc2V0KGRhdGFzZXRfcm9vdCkKICAgIGRhdGFfaW52ZW50b3J5ID0gaW52ZW50b3J5KGRhdGFzZXQpCiAgICBjb25maWdfcGF0aCA9IHJlcG9fcm9vdCAvICJjb25maWdzIiAvICJ2YWxpZGF0aW9uLWJhc2VsaW5lLmpzb24iCiAgICBjb25maWcgPSBqc29uLmxvYWRzKGNvbmZpZ19wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHNwbGl0ID0gY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInNwbGl0cyIsCiAgICAgICAgc2VlZD1pbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgIHZhbGlkYXRpb25fZnJhY3Rpb249ZmxvYXQoY29uZmlnWyJ2YWxpZGF0aW9uX2ZyYWN0aW9uIl0pLAogICAgKQogICAgaWYgc3BsaXQuc2hhMjU2ICE9IGNvbmZpZ1siZXhwZWN0ZWRfc3BsaXRfc2hhMjU2Il06CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGUgZnJvemVuIHZhbGlkYXRpb24gc3BsaXQgZGlnZXN0IGNoYW5nZWQiKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIGlmIGxlbih2YWxpZGF0aW9uX25hbWVzKSAhPSAxNDI6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiRXhwZWN0ZWQgYWxsIDE0MiBob2xkb3V0IG9ic2VydmF0aW9ucywgZm91bmQge2xlbih2YWxpZGF0aW9uX25hbWVzKX0iKQoKICAgIG91dHB1dF9yb290ID0gbW9kZWxfcnVuX3Jvb3QgLyBmInR0YS17dHRhfSIKICAgIGlmIG91dHB1dF9yb290LmV4aXN0cygpIGFuZCBhbnkob3V0cHV0X3Jvb3QuaXRlcmRpcigpKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZiJUVEEgb3V0cHV0IGFscmVhZHkgZXhpc3RzOyBjaG9vc2Ugb3IgY2xlYXIgYSBuZXcgcnVuIGRpcmVjdG9yeToge291dHB1dF9yb290fSIpCiAgICBwcm9iYWJpbGl0eV9kaXIgPSBvdXRwdXRfcm9vdCAvICJwcm9iYWJpbGl0aWVzIgogICAgcHJvYmFiaWxpdHlfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYTowIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBtb2RlbCA9IF9sb2FkX21vZGVsKGNoZWNrcG9pbnRfcGF0aCwgZGV2aWNlKQogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoZGV2aWNlKQogICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXV0gPSB7fQogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGluZmVyZW5jZV9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIGluZGV4LCBmaWxlX25hbWUgaW4gZW51bWVyYXRlKHZhbGlkYXRpb25fbmFtZXMsIHN0YXJ0PTEpOgogICAgICAgIGltYWdlID0gcmVhZF9ncmF5c2NhbGUoZGF0YXNldC50cmFpbl9pbWFnZV9kaXIgLyBmaWxlX25hbWUpCiAgICAgICAgZm9yZWdyb3VuZCwgYm91bmRhcnkgPSBwcmVkaWN0X3Byb2JhYmlsaXRpZXMoCiAgICAgICAgICAgIG1vZGVsLAogICAgICAgICAgICBpbWFnZSwKICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgcGF0Y2hfc2l6ZT1wYXRjaF9zaXplLAogICAgICAgICAgICBzdHJpZGU9bWF4KHBhdGNoX3NpemUgLy8gMiwgaW50KHBhdGNoX3NpemUgKiAwLjc1KSksCiAgICAgICAgICAgIHR0YT10dGEsCiAgICAgICAgKQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoCiAgICAgICAgICAgIHByb2JhYmlsaXR5X2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiLAogICAgICAgICAgICBmb3JlZ3JvdW5kPWZvcmVncm91bmQuYXN0eXBlKG5wLmZsb2F0MTYpLAogICAgICAgICAgICBib3VuZGFyeT1ib3VuZGFyeS5hc3R5cGUobnAuZmxvYXQxNiksCiAgICAgICAgKQogICAgICAgIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0gPSBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzKAogICAgICAgICAgICBmb3JlZ3JvdW5kLAogICAgICAgICAgICBib3VuZGFyeSwKICAgICAgICAgICAgZmlsZV9uYW1lLAogICAgICAgICAgICBJbnN0YW5jZUNvbmZpZygpLAogICAgICAgICkKICAgICAgICBpZiBpbmRleCAlIDIwID09IDAgb3IgaW5kZXggPT0gbGVuKHZhbGlkYXRpb25fbmFtZXMpOgogICAgICAgICAgICBwcmludChmIlJlLWluZmVycmVkIHtpbmRleH0ve2xlbih2YWxpZGF0aW9uX25hbWVzKX0gb2JzZXJ2YXRpb25zIHdpdGggVFRBPXt0dGF9IiwgZmx1c2g9VHJ1ZSkKICAgIGluZmVyZW5jZV9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGluZmVyZW5jZV9zdGFydGVkCiAgICBwZWFrX2dwdV9tZW1vcnlfYnl0ZXMgPSAoCiAgICAgICAgaW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoZGV2aWNlKSkgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGVsc2UgMAogICAgKQoKICAgIHNjb3Jpbmdfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIG1ldHJpY3MsIG9ic2VydmF0aW9uX3Jvd3MgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgcHJlZGljdGlvbnMsCiAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICBpb3VfdGhyZXNob2xkPWZsb2F0KGNvbmZpZ1siaW91X3RocmVzaG9sZCJdKSwKICAgICAgICBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkPWZsb2F0KGNvbmZpZ1siZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZCJdKSwKICAgICkKICAgIHNjb3Jpbmdfc2Vjb25kcyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzY29yaW5nX3N0YXJ0ZWQKICAgIF93cml0ZV9wcmVkaWN0aW9ucyhvdXRwdXRfcm9vdCAvICJwcmVkaWN0aW9ucy5jc3YiLCBwcmVkaWN0aW9ucykKICAgIF93cml0ZV9vYnNlcnZhdGlvbl9zY29yZXMob3V0cHV0X3Jvb3QgLyAicGVyX29ic2VydmF0aW9uLmNzdiIsIG9ic2VydmF0aW9uX3Jvd3MsIHByZWRpY3Rpb25zKQogICAgX3dyaXRlX292ZXJsYXlzKGRhdGFzZXQsIHByZWRpY3Rpb25zLCBvdXRwdXRfcm9vdCAvICJvdmVybGF5cyIsIG92ZXJsYXlfY291bnQpCiAgICBjb21wYXJpc29uX292ZXJsYXlzID0gd3JpdGVfY29tcGFyaXNvbl9vdmVybGF5cygKICAgICAgICBkYXRhc2V0LCBwcmVkaWN0aW9ucywgb3V0cHV0X3Jvb3QgLyAiY29tcGFyaXNvbi1vdmVybGF5cyIsIG92ZXJsYXlfY291bnQKICAgICkKCiAgICByZXBvc2l0b3J5X2luZm8gPSBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3QpCiAgICByZXN1bHQgPSB7CiAgICAgICAgInN0YXR1cyI6ICJjb21wbGV0ZSIsCiAgICAgICAgInJ1bl9uYW1lIjogZiJ7bW9kZWxfcnVuX3Jvb3QubmFtZX0tdHRhLXt0dGF9IiwKICAgICAgICAicGFyZW50X21vZGVsX3J1biI6IHN0cihtb2RlbF9ydW5fcm9vdC5yZWxhdGl2ZV90byhyZXBvX3Jvb3QpKQogICAgICAgIGlmIF93aXRoaW4obW9kZWxfcnVuX3Jvb3QsIHJlcG9fcm9vdCkKICAgICAgICBlbHNlIHN0cihtb2RlbF9ydW5fcm9vdCksCiAgICAgICAgInBhcmVudF9tb2RlbF9jb25maWd1cmF0aW9uIjogbW9kZWxfY29uZmlndXJhdGlvbiwKICAgICAgICAiY2hlY2twb2ludF9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihjaGVja3BvaW50X3BhdGgucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAidHRhIjogdHRhLAogICAgICAgICJwYXRjaF9zaXplIjogcGF0Y2hfc2l6ZSwKICAgICAgICAiZGVmYXVsdF9pbnN0YW5jZV9jb25maWd1cmF0aW9uIjogSW5zdGFuY2VDb25maWcoKS5fX2RpY3RfXywKICAgICAgICAicmVwb3NpdG9yeSI6IHJlcG9zaXRvcnlfaW5mbywKICAgICAgICAiY29uZmlndXJhdGlvbl9zaGEyNTYiOiBoYXNobGliLnNoYTI1Nihjb25maWdfcGF0aC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICJkYXRhc2V0X2ludmVudG9yeSI6IGRhdGFfaW52ZW50b3J5LAogICAgICAgICJzcGxpdCI6IHsKICAgICAgICAgICAgInNoYTI1NiI6IHNwbGl0LnNoYTI1NiwKICAgICAgICAgICAgInRyYWluaW5nX29ic2VydmF0aW9ucyI6IGxlbihzcGxpdC50cmFpbmluZ19uYW1lcyksCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX29ic2VydmF0aW9ucyI6IGxlbih2YWxpZGF0aW9uX25hbWVzKSwKICAgICAgICB9LAogICAgICAgICJtZXRyaWNzIjogbWV0cmljcywKICAgICAgICAicnVudGltZSI6IHsKICAgICAgICAgICAgKipoYXJkd2FyZV9pbmZvKCksCiAgICAgICAgICAgICJwYWNrYWdlX3ZlcnNpb25zIjogcGFja2FnZV92ZXJzaW9ucygpLAogICAgICAgICAgICAiZGV2aWNlIjogc3RyKGRldmljZSksCiAgICAgICAgICAgICJpbmZlcmVuY2Vfc2Vjb25kcyI6IGluZmVyZW5jZV9zZWNvbmRzLAogICAgICAgICAgICAic2NvcmluZ19zZWNvbmRzIjogc2NvcmluZ19zZWNvbmRzLAogICAgICAgICAgICAiZW5kX3RvX2VuZF9zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQsCiAgICAgICAgICAgICJwZWFrX2dwdV9tZW1vcnlfYnl0ZXMiOiBwZWFrX2dwdV9tZW1vcnlfYnl0ZXMsCiAgICAgICAgICAgICJjb21wYXJpc29uX292ZXJsYXlzIjogY29tcGFyaXNvbl9vdmVybGF5cywKICAgICAgICB9LAogICAgfQogICAgKG91dHB1dF9yb290IC8gIm1ldHJpY3MuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcmludChqc29uLmR1bXBzKHsicnVuX25hbWUiOiByZXN1bHRbInJ1bl9uYW1lIl0sICJtZXRyaWNzIjogbWV0cmljc1siZ2xvYmFsIl0sICJzcGxpdF9zaGEyNTYiOiBzcGxpdC5zaGEyNTZ9LCBpbmRlbnQ9MikpCiAgICByZXR1cm4gcmVzdWx0Cg==\", \"src/filament_baseline/runner.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSAuZGF0YSBpbXBvcnQgQ29tcGV0aXRpb25EYXRhc2V0LCBmaW5kX2RhdGFzZXRfcm9vdCwgaW52ZW50b3J5LCBsb2FkX2RhdGFzZXQKZnJvbSAucHJlZGljdG9yIGltcG9ydCBCYXNlbGluZUNvbmZpZywgUHJlZGljdGVkSW5zdGFuY2UsIHByZWRpY3RfaW1hZ2UsIHJlYWRfZ3JheXNjYWxlLCB3cml0ZV9wcmVkaWN0aW9uX292ZXJsYXkKZnJvbSAucnVudGltZSBpbXBvcnQgZ2l0X2luZm8sIGhhcmR3YXJlX2luZm8sIHBhY2thZ2VfdmVyc2lvbnMKZnJvbSAuc2NvcmVyIGltcG9ydCBPYnNlcnZhdGlvbkNvdW50cywgc2NvcmVfdmFsaWRhdGlvbgpmcm9tIC5zcGxpdCBpbXBvcnQgY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0CgpPRkZJQ0lBTF9TQ09SRVJfVVJMID0gKAogICAgImh0dHBzOi8vd3d3LmthZ2dsZS5jb20vY29kZS9hemltYWhtYWR6YWRlaC9zZWxmLWV2YWx1YXRpb24tbm90ZWJvb2siCikKCgpkZWYgX3dpdGhpbihwYXRoOiBQYXRoLCBwYXJlbnQ6IFBhdGgpIC0+IGJvb2w6CiAgICB0cnk6CiAgICAgICAgcGF0aC5yZXNvbHZlKCkucmVsYXRpdmVfdG8ocGFyZW50LnJlc29sdmUoKSkKICAgICAgICByZXR1cm4gVHJ1ZQogICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIF9pc19rYWdnbGVfcnVudGltZSgpIC0+IGJvb2w6CiAgICByZXR1cm4gYm9vbChvcy5lbnZpcm9uLmdldCgiS0FHR0xFX0tFUk5FTF9SVU5fVFlQRSIpKSBvciAoCiAgICAgICAgUGF0aCgiL2thZ2dsZS9pbnB1dCIpLmlzX2RpcigpIGFuZCBQYXRoKCIva2FnZ2xlL3dvcmtpbmciKS5pc19kaXIoKQogICAgKQoKCmRlZiBfc291cmNlX2ZpbmdlcnByaW50KHJlcG9fcm9vdDogUGF0aCkgLT4gc3RyOgogICAgcGF0aHMgPSBbCiAgICAgICAgKnNvcnRlZCgocmVwb19yb290IC8gInNyYyIpLnJnbG9iKCIqLnB5IikpLAogICAgICAgICpzb3J0ZWQoKHJlcG9fcm9vdCAvICJzY3JpcHRzIikuZ2xvYigiKi5weSIpKSwKICAgICAgICByZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIiwKICAgICAgICByZXBvX3Jvb3QgLyAibm90ZWJvb2tzIiAvICJ2YWxpZGF0aW9uX2Jhc2VsaW5lLmlweW5iIiwKICAgICAgICByZXBvX3Jvb3QgLyAicmVxdWlyZW1lbnRzLnR4dCIsCiAgICAgICAgcmVwb19yb290IC8gInJlcXVpcmVtZW50cy1ncHUudHh0IiwKICAgIF0KICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIGZvciBwYXRoIGluIHBhdGhzOgogICAgICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBkaWdlc3QudXBkYXRlKHBhdGgucmVsYXRpdmVfdG8ocmVwb19yb290KS5hc19wb3NpeCgpLmVuY29kZSgidXRmLTgiKSkKICAgICAgICBkaWdlc3QudXBkYXRlKGIiXDAiKQogICAgICAgIGRpZ2VzdC51cGRhdGUocGF0aC5yZWFkX2J5dGVzKCkpCiAgICAgICAgZGlnZXN0LnVwZGF0ZShiIlwwIikKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgX2xvYWRfY29uZmlnKHJlcG9fcm9vdDogUGF0aCkgLT4gdHVwbGVbZGljdFtzdHIsIEFueV0sIEJhc2VsaW5lQ29uZmlnXToKICAgIHBhdGggPSByZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIgogICAgcGF5bG9hZCA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBiYXNlbGluZSA9IEJhc2VsaW5lQ29uZmlnKAogICAgICAgIGJsdXJfa2VybmVsPWludChwYXlsb2FkWyJibHVyX2tlcm5lbCJdKSwKICAgICAgICBzb2xhcl9kaXNrX3JhZGl1c19mcmFjdGlvbj1mbG9hdChwYXlsb2FkWyJzb2xhcl9kaXNrX3JhZGl1c19mcmFjdGlvbiJdKSwKICAgICAgICBjbG9zaW5nX2tlcm5lbD1pbnQocGF5bG9hZFsiY2xvc2luZ19rZXJuZWwiXSksCiAgICAgICAgY29tcG9uZW50X2Nvbm5lY3Rpdml0eT1pbnQocGF5bG9hZFsiY29tcG9uZW50X2Nvbm5lY3Rpdml0eSJdKSwKICAgICAgICBtaW5fY29tcG9uZW50X2FyZWE9aW50KHBheWxvYWRbIm1pbl9jb21wb25lbnRfYXJlYSJdKSwKICAgICAgICBtYXhfY29tcG9uZW50X2FyZWE9aW50KHBheWxvYWRbIm1heF9jb21wb25lbnRfYXJlYSJdKSwKICAgICkKICAgIHJldHVybiBwYXlsb2FkLCBiYXNlbGluZQoKCmRlZiBfcmVzb2x2ZV9kYXRhX3Jvb3QoZGF0YV9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSwgcmVwb19yb290OiBQYXRoKSAtPiBQYXRoOgogICAgY2FuZGlkYXRlczogbGlzdFtQYXRoXSA9IFtdCiAgICBpZiBkYXRhX3Jvb3QgaXMgbm90IE5vbmU6CiAgICAgICAgY2FuZGlkYXRlcy5hcHBlbmQoUGF0aChkYXRhX3Jvb3QpKQogICAgZW52X3Jvb3QgPSBvcy5lbnZpcm9uLmdldCgiRVFVSU5FWFVTX0RBVEFfUk9PVCIpCiAgICBpZiBlbnZfcm9vdDoKICAgICAgICBjYW5kaWRhdGVzLmFwcGVuZChQYXRoKGVudl9yb290KSkKICAgIGNhbmRpZGF0ZXMuZXh0ZW5kKAogICAgICAgIFsKICAgICAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInJhdyIgLyAiTUFHRmlMT18xLjBfS2FnZ2xlXzIwMjYiLAogICAgICAgICAgICBQYXRoKCIva2FnZ2xlL2lucHV0IiksCiAgICAgICAgXQogICAgKQogICAgZXJyb3JzID0gW10KICAgIGZvciBjYW5kaWRhdGUgaW4gY2FuZGlkYXRlczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBmaW5kX2RhdGFzZXRfcm9vdChjYW5kaWRhdGUpCiAgICAgICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvciwgUnVudGltZUVycm9yKSBhcyBleGM6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7Y2FuZGlkYXRlfToge2V4Y30iKQogICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgIk5vIGNvbXBldGl0aW9uIGRhdGFzZXQgd2FzIGZvdW5kLiBTZXQgRVFVSU5FWFVTX0RBVEFfUk9PVCBvciBwYXNzIC0tZGF0YS1yb290LlxuIgogICAgICAgICsgIlxuIi5qb2luKGVycm9ycykKICAgICkKCgpkZWYgX3dyaXRlX3ByZWRpY3Rpb25zKAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICBwcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RbUHJlZGljdGVkSW5zdGFuY2VdXSwKKSAtPiBOb25lOgogICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3V0cHV0X3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICB3cml0ZXIgPSBjc3Yud3JpdGVyKHN0cmVhbSwgbGluZXRlcm1pbmF0b3I9IlxuIikKICAgICAgICB3cml0ZXIud3JpdGVyb3coKCJmaWxhbWVudF9pZCIsICJzZWdtZW50YXRpb25fcmxlIikpCiAgICAgICAgZm9yIGZpbGVfbmFtZSBpbiBzb3J0ZWQocHJlZGljdGlvbnMpOgogICAgICAgICAgICBmb3IgcHJlZGljdGlvbiBpbiBwcmVkaWN0aW9uc1tmaWxlX25hbWVdOgogICAgICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KChwcmVkaWN0aW9uLmZpbGFtZW50X2lkLCBwcmVkaWN0aW9uLnJsZVsiY291bnRzIl0pKQoKCmRlZiBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzKAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICByb3dzOiBsaXN0W09ic2VydmF0aW9uQ291bnRzXSwKICAgIHByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV1dLAopIC0+IE5vbmU6CiAgICBvdXRwdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZmllbGRzID0gKAogICAgICAgICJmaWxlX25hbWUiLAogICAgICAgICJ5ZWFyIiwKICAgICAgICAic2l0ZSIsCiAgICAgICAgImFubm90YXRvcl92aWV3cyIsCiAgICAgICAgImdyb3VuZF90cnV0aF9pbnN0YW5jZXNfYWNyb3NzX3ZpZXdzIiwKICAgICAgICAicHJlZGljdGVkX2luc3RhbmNlc19wZXJfb2JzZXJ2YXRpb24iLAogICAgICAgICJ0cF9lZGdlcyIsCiAgICAgICAgInRwX2lvdV9zdW0iLAogICAgICAgICJmcCIsCiAgICAgICAgImZuIiwKICAgICAgICAicHEiLAogICAgKQogICAgd2l0aCBvdXRwdXRfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1maWVsZHMsIGxpbmV0ZXJtaW5hdG9yPSJcbiIpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAiZmlsZV9uYW1lIjogcm93LmZpbGVfbmFtZSwKICAgICAgICAgICAgICAgICAgICAieWVhciI6IHJvdy55ZWFyLAogICAgICAgICAgICAgICAgICAgICJzaXRlIjogcm93LnNpdGUsCiAgICAgICAgICAgICAgICAgICAgImFubm90YXRvcl92aWV3cyI6IHJvdy5hbm5vdGF0b3Jfdmlld3MsCiAgICAgICAgICAgICAgICAgICAgImdyb3VuZF90cnV0aF9pbnN0YW5jZXNfYWNyb3NzX3ZpZXdzIjogcm93Lmd0X2NvdW50LAogICAgICAgICAgICAgICAgICAgICJwcmVkaWN0ZWRfaW5zdGFuY2VzX3Blcl9vYnNlcnZhdGlvbiI6IGxlbigKICAgICAgICAgICAgICAgICAgICAgICAgcHJlZGljdGlvbnMuZ2V0KHJvdy5maWxlX25hbWUsIFtdKQogICAgICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICAgICAgInRwX2VkZ2VzIjogcm93LnRwLAogICAgICAgICAgICAgICAgICAgICJ0cF9pb3Vfc3VtIjogZiJ7cm93LnRwX2lvdV9zdW06LjlmfSIsCiAgICAgICAgICAgICAgICAgICAgImZwIjogcm93LmZwLAogICAgICAgICAgICAgICAgICAgICJmbiI6IHJvdy5mbiwKICAgICAgICAgICAgICAgICAgICAicHEiOiBmIntyb3cucHE6LjlmfSIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKCgpkZWYgX3dyaXRlX292ZXJsYXlzKAogICAgZGF0YXNldDogQ29tcGV0aXRpb25EYXRhc2V0LAogICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXV0sCiAgICBvdXRwdXRfZGlyOiBQYXRoLAogICAgbWF4aW11bTogaW50LAopIC0+IGxpc3Rbc3RyXToKICAgIG5hbWVzID0gc29ydGVkKHByZWRpY3Rpb25zKQogICAgaWYgbm90IG5hbWVzIG9yIG1heGltdW0gPD0gMDoKICAgICAgICByZXR1cm4gW10KICAgIGlmIGxlbihuYW1lcykgPD0gbWF4aW11bToKICAgICAgICBzZWxlY3RlZCA9IG5hbWVzCiAgICBlbGlmIG1heGltdW0gPT0gMToKICAgICAgICBzZWxlY3RlZCA9IFtuYW1lc1tsZW4obmFtZXMpIC8vIDJdXQogICAgZWxzZToKICAgICAgICBpbmRpY2VzID0ge3JvdW5kKGluZGV4ICogKGxlbihuYW1lcykgLSAxKSAvIChtYXhpbXVtIC0gMSkpIGZvciBpbmRleCBpbiByYW5nZShtYXhpbXVtKX0KICAgICAgICBzZWxlY3RlZCA9IFtuYW1lc1tpbmRleF0gZm9yIGluZGV4IGluIHNvcnRlZChpbmRpY2VzKV0KICAgIG91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZm9yIGZpbGVfbmFtZSBpbiBzZWxlY3RlZDoKICAgICAgICBncmF5ID0gcmVhZF9ncmF5c2NhbGUoZGF0YXNldC50cmFpbl9pbWFnZV9kaXIgLyBmaWxlX25hbWUpCiAgICAgICAgb3V0cHV0X3BhdGggPSBvdXRwdXRfZGlyIC8gZiJ7UGF0aChmaWxlX25hbWUpLnN0ZW19X3ByZWRpY3Rpb24ucG5nIgogICAgICAgIHdyaXRlX3ByZWRpY3Rpb25fb3ZlcmxheShncmF5LCBwcmVkaWN0aW9uc1tmaWxlX25hbWVdLCBvdXRwdXRfcGF0aCkKICAgIHJldHVybiBzZWxlY3RlZAoKCmRlZiBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3Q6IFBhdGgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgaW5mbyA9IGdpdF9pbmZvKHJlcG9fcm9vdCkKICAgIGluZm9bInNvdXJjZV90cmVlX3NoYTI1NiJdID0gX3NvdXJjZV9maW5nZXJwcmludChyZXBvX3Jvb3QpCiAgICByZXR1cm4gaW5mbwoKCmRlZiBydW5fdmFsaWRhdGlvbigKICAgICosCiAgICByZXBvX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKICAgIG91dHB1dF9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUsCiAgICBsaW1pdF9vYnNlcnZhdGlvbnM6IGludCB8IE5vbmUgPSBOb25lLAogICAgb3ZlcmxheV9jb3VudDogaW50ID0gMTIsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBydW5fc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIHJlcG9fcm9vdCA9IFBhdGgocmVwb19yb290KS5yZXNvbHZlKCkKICAgIGthZ2dsZV9ydW50aW1lID0gX2lzX2thZ2dsZV9ydW50aW1lKCkKICAgIGRhdGFzZXRfcm9vdCA9IF9yZXNvbHZlX2RhdGFfcm9vdChkYXRhX3Jvb3QsIHJlcG9fcm9vdCkKICAgIGlmIG5vdCBrYWdnbGVfcnVudGltZSBhbmQgbm90IF93aXRoaW4oZGF0YXNldF9yb290LCByZXBvX3Jvb3QgLyAiZGF0YSIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiTG9jYWwgY29tcGV0aXRpb24gZGF0YSBtdXN0IHN0YXkgdW5kZXIgZGF0YS86IHtkYXRhc2V0X3Jvb3R9IgogICAgICAgICkKCiAgICBpZiBvdXRwdXRfcm9vdCBpcyBOb25lOgogICAgICAgIG91dHB1dF9yb290ID0gKAogICAgICAgICAgICBQYXRoKCIva2FnZ2xlL3dvcmtpbmcvZmlsYW1lbnQtdmFsaWRhdGlvbi1iYXNlbGluZSIpCiAgICAgICAgICAgIGlmIGthZ2dsZV9ydW50aW1lCiAgICAgICAgICAgIGVsc2UgcmVwb19yb290IC8gImFydGlmYWN0cyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZSIKICAgICAgICApCiAgICBvdXRwdXRfcm9vdCA9IFBhdGgob3V0cHV0X3Jvb3QpCiAgICBpZiBub3Qgb3V0cHV0X3Jvb3QuaXNfYWJzb2x1dGUoKToKICAgICAgICBvdXRwdXRfcm9vdCA9IHJlcG9fcm9vdCAvIG91dHB1dF9yb290CiAgICBvdXRwdXRfcm9vdCA9IG91dHB1dF9yb290LnJlc29sdmUoKQogICAgaWYgbm90IGthZ2dsZV9ydW50aW1lIGFuZCBub3QgX3dpdGhpbihvdXRwdXRfcm9vdCwgcmVwb19yb290KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgcnVuIG91dHB1dCBtdXN0IHN0YXkgaW5zaWRlIHRoZSByZXBvc2l0b3J5OiB7b3V0cHV0X3Jvb3R9IikKICAgIG91dHB1dF9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBjb25maWdfanNvbiwgYmFzZWxpbmVfY29uZmlnID0gX2xvYWRfY29uZmlnKHJlcG9fcm9vdCkKICAgIGRhdGFzZXQgPSBsb2FkX2RhdGFzZXQoZGF0YXNldF9yb290KQogICAgaW5wdXRfaW52ZW50b3J5ID0gaW52ZW50b3J5KGRhdGFzZXQpCiAgICBpZiBpbnB1dF9pbnZlbnRvcnlbIm9ic2VydmF0aW9uX2NvdW50Il0gIT0gNzA3OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJFeHBlY3RlZCA3MDcgY29tcGV0aXRpb24gdHJhaW5pbmcgb2JzZXJ2YXRpb25zLCBmb3VuZCAiCiAgICAgICAgICAgIGYie2lucHV0X2ludmVudG9yeVsnb2JzZXJ2YXRpb25fY291bnQnXX0iCiAgICAgICAgKQogICAgaWYgaW5wdXRfaW52ZW50b3J5WyJhbm5vdGF0b3Jfdmlld19jb3VudCJdICE9IDExNTQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgIkV4cGVjdGVkIDEsMTU0IGFubm90YXRvciB2aWV3cywgZm91bmQgIgogICAgICAgICAgICBmIntpbnB1dF9pbnZlbnRvcnlbJ2Fubm90YXRvcl92aWV3X2NvdW50J119IgogICAgICAgICkKICAgIGlmIGlucHV0X2ludmVudG9yeVsiYW5ub3RhdGlvbl9jb3VudCJdICE9IDgxOTk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgZiJFeHBlY3RlZCA4LDE5OSB0cmFpbmluZyBhbm5vdGF0aW9ucywgZm91bmQge2lucHV0X2ludmVudG9yeVsnYW5ub3RhdGlvbl9jb3VudCddfSIKICAgICAgICApCgogICAgc3BsaXQgPSBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICByZXBvX3Jvb3QgLyAiZGF0YSIgLyAic3BsaXRzIiwKICAgICAgICBzZWVkPWludChjb25maWdfanNvblsic2VlZCJdKSwKICAgICAgICB2YWxpZGF0aW9uX2ZyYWN0aW9uPWZsb2F0KGNvbmZpZ19qc29uWyJ2YWxpZGF0aW9uX2ZyYWN0aW9uIl0pLAogICAgKQogICAgZXhwZWN0ZWRfc3BsaXRfc2hhMjU2ID0gY29uZmlnX2pzb24uZ2V0KCJleHBlY3RlZF9zcGxpdF9zaGEyNTYiKQogICAgaWYgZXhwZWN0ZWRfc3BsaXRfc2hhMjU2IGFuZCBzcGxpdC5zaGEyNTYgIT0gZXhwZWN0ZWRfc3BsaXRfc2hhMjU2OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgIk9ic2VydmF0aW9uIHNwbGl0IGRpZ2VzdCBkaWZmZXJzIGZyb20gdGhlIGZyb3plbiBiYXNlbGluZTogIgogICAgICAgICAgICBmImV4cGVjdGVkIHtleHBlY3RlZF9zcGxpdF9zaGEyNTZ9LCBnb3Qge3NwbGl0LnNoYTI1Nn0iCiAgICAgICAgKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIGZ1bGxfaG9sZG91dF9jb3VudCA9IGxlbih2YWxpZGF0aW9uX25hbWVzKQogICAgaWYgbGltaXRfb2JzZXJ2YXRpb25zIGlzIG5vdCBOb25lOgogICAgICAgIGlmIGxpbWl0X29ic2VydmF0aW9ucyA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxpbWl0X29ic2VydmF0aW9ucyBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICB2YWxpZGF0aW9uX25hbWVzID0gdmFsaWRhdGlvbl9uYW1lc1s6bGltaXRfb2JzZXJ2YXRpb25zXQoKICAgIGhhcmR3YXJlID0gaGFyZHdhcmVfaW5mbygpCiAgICBwcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RbUHJlZGljdGVkSW5zdGFuY2VdXSA9IHt9CiAgICBwcmVkaWN0aW9uX3N0YXJ0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIGluZGV4LCBmaWxlX25hbWUgaW4gZW51bWVyYXRlKHZhbGlkYXRpb25fbmFtZXMsIHN0YXJ0PTEpOgogICAgICAgIGltYWdlID0gcmVhZF9ncmF5c2NhbGUoZGF0YXNldC50cmFpbl9pbWFnZV9kaXIgLyBmaWxlX25hbWUpCiAgICAgICAgcHJlZGljdGlvbnNbZmlsZV9uYW1lXSA9IHByZWRpY3RfaW1hZ2UoaW1hZ2UsIGZpbGVfbmFtZSwgYmFzZWxpbmVfY29uZmlnKQogICAgICAgIGlmIGluZGV4ICUgMjAgPT0gMCBvciBpbmRleCA9PSBsZW4odmFsaWRhdGlvbl9uYW1lcyk6CiAgICAgICAgICAgIHByaW50KGYiUHJlZGljdGVkIHtpbmRleH0ve2xlbih2YWxpZGF0aW9uX25hbWVzKX0gdmFsaWRhdGlvbiBvYnNlcnZhdGlvbnMiKQogICAgcHJlZGljdGlvbl9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHByZWRpY3Rpb25fc3RhcnQKCiAgICBzY29yZXJfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBtZXRyaWNfcmVzdWx0LCBvYnNlcnZhdGlvbl9yb3dzID0gc2NvcmVfdmFsaWRhdGlvbigKICAgICAgICBkYXRhc2V0LAogICAgICAgIHByZWRpY3Rpb25zLAogICAgICAgIHZhbGlkYXRpb25fbmFtZXMsCiAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdChjb25maWdfanNvblsiaW91X3RocmVzaG9sZCJdKSwKICAgICAgICBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkPWZsb2F0KGNvbmZpZ19qc29uWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0pLAogICAgKQogICAgc2NvcmluZ19zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHNjb3Jlcl9zdGFydAoKICAgIF93cml0ZV9wcmVkaWN0aW9ucyhvdXRwdXRfcm9vdCAvICJwcmVkaWN0aW9ucy5jc3YiLCBwcmVkaWN0aW9ucykKICAgIF93cml0ZV9vYnNlcnZhdGlvbl9zY29yZXMoCiAgICAgICAgb3V0cHV0X3Jvb3QgLyAicGVyX29ic2VydmF0aW9uLmNzdiIsIG9ic2VydmF0aW9uX3Jvd3MsIHByZWRpY3Rpb25zCiAgICApCiAgICBvdmVybGF5X25hbWVzID0gX3dyaXRlX292ZXJsYXlzKAogICAgICAgIGRhdGFzZXQsIHByZWRpY3Rpb25zLCBvdXRwdXRfcm9vdCAvICJvdmVybGF5cyIsIG92ZXJsYXlfY291bnQKICAgICkKICAgIHNwbGl0X2JhbGFuY2U6IGRpY3Rbc3RyLCBkaWN0W3N0ciwgaW50XV0gPSB7InllYXIiOiB7fSwgInNpdGUiOiB7fX0KICAgIGZvciBmaWxlX25hbWUgaW4gc3BsaXQudmFsaWRhdGlvbl9uYW1lczoKICAgICAgICB2aWV3ID0gZGF0YXNldC52aWV3c19ieV9vYnNlcnZhdGlvbltmaWxlX25hbWVdWzBdCiAgICAgICAgc3BsaXRfYmFsYW5jZVsieWVhciJdW3N0cih2aWV3LnllYXIpXSA9ICgKICAgICAgICAgICAgc3BsaXRfYmFsYW5jZVsieWVhciJdLmdldChzdHIodmlldy55ZWFyKSwgMCkgKyAxCiAgICAgICAgKQogICAgICAgIHNwbGl0X2JhbGFuY2VbInNpdGUiXVt2aWV3LnNpdGVdID0gc3BsaXRfYmFsYW5jZVsic2l0ZSJdLmdldCh2aWV3LnNpdGUsIDApICsgMQoKICAgIHRvdGFsX3NlY29uZHMgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gcnVuX3N0YXJ0ZWQKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKCiAgICAgICAgcGVha19ncHVfYnl0ZXMgPSAoCiAgICAgICAgICAgIGludCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkpCiAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkKICAgICAgICAgICAgZWxzZSAwCiAgICAgICAgKQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHBlYWtfZ3B1X2J5dGVzID0gMAoKICAgIHJ1bl9yZXN1bHQ6IGRpY3Rbc3RyLCBBbnldID0gewogICAgICAgICJzdGF0dXMiOiAiY29tcGxldGUiIGlmIGxpbWl0X29ic2VydmF0aW9ucyBpcyBOb25lIGVsc2UgInNtb2tlX3N1YnNldCIsCiAgICAgICAgImNyZWF0ZWRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiAgICAgICAgInJlcG9zaXRvcnkiOiBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3QpLAogICAgICAgICJrYWdnbGVfbm90ZWJvb2tfdmVyc2lvbiI6IG9zLmVudmlyb24uZ2V0KAogICAgICAgICAgICAiS0FHR0xFX0tFUk5FTF9WRVJTSU9OIiwgb3MuZW52aXJvbi5nZXQoIkVRVUlORVhVU19OT1RFQk9PS19WRVJTSU9OIikKICAgICAgICApLAogICAgICAgICJkYXRhX3Jvb3QiOiBzdHIoZGF0YXNldC5yb290KSwKICAgICAgICAib3V0cHV0X3Jvb3QiOiBzdHIob3V0cHV0X3Jvb3QpLAogICAgICAgICJkYXRhc2V0X2ludmVudG9yeSI6IGlucHV0X2ludmVudG9yeSwKICAgICAgICAic3BsaXQiOiB7CiAgICAgICAgICAgICJtYW5pZmVzdCI6IHN0cihzcGxpdC5tYW5pZmVzdF9wYXRoKSwKICAgICAgICAgICAgInNoYTI1NiI6IHNwbGl0LnNoYTI1NiwKICAgICAgICAgICAgIm1hdGNoZXNfZXhwZWN0ZWRfc2hhMjU2IjogKAogICAgICAgICAgICAgICAgc3BsaXQuc2hhMjU2ID09IGV4cGVjdGVkX3NwbGl0X3NoYTI1NgogICAgICAgICAgICAgICAgaWYgZXhwZWN0ZWRfc3BsaXRfc2hhMjU2CiAgICAgICAgICAgICAgICBlbHNlIE5vbmUKICAgICAgICAgICAgKSwKICAgICAgICAgICAgInNlZWQiOiBpbnQoY29uZmlnX2pzb25bInNlZWQiXSksCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX2ZyYWN0aW9uIjogZmxvYXQoY29uZmlnX2pzb25bInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICAgICAgICAgICJzdHJhdGlmaWNhdGlvbiI6IGNvbmZpZ19qc29uWyJzdHJhdGlmeV9ieSJdLAogICAgICAgICAgICAidHJhaW5pbmdfb2JzZXJ2YXRpb25zIjogbGVuKHNwbGl0LnRyYWluaW5nX25hbWVzKSwKICAgICAgICAgICAgInZhbGlkYXRpb25fb2JzZXJ2YXRpb25zIjogZnVsbF9ob2xkb3V0X2NvdW50LAogICAgICAgICAgICAiZXZhbHVhdGVkX29ic2VydmF0aW9ucyI6IGxlbih2YWxpZGF0aW9uX25hbWVzKSwKICAgICAgICAgICAgInZhbGlkYXRpb25fYmFsYW5jZSI6IHNwbGl0X2JhbGFuY2UsCiAgICAgICAgfSwKICAgICAgICAiYmFzZWxpbmUiOiB7CiAgICAgICAgICAgICoqY29uZmlnX2pzb24sCiAgICAgICAgICAgICJuYW1lIjogImxvY2FsX2NvbnRyYXN0X290c3VfY29ubmVjdGVkX2NvbXBvbmVudHMiLAogICAgICAgICAgICAidHJhaW5lZCI6IEZhbHNlLAogICAgICAgICAgICAiZGV2aWNlX3VzZWQiOiAiY3B1IiwKICAgICAgICB9LAogICAgICAgICJzY29yZXIiOiB7CiAgICAgICAgICAgICJuYW1lIjogIm9yZ2FuaXplcl9zZWxmX2V2YWx1YXRpb25fbm90ZWJvb2tfc2VtYW50aWNzIiwKICAgICAgICAgICAgInJlZmVyZW5jZV91cmwiOiBPRkZJQ0lBTF9TQ09SRVJfVVJMLAogICAgICAgICAgICAiaW91X3RocmVzaG9sZF9pc19zdHJpY3QiOiBUcnVlLAogICAgICAgICAgICAib25lX3RvX29uZV9hc3NpZ25tZW50IjogRmFsc2UsCiAgICAgICAgICAgICJwcmVkaWN0aW9uc19yZXVzZWRfZm9yX2VhY2hfYW5ub3RhdG9yX3ZpZXciOiBUcnVlLAogICAgICAgIH0sCiAgICAgICAgInJ1bnRpbWUiOiB7CiAgICAgICAgICAgICoqaGFyZHdhcmUsCiAgICAgICAgICAgICJjdWRhX3VzZWRfYnlfYmFzZWxpbmUiOiBGYWxzZSwKICAgICAgICAgICAgInBlYWtfZ3B1X21lbW9yeV9ieXRlcyI6IHBlYWtfZ3B1X2J5dGVzLAogICAgICAgICAgICAicHJlZGljdGlvbl9zZWNvbmRzIjogcHJlZGljdGlvbl9zZWNvbmRzLAogICAgICAgICAgICAic2NvcmluZ19zZWNvbmRzIjogc2NvcmluZ19zZWNvbmRzLAogICAgICAgICAgICAidG90YWxfc2Vjb25kcyI6IHRvdGFsX3NlY29uZHMsCiAgICAgICAgICAgICJwYWNrYWdlX3ZlcnNpb25zIjogcGFja2FnZV92ZXJzaW9ucygpLAogICAgICAgIH0sCiAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNfcmVzdWx0LAogICAgICAgICJwcmVkaWN0aW9uX2luc3RhbmNlX2NvdW50Ijogc3VtKGxlbihpdGVtcykgZm9yIGl0ZW1zIGluIHByZWRpY3Rpb25zLnZhbHVlcygpKSwKICAgICAgICAib3ZlcmxheV9vYnNlcnZhdGlvbnMiOiBvdmVybGF5X25hbWVzLAogICAgICAgICJhcnRpZmFjdHMiOiB7CiAgICAgICAgICAgICJwcmVkaWN0aW9uc19jc3YiOiAicHJlZGljdGlvbnMuY3N2IiwKICAgICAgICAgICAgInBlcl9vYnNlcnZhdGlvbl9jc3YiOiAicGVyX29ic2VydmF0aW9uLmNzdiIsCiAgICAgICAgICAgICJvdmVybGF5c19kaXIiOiAib3ZlcmxheXMiLAogICAgICAgIH0sCiAgICB9CiAgICAob3V0cHV0X3Jvb3QgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKHJ1bl9yZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLAogICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICApCiAgICBwcmludCgKICAgICAgICAiVmFsaWRhdGlvbiBQUTogIgogICAgICAgIGYie21ldHJpY19yZXN1bHRbJ2dsb2JhbCddWydwcSddOi42Zn0gIgogICAgICAgIGYiKHttZXRyaWNfcmVzdWx0WydnbG9iYWwnXVsndHAnXX0gVFAgZWRnZXMsICIKICAgICAgICBmInttZXRyaWNfcmVzdWx0WydnbG9iYWwnXVsnZnAnXX0gRlAsIHttZXRyaWNfcmVzdWx0WydnbG9iYWwnXVsnZm4nXX0gRk4pIgogICAgKQogICAgcHJpbnQoZiJTcGxpdCBTSEEtMjU2OiB7c3BsaXQuc2hhMjU2fSIpCiAgICBwcmludChmIlJ1biBhcnRpZmFjdHM6IHtvdXRwdXRfcm9vdH0iKQogICAgcmV0dXJuIHJ1bl9yZXN1bHQKCgpkZWYgbWFpbihhcmd2OiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSkgLT4gaW50OgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlJ1biB0aGUgZml4ZWQgZ3JvdXBlZCB2YWxpZGF0aW9uIGJhc2VsaW5lLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlcG8tcm9vdCIsIGRlZmF1bHQ9c3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbGltaXQtb2JzZXJ2YXRpb25zIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3ZlcmxheS1jb3VudCIsIHR5cGU9aW50LCBkZWZhdWx0PTEyKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKGFyZ3YpCiAgICByZXN1bHQgPSBydW5fdmFsaWRhdGlvbigKICAgICAgICByZXBvX3Jvb3Q9YXJncy5yZXBvX3Jvb3QsCiAgICAgICAgZGF0YV9yb290PWFyZ3MuZGF0YV9yb290LAogICAgICAgIG91dHB1dF9yb290PWFyZ3Mub3V0cHV0X3Jvb3QsCiAgICAgICAgbGltaXRfb2JzZXJ2YXRpb25zPWFyZ3MubGltaXRfb2JzZXJ2YXRpb25zLAogICAgICAgIG92ZXJsYXlfY291bnQ9YXJncy5vdmVybGF5X2NvdW50LAogICAgKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXN1bHRbIm1ldHJpY3MiXVsiZ2xvYmFsIl0sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK\", \"src/filament_baseline/runtime.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKClRSQUNLRURfUlVOVElNRV9QQUNLQUdFUyA9ICgKICAgICJudW1weSIsCiAgICAib3BlbmN2LXB5dGhvbi1oZWFkbGVzcyIsCiAgICAicGFuZGFzIiwKICAgICJzY2lraXQtbGVhcm4iLAogICAgInNjaXB5IiwKICAgICJtYXRwbG90bGliIiwKICAgICJweWNvY290b29scyIsCiAgICAidG9yY2giLAopCgoKZGVmIHBhY2thZ2VfdmVyc2lvbnMoKSAtPiBkaWN0W3N0ciwgc3RyIHwgTm9uZV06CiAgICB2ZXJzaW9uczogZGljdFtzdHIsIHN0ciB8IE5vbmVdID0ge30KICAgIGZvciBwYWNrYWdlIGluIFRSQUNLRURfUlVOVElNRV9QQUNLQUdFUzoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHZlcnNpb25zW3BhY2thZ2VdID0gaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24ocGFja2FnZSkKICAgICAgICBleGNlcHQgaW1wb3J0bGliLm1ldGFkYXRhLlBhY2thZ2VOb3RGb3VuZEVycm9yOgogICAgICAgICAgICB2ZXJzaW9uc1twYWNrYWdlXSA9IE5vbmUKICAgIHJldHVybiB2ZXJzaW9ucwoKCmRlZiBoYXJkd2FyZV9pbmZvKCkgLT4gZGljdFtzdHIsIEFueV06CiAgICByZXN1bHQ6IGRpY3Rbc3RyLCBBbnldID0gewogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksCiAgICAgICAgInB5dGhvbiI6IHN5cy52ZXJzaW9uLnNwbGl0KClbMF0sCiAgICAgICAgImN1ZGFfYXZhaWxhYmxlIjogRmFsc2UsCiAgICAgICAgInRvcmNoX3ZlcnNpb24iOiBOb25lLAogICAgICAgICJ0b3JjaF9jdWRhX3ZlcnNpb24iOiBOb25lLAogICAgICAgICJncHVfZGV2aWNlcyI6IFtdLAogICAgICAgICJudmlkaWFfc21pX3N1bW1hcnkiOiBOb25lLAogICAgfQogICAgdHJ5OgogICAgICAgIHJlc3VsdFsibnZpZGlhX3NtaV9zdW1tYXJ5Il0gPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWwogICAgICAgICAgICAgICAgIm52aWRpYS1zbWkiLAogICAgICAgICAgICAgICAgIi0tcXVlcnktZ3B1PW5hbWUsbWVtb3J5LnRvdGFsLGRyaXZlcl92ZXJzaW9uIiwKICAgICAgICAgICAgICAgICItLWZvcm1hdD1jc3Ysbm9oZWFkZXIiLAogICAgICAgICAgICBdLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgKS5zdGRvdXQuc3RyaXAoKS5zcGxpdGxpbmVzKCkKICAgIGV4Y2VwdCAoT1NFcnJvciwgc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3IpOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICByZXR1cm4gcmVzdWx0CgogICAgcmVzdWx0WyJ0b3JjaF92ZXJzaW9uIl0gPSB0b3JjaC5fX3ZlcnNpb25fXwogICAgcmVzdWx0WyJ0b3JjaF9jdWRhX3ZlcnNpb24iXSA9IHRvcmNoLnZlcnNpb24uY3VkYQogICAgcmVzdWx0WyJjdWRhX2F2YWlsYWJsZSJdID0gYm9vbCh0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQogICAgaWYgcmVzdWx0WyJjdWRhX2F2YWlsYWJsZSJdOgogICAgICAgIGRldmljZXMgPSBbXQogICAgICAgIGZvciBpbmRleCBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKToKICAgICAgICAgICAgcHJvcHMgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhpbmRleCkKICAgICAgICAgICAgZGV2aWNlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImluZGV4IjogaW5kZXgsCiAgICAgICAgICAgICAgICAgICAgIm5hbWUiOiBwcm9wcy5uYW1lLAogICAgICAgICAgICAgICAgICAgICJ0b3RhbF9tZW1vcnlfYnl0ZXMiOiBpbnQocHJvcHMudG90YWxfbWVtb3J5KSwKICAgICAgICAgICAgICAgICAgICAiY29tcHV0ZV9jYXBhYmlsaXR5IjogZiJ7cHJvcHMubWFqb3J9Lntwcm9wcy5taW5vcn0iLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICAgICAgcmVzdWx0WyJncHVfZGV2aWNlcyJdID0gZGV2aWNlcwogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBnaXRfaW5mbyhyZXBvX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgcmVwb19yb290ID0gUGF0aChyZXBvX3Jvb3QpCiAgICB0cnk6CiAgICAgICAgY29tbWl0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgIFsiZ2l0IiwgIi1DIiwgc3RyKHJlcG9fcm9vdCksICJyZXYtcGFyc2UiLCAiSEVBRCJdLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgKS5zdGRvdXQuc3RyaXAoKQogICAgICAgIHN0YXR1cyA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICBbImdpdCIsICItQyIsIHN0cihyZXBvX3Jvb3QpLCAic3RhdHVzIiwgIi0tcG9yY2VsYWluIl0sCiAgICAgICAgICAgIGNoZWNrPVRydWUsCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICApLnN0ZG91dAogICAgICAgIHJldHVybiB7ImNvbW1pdCI6IGNvbW1pdCwgIndvcmt0cmVlX2RpcnR5IjogYm9vbChzdGF0dXMuc3RyaXAoKSl9CiAgICBleGNlcHQgKE9TRXJyb3IsIHN1YnByb2Nlc3MuQ2FsbGVkUHJvY2Vzc0Vycm9yKToKICAgICAgICByZXR1cm4geyJjb21taXQiOiBOb25lLCAid29ya3RyZWVfZGlydHkiOiBOb25lfQo=\", \"src/filament_baseline/scorer.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlciwgZGVmYXVsdGRpY3QKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBJdGVyYWJsZQoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gcHljb2NvdG9vbHMgaW1wb3J0IG1hc2sgYXMgbWFza191dGlscwoKZnJvbSAuZGF0YSBpbXBvcnQgQ29tcGV0aXRpb25EYXRhc2V0LCBJbWFnZVZpZXcKZnJvbSAucHJlZGljdG9yIGltcG9ydCBQcmVkaWN0ZWRJbnN0YW5jZQoKCkBkYXRhY2xhc3MKY2xhc3MgT2JzZXJ2YXRpb25Db3VudHM6CiAgICBmaWxlX25hbWU6IHN0cgogICAgeWVhcjogaW50CiAgICBzaXRlOiBzdHIKICAgIGFubm90YXRvcl92aWV3czogaW50ID0gMAogICAgZ3RfY291bnQ6IGludCA9IDAKICAgIHByZWRpY3Rpb25fY291bnQ6IGludCA9IDAKICAgIHRwX2lvdXM6IGxpc3RbZmxvYXRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBmcDogaW50ID0gMAogICAgZm46IGludCA9IDAKCiAgICBAcHJvcGVydHkKICAgIGRlZiB0cChzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLnRwX2lvdXMpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdHBfaW91X3N1bShzZWxmKSAtPiBmbG9hdDoKICAgICAgICByZXR1cm4gZmxvYXQoc3VtKHNlbGYudHBfaW91cykpCgogICAgQHByb3BlcnR5CiAgICBkZWYgcHEoc2VsZikgLT4gZmxvYXQ6CiAgICAgICAgcmV0dXJuIHBxX3ZhbHVlKHNlbGYudHBfaW91cywgc2VsZi5mcCwgc2VsZi5mbikKCgpAZGF0YWNsYXNzCmNsYXNzIFNjb3JlckRpYWdub3N0aWNzOgogICAgcG9zaXRpdmVfaW91czogbGlzdFtmbG9hdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHBvc2l0aXZlX2RpY2VzOiBsaXN0W2Zsb2F0XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgZ3RfbWF0Y2hfZGVncmVlczogQ291bnRlcltpbnRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PUNvdW50ZXIpCiAgICBwcmVkaWN0aW9uX21hdGNoX2RlZ3JlZXM6IENvdW50ZXJbaW50XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1Db3VudGVyKQoKCmRlZiBfY29jb19ybGUocmxlOiBkaWN0W3N0ciwgQW55XSkgLT4gZGljdFtzdHIsIEFueV06CiAgICBjb3VudHMgPSBybGVbImNvdW50cyJdCiAgICBpZiBpc2luc3RhbmNlKGNvdW50cywgc3RyKToKICAgICAgICBjb3VudHMgPSBjb3VudHMuZW5jb2RlKCJhc2NpaSIpCiAgICByZXR1cm4geyJzaXplIjogW2ludChybGVbInNpemUiXVswXSksIGludChybGVbInNpemUiXVsxXSldLCAiY291bnRzIjogY291bnRzfQoKCmRlZiBhbm5vdGF0aW9uX3RvX3JsZShhbm5vdGF0aW9uOiBkaWN0W3N0ciwgQW55XSwgaGVpZ2h0OiBpbnQsIHdpZHRoOiBpbnQpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgc2VnbWVudGF0aW9uID0gYW5ub3RhdGlvblsic2VnbWVudGF0aW9uIl0KICAgIGlmIGlzaW5zdGFuY2Uoc2VnbWVudGF0aW9uLCBkaWN0KToKICAgICAgICByZXR1cm4gX2NvY29fcmxlKHNlZ21lbnRhdGlvbikKICAgIGlmIG5vdCBpc2luc3RhbmNlKHNlZ21lbnRhdGlvbiwgbGlzdCkgb3Igbm90IHNlZ21lbnRhdGlvbjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgb3IgZW1wdHkgc2VnbWVudGF0aW9uIGZvciBhbm5vdGF0aW9uIHthbm5vdGF0aW9uLmdldCgnaWQnKX0iKQogICAgcmxlcyA9IG1hc2tfdXRpbHMuZnJQeU9iamVjdHMoc2VnbWVudGF0aW9uLCBoZWlnaHQsIHdpZHRoKQogICAgbWVyZ2VkID0gbWFza191dGlscy5tZXJnZShybGVzKSBpZiBpc2luc3RhbmNlKHJsZXMsIGxpc3QpIGVsc2UgcmxlcwogICAgcmV0dXJuIF9jb2NvX3JsZShtZXJnZWQpCgoKZGVmIHBhaXJ3aXNlX2lvdSgKICAgIGd0X3JsZXM6IGxpc3RbZGljdFtzdHIsIEFueV1dLAogICAgcHJlZGljdGlvbl9ybGVzOiBsaXN0W2RpY3Rbc3RyLCBBbnldXSwKKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJuIChuX2d0LCBuX3ByZWQpIGZsb2F0MzIgSW9VcywgbWF0Y2hpbmcgdGhlIG9yZ2FuaXplcidzIHRlbnNvciBkdHlwZS4iIiIKICAgIGlmIG5vdCBndF9ybGVzIG9yIG5vdCBwcmVkaWN0aW9uX3JsZXM6CiAgICAgICAgcmV0dXJuIG5wLnplcm9zKChsZW4oZ3RfcmxlcyksIGxlbihwcmVkaWN0aW9uX3JsZXMpKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICMgcHljb2NvdG9vbHMgcmV0dXJucyAoZGV0ZWN0aW9ucywgZ3JvdW5kIHRydXRoKSBpbiBmbG9hdDY0LiBUaGUgb3JnYW5pemVyCiAgICAjIGRlY29kZXMgdGhlIHNhbWUgUkxFcyB0byBmbG9hdDMyIGxheWVycyBhbmQgY29tcHV0ZXMgZmxvYXQzMiBJb1VzLgogICAgbWF0cml4ID0gbWFza191dGlscy5pb3UoCiAgICAgICAgW19jb2NvX3JsZShybGUpIGZvciBybGUgaW4gcHJlZGljdGlvbl9ybGVzXSwKICAgICAgICBbX2NvY29fcmxlKHJsZSkgZm9yIHJsZSBpbiBndF9ybGVzXSwKICAgICAgICBbMF0gKiBsZW4oZ3RfcmxlcyksCiAgICApCiAgICByZXR1cm4gbnAuYXNhcnJheShtYXRyaXgsIGR0eXBlPW5wLmZsb2F0MzIpLlQKCgpkZWYgcHFfdmFsdWUodHBfaW91czogSXRlcmFibGVbZmxvYXRdLCBmcDogaW50LCBmbjogaW50KSAtPiBmbG9hdDoKICAgIHRwX2lvdXMgPSBsaXN0KHRwX2lvdXMpCiAgICBkZW5vbWluYXRvciA9IGxlbih0cF9pb3VzKSArIDAuNSAqIGZwICsgMC41ICogZm4KICAgIHJldHVybiBmbG9hdChzdW0odHBfaW91cykgLyBkZW5vbWluYXRvcikgaWYgZGVub21pbmF0b3IgPiAwIGVsc2UgMC4wCgoKZGVmIF9hY2N1bXVsYXRlX3ZpZXcoCiAgICBjb3VudHM6IE9ic2VydmF0aW9uQ291bnRzLAogICAgZGlhZ25vc3RpY3M6IFNjb3JlckRpYWdub3N0aWNzLAogICAgZ3RfcmxlczogbGlzdFtkaWN0W3N0ciwgQW55XV0sCiAgICBwcmVkaWN0aW9uX3JsZXM6IGxpc3RbZGljdFtzdHIsIEFueV1dLAogICAgaW91X3RocmVzaG9sZDogZmxvYXQsCiAgICBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkOiBmbG9hdCwKKSAtPiBOb25lOgogICAgY291bnRzLmd0X2NvdW50ICs9IGxlbihndF9ybGVzKQogICAgY291bnRzLnByZWRpY3Rpb25fY291bnQgKz0gbGVuKHByZWRpY3Rpb25fcmxlcykKICAgIGlmIG5vdCBndF9ybGVzOgogICAgICAgIGNvdW50cy5mcCArPSBsZW4ocHJlZGljdGlvbl9ybGVzKQogICAgICAgIGRpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcy51cGRhdGUoezA6IGxlbihwcmVkaWN0aW9uX3JsZXMpfSkKICAgICAgICByZXR1cm4KICAgIGlmIG5vdCBwcmVkaWN0aW9uX3JsZXM6CiAgICAgICAgY291bnRzLmZuICs9IGxlbihndF9ybGVzKQogICAgICAgIGRpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMudXBkYXRlKHswOiBsZW4oZ3Rfcmxlcyl9KQogICAgICAgIHJldHVybgoKICAgIGlvdXMgPSBwYWlyd2lzZV9pb3UoZ3RfcmxlcywgcHJlZGljdGlvbl9ybGVzKQogICAgaGl0cyA9IGlvdXMgPiBpb3VfdGhyZXNob2xkCgogICAgIyBUaGlzIGludGVudGlvbmFsbHkgbWlycm9ycyB0aGUgcHJvdmlkZWQgc2NvcmVyOiBldmVyeSBxdWFsaWZ5aW5nIGVkZ2UgaXMKICAgICMgYSBUUDsgaXQgZG9lcyBub3QgcGVyZm9ybSBvbmUtdG8tb25lIGFzc2lnbm1lbnQuCiAgICBjb3VudHMudHBfaW91cy5leHRlbmQoZmxvYXQodmFsdWUpIGZvciB2YWx1ZSBpbiBpb3VzW2hpdHNdKQogICAgY291bnRzLmZwICs9IGludChucC5jb3VudF9ub256ZXJvKG5wLnN1bShoaXRzLCBheGlzPTApID09IDApKQogICAgY291bnRzLmZuICs9IGludChucC5jb3VudF9ub256ZXJvKG5wLnN1bShoaXRzLCBheGlzPTEpID09IDApKQoKICAgIHBvc2l0aXZlID0gaW91c1tpb3VzID4gMF0KICAgIGRpYWdub3N0aWNzLnBvc2l0aXZlX2lvdXMuZXh0ZW5kKGZsb2F0KHZhbHVlKSBmb3IgdmFsdWUgaW4gcG9zaXRpdmUpCiAgICBkaWFnbm9zdGljcy5wb3NpdGl2ZV9kaWNlcy5leHRlbmQoCiAgICAgICAgZmxvYXQodmFsdWUpIGZvciB2YWx1ZSBpbiAoMi4wICogcG9zaXRpdmUgLyAoMS4wICsgcG9zaXRpdmUpKQogICAgKQoKICAgIGFueV9vdmVybGFwID0gaW91cyA+IGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQKICAgIGRpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMudXBkYXRlKGludCh2YWx1ZSkgZm9yIHZhbHVlIGluIG5wLnN1bShhbnlfb3ZlcmxhcCwgYXhpcz0xKSkKICAgIGRpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcy51cGRhdGUoCiAgICAgICAgaW50KHZhbHVlKSBmb3IgdmFsdWUgaW4gbnAuc3VtKGFueV9vdmVybGFwLCBheGlzPTApCiAgICApCgoKZGVmIHNjb3JlX29ic2VydmF0aW9uKAogICAgZGF0YXNldDogQ29tcGV0aXRpb25EYXRhc2V0LAogICAgZmlsZV9uYW1lOiBzdHIsCiAgICBwcmVkaWN0aW9uczogbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV0sCiAgICAqLAogICAgaW91X3RocmVzaG9sZDogZmxvYXQgPSAwLjUsCiAgICBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkOiBmbG9hdCA9IDAuMCwKKSAtPiB0dXBsZVtPYnNlcnZhdGlvbkNvdW50cywgU2NvcmVyRGlhZ25vc3RpY3NdOgogICAgdmlld3MgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV0KICAgIGNvdW50cyA9IE9ic2VydmF0aW9uQ291bnRzKAogICAgICAgIGZpbGVfbmFtZT1maWxlX25hbWUsCiAgICAgICAgeWVhcj12aWV3c1swXS55ZWFyLAogICAgICAgIHNpdGU9dmlld3NbMF0uc2l0ZSwKICAgICAgICBhbm5vdGF0b3Jfdmlld3M9bGVuKHZpZXdzKSwKICAgICkKICAgIGRpYWdub3N0aWNzID0gU2NvcmVyRGlhZ25vc3RpY3MoKQogICAgcHJlZGljdGlvbl9ybGVzID0gW3ByZWRpY3Rpb24ucmxlIGZvciBwcmVkaWN0aW9uIGluIHByZWRpY3Rpb25zXQogICAgZm9yIHZpZXcgaW4gdmlld3M6CiAgICAgICAgYW5ub3RhdGlvbnMgPSBkYXRhc2V0LmFubm90YXRpb25zX2J5X2ltYWdlX2lkW3ZpZXcuaW1hZ2VfaWRdCiAgICAgICAgZ3RfcmxlcyA9IFsKICAgICAgICAgICAgYW5ub3RhdGlvbl90b19ybGUoYW5ub3RhdGlvbiwgdmlldy5oZWlnaHQsIHZpZXcud2lkdGgpCiAgICAgICAgICAgIGZvciBhbm5vdGF0aW9uIGluIGFubm90YXRpb25zCiAgICAgICAgXQogICAgICAgIF9hY2N1bXVsYXRlX3ZpZXcoCiAgICAgICAgICAgIGNvdW50cywKICAgICAgICAgICAgZGlhZ25vc3RpY3MsCiAgICAgICAgICAgIGd0X3JsZXMsCiAgICAgICAgICAgIHByZWRpY3Rpb25fcmxlcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZCwKICAgICAgICAgICAgZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZCwKICAgICAgICApCiAgICByZXR1cm4gY291bnRzLCBkaWFnbm9zdGljcwoKCmRlZiBfYWdncmVnYXRlX2NvdW50cyhyb3dzOiBJdGVyYWJsZVtPYnNlcnZhdGlvbkNvdW50c10pIC0+IGRpY3Rbc3RyLCBmbG9hdCB8IGludF06CiAgICB0cF9pb3VzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICBmcCA9IDAKICAgIGZuID0gMAogICAgZ3RfY291bnQgPSAwCiAgICBwcmVkaWN0aW9uX2NvdW50ID0gMAogICAgYW5ub3RhdG9yX3ZpZXdzID0gMAogICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgIHRwX2lvdXMuZXh0ZW5kKHJvdy50cF9pb3VzKQogICAgICAgIGZwICs9IHJvdy5mcAogICAgICAgIGZuICs9IHJvdy5mbgogICAgICAgIGd0X2NvdW50ICs9IHJvdy5ndF9jb3VudAogICAgICAgIHByZWRpY3Rpb25fY291bnQgKz0gcm93LnByZWRpY3Rpb25fY291bnQKICAgICAgICBhbm5vdGF0b3Jfdmlld3MgKz0gcm93LmFubm90YXRvcl92aWV3cwogICAgcmV0dXJuIHsKICAgICAgICAicHEiOiBwcV92YWx1ZSh0cF9pb3VzLCBmcCwgZm4pLAogICAgICAgICJ0cCI6IGxlbih0cF9pb3VzKSwKICAgICAgICAidHBfaW91X3N1bSI6IGZsb2F0KHN1bSh0cF9pb3VzKSksCiAgICAgICAgImZwIjogZnAsCiAgICAgICAgImZuIjogZm4sCiAgICAgICAgImd0X2NvdW50X2Fjcm9zc19hbm5vdGF0b3Jfdmlld3MiOiBndF9jb3VudCwKICAgICAgICAicHJlZGljdGlvbl9jb3VudF9hY3Jvc3NfYW5ub3RhdG9yX3ZpZXdzIjogcHJlZGljdGlvbl9jb3VudCwKICAgICAgICAiYW5ub3RhdG9yX3ZpZXdfY291bnQiOiBhbm5vdGF0b3Jfdmlld3MsCiAgICB9CgoKZGVmIF9oaXN0b2dyYW0odmFsdWVzOiBsaXN0W2Zsb2F0XSkgLT4gZGljdFtzdHIsIEFueV06CiAgICBiaW5zID0gbnAubGluc3BhY2UoMC4wLCAxLjAsIDUxLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgY291bnRzLCBlZGdlcyA9IG5wLmhpc3RvZ3JhbSh2YWx1ZXMsIGJpbnM9YmlucykKICAgIHRvdGFsID0gaW50KGNvdW50cy5zdW0oKSkKICAgIHJldHVybiB7CiAgICAgICAgImNvdW50IjogaW50KGxlbih2YWx1ZXMpKSwKICAgICAgICAibWVhbiI6IGZsb2F0KG5wLm1lYW4odmFsdWVzKSkgaWYgdmFsdWVzIGVsc2UgMC4wLAogICAgICAgICJiaW5zIjogW2Zsb2F0KHZhbHVlKSBmb3IgdmFsdWUgaW4gZWRnZXNdLAogICAgICAgICJjb3VudHMiOiBbaW50KHZhbHVlKSBmb3IgdmFsdWUgaW4gY291bnRzXSwKICAgICAgICAicHJvYmFiaWxpdGllcyI6IFtmbG9hdCh2YWx1ZSAvIHRvdGFsKSBmb3IgdmFsdWUgaW4gY291bnRzXSBpZiB0b3RhbCBlbHNlIFswLjBdICogNTAsCiAgICB9CgoKZGVmIF9kZWdyZWVfZGlzdHJpYnV0aW9uKGNvdW50ZXI6IENvdW50ZXJbaW50XSkgLT4gZGljdFtzdHIsIGludF06CiAgICByZXR1cm4ge3N0cihkZWdyZWUpOiBpbnQoY291bnRlcltkZWdyZWVdKSBmb3IgZGVncmVlIGluIHNvcnRlZChjb3VudGVyKX0KCgpkZWYgc2NvcmVfdmFsaWRhdGlvbigKICAgIGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwKICAgIHByZWRpY3Rpb25zX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV1dLAogICAgdmFsaWRhdGlvbl9uYW1lczogSXRlcmFibGVbc3RyXSwKICAgICosCiAgICBpb3VfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwKICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ6IGZsb2F0ID0gMC4wLAopIC0+IHR1cGxlW2RpY3Rbc3RyLCBBbnldLCBsaXN0W09ic2VydmF0aW9uQ291bnRzXV06CiAgICByb3dzOiBsaXN0W09ic2VydmF0aW9uQ291bnRzXSA9IFtdCiAgICBhbGxfZGlhZ25vc3RpY3MgPSBTY29yZXJEaWFnbm9zdGljcygpCiAgICBmb3IgZmlsZV9uYW1lIGluIHNvcnRlZCh2YWxpZGF0aW9uX25hbWVzKToKICAgICAgICBjb3VudHMsIGRpYWdub3N0aWNzID0gc2NvcmVfb2JzZXJ2YXRpb24oCiAgICAgICAgICAgIGRhdGFzZXQsCiAgICAgICAgICAgIGZpbGVfbmFtZSwKICAgICAgICAgICAgcHJlZGljdGlvbnNfYnlfb2JzZXJ2YXRpb24uZ2V0KGZpbGVfbmFtZSwgW10pLAogICAgICAgICAgICBpb3VfdGhyZXNob2xkPWlvdV90aHJlc2hvbGQsCiAgICAgICAgICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ9ZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZCwKICAgICAgICApCiAgICAgICAgcm93cy5hcHBlbmQoY291bnRzKQogICAgICAgIGFsbF9kaWFnbm9zdGljcy5wb3NpdGl2ZV9pb3VzLmV4dGVuZChkaWFnbm9zdGljcy5wb3NpdGl2ZV9pb3VzKQogICAgICAgIGFsbF9kaWFnbm9zdGljcy5wb3NpdGl2ZV9kaWNlcy5leHRlbmQoZGlhZ25vc3RpY3MucG9zaXRpdmVfZGljZXMpCiAgICAgICAgYWxsX2RpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMudXBkYXRlKGRpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMpCiAgICAgICAgYWxsX2RpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcy51cGRhdGUoCiAgICAgICAgICAgIGRpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcwogICAgICAgICkKCiAgICBncm91cGVkOiBkaWN0W3N0ciwgZGljdFtzdHIsIGxpc3RbT2JzZXJ2YXRpb25Db3VudHNdXV0gPSB7CiAgICAgICAgInllYXIiOiBkZWZhdWx0ZGljdChsaXN0KSwKICAgICAgICAic2l0ZSI6IGRlZmF1bHRkaWN0KGxpc3QpLAogICAgfQogICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgIGdyb3VwZWRbInllYXIiXVtzdHIocm93LnllYXIpXS5hcHBlbmQocm93KQogICAgICAgIGdyb3VwZWRbInNpdGUiXVtyb3cuc2l0ZV0uYXBwZW5kKHJvdykKCiAgICByZXN1bHQgPSB7CiAgICAgICAgImdsb2JhbCI6IF9hZ2dyZWdhdGVfY291bnRzKHJvd3MpLAogICAgICAgICJieV95ZWFyIjogewogICAgICAgICAgICBsYWJlbDogX2FnZ3JlZ2F0ZV9jb3VudHMoZ3JvdXBfcm93cykKICAgICAgICAgICAgZm9yIGxhYmVsLCBncm91cF9yb3dzIGluIHNvcnRlZChncm91cGVkWyJ5ZWFyIl0uaXRlbXMoKSkKICAgICAgICB9LAogICAgICAgICJieV9zaXRlIjogewogICAgICAgICAgICBsYWJlbDogX2FnZ3JlZ2F0ZV9jb3VudHMoZ3JvdXBfcm93cykKICAgICAgICAgICAgZm9yIGxhYmVsLCBncm91cF9yb3dzIGluIHNvcnRlZChncm91cGVkWyJzaXRlIl0uaXRlbXMoKSkKICAgICAgICB9LAogICAgICAgICJpb3VfcG9zaXRpdmVfcGFpcnMiOiBfaGlzdG9ncmFtKGFsbF9kaWFnbm9zdGljcy5wb3NpdGl2ZV9pb3VzKSwKICAgICAgICAiZGljZV9wb3NpdGl2ZV9wYWlycyI6IF9oaXN0b2dyYW0oYWxsX2RpYWdub3N0aWNzLnBvc2l0aXZlX2RpY2VzKSwKICAgICAgICAib25lX3RvX21hbnlfbWFueV90b19vbmUiOiB7CiAgICAgICAgICAgICJ0aHJlc2hvbGQiOiBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkLAogICAgICAgICAgICAiZ3RfbWF0Y2hfZGVncmVlX2NvdW50cyI6IF9kZWdyZWVfZGlzdHJpYnV0aW9uKAogICAgICAgICAgICAgICAgYWxsX2RpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMKICAgICAgICAgICAgKSwKICAgICAgICAgICAgInByZWRpY3Rpb25fbWF0Y2hfZGVncmVlX2NvdW50cyI6IF9kZWdyZWVfZGlzdHJpYnV0aW9uKAogICAgICAgICAgICAgICAgYWxsX2RpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcwogICAgICAgICAgICApLAogICAgICAgIH0sCiAgICB9CiAgICByZXR1cm4gcmVzdWx0LCByb3dzCgoKZGVmIHZpZXdzX2Zvcl9vYnNlcnZhdGlvbihkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsIGZpbGVfbmFtZTogc3RyKSAtPiB0dXBsZVtJbWFnZVZpZXcsIC4uLl06CiAgICByZXR1cm4gZGF0YXNldC52aWV3c19ieV9vYnNlcnZhdGlvbltmaWxlX25hbWVdCg==\", \"src/filament_baseline/solar_ssl.py\": \"...\", \"src/filament_baseline/split.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBTdHJhdGlmaWVkU2h1ZmZsZVNwbGl0Cgpmcm9tIC5kYXRhIGltcG9ydCBDb21wZXRpdGlvbkRhdGFzZXQKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBPYnNlcnZhdGlvblNwbGl0OgogICAgYXNzaWdubWVudHM6IGRpY3Rbc3RyLCBzdHJdCiAgICBtYW5pZmVzdF9wYXRoOiBQYXRoCiAgICBzaGEyNTY6IHN0cgoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHZhbGlkYXRpb25fbmFtZXMoc2VsZikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgICAgIHJldHVybiB0dXBsZSgKICAgICAgICAgICAgc29ydGVkKG5hbWUgZm9yIG5hbWUsIHBhcnRpdGlvbiBpbiBzZWxmLmFzc2lnbm1lbnRzLml0ZW1zKCkgaWYgcGFydGl0aW9uID09ICJ2YWxpZGF0aW9uIikKICAgICAgICApCgogICAgQHByb3BlcnR5CiAgICBkZWYgdHJhaW5pbmdfbmFtZXMoc2VsZikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgICAgIHJldHVybiB0dXBsZSgKICAgICAgICAgICAgc29ydGVkKG5hbWUgZm9yIG5hbWUsIHBhcnRpdGlvbiBpbiBzZWxmLmFzc2lnbm1lbnRzLml0ZW1zKCkgaWYgcGFydGl0aW9uID09ICJ0cmFpbiIpCiAgICAgICAgKQoKCmRlZiBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICBkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsCiAgICBzcGxpdF9kaXI6IHN0ciB8IFBhdGgsCiAgICAqLAogICAgc2VlZDogaW50ID0gNDIsCiAgICB2YWxpZGF0aW9uX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMiwKKSAtPiBPYnNlcnZhdGlvblNwbGl0OgogICAgbmFtZXMgPSBsaXN0KGRhdGFzZXQub2JzZXJ2YXRpb25fbmFtZXMpCiAgICB5ZWFycyA9IFtkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdLnllYXIgZm9yIG5hbWUgaW4gbmFtZXNdCiAgICBzcGxpdHRlciA9IFN0cmF0aWZpZWRTaHVmZmxlU3BsaXQoCiAgICAgICAgbl9zcGxpdHM9MSwgdGVzdF9zaXplPXZhbGlkYXRpb25fZnJhY3Rpb24sIHJhbmRvbV9zdGF0ZT1zZWVkCiAgICApCiAgICB0cmFpbl9pbmRpY2VzLCB2YWxpZGF0aW9uX2luZGljZXMgPSBuZXh0KHNwbGl0dGVyLnNwbGl0KG5hbWVzLCB5ZWFycykpCiAgICBhc3NpZ25tZW50cyA9IHtuYW1lOiAidHJhaW4iIGZvciBuYW1lIGluIG5hbWVzfQogICAgZm9yIGluZGV4IGluIHZhbGlkYXRpb25faW5kaWNlczoKICAgICAgICBhc3NpZ25tZW50c1tuYW1lc1tpbnQoaW5kZXgpXV0gPSAidmFsaWRhdGlvbiIKCiAgICByb3dzID0gW10KICAgIGZvciBuYW1lIGluIG5hbWVzOgogICAgICAgIHZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJmaWxlX25hbWUiOiBuYW1lLAogICAgICAgICAgICAgICAgInllYXIiOiB2aWV3LnllYXIsCiAgICAgICAgICAgICAgICAic2l0ZSI6IHZpZXcuc2l0ZSwKICAgICAgICAgICAgICAgICJwYXJ0aXRpb24iOiBhc3NpZ25tZW50c1tuYW1lXSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICBzcGxpdF9kaXIgPSBQYXRoKHNwbGl0X2RpcikKICAgIHNwbGl0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBtYW5pZmVzdF9wYXRoID0gc3BsaXRfZGlyIC8gZiJvYnNlcnZhdGlvbl9zcGxpdF9zZWVke3NlZWR9LmNzdiIKICAgIGZyb20gaW8gaW1wb3J0IFN0cmluZ0lPCgogICAgYnVmZmVyID0gU3RyaW5nSU8obmV3bGluZT0iIikKICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKAogICAgICAgIGJ1ZmZlciwKICAgICAgICBmaWVsZG5hbWVzPSgiZmlsZV9uYW1lIiwgInllYXIiLCAic2l0ZSIsICJwYXJ0aXRpb24iKSwKICAgICAgICBsaW5ldGVybWluYXRvcj0iXG4iLAogICAgKQogICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgIHdyaXRlci53cml0ZXJvd3Mocm93cykKICAgIG1hbmlmZXN0X2J5dGVzID0gYnVmZmVyLmdldHZhbHVlKCkuZW5jb2RlKCJ1dGYtOCIpCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihtYW5pZmVzdF9ieXRlcykuaGV4ZGlnZXN0KCkKCiAgICBpZiBtYW5pZmVzdF9wYXRoLmV4aXN0cygpOgogICAgICAgIGV4aXN0aW5nID0gbWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkKICAgICAgICBpZiBleGlzdGluZyAhPSBtYW5pZmVzdF9ieXRlczoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgZiJFeGlzdGluZyBzcGxpdCBtYW5pZmVzdCBkaWZmZXJzIGZyb20gdGhlIGRldGVybWluaXN0aWMgc3BsaXQ6IHttYW5pZmVzdF9wYXRofSIKICAgICAgICAgICAgKQogICAgZWxzZToKICAgICAgICBtYW5pZmVzdF9wYXRoLndyaXRlX2J5dGVzKG1hbmlmZXN0X2J5dGVzKQoKICAgIGlmIHNldChhc3NpZ25tZW50cy52YWx1ZXMoKSkgIT0geyJ0cmFpbiIsICJ2YWxpZGF0aW9uIn06CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIlRoZSBzcGxpdCBtdXN0IGNvbnRhaW4gYm90aCB0cmFpbmluZyBhbmQgdmFsaWRhdGlvbiBvYnNlcnZhdGlvbnMiKQogICAgaWYgc2V0KGFzc2lnbm1lbnRzKSAhPSBzZXQobmFtZXMpOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJUaGUgc3BsaXQgbWFuaWZlc3QgZG9lcyBub3QgY292ZXIgZXZlcnkgb3JpZ2luYWwgb2JzZXJ2YXRpb24iKQogICAgaWYgc2V0KGRhdGFzZXQub2JzZXJ2YXRpb25fbmFtZXMpICE9IHNldChhc3NpZ25tZW50cyk6CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIk9ic2VydmF0aW9uIHNwbGl0IGtleXMgZG8gbm90IG1hdGNoIENPQ08gZmlsZV9uYW1lIGdyb3VwcyIpCgogICAgcmV0dXJuIE9ic2VydmF0aW9uU3BsaXQoYXNzaWdubWVudHMsIG1hbmlmZXN0X3BhdGgsIGRpZ2VzdCkKCgpkZWYgbG9hZF9vYnNlcnZhdGlvbl9zcGxpdF9tYW5pZmVzdCgKICAgIGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwKICAgIG1hbmlmZXN0X3BhdGg6IHN0ciB8IFBhdGgsCiAgICAqLAogICAgYWxsb3dfc3Vic2V0OiBib29sID0gRmFsc2UsCikgLT4gT2JzZXJ2YXRpb25TcGxpdDoKICAgICIiIkxvYWQgYW4gZXhwbGljaXRseSBncm91cGVkIHRyYWluL3ZhbGlkYXRpb24gbWFuaWZlc3QgdW5kZXIgZGF0YS9zcGxpdHMuIiIiCiAgICBtYW5pZmVzdF9wYXRoID0gUGF0aChtYW5pZmVzdF9wYXRoKS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICBjb250ZW50ID0gbWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkKICAgIGFzc2lnbm1lbnRzOiBkaWN0W3N0ciwgc3RyXSA9IHt9CiAgICB3aXRoIG1hbmlmZXN0X3BhdGgub3BlbigiciIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICByZWFkZXIgPSBjc3YuRGljdFJlYWRlcihzdHJlYW0pCiAgICAgICAgaWYgcmVhZGVyLmZpZWxkbmFtZXMgIT0gWyJmaWxlX25hbWUiLCAieWVhciIsICJzaXRlIiwgInBhcnRpdGlvbiJdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTcGxpdCBtYW5pZmVzdCBtdXN0IGNvbnRhaW4gZmlsZV9uYW1lLCB5ZWFyLCBzaXRlLCBwYXJ0aXRpb24gY29sdW1ucyIpCiAgICAgICAgZm9yIHJvdyBpbiByZWFkZXI6CiAgICAgICAgICAgIG5hbWUgPSByb3dbImZpbGVfbmFtZSJdCiAgICAgICAgICAgIGlmIG5hbWUgaW4gYXNzaWdubWVudHM6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRHVwbGljYXRlIG9ic2VydmF0aW9uIGluIHNwbGl0IG1hbmlmZXN0OiB7bmFtZX0iKQogICAgICAgICAgICBpZiBuYW1lIG5vdCBpbiBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gb2JzZXJ2YXRpb24gaW4gc3BsaXQgbWFuaWZlc3Q6IHtuYW1lfSIpCiAgICAgICAgICAgIG9yaWdpbmFsX3ZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdCiAgICAgICAgICAgIGlmIGludChyb3dbInllYXIiXSkgIT0gb3JpZ2luYWxfdmlldy55ZWFyIG9yIHJvd1sic2l0ZSJdICE9IG9yaWdpbmFsX3ZpZXcuc2l0ZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJZZWFyL3NpdGUgbWV0YWRhdGEgY2hhbmdlZCBmb3Igb2JzZXJ2YXRpb24ge25hbWV9IikKICAgICAgICAgICAgaWYgcm93WyJwYXJ0aXRpb24iXSBub3QgaW4geyJ0cmFpbiIsICJ2YWxpZGF0aW9uIn06CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBwYXJ0aXRpb24gZm9yIG9ic2VydmF0aW9uIHtuYW1lfToge3Jvd1sncGFydGl0aW9uJ119IikKICAgICAgICAgICAgYXNzaWdubWVudHNbbmFtZV0gPSByb3dbInBhcnRpdGlvbiJdCgogICAgZGF0YXNldF9uYW1lcyA9IHNldChkYXRhc2V0Lm9ic2VydmF0aW9uX25hbWVzKQogICAgbWFuaWZlc3RfbmFtZXMgPSBzZXQoYXNzaWdubWVudHMpCiAgICBpZiBub3QgbWFuaWZlc3RfbmFtZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3BsaXQgbWFuaWZlc3QgY29udGFpbnMgbm8gb2JzZXJ2YXRpb25zIikKICAgIGlmIGFsbG93X3N1YnNldDoKICAgICAgICBpZiBub3QgbWFuaWZlc3RfbmFtZXMuaXNzdWJzZXQoZGF0YXNldF9uYW1lcyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNwbGl0IG1hbmlmZXN0IGNvbnRhaW5zIG9ic2VydmF0aW9ucyBvdXRzaWRlIHRoZSBjb21wZXRpdGlvbiBkYXRhc2V0IikKICAgIGVsaWYgbWFuaWZlc3RfbmFtZXMgIT0gZGF0YXNldF9uYW1lczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTcGxpdCBtYW5pZmVzdCBtdXN0IGNvdmVyIGV2ZXJ5IGNvbXBldGl0aW9uIHRyYWluaW5nIG9ic2VydmF0aW9uIikKICAgIGlmIHNldChhc3NpZ25tZW50cy52YWx1ZXMoKSkgIT0geyJ0cmFpbiIsICJ2YWxpZGF0aW9uIn06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3BsaXQgbWFuaWZlc3QgbXVzdCBjb250YWluIGJvdGggdHJhaW4gYW5kIHZhbGlkYXRpb24gb2JzZXJ2YXRpb25zIikKCiAgICByZXR1cm4gT2JzZXJ2YXRpb25TcGxpdCgKICAgICAgICBhc3NpZ25tZW50cz1hc3NpZ25tZW50cywKICAgICAgICBtYW5pZmVzdF9wYXRoPW1hbmlmZXN0X3BhdGgsCiAgICAgICAgc2hhMjU2PWhhc2hsaWIuc2hhMjU2KGNvbnRlbnQpLmhleGRpZ2VzdCgpLAogICAgKQo=\", \"src/filament_baseline/structure_sweep.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmRhdGEgaW1wb3J0IGxvYWRfZGF0YXNldApmcm9tIC5pbmZlcmVuY2UgaW1wb3J0IEluc3RhbmNlQ29uZmlnLCBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzCmZyb20gLnJ1bm5lciBpbXBvcnQgKAogICAgX3Jlc29sdmVfZGF0YV9yb290LAogICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcywKICAgIF93cml0ZV9vdmVybGF5cywKICAgIF93cml0ZV9wcmVkaWN0aW9ucywKKQpmcm9tIC5zY29yZXIgaW1wb3J0IHNjb3JlX3ZhbGlkYXRpb24KZnJvbSAuc3BsaXQgaW1wb3J0IGNyZWF0ZV9vYnNlcnZhdGlvbl9zcGxpdApmcm9tIC52aXN1YWxzIGltcG9ydCB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzCgoKZGVmIHJ1bl9pbnN0YW5jZV9zdHJ1Y3R1cmVfc3dlZXAoCiAgICAqLAogICAgcmVwb19yb290OiBzdHIgfCBQYXRoLAogICAgZGF0YV9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSwKICAgIHJ1bl9yb290OiBzdHIgfCBQYXRoLAogICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ6IGZsb2F0ID0gMC41NSwKICAgIGJvdW5kYXJ5X3RocmVzaG9sZDogZmxvYXQgPSAwLjYwLAogICAgYm91bmRhcnlfd2VpZ2h0OiBmbG9hdCA9IDEuNSwKICAgIGJvdW5kYXJ5X3dlaWdodHM6IHR1cGxlW2Zsb2F0LCAuLi5dIHwgTm9uZSA9IE5vbmUsCiAgICBzZWVkX3RocmVzaG9sZDogZmxvYXQgPSAwLjQ1LAopIC0+IGxpc3RbZGljdFtzdHIsIEFueV1dOgogICAgcmVwb19yb290ID0gUGF0aChyZXBvX3Jvb3QpLnJlc29sdmUoKQogICAgcnVuX3Jvb3QgPSBQYXRoKHJ1bl9yb290KS5yZXNvbHZlKCkKICAgIGRhdGFzZXQgPSBsb2FkX2RhdGFzZXQoX3Jlc29sdmVfZGF0YV9yb290KGRhdGFfcm9vdCwgcmVwb19yb290KSkKICAgIHZhbGlkYXRpb25fY29uZmlnID0ganNvbi5sb2FkcygKICAgICAgICAocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfdGV4dCgKICAgICAgICAgICAgZW5jb2Rpbmc9InV0Zi04IgogICAgICAgICkKICAgICkKICAgIHNwbGl0ID0gY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInNwbGl0cyIsCiAgICAgICAgc2VlZD1pbnQodmFsaWRhdGlvbl9jb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdCh2YWxpZGF0aW9uX2NvbmZpZ1sidmFsaWRhdGlvbl9mcmFjdGlvbiJdKSwKICAgICkKICAgIGlmIHNwbGl0LnNoYTI1NiAhPSB2YWxpZGF0aW9uX2NvbmZpZ1siZXhwZWN0ZWRfc3BsaXRfc2hhMjU2Il06CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGUgc3RydWN0dXJlIHN3ZWVwIHJlcXVpcmVzIHRoZSBmcm96ZW4gdmFsaWRhdGlvbiBzcGxpdCIpCgogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIHByb2JhYmlsaXR5X2RpciA9IHJ1bl9yb290IC8gInByb2JhYmlsaXRpZXMiCiAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgY2FjaGVfcGF0aCA9IHByb2JhYmlsaXR5X2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiCiAgICAgICAgaWYgbm90IGNhY2hlX3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk1pc3NpbmcgcHJvYmFiaWxpdHkgY2FjaGU6IHtjYWNoZV9wYXRofSIpCgogICAgd2VpZ2h0cyA9IGJvdW5kYXJ5X3dlaWdodHMgb3IgKGJvdW5kYXJ5X3dlaWdodCwpCiAgICBjb25maWd1cmF0aW9ucyA9IFsKICAgICAgICBJbnN0YW5jZUNvbmZpZygKICAgICAgICAgICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ9Zm9yZWdyb3VuZF90aHJlc2hvbGQsCiAgICAgICAgICAgIGJvdW5kYXJ5X3RocmVzaG9sZD1ib3VuZGFyeV90aHJlc2hvbGQsCiAgICAgICAgICAgIGJvdW5kYXJ5X3dlaWdodD13ZWlnaHQsCiAgICAgICAgICAgIHNlZWRfdGhyZXNob2xkPXNlZWRfdGhyZXNob2xkLAogICAgICAgICAgICBzZWVkX21pbl9hcmVhPXNlZWRfYXJlYSwKICAgICAgICAgICAgbWluX2luc3RhbmNlX2FyZWE9aW5zdGFuY2VfYXJlYSwKICAgICAgICApCiAgICAgICAgZm9yIHdlaWdodCwgKGluc3RhbmNlX2FyZWEsIHNlZWRfYXJlYSkgaW4gaXRlcnRvb2xzLnByb2R1Y3QoCiAgICAgICAgICAgIHdlaWdodHMsCiAgICAgICAgICAgIGl0ZXJ0b29scy5wcm9kdWN0KAogICAgICAgICAgICAgICAgKDUwLCAxMDAsIDIwMCwgMzAwLCA0MDAsIDUwMCwgNjAwLCA3MDAsIDgwMCwgMV8wMDAsIDFfMjAwKSwKICAgICAgICAgICAgICAgICg4LCAzMiwgMTI4KSwKICAgICAgICAgICAgKSwKICAgICAgICApCiAgICBdCiAgICByZXN1bHRzID0gW10KICAgIGJlc3RfcHEgPSAtMS4wCiAgICBmb3IgaW5kZXgsIGNhbmRpZGF0ZSBpbiBlbnVtZXJhdGUoY29uZmlndXJhdGlvbnMsIHN0YXJ0PTEpOgogICAgICAgIHByZWRpY3Rpb25zID0ge30KICAgICAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgICAgIHdpdGggbnAubG9hZChwcm9iYWJpbGl0eV9kaXIgLyBmIntQYXRoKGZpbGVfbmFtZSkuc3RlbX0ubnB6IikgYXMgY2FjaGU6CiAgICAgICAgICAgICAgICBwcmVkaWN0aW9uc1tmaWxlX25hbWVdID0gaW5zdGFuY2VzX2Zyb21fcHJvYmFiaWxpdGllcygKICAgICAgICAgICAgICAgICAgICBjYWNoZVsiZm9yZWdyb3VuZCJdLmFzdHlwZShucC5mbG9hdDMyKSwKICAgICAgICAgICAgICAgICAgICBjYWNoZVsiYm91bmRhcnkiXS5hc3R5cGUobnAuZmxvYXQzMiksCiAgICAgICAgICAgICAgICAgICAgZmlsZV9uYW1lLAogICAgICAgICAgICAgICAgICAgIGNhbmRpZGF0ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICBtZXRyaWNzLCBvYnNlcnZhdGlvbnMgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgICAgICBkYXRhc2V0LAogICAgICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdCh2YWxpZGF0aW9uX2NvbmZpZ1siaW91X3RocmVzaG9sZCJdKSwKICAgICAgICAgICAgZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZD1mbG9hdCgKICAgICAgICAgICAgICAgIHZhbGlkYXRpb25fY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0KICAgICAgICAgICAgKSwKICAgICAgICApCiAgICAgICAgZ2xvYmFsX21ldHJpY3MgPSBtZXRyaWNzWyJnbG9iYWwiXQogICAgICAgIHJvdyA9IHsKICAgICAgICAgICAgImNvbmZpZ3VyYXRpb24iOiBjYW5kaWRhdGUuX19kaWN0X18sCiAgICAgICAgICAgICJwcSI6IGdsb2JhbF9tZXRyaWNzWyJwcSJdLAogICAgICAgICAgICAidHAiOiBnbG9iYWxfbWV0cmljc1sidHAiXSwKICAgICAgICAgICAgImZwIjogZ2xvYmFsX21ldHJpY3NbImZwIl0sCiAgICAgICAgICAgICJmbiI6IGdsb2JhbF9tZXRyaWNzWyJmbiJdLAogICAgICAgICAgICAicHJlZGljdGlvbl9jb3VudF9hY3Jvc3NfYW5ub3RhdG9yX3ZpZXdzIjogZ2xvYmFsX21ldHJpY3NbCiAgICAgICAgICAgICAgICAicHJlZGljdGlvbl9jb3VudF9hY3Jvc3NfYW5ub3RhdG9yX3ZpZXdzIgogICAgICAgICAgICBdLAogICAgICAgICAgICAibWV0cmljcyI6IG1ldHJpY3MsCiAgICAgICAgfQogICAgICAgIHJlc3VsdHMuYXBwZW5kKHJvdykKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJbe2luZGV4fS97bGVuKGNvbmZpZ3VyYXRpb25zKX1dIG1pbl9hcmVhPXtjYW5kaWRhdGUubWluX2luc3RhbmNlX2FyZWF9ICIKICAgICAgICAgICAgZiJzZWVkX2FyZWE9e2NhbmRpZGF0ZS5zZWVkX21pbl9hcmVhfSBQUT17cm93WydwcSddOi42Zn0gIgogICAgICAgICAgICBmIlRQPXtyb3dbJ3RwJ119IEZQPXtyb3dbJ2ZwJ119IEZOPXtyb3dbJ2ZuJ119IiwKICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICApCiAgICAgICAgaWYgcm93WyJwcSJdID4gYmVzdF9wcToKICAgICAgICAgICAgYmVzdF9wcSA9IHJvd1sicHEiXQogICAgICAgICAgICBiZXN0X2RpciA9IHJ1bl9yb290IC8gImJlc3QtaW5zdGFuY2Utc3RydWN0dXJlIgogICAgICAgICAgICBiZXN0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIChiZXN0X2RpciAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICJzcGxpdF9zaGEyNTYiOiBzcGxpdC5zaGEyNTYsCiAgICAgICAgICAgICAgICAgICAgICAgICJjb25maWd1cmF0aW9uIjogY2FuZGlkYXRlLl9fZGljdF9fLAogICAgICAgICAgICAgICAgICAgICAgICAibWV0cmljcyI6IG1ldHJpY3MsCiAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICAgICAgICAgICAgICBzb3J0X2tleXM9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICsgIlxuIiwKICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgICAgICkKICAgICAgICAgICAgX3dyaXRlX3ByZWRpY3Rpb25zKGJlc3RfZGlyIC8gInByZWRpY3Rpb25zLmNzdiIsIHByZWRpY3Rpb25zKQogICAgICAgICAgICBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzKAogICAgICAgICAgICAgICAgYmVzdF9kaXIgLyAicGVyX29ic2VydmF0aW9uLmNzdiIsIG9ic2VydmF0aW9ucywgcHJlZGljdGlvbnMKICAgICAgICAgICAgKQogICAgICAgICAgICBfd3JpdGVfb3ZlcmxheXMoZGF0YXNldCwgcHJlZGljdGlvbnMsIGJlc3RfZGlyIC8gIm92ZXJsYXlzIiwgMTIpCiAgICAgICAgICAgIHdyaXRlX2NvbXBhcmlzb25fb3ZlcmxheXMoCiAgICAgICAgICAgICAgICBkYXRhc2V0LCBwcmVkaWN0aW9ucywgYmVzdF9kaXIgLyAiY29tcGFyaXNvbi1vdmVybGF5cyIsIDEyCiAgICAgICAgICAgICkKCiAgICByZXN1bHRzLnNvcnQoa2V5PWxhbWJkYSBpdGVtOiBpdGVtWyJwcSJdLCByZXZlcnNlPVRydWUpCiAgICAocnVuX3Jvb3QgLyAiaW5zdGFuY2Utc3RydWN0dXJlLXN3ZWVwLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVzdWx0cywgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHdpdGggKHJ1bl9yb290IC8gImluc3RhbmNlLXN0cnVjdHVyZS1zd2VlcC5jc3YiKS5vcGVuKAogICAgICAgICJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIgogICAgKSBhcyBzdHJlYW06CiAgICAgICAgZmllbGRzID0gKAogICAgICAgICAgICAqSW5zdGFuY2VDb25maWcoKS5fX2RpY3RfXy5rZXlzKCksCiAgICAgICAgICAgICJwcSIsCiAgICAgICAgICAgICJ0cCIsCiAgICAgICAgICAgICJmcCIsCiAgICAgICAgICAgICJmbiIsCiAgICAgICAgICAgICJwcmVkaWN0aW9uX2NvdW50X2Fjcm9zc19hbm5vdGF0b3Jfdmlld3MiLAogICAgICAgICkKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihzdHJlYW0sIGZpZWxkbmFtZXM9ZmllbGRzLCBsaW5ldGVybWluYXRvcj0iXG4iKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIHJvdyBpbiByZXN1bHRzOgogICAgICAgICAgICBjb25maWdfdmFsdWVzID0gcm93WyJjb25maWd1cmF0aW9uIl0KICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICoqY29uZmlnX3ZhbHVlcywKICAgICAgICAgICAgICAgICAgICAqKnsKICAgICAgICAgICAgICAgICAgICAgICAgZmllbGQ6IHJvd1tmaWVsZF0KICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGZpZWxkIGluIGZpZWxkc1tsZW4oY29uZmlnX3ZhbHVlcykgOl0KICAgICAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICByZXR1cm4gcmVzdWx0cwo=\", \"src/filament_baseline/submission.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIC5kYXRhIGltcG9ydCBJTUFHRV9TSVpFLCBpbnZlbnRvcnksIGxvYWRfZGF0YXNldApmcm9tIC5mZWF0dXJlcyBpbXBvcnQgSW5wdXRNb2RlCmZyb20gLmluZmVyZW5jZSBpbXBvcnQgSW5zdGFuY2VDb25maWcsIGluc3RhbmNlc19mcm9tX3Byb2JhYmlsaXRpZXMsIHByZWRpY3RfcHJvYmFiaWxpdGllcwpmcm9tIC5sZWFybmluZyBpbXBvcnQgRmlsYW1lbnRQYXRjaERhdGFzZXQsIFRhcmdldE1vZGUKZnJvbSAubW9kZWxfcnVubmVyIGltcG9ydCBfZmlsZV9zaGEyNTYsIF9pc19rYWdnbGVfcnVudGltZSwgX2xvYWRfbW9kZWwsIF93aXRoaW4KZnJvbSAubW9kZWwgaW1wb3J0IERJTk9fVjJfUkVQT1NJVE9SWQpmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IGRlY29kZV9ybGUsIHJlYWRfZ3JheXNjYWxlCmZyb20gLnJ1bm5lciBpbXBvcnQgX2dpdF9zdGF0dXMsIF9yZXNvbHZlX2RhdGFfcm9vdCwgX3dyaXRlX3ByZWRpY3Rpb25zCmZyb20gLnJ1bnRpbWUgaW1wb3J0IGhhcmR3YXJlX2luZm8sIHBhY2thZ2VfdmVyc2lvbnMKZnJvbSAudHJhaW5pbmcgaW1wb3J0IHRyYWluX21vZGVsCgoKZGVmIF9jaGVja19zdWJtaXNzaW9uKAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICBkYXRhc2V0X3Jvb3Q6IFBhdGgsCiAgICB0ZXN0X25hbWVzOiBsaXN0W3N0cl0sCiAgICBwcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RdLAopIC0+IGRpY3Rbc3RyLCBpbnRdOgogICAgZXhwZWN0ZWRfY29sdW1ucyA9IFsiZmlsYW1lbnRfaWQiLCAic2VnbWVudGF0aW9uX3JsZSJdCiAgICB3aXRoIG91dHB1dF9wYXRoLm9wZW4oInIiLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgcmVhZGVyID0gY3N2LkRpY3RSZWFkZXIoc3RyZWFtKQogICAgICAgIGlmIHJlYWRlci5maWVsZG5hbWVzICE9IGV4cGVjdGVkX2NvbHVtbnM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmV4cGVjdGVkIHN1Ym1pc3Npb24gY29sdW1uczoge3JlYWRlci5maWVsZG5hbWVzfSIpCiAgICAgICAgcm93cyA9IGxpc3QocmVhZGVyKQoKICAgIGV4cGVjdGVkX2lkcyA9IHsKICAgICAgICBpbnN0YW5jZS5maWxhbWVudF9pZAogICAgICAgIGZvciBmaWxlX25hbWUgaW4gdGVzdF9uYW1lcwogICAgICAgIGZvciBpbnN0YW5jZSBpbiBwcmVkaWN0aW9uc1tmaWxlX25hbWVdCiAgICB9CiAgICByb3dfaWRzID0gW3Jvd1siZmlsYW1lbnRfaWQiXSBmb3Igcm93IGluIHJvd3NdCiAgICBpZiBsZW4ocm93X2lkcykgIT0gbGVuKHNldChyb3dfaWRzKSkgb3Igc2V0KHJvd19pZHMpICE9IGV4cGVjdGVkX2lkczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTdWJtaXNzaW9uIGZpbGFtZW50IElEcyBhcmUgZHVwbGljYXRlZCBvciBkaWZmZXIgZnJvbSBwcmVkaWN0aW9ucyIpCiAgICBpZiBhbnkobm90IHJvd1sic2VnbWVudGF0aW9uX3JsZSJdIGZvciByb3cgaW4gcm93cyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3VibWlzc2lvbiBjb250YWlucyBhbiBlbXB0eSBSTEUgc3RyaW5nIikKCiAgICBpbnN0YW5jZV9jb3VudCA9IDAKICAgIGZvciBmaWxlX25hbWUgaW4gdGVzdF9uYW1lczoKICAgICAgICBpbWFnZSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXRfcm9vdCAvICJ0ZXN0IiAvICJ0ZXN0X2ltYWdlcyIgLyBmaWxlX25hbWUpCiAgICAgICAgaWYgaW1hZ2Uuc2hhcGUgIT0gKElNQUdFX1NJWkUsIElNQUdFX1NJWkUpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCB0ZXN0IGltYWdlIHNpemUgZm9yIHtmaWxlX25hbWV9OiB7aW1hZ2Uuc2hhcGV9IikKICAgICAgICBvY2N1cGllZCA9IG5wLnplcm9zKChJTUFHRV9TSVpFLCBJTUFHRV9TSVpFKSwgZHR5cGU9Ym9vbCkKICAgICAgICBzZWVuX2Zvcl9pbWFnZSA9IHNldCgpCiAgICAgICAgZm9yIGluc3RhbmNlIGluIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV06CiAgICAgICAgICAgIGlmIG5vdCBpbnN0YW5jZS5maWxhbWVudF9pZC5zdGFydHN3aXRoKGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfV8iKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJGaWxhbWVudCBJRCBkb2VzIG5vdCBiZWxvbmcgdG8ge2ZpbGVfbmFtZX0iKQogICAgICAgICAgICBpZiBpbnN0YW5jZS5maWxhbWVudF9pZCBpbiBzZWVuX2Zvcl9pbWFnZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJEdXBsaWNhdGUgZmlsYW1lbnQgSUQgaW4ge2ZpbGVfbmFtZX0iKQogICAgICAgICAgICBzZWVuX2Zvcl9pbWFnZS5hZGQoaW5zdGFuY2UuZmlsYW1lbnRfaWQpCiAgICAgICAgICAgIG1hc2sgPSBkZWNvZGVfcmxlKGluc3RhbmNlLnJsZSkKICAgICAgICAgICAgaWYgbWFzay5zaGFwZSAhPSAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBtYXNrIGRpbWVuc2lvbnMgZm9yIHtpbnN0YW5jZS5maWxhbWVudF9pZH0iKQogICAgICAgICAgICBpZiBpbnQobWFzay5zdW0oKSkgIT0gaW5zdGFuY2UuYXJlYToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJSTEUgYXJlYSBtaXNtYXRjaCBmb3Ige2luc3RhbmNlLmZpbGFtZW50X2lkfSIpCiAgICAgICAgICAgIGlmIG5wLmFueShvY2N1cGllZCAmIG1hc2suYXN0eXBlKGJvb2wpKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJQcmVkaWN0ZWQgaW5zdGFuY2VzIG92ZXJsYXAgaW4ge2ZpbGVfbmFtZX0iKQogICAgICAgICAgICBvY2N1cGllZCB8PSBtYXNrLmFzdHlwZShib29sKQogICAgICAgICAgICBpbnN0YW5jZV9jb3VudCArPSAxCgogICAgaWYgbGVuKHJvd3MpICE9IGluc3RhbmNlX2NvdW50OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlN1Ym1pc3Npb24gcm93IGNvdW50IGRvZXMgbm90IG1hdGNoIHZhbGlkYXRlZCB0ZXN0IHByZWRpY3Rpb25zIikKICAgIHJldHVybiB7InRlc3RfaW1hZ2VzIjogbGVuKHRlc3RfbmFtZXMpLCAicHJlZGljdGVkX2luc3RhbmNlcyI6IGluc3RhbmNlX2NvdW50fQoKCmRlZiBydW5fZmluYWxfc3VibWlzc2lvbigKICAgICosCiAgICByZXBvX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lLAogICAgcnVuX25hbWU6IHN0ciwKICAgIHRhcmdldF9tb2RlOiBUYXJnZXRNb2RlID0gImNvbnNlbnN1cyIsCiAgICBzdGVwczogaW50ID0gNV8wMDAsCiAgICBiYXRjaF9zaXplOiBpbnQgPSAyLAogICAgcGF0Y2hfc2l6ZTogaW50ID0gNTEyLAogICAgYmFzZV9jaGFubmVsczogaW50ID0gMTYsCiAgICBtb2RlbF90eXBlOiBzdHIgPSAidW5ldCIsCiAgICBwcmV0cmFpbmVkX2JhY2tib25lOiBib29sID0gRmFsc2UsCiAgICBodWJfcmVwb3NpdG9yeTogc3RyID0gRElOT19WMl9SRVBPU0lUT1JZLAogICAgdHRhOiBpbnQgPSAxLAogICAgaW5wdXRfbW9kZTogSW5wdXRNb2RlID0gInJhdyIsCiAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZDogZmxvYXQgPSAwLjQ1LAogICAgYm91bmRhcnlfdGhyZXNob2xkOiBmbG9hdCA9IDAuNjUsCiAgICBib3VuZGFyeV93ZWlnaHQ6IGZsb2F0ID0gMS41LAogICAgc2VlZF90aHJlc2hvbGQ6IGZsb2F0ID0gMC40NSwKICAgIHNlZWRfbWluX2FyZWE6IGludCA9IDgsCiAgICBtaW5faW5zdGFuY2VfYXJlYTogaW50ID0gNTAsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBpZiBub3QgcmUuZnVsbG1hdGNoKHIiW0EtWmEtejAtOV1bQS1aYS16MC05Ll8tXXswLDYzfSIsIHJ1bl9uYW1lKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJydW5fbmFtZSBtdXN0IHVzZSBvbmx5IGxldHRlcnMsIGRpZ2l0cywgZG90LCB1bmRlcnNjb3JlLCBvciBkYXNoIikKICAgIHJlcG9fcm9vdCA9IFBhdGgocmVwb19yb290KS5yZXNvbHZlKCkKICAgIGthZ2dsZV9ydW50aW1lID0gX2lzX2thZ2dsZV9ydW50aW1lKCkKICAgIGRhdGFzZXRfcm9vdCA9IF9yZXNvbHZlX2RhdGFfcm9vdChkYXRhX3Jvb3QsIHJlcG9fcm9vdCkKICAgIGlmIG5vdCBrYWdnbGVfcnVudGltZSBhbmQgbm90IF93aXRoaW4oZGF0YXNldF9yb290LCByZXBvX3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJMb2NhbCBjb21wZXRpdGlvbiBkYXRhIG11c3Qgc3RheSBpbiB0aGUgcmVwb3NpdG9yeToge2RhdGFzZXRfcm9vdH0iKQogICAgcnVuX3Jvb3QgPSAoCiAgICAgICAgUGF0aCgiL2thZ2dsZS93b3JraW5nIikgLyAiZXF1aW5leHVzIiAvIHJ1bl9uYW1lCiAgICAgICAgaWYga2FnZ2xlX3J1bnRpbWUKICAgICAgICBlbHNlIHJlcG9fcm9vdCAvICJhcnRpZmFjdHMiIC8gInN1Ym1pc3Npb25zIiAvIHJ1bl9uYW1lCiAgICApCiAgICBpZiBub3Qga2FnZ2xlX3J1bnRpbWUgYW5kIG5vdCBfd2l0aGluKHJ1bl9yb290LCByZXBvX3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJMb2NhbCBvdXRwdXRzIG11c3Qgc3RheSBpbiB0aGUgcmVwb3NpdG9yeToge3J1bl9yb290fSIpCiAgICBydW5fcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBodWJfZGlyID0gKAogICAgICAgIFBhdGgoIi9rYWdnbGUvd29ya2luZy9lcXVpbmV4dXMvcHJldHJhaW5lZC90b3JjaC9odWIiKQogICAgICAgIGlmIGthZ2dsZV9ydW50aW1lCiAgICAgICAgZWxzZSByZXBvX3Jvb3QgLyAiYXJ0aWZhY3RzIiAvICJwcmV0cmFpbmVkIiAvICJ0b3JjaCIgLyAiaHViIgogICAgKQogICAgcmVwb19pbmZvID0gX2dpdF9zdGF0dXMocmVwb19yb290KQoKICAgIGNvbmZpZ19qc29uID0ganNvbi5sb2FkcygKICAgICAgICAocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgKQogICAgY29uZmlnX3NoYTI1NiA9IGhhc2hsaWIuc2hhMjU2KAogICAgICAgIChyZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIikucmVhZF9ieXRlcygpCiAgICApLmhleGRpZ2VzdCgpCiAgICBkYXRhc2V0ID0gbG9hZF9kYXRhc2V0KGRhdGFzZXRfcm9vdCkKICAgIGRhdGFfaW52ZW50b3J5ID0gaW52ZW50b3J5KGRhdGFzZXQpCiAgICBpZiBkYXRhX2ludmVudG9yeVsib2JzZXJ2YXRpb25fY291bnQiXSAhPSA3MDc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRmluYWwgdHJhaW5pbmcgbXVzdCB1c2UgYWxsIDcwNyBvZmZpY2lhbCBjb21wZXRpdGlvbiBvYnNlcnZhdGlvbnMiKQogICAgaWYgZGF0YV9pbnZlbnRvcnlbImFubm90YXRvcl92aWV3X2NvdW50Il0gIT0gMV8xNTQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRmluYWwgdHJhaW5pbmcgbXVzdCB1c2UgYWxsIDEsMTU0IG9mZmljaWFsIGFubm90YXRvciB2aWV3cyIpCiAgICBpZiBkYXRhX2ludmVudG9yeVsiYW5ub3RhdGlvbl9jb3VudCJdICE9IDhfMTk5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuZXhwZWN0ZWQgbnVtYmVyIG9mIGNvbXBldGl0aW9uLXByb3ZpZGVkIHRyYWluaW5nIGFubm90YXRpb25zIikKICAgIGlmIGRhdGFfaW52ZW50b3J5WyJ0ZXN0X2ltYWdlX2ZpbGVfY291bnQiXSAhPSAxODA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgMTgwIHN1cHBsaWVkIEgtYWxwaGEgdGVzdCBpbWFnZXMiKQoKICAgIHRyYWluX25hbWVzID0gbGlzdChkYXRhc2V0Lm9ic2VydmF0aW9uX25hbWVzKQogICAgdHJhaW5fZGF0YXNldCA9IEZpbGFtZW50UGF0Y2hEYXRhc2V0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgdHJhaW5fbmFtZXMsCiAgICAgICAgdGFyZ2V0X21vZGU9dGFyZ2V0X21vZGUsCiAgICAgICAgcGF0Y2hfc2l6ZT1wYXRjaF9zaXplLAogICAgICAgIGF1Z21lbnQ9VHJ1ZSwKICAgICAgICBzZWVkPWludChjb25maWdfanNvblsic2VlZCJdKSwKICAgICAgICBpbnB1dF9tb2RlPWlucHV0X21vZGUsCiAgICApCiAgICBjaGVja3BvaW50X3BhdGggPSBydW5fcm9vdCAvICJmaW5hbF9tb2RlbC5wdCIKICAgIHN0YXJ0X3RpbWUgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICB0cmFpbmluZyA9IHRyYWluX21vZGVsKAogICAgICAgIHRyYWluX2RhdGFzZXQ9dHJhaW5fZGF0YXNldCwKICAgICAgICB2YWxpZGF0aW9uX2RhdGFzZXQ9Tm9uZSwKICAgICAgICBvdXRwdXRfcGF0aD1jaGVja3BvaW50X3BhdGgsCiAgICAgICAgc3RlcHM9c3RlcHMsCiAgICAgICAgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLAogICAgICAgIGJhc2VfY2hhbm5lbHM9YmFzZV9jaGFubmVscywKICAgICAgICBzZWVkPWludChjb25maWdfanNvblsic2VlZCJdKSwKICAgICAgICB2YWxpZGF0aW9uX2ludGVydmFsPW1heCgxMDAsIG1pbig1MDAsIHN0ZXBzIC8vIDUpKSwKICAgICAgICBtb2RlbF90eXBlPW1vZGVsX3R5cGUsCiAgICAgICAgcHJldHJhaW5lZF9iYWNrYm9uZT1wcmV0cmFpbmVkX2JhY2tib25lLAogICAgICAgIGh1Yl9yZXBvc2l0b3J5PWh1Yl9yZXBvc2l0b3J5LAogICAgICAgIGh1Yl9kaXI9aHViX2RpciwKICAgICkKCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGE6MCIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgbW9kZWwgPSBfbG9hZF9tb2RlbChjaGVja3BvaW50X3BhdGgsIGRldmljZSwgaHViX2Rpcj1odWJfZGlyKQogICAgcHJldHJhaW5lZF9jaGVja3BvaW50ID0gaHViX2RpciAvICJjaGVja3BvaW50cyIgLyAiZGlub3YyX3ZpdHMxNF9wcmV0cmFpbi5wdGgiCiAgICBwcmV0cmFpbmVkX2NoZWNrcG9pbnRfc2hhMjU2ID0gKAogICAgICAgIF9maWxlX3NoYTI1NihwcmV0cmFpbmVkX2NoZWNrcG9pbnQpCiAgICAgICAgaWYgbW9kZWxfdHlwZSA9PSAiZGlub3YyX3ZpdHMxNCIgYW5kIHByZXRyYWluZWRfYmFja2JvbmUKICAgICAgICBlbHNlIE5vbmUKICAgICkKICAgIGFkYXB0YXRpb25fbWV0YWRhdGEgPSBOb25lCiAgICBhZGFwdGF0aW9uX21ldGFkYXRhX3BhdGggPSBwcmV0cmFpbmVkX2NoZWNrcG9pbnQud2l0aF9zdWZmaXgoIi5qc29uIikKICAgIGlmICgKICAgICAgICBtb2RlbF90eXBlID09ICJkaW5vdjJfdml0czE0IgogICAgICAgIGFuZCBwcmV0cmFpbmVkX2JhY2tib25lCiAgICAgICAgYW5kIGFkYXB0YXRpb25fbWV0YWRhdGFfcGF0aC5pc19maWxlKCkKICAgICk6CiAgICAgICAgY2FuZGlkYXRlX2FkYXB0YXRpb24gPSBqc29uLmxvYWRzKGFkYXB0YXRpb25fbWV0YWRhdGFfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgaWYgY2FuZGlkYXRlX2FkYXB0YXRpb24uZ2V0KCJhZGFwdGVkX2NoZWNrcG9pbnRfc2hhMjU2IikgPT0gcHJldHJhaW5lZF9jaGVja3BvaW50X3NoYTI1NjoKICAgICAgICAgICAgYWRhcHRhdGlvbl9tZXRhZGF0YSA9IGNhbmRpZGF0ZV9hZGFwdGF0aW9uCiAgICBpbnN0YW5jZV9jb25maWcgPSBJbnN0YW5jZUNvbmZpZygKICAgICAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZD1mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV90aHJlc2hvbGQ9Ym91bmRhcnlfdGhyZXNob2xkLAogICAgICAgIGJvdW5kYXJ5X3dlaWdodD1ib3VuZGFyeV93ZWlnaHQsCiAgICAgICAgc2VlZF90aHJlc2hvbGQ9c2VlZF90aHJlc2hvbGQsCiAgICAgICAgc2VlZF9taW5fYXJlYT1zZWVkX21pbl9hcmVhLAogICAgICAgIG1pbl9pbnN0YW5jZV9hcmVhPW1pbl9pbnN0YW5jZV9hcmVhLAogICAgKQogICAgdGVzdF9uYW1lcyA9IHNvcnRlZCgKICAgICAgICBwYXRoLm5hbWUgZm9yIHBhdGggaW4gZGF0YXNldC50ZXN0X2ltYWdlX2Rpci5pdGVyZGlyKCkgaWYgcGF0aC5pc19maWxlKCkKICAgICkKICAgIHByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbGlzdF0gPSB7fQogICAgY292ZXJhZ2Vfcm93cyA9IFtdCiAgICBpbmZlcmVuY2Vfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBmb3IgaW5kZXgsIGZpbGVfbmFtZSBpbiBlbnVtZXJhdGUodGVzdF9uYW1lcywgc3RhcnQ9MSk6CiAgICAgICAgaW1hZ2UgPSByZWFkX2dyYXlzY2FsZShkYXRhc2V0LnRlc3RfaW1hZ2VfZGlyIC8gZmlsZV9uYW1lKQogICAgICAgIGZvcmVncm91bmQsIGJvdW5kYXJ5ID0gcHJlZGljdF9wcm9iYWJpbGl0aWVzKAogICAgICAgICAgICBtb2RlbCwKICAgICAgICAgICAgaW1hZ2UsCiAgICAgICAgICAgIGRldmljZT1kZXZpY2UsCiAgICAgICAgICAgIHBhdGNoX3NpemU9cGF0Y2hfc2l6ZSwKICAgICAgICAgICAgc3RyaWRlPW1heChwYXRjaF9zaXplIC8vIDIsIGludChwYXRjaF9zaXplICogMC43NSkpLAogICAgICAgICAgICB0dGE9dHRhLAogICAgICAgICkKICAgICAgICBwcmVkaWN0aW9uc1tmaWxlX25hbWVdID0gaW5zdGFuY2VzX2Zyb21fcHJvYmFiaWxpdGllcygKICAgICAgICAgICAgZm9yZWdyb3VuZCwgYm91bmRhcnksIGZpbGVfbmFtZSwgaW5zdGFuY2VfY29uZmlnCiAgICAgICAgKQogICAgICAgIGNvdmVyYWdlX3Jvd3MuYXBwZW5kKAogICAgICAgICAgICB7ImZpbGVfbmFtZSI6IGZpbGVfbmFtZSwgImluc3RhbmNlX2NvdW50IjogbGVuKHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0pfQogICAgICAgICkKICAgICAgICBpZiBpbmRleCAlIDEwID09IDAgb3IgaW5kZXggPT0gbGVuKHRlc3RfbmFtZXMpOgogICAgICAgICAgICBwcmludChmIlByZWRpY3RlZCB7aW5kZXh9L3tsZW4odGVzdF9uYW1lcyl9IHRlc3QgaW1hZ2VzIiwgZmx1c2g9VHJ1ZSkKICAgIGluZmVyZW5jZV9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGluZmVyZW5jZV9zdGFydAoKICAgIHN1Ym1pc3Npb25fcGF0aCA9IHJ1bl9yb290IC8gInN1Ym1pc3Npb24uY3N2IgogICAgX3dyaXRlX3ByZWRpY3Rpb25zKHN1Ym1pc3Npb25fcGF0aCwgcHJlZGljdGlvbnMpCiAgICB2YWxpZGF0aW9uID0gX2NoZWNrX3N1Ym1pc3Npb24oCiAgICAgICAgc3VibWlzc2lvbl9wYXRoLAogICAgICAgIGRhdGFzZXQucm9vdCwKICAgICAgICB0ZXN0X25hbWVzLAogICAgICAgIHByZWRpY3Rpb25zLAogICAgKQogICAgY292ZXJhZ2VfcGF0aCA9IHJ1bl9yb290IC8gInRlc3RfY292ZXJhZ2UuY3N2IgogICAgd2l0aCBjb3ZlcmFnZV9wYXRoLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoCiAgICAgICAgICAgIHN0cmVhbSwgZmllbGRuYW1lcz0oImZpbGVfbmFtZSIsICJpbnN0YW5jZV9jb3VudCIpLCBsaW5ldGVybWluYXRvcj0iXG4iCiAgICAgICAgKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhjb3ZlcmFnZV9yb3dzKQoKICAgIGlmIG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfU09VUkNFX0NPTU1JVCIpOgogICAgICAgIHJlcG9faW5mb1siY29tbWl0Il0gPSBvcy5lbnZpcm9uWyJFUVVJTkVYVVNfU09VUkNFX0NPTU1JVCJdCiAgICAgICAgcmVwb19pbmZvWyJ3b3JrdHJlZV9kaXJ0eSJdID0gVHJ1ZQogICAgcmVwb19pbmZvWyJzb3VyY2VfYnVuZGxlX3NoYTI1NiJdID0gb3MuZW52aXJvbi5nZXQoCiAgICAgICAgIkVRVUlORVhVU19TT1VSQ0VfQlVORExFX1NIQTI1NiIKICAgICkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic3RhdHVzIjogImNvbXBsZXRlIiwKICAgICAgICAiY3JlYXRlZF91dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKSwKICAgICAgICAicnVuX25hbWUiOiBydW5fbmFtZSwKICAgICAgICAicmVwb3NpdG9yeSI6IHJlcG9faW5mbywKICAgICAgICAiY29uZmlndXJhdGlvbl9zaGEyNTYiOiBjb25maWdfc2hhMjU2LAogICAgICAgICJrYWdnbGVfbm90ZWJvb2tfdmVyc2lvbiI6IG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfTk9URUJPT0tfVkVSU0lPTiIpLAogICAgICAgICJkYXRhX2JvdW5kYXJ5IjogewogICAgICAgICAgICAidHJhaW5pbmdfbGFiZWxzIjogImNvbXBldGl0aW9uLXByb3ZpZGVkIHRyYWluIGFubm90YXRpb25zIG9ubHkiLAogICAgICAgICAgICAiaW5mZXJlbmNlX2lucHV0cyI6ICJhdHRhY2hlZCBjb21wZXRpdGlvbiBILWFscGhhIHRlc3QgaW1hZ2VzIG9ubHkiLAogICAgICAgICAgICAiZXh0ZXJuYWxfZ3JvdW5kX3RydXRoIjogRmFsc2UsCiAgICAgICAgfSwKICAgICAgICAiZGF0YXNldF9pbnZlbnRvcnkiOiBkYXRhX2ludmVudG9yeSwKICAgICAgICAidmFsaWRhdGlvbl9yZWZlcmVuY2UiOiB7CiAgICAgICAgICAgICJzcGxpdF9zaGEyNTYiOiBjb25maWdfanNvblsiZXhwZWN0ZWRfc3BsaXRfc2hhMjU2Il0sCiAgICAgICAgICAgICJzZWxlY3Rpb25fbm90ZSI6ICJQb3N0cHJvY2Vzc2luZyBzZWxlY3RlZCBvbiB0aGUgZnJvemVuIG9ic2VydmF0aW9uLWdyb3VwZWQgaG9sZG91dCBiZWZvcmUgcmVmaXR0aW5nIG9uIGFsbCBvZmZpY2lhbCB0cmFpbmluZyBvYnNlcnZhdGlvbnMuIiwKICAgICAgICB9LAogICAgICAgICJ0cmFpbmluZyI6IHsqKnRyYWluaW5nLCAidHJhaW5pbmdfb2JzZXJ2YXRpb25zIjogbGVuKHRyYWluX25hbWVzKX0sCiAgICAgICAgIm1ldGhvZCI6IHsKICAgICAgICAgICAgImFyY2hpdGVjdHVyZSI6ICgKICAgICAgICAgICAgICAgICJkaW5vdjJfdml0czE0X2Zyb3plbl9iYWNrYm9uZV9jbm5fZGVjb2Rlcl9mb3JlZ3JvdW5kX2JvdW5kYXJ5IgogICAgICAgICAgICAgICAgaWYgbW9kZWxfdHlwZSA9PSAiZGlub3YyX3ZpdHMxNCIKICAgICAgICAgICAgICAgIGVsc2UgImZyb21fc2NyYXRjaF9maXZlX2xldmVsX3VuZXRfZm9yZWdyb3VuZF9ib3VuZGFyeSIKICAgICAgICAgICAgKSwKICAgICAgICAgICAgIm1vZGVsX3R5cGUiOiBtb2RlbF90eXBlLAogICAgICAgICAgICAidGFyZ2V0X21vZGUiOiB0YXJnZXRfbW9kZSwKICAgICAgICAgICAgInBhdGNoX3NpemUiOiBwYXRjaF9zaXplLAogICAgICAgICAgICAiYmFzZV9jaGFubmVscyI6IGJhc2VfY2hhbm5lbHMsCiAgICAgICAgICAgICJpbnB1dF9tb2RlIjogaW5wdXRfbW9kZSwKICAgICAgICAgICAgInN0ZXBzIjogc3RlcHMsCiAgICAgICAgICAgICJiYXRjaF9zaXplIjogYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgInR0YSI6IHR0YSwKICAgICAgICAgICAgImluc3RhbmNlX2NvbmZpZyI6IGluc3RhbmNlX2NvbmZpZy5fX2RpY3RfXywKICAgICAgICAgICAgInByZXRyYWluZWRfYmFja2JvbmUiOiBwcmV0cmFpbmVkX2JhY2tib25lLAogICAgICAgICAgICAicHJldHJhaW5lZF9iYWNrYm9uZV9yZXBvc2l0b3J5IjogaHViX3JlcG9zaXRvcnkgaWYgcHJldHJhaW5lZF9iYWNrYm9uZSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJwcmV0cmFpbmVkX2JhY2tib25lX2xpY2Vuc2UiOiAiQXBhY2hlLTIuMCIKICAgICAgICAgICAgaWYgbW9kZWxfdHlwZSA9PSAiZGlub3YyX3ZpdHMxNCIgYW5kIHByZXRyYWluZWRfYmFja2JvbmUKICAgICAgICAgICAgZWxzZSBOb25lLAogICAgICAgICAgICAicHJldHJhaW5lZF9iYWNrYm9uZV9jaGVja3BvaW50X3NoYTI1NiI6IHByZXRyYWluZWRfY2hlY2twb2ludF9zaGEyNTYsCiAgICAgICAgICAgICJwcmV0cmFpbmluZ19kYXRhIjogIkRJTk92MiBMVkQtMTQyTSwgc2VsZi1zdXBlcnZpc2VkIHdpdGhvdXQgbGFiZWxzIgogICAgICAgICAgICBpZiBwcmV0cmFpbmVkX2JhY2tib25lCiAgICAgICAgICAgIGVsc2UgTm9uZSwKICAgICAgICAgICAgImV4dGVybmFsX3VubGFiZWxlZF9kYXRhIjogYWRhcHRhdGlvbl9tZXRhZGF0YSwKICAgICAgICAgICAgImV4dGVybmFsX3N1cGVydmlzZWRfd2VpZ2h0cyI6IEZhbHNlLAogICAgICAgIH0sCiAgICAgICAgInRlc3RfdmFsaWRhdGlvbiI6IHZhbGlkYXRpb24sCiAgICAgICAgInJ1bnRpbWUiOiB7CiAgICAgICAgICAgICoqaGFyZHdhcmVfaW5mbygpLAogICAgICAgICAgICAiaW5mZXJlbmNlX3NlY29uZHMiOiBpbmZlcmVuY2Vfc2Vjb25kcywKICAgICAgICAgICAgImVuZF90b19lbmRfc2Vjb25kcyI6IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydF90aW1lLAogICAgICAgICAgICAicGFja2FnZV92ZXJzaW9ucyI6IHBhY2thZ2VfdmVyc2lvbnMoKSwKICAgICAgICB9LAogICAgICAgICJhcnRpZmFjdHMiOiB7CiAgICAgICAgICAgICJjaGVja3BvaW50IjogY2hlY2twb2ludF9wYXRoLm5hbWUsCiAgICAgICAgICAgICJ0cmFpbmluZ19oaXN0b3J5IjogY2hlY2twb2ludF9wYXRoLndpdGhfc3VmZml4KCIuanNvbmwiKS5uYW1lLAogICAgICAgICAgICAic3VibWlzc2lvbiI6IHN1Ym1pc3Npb25fcGF0aC5uYW1lLAogICAgICAgICAgICAiY292ZXJhZ2UiOiBjb3ZlcmFnZV9wYXRoLm5hbWUsCiAgICAgICAgfSwKICAgIH0KICAgIChydW5fcm9vdCAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IgogICAgKQogICAgcHJpbnQoCiAgICAgICAgZiJWYWxpZGF0ZWQgc3VibWlzc2lvbiB3aXRoIHt2YWxpZGF0aW9uWydwcmVkaWN0ZWRfaW5zdGFuY2VzJ119IGluc3RhbmNlcyAiCiAgICAgICAgZiJmb3Ige3ZhbGlkYXRpb25bJ3Rlc3RfaW1hZ2VzJ119IGltYWdlczoge3N1Ym1pc3Npb25fcGF0aH0iLAogICAgICAgIGZsdXNoPVRydWUsCiAgICApCiAgICByZXR1cm4gcmVzdWx0Cg==\", \"src/filament_baseline/sweep.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmRhdGEgaW1wb3J0IGxvYWRfZGF0YXNldApmcm9tIC5pbmZlcmVuY2UgaW1wb3J0IEluc3RhbmNlQ29uZmlnLCBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzCmZyb20gLnJ1bm5lciBpbXBvcnQgX3Jlc29sdmVfZGF0YV9yb290LCBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzLCBfd3JpdGVfb3ZlcmxheXMsIF93cml0ZV9wcmVkaWN0aW9ucwpmcm9tIC5zY29yZXIgaW1wb3J0IHNjb3JlX3ZhbGlkYXRpb24KZnJvbSAuc3BsaXQgaW1wb3J0IGNyZWF0ZV9vYnNlcnZhdGlvbl9zcGxpdApmcm9tIC52aXN1YWxzIGltcG9ydCB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzCgoKZGVmIHJ1bl9wb3N0cHJvY2Vzc2luZ19zd2VlcCgKICAgICosCiAgICByZXBvX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lLAogICAgcnVuX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBvYnNlcnZhdGlvbl9saW1pdDogaW50IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtkaWN0W3N0ciwgQW55XV06CiAgICByZXBvX3Jvb3QgPSBQYXRoKHJlcG9fcm9vdCkucmVzb2x2ZSgpCiAgICBydW5fcm9vdCA9IFBhdGgocnVuX3Jvb3QpLnJlc29sdmUoKQogICAgZGF0YXNldCA9IGxvYWRfZGF0YXNldChfcmVzb2x2ZV9kYXRhX3Jvb3QoZGF0YV9yb290LCByZXBvX3Jvb3QpKQogICAgY29uZmlnID0ganNvbi5sb2FkcygocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgc3BsaXQgPSBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICByZXBvX3Jvb3QgLyAiZGF0YSIgLyAic3BsaXRzIiwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdChjb25maWdbInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICApCiAgICBpZiBzcGxpdC5zaGEyNTYgIT0gY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBwb3N0cHJvY2Vzc2luZyBzd2VlcCByZXF1aXJlcyB0aGUgZnJvemVuIHZhbGlkYXRpb24gc3BsaXQiKQogICAgYmFzZV9pbnN0YW5jZV9jb25maWcgPSBJbnN0YW5jZUNvbmZpZygpCiAgICBtZXRyaWNzX3BhdGggPSBydW5fcm9vdCAvICJtZXRyaWNzLmpzb24iCiAgICBpZiBtZXRyaWNzX3BhdGguaXNfZmlsZSgpOgogICAgICAgIHJ1bl9tZXRyaWNzID0ganNvbi5sb2FkcyhtZXRyaWNzX3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGNvbmZpZ3VyZWQgPSBydW5fbWV0cmljcy5nZXQoIm1ldGhvZCIsIHt9KS5nZXQoImluc3RhbmNlX2NvbmZpZyIpCiAgICAgICAgaWYgaXNpbnN0YW5jZShjb25maWd1cmVkLCBkaWN0KToKICAgICAgICAgICAgZmllbGRzID0gSW5zdGFuY2VDb25maWcuX19kYXRhY2xhc3NfZmllbGRzX18KICAgICAgICAgICAgYmFzZV9pbnN0YW5jZV9jb25maWcgPSBJbnN0YW5jZUNvbmZpZygKICAgICAgICAgICAgICAgICoqe25hbWU6IGNvbmZpZ3VyZWRbbmFtZV0gZm9yIG5hbWUgaW4gZmllbGRzIGlmIG5hbWUgaW4gY29uZmlndXJlZH0KICAgICAgICAgICAgKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIGlmIG9ic2VydmF0aW9uX2xpbWl0IGlzIG5vdCBOb25lOgogICAgICAgIHZhbGlkYXRpb25fbmFtZXMgPSB2YWxpZGF0aW9uX25hbWVzWzpvYnNlcnZhdGlvbl9saW1pdF0KICAgIGNhY2hlX2RpciA9IHJ1bl9yb290IC8gInByb2JhYmlsaXRpZXMiCiAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgY2FjaGVfcGF0aCA9IGNhY2hlX2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiCiAgICAgICAgaWYgbm90IGNhY2hlX3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk1pc3NpbmcgY2FjaGVkIHZhbGlkYXRpb24gcHJvYmFiaWxpdHkgbWFwOiB7Y2FjaGVfcGF0aH0iKQoKICAgIGNvbmZpZ3VyYXRpb25zID0gWwogICAgICAgIEluc3RhbmNlQ29uZmlnKAogICAgICAgICAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZD1mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICAgICAgYm91bmRhcnlfdGhyZXNob2xkPWJvdW5kYXJ5X3RocmVzaG9sZCwKICAgICAgICAgICAgYm91bmRhcnlfd2VpZ2h0PWJhc2VfaW5zdGFuY2VfY29uZmlnLmJvdW5kYXJ5X3dlaWdodCwKICAgICAgICAgICAgc2VlZF90aHJlc2hvbGQ9YmFzZV9pbnN0YW5jZV9jb25maWcuc2VlZF90aHJlc2hvbGQsCiAgICAgICAgICAgIHNlZWRfbWluX2FyZWE9YmFzZV9pbnN0YW5jZV9jb25maWcuc2VlZF9taW5fYXJlYSwKICAgICAgICAgICAgbWluX2luc3RhbmNlX2FyZWE9YmFzZV9pbnN0YW5jZV9jb25maWcubWluX2luc3RhbmNlX2FyZWEsCiAgICAgICAgICAgIHNvbGFyX3JhZGl1cz1iYXNlX2luc3RhbmNlX2NvbmZpZy5zb2xhcl9yYWRpdXMsCiAgICAgICAgKQogICAgICAgIGZvciBmb3JlZ3JvdW5kX3RocmVzaG9sZCwgYm91bmRhcnlfdGhyZXNob2xkIGluIGl0ZXJ0b29scy5wcm9kdWN0KAogICAgICAgICAgICAoMC4yNSwgMC4zNSwgMC40NSwgMC41NSwgMC42NSksCiAgICAgICAgICAgICgwLjQ1LCAwLjUwLCAwLjU1LCAwLjYwLCAwLjY1LCAwLjcwLCAwLjc1LCAwLjgwLCAwLjkwKSwKICAgICAgICApCiAgICBdCiAgICByZXN1bHRzID0gW10KICAgIGZvciBpbmRleCwgaW5zdGFuY2VfY29uZmlnIGluIGVudW1lcmF0ZShjb25maWd1cmF0aW9ucywgc3RhcnQ9MSk6CiAgICAgICAgcHJlZGljdGlvbnMgPSB7fQogICAgICAgIGZvciBmaWxlX25hbWUgaW4gdmFsaWRhdGlvbl9uYW1lczoKICAgICAgICAgICAgY2FjaGVfcGF0aCA9IGNhY2hlX2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiCiAgICAgICAgICAgIHdpdGggbnAubG9hZChjYWNoZV9wYXRoKSBhcyBjYWNoZToKICAgICAgICAgICAgICAgIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0gPSBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzKAogICAgICAgICAgICAgICAgICAgIGNhY2hlWyJmb3JlZ3JvdW5kIl0uYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICAgICAgICAgIGNhY2hlWyJib3VuZGFyeSJdLmFzdHlwZShucC5mbG9hdDMyKSwKICAgICAgICAgICAgICAgICAgICBmaWxlX25hbWUsCiAgICAgICAgICAgICAgICAgICAgaW5zdGFuY2VfY29uZmlnLAogICAgICAgICAgICAgICAgKQogICAgICAgIG1ldHJpY3MsIHJvd3MgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgICAgICBkYXRhc2V0LAogICAgICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdChjb25maWdbImlvdV90aHJlc2hvbGQiXSksCiAgICAgICAgICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ9ZmxvYXQoY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0pLAogICAgICAgICkKICAgICAgICBnbG9iYWxfbWV0cmljcyA9IG1ldHJpY3NbImdsb2JhbCJdCiAgICAgICAgcm93ID0gewogICAgICAgICAgICAiY29uZmlndXJhdGlvbiI6IGluc3RhbmNlX2NvbmZpZy5fX2RpY3RfXywKICAgICAgICAgICAgInBxIjogZ2xvYmFsX21ldHJpY3NbInBxIl0sCiAgICAgICAgICAgICJ0cCI6IGdsb2JhbF9tZXRyaWNzWyJ0cCJdLAogICAgICAgICAgICAiZnAiOiBnbG9iYWxfbWV0cmljc1siZnAiXSwKICAgICAgICAgICAgImZuIjogZ2xvYmFsX21ldHJpY3NbImZuIl0sCiAgICAgICAgICAgICJwcmVkaWN0aW9uX2NvdW50X2Fjcm9zc192aWV3cyI6IGdsb2JhbF9tZXRyaWNzWwogICAgICAgICAgICAgICAgInByZWRpY3Rpb25fY291bnRfYWNyb3NzX2Fubm90YXRvcl92aWV3cyIKICAgICAgICAgICAgXSwKICAgICAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLAogICAgICAgIH0KICAgICAgICByZXN1bHRzLmFwcGVuZChyb3cpCiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiW3tpbmRleH0ve2xlbihjb25maWd1cmF0aW9ucyl9XSBmZz17aW5zdGFuY2VfY29uZmlnLmZvcmVncm91bmRfdGhyZXNob2xkOi4yZn0gIgogICAgICAgICAgICBmImJvdW5kYXJ5PXtpbnN0YW5jZV9jb25maWcuYm91bmRhcnlfdGhyZXNob2xkOi4yZn0gIgogICAgICAgICAgICBmIlBRPXtnbG9iYWxfbWV0cmljc1sncHEnXTouNmZ9IFRQPXtnbG9iYWxfbWV0cmljc1sndHAnXX0gIgogICAgICAgICAgICBmIkZQPXtnbG9iYWxfbWV0cmljc1snZnAnXX0gRk49e2dsb2JhbF9tZXRyaWNzWydmbiddfSIsCiAgICAgICAgICAgIGZsdXNoPVRydWUsCiAgICAgICAgKQoKICAgICAgICBpZiBpbmRleCA9PSAxIG9yIHJvd1sicHEiXSA+IG1heChpdGVtWyJwcSJdIGZvciBpdGVtIGluIHJlc3VsdHNbOi0xXSk6CiAgICAgICAgICAgIGJlc3RfZGlyID0gcnVuX3Jvb3QgLyAiYmVzdC1wb3N0cHJvY2Vzc2luZyIKICAgICAgICAgICAgYmVzdF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICAoYmVzdF9kaXIgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgIGpzb24uZHVtcHMoCiAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAic3BsaXRfc2hhMjU2Ijogc3BsaXQuc2hhMjU2LAogICAgICAgICAgICAgICAgICAgICAgICAiYmFzZV9pbnN0YW5jZV9jb25maWd1cmF0aW9uIjogYmFzZV9pbnN0YW5jZV9jb25maWcuX19kaWN0X18sCiAgICAgICAgICAgICAgICAgICAgICAgICJjb25maWd1cmF0aW9uIjogaW5zdGFuY2VfY29uZmlnLl9fZGljdF9fLAogICAgICAgICAgICAgICAgICAgICAgICAibWV0cmljcyI6IG1ldHJpY3MsCiAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICAgICAgICAgICAgICBzb3J0X2tleXM9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICsgIlxuIiwKICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgICAgICkKICAgICAgICAgICAgX3dyaXRlX3ByZWRpY3Rpb25zKGJlc3RfZGlyIC8gInByZWRpY3Rpb25zLmNzdiIsIHByZWRpY3Rpb25zKQogICAgICAgICAgICBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzKGJlc3RfZGlyIC8gInBlcl9vYnNlcnZhdGlvbi5jc3YiLCByb3dzLCBwcmVkaWN0aW9ucykKICAgICAgICAgICAgX3dyaXRlX292ZXJsYXlzKGRhdGFzZXQsIHByZWRpY3Rpb25zLCBiZXN0X2RpciAvICJvdmVybGF5cyIsIDEyKQogICAgICAgICAgICB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzKAogICAgICAgICAgICAgICAgZGF0YXNldCwgcHJlZGljdGlvbnMsIGJlc3RfZGlyIC8gImNvbXBhcmlzb24tb3ZlcmxheXMiLCAxMgogICAgICAgICAgICApCgogICAgcmVzdWx0X3BhdGggPSBydW5fcm9vdCAvICJwb3N0cHJvY2Vzc2luZ19zd2VlcC5qc29uIgogICAgcmVzdWx0X3BhdGgud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKHJlc3VsdHMsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICB3aXRoIChydW5fcm9vdCAvICJwb3N0cHJvY2Vzc2luZ19zd2VlcC5jc3YiKS5vcGVuKAogICAgICAgICJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIgogICAgKSBhcyBzdHJlYW06CiAgICAgICAgZmllbGRzID0gKAogICAgICAgICAgICAqSW5zdGFuY2VDb25maWcoKS5fX2RpY3RfXy5rZXlzKCksCiAgICAgICAgICAgICJwcSIsCiAgICAgICAgICAgICJ0cCIsCiAgICAgICAgICAgICJmcCIsCiAgICAgICAgICAgICJmbiIsCiAgICAgICAgICAgICJwcmVkaWN0aW9uX2NvdW50X2Fjcm9zc192aWV3cyIsCiAgICAgICAgKQogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1maWVsZHMsIGxpbmV0ZXJtaW5hdG9yPSJcbiIpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIHJlc3VsdHM6CiAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAqKnJvd1siY29uZmlndXJhdGlvbiJdLAogICAgICAgICAgICAgICAgICAgICoqe2tleTogcm93W2tleV0gZm9yIGtleSBpbiBmaWVsZHNbbGVuKEluc3RhbmNlQ29uZmlnKCkuX19kaWN0X18pIDpdfSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQogICAgcmV0dXJuIHNvcnRlZChyZXN1bHRzLCBrZXk9bGFtYmRhIGl0ZW06IGl0ZW1bInBxIl0sIHJldmVyc2U9VHJ1ZSkK\", \"src/filament_baseline/training.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJhbmRvbQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKCmZyb20gLmxlYXJuaW5nIGltcG9ydCBGaWxhbWVudFBhdGNoRGF0YXNldCwgVGFyZ2V0TW9kZSwgdHJhaW5pbmdfbG9zcwpmcm9tIC5tb2RlbCBpbXBvcnQgRElOT19WMl9SRVBPU0lUT1JZLCBidWlsZF9maWxhbWVudF9tb2RlbAoKCmRlZiBzZWVkX2V2ZXJ5dGhpbmcoc2VlZDogaW50KSAtPiBOb25lOgogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIG5wLnJhbmRvbS5zZWVkKHNlZWQpCiAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQogICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQogICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmJlbmNobWFyayA9IFRydWUKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBldmFsdWF0ZV9wYXRjaF9kaWNlKAogICAgbW9kZWw6IHRvcmNoLm5uLk1vZHVsZSwKICAgIGxvYWRlcjogRGF0YUxvYWRlciwKICAgIGRldmljZTogdG9yY2guZGV2aWNlLAogICAgKiwKICAgIG1heF9iYXRjaGVzOiBpbnQgPSA2NCwKKSAtPiBmbG9hdDoKICAgIG1vZGVsLmV2YWwoKQogICAgc2NvcmVzID0gW10KICAgIGZvciBpbmRleCwgKGltYWdlcywgdGFyZ2V0cykgaW4gZW51bWVyYXRlKGxvYWRlcik6CiAgICAgICAgaWYgaW5kZXggPj0gbWF4X2JhdGNoZXM6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgaW1hZ2VzID0gaW1hZ2VzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgdGFyZ2V0cyA9IHRhcmdldHMudG8oZGV2aWNlLCBub25fYmxvY2tpbmc9VHJ1ZSkKICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KAogICAgICAgICAgICBkZXZpY2VfdHlwZT1kZXZpY2UudHlwZSwKICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICAgICAgZW5hYmxlZD1kZXZpY2UudHlwZSA9PSAiY3VkYSIsCiAgICAgICAgKToKICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2VzKQogICAgICAgIHByb2JhYmlsaXRpZXMgPSBsb2dpdHNbOiwgMDoxXS5mbG9hdCgpLnNpZ21vaWQoKQogICAgICAgIGV4cGVjdGVkID0gdGFyZ2V0c1s6LCAwOjFdCiAgICAgICAgaW50ZXJzZWN0aW9uID0gKHByb2JhYmlsaXRpZXMgKiBleHBlY3RlZCkuc3VtKGRpbT0oMSwgMiwgMykpCiAgICAgICAgZGVub21pbmF0b3IgPSBwcm9iYWJpbGl0aWVzLnN1bShkaW09KDEsIDIsIDMpKSArIGV4cGVjdGVkLnN1bShkaW09KDEsIDIsIDMpKQogICAgICAgIHNjb3Jlcy5leHRlbmQoKCgyICogaW50ZXJzZWN0aW9uICsgMSkgLyAoZGVub21pbmF0b3IgKyAxKSkuY3B1KCkudG9saXN0KCkpCiAgICByZXR1cm4gZmxvYXQobnAubWVhbihzY29yZXMpKSBpZiBzY29yZXMgZWxzZSAwLjAKCgpkZWYgdHJhaW5fbW9kZWwoCiAgICAqLAogICAgdHJhaW5fZGF0YXNldDogRmlsYW1lbnRQYXRjaERhdGFzZXQsCiAgICB2YWxpZGF0aW9uX2RhdGFzZXQ6IEZpbGFtZW50UGF0Y2hEYXRhc2V0IHwgTm9uZSwKICAgIG91dHB1dF9wYXRoOiBzdHIgfCBQYXRoLAogICAgc3RlcHM6IGludCwKICAgIGJhdGNoX3NpemU6IGludCwKICAgIGxlYXJuaW5nX3JhdGU6IGZsb2F0ID0gM2UtNCwKICAgIHdlaWdodF9kZWNheTogZmxvYXQgPSAxZS00LAogICAgc2VlZDogaW50ID0gNDIsCiAgICB2YWxpZGF0aW9uX2ludGVydmFsOiBpbnQgPSA1MDAsCiAgICBncmFkaWVudF9hY2N1bXVsYXRpb246IGludCA9IDEsCiAgICBiYXNlX2NoYW5uZWxzOiBpbnQgPSAxNiwKICAgIG1vZGVsX3R5cGU6IHN0ciA9ICJ1bmV0IiwKICAgIHByZXRyYWluZWRfYmFja2JvbmU6IGJvb2wgPSBGYWxzZSwKICAgIGh1Yl9yZXBvc2l0b3J5OiBzdHIgPSBESU5PX1YyX1JFUE9TSVRPUlksCiAgICBodWJfZGlyOiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUsCiAgICBpbml0aWFsX3dlaWdodHM6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGlmIHN0ZXBzIDwgMSBvciBiYXRjaF9zaXplIDwgMSBvciBncmFkaWVudF9hY2N1bXVsYXRpb24gPCAxIG9yIGJhc2VfY2hhbm5lbHMgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJzdGVwcywgYmF0Y2hfc2l6ZSwgZ3JhZGllbnRfYWNjdW11bGF0aW9uLCBhbmQgYmFzZV9jaGFubmVscyBtdXN0IGJlIHBvc2l0aXZlIgogICAgICAgICkKICAgIHNlZWRfZXZlcnl0aGluZyhzZWVkKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhOjAiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIGlucHV0X2NoYW5uZWxzID0gaW50KGdldGF0dHIodHJhaW5fZGF0YXNldCwgImluX2NoYW5uZWxzIiwgMSkpCiAgICBpZiBtb2RlbF90eXBlID09ICJkaW5vdjJfdml0czE0IjoKICAgICAgICBwYXRjaF9zaXplID0gaW50KGdldGF0dHIodHJhaW5fZGF0YXNldCwgInBhdGNoX3NpemUiLCAwKSkKICAgICAgICBpZiBwYXRjaF9zaXplICUgMjI0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJESU5PdjIgcGF0Y2ggc2l6ZSBtdXN0IGJlIGRpdmlzaWJsZSBieSAyMjQsIHJlY2VpdmVkIHtwYXRjaF9zaXplfSIKICAgICAgICAgICAgKQogICAgaWYgaHViX2RpciBpcyBub3QgTm9uZToKICAgICAgICBQYXRoKGh1Yl9kaXIpLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICB0b3JjaC5odWIuc2V0X2RpcihzdHIoaHViX2RpcikpCiAgICBtb2RlbCA9IGJ1aWxkX2ZpbGFtZW50X21vZGVsKAogICAgICAgIG1vZGVsX3R5cGU9bW9kZWxfdHlwZSwKICAgICAgICBpbl9jaGFubmVscz1pbnB1dF9jaGFubmVscywKICAgICAgICBiYXNlX2NoYW5uZWxzPWJhc2VfY2hhbm5lbHMsCiAgICAgICAgcHJldHJhaW5lZF9iYWNrYm9uZT1wcmV0cmFpbmVkX2JhY2tib25lLAogICAgICAgIGh1Yl9yZXBvc2l0b3J5PWh1Yl9yZXBvc2l0b3J5LAogICAgKS50byhkZXZpY2UpCiAgICBpZiBpbml0aWFsX3dlaWdodHMgaXMgbm90IE5vbmU6CiAgICAgICAgaW5pdGlhbF9wYXRoID0gUGF0aChpbml0aWFsX3dlaWdodHMpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgICAgICBpZiBub3QgaW5pdGlhbF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJJbml0aWFsIG1vZGVsIGNoZWNrcG9pbnQgZG9lcyBub3QgZXhpc3Q6IHtpbml0aWFsX3BhdGh9IikKICAgICAgICBjaGVja3BvaW50ID0gdG9yY2gubG9hZChpbml0aWFsX3BhdGgsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICAgICAgZXhwZWN0ZWRfY29uZmlnID0gewogICAgICAgICAgICAibW9kZWxfdHlwZSI6IG1vZGVsX3R5cGUsCiAgICAgICAgICAgICJiYXNlX2NoYW5uZWxzIjogYmFzZV9jaGFubmVscywKICAgICAgICAgICAgImluX2NoYW5uZWxzIjogaW5wdXRfY2hhbm5lbHMsCiAgICAgICAgICAgICJodWJfcmVwb3NpdG9yeSI6IGh1Yl9yZXBvc2l0b3J5LAogICAgICAgIH0KICAgICAgICBpZiBjaGVja3BvaW50LmdldCgibW9kZWxfY29uZmlnIikgIT0gZXhwZWN0ZWRfY29uZmlnOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgIkluaXRpYWwgY2hlY2twb2ludCBtb2RlbCBjb25maWd1cmF0aW9uIGRvZXMgbm90IG1hdGNoIHRoZSByZXF1ZXN0ZWQgbW9kZWw6ICIKICAgICAgICAgICAgICAgIGYie2NoZWNrcG9pbnQuZ2V0KCdtb2RlbF9jb25maWcnKX0gIT0ge2V4cGVjdGVkX2NvbmZpZ30iCiAgICAgICAgICAgICkKICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2hlY2twb2ludFsibW9kZWwiXSwgc3RyaWN0PVRydWUpCiAgICAgICAgcHJpbnQoZiJJbml0aWFsaXplZCBtb2RlbCBmcm9tIHtpbml0aWFsX3BhdGh9IiwgZmx1c2g9VHJ1ZSkKICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICBtb2RlbCA9IG1vZGVsLnRvKG1lbW9yeV9mb3JtYXQ9dG9yY2guY2hhbm5lbHNfbGFzdCkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKAogICAgICAgIChwYXJhbWV0ZXIgZm9yIHBhcmFtZXRlciBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcGFyYW1ldGVyLnJlcXVpcmVzX2dyYWQpLAogICAgICAgIGxyPWxlYXJuaW5nX3JhdGUsCiAgICAgICAgd2VpZ2h0X2RlY2F5PXdlaWdodF9kZWNheSwKICAgICkKICAgIHdhcm11cF9zdGVwcyA9IG1pbigzMDAsIG1heCgxLCBzdGVwcyAvLyAxMCkpCgogICAgZGVmIHNjaGVkdWxlKHN0ZXA6IGludCkgLT4gZmxvYXQ6CiAgICAgICAgaWYgc3RlcCA8IHdhcm11cF9zdGVwczoKICAgICAgICAgICAgcmV0dXJuIG1heCgwLjA1LCAoc3RlcCArIDEpIC8gd2FybXVwX3N0ZXBzKQogICAgICAgIHByb2dyZXNzID0gKHN0ZXAgLSB3YXJtdXBfc3RlcHMpIC8gbWF4KDEsIHN0ZXBzIC0gd2FybXVwX3N0ZXBzKQogICAgICAgIHJldHVybiAwLjAxICsgMC45OSAqIDAuNSAqICgxLjAgKyBtYXRoLmNvcyhtYXRoLnBpICogcHJvZ3Jlc3MpKQoKICAgIHNjaGVkdWxlciA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5MYW1iZGFMUihvcHRpbWl6ZXIsIHNjaGVkdWxlKQogICAgc2NhbGVyID0gdG9yY2guYW1wLkdyYWRTY2FsZXIoImN1ZGEiLCBlbmFibGVkPWRldmljZS50eXBlID09ICJjdWRhIikKICAgIHRyYWluX2xvYWRlciA9IERhdGFMb2FkZXIoCiAgICAgICAgdHJhaW5fZGF0YXNldCwKICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsCiAgICAgICAgc2h1ZmZsZT1UcnVlLAogICAgICAgIG51bV93b3JrZXJzPTAsCiAgICAgICAgcGluX21lbW9yeT1kZXZpY2UudHlwZSA9PSAiY3VkYSIsCiAgICAgICAgZHJvcF9sYXN0PVRydWUsCiAgICApCiAgICB2YWxpZGF0aW9uX2xvYWRlciA9ICgKICAgICAgICBEYXRhTG9hZGVyKAogICAgICAgICAgICB2YWxpZGF0aW9uX2RhdGFzZXQsCiAgICAgICAgICAgIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwKICAgICAgICAgICAgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgbnVtX3dvcmtlcnM9MCwKICAgICAgICAgICAgcGluX21lbW9yeT1kZXZpY2UudHlwZSA9PSAiY3VkYSIsCiAgICAgICAgKQogICAgICAgIGlmIHZhbGlkYXRpb25fZGF0YXNldCBpcyBub3QgTm9uZQogICAgICAgIGVsc2UgTm9uZQogICAgKQogICAgaWYgbGVuKHRyYWluX2xvYWRlcikgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaGUgdHJhaW5pbmcgcGF0Y2ggbG9hZGVyIHByb2R1Y2VkIG5vIGJhdGNoZXMiKQoKICAgIG91dHB1dF9wYXRoID0gUGF0aChvdXRwdXRfcGF0aCkKICAgIG91dHB1dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBoaXN0b3J5X3BhdGggPSBvdXRwdXRfcGF0aC53aXRoX3N1ZmZpeCgiLmpzb25sIikKICAgIGJlc3RfZGljZSA9IC0xLjAKICAgIGJlc3Rfc3RlcCA9IDAKICAgIHJ1bm5pbmdfbG9zcyA9IDAuMAogICAgbG9nZ2VkX3N0ZXBzID0gMAogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGl0ZXJhdG9yID0gaXRlcih0cmFpbl9sb2FkZXIpCiAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICBpZiBkZXZpY2UudHlwZSA9PSAiY3VkYSI6CiAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cyhkZXZpY2UpCgogICAgd2l0aCBoaXN0b3J5X3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhpc3Rvcnk6CiAgICAgICAgZm9yIHN0ZXAgaW4gcmFuZ2UoMSwgc3RlcHMgKyAxKToKICAgICAgICAgICAgbW9kZWwudHJhaW4oKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBpbWFnZXMsIHRhcmdldHMgPSBuZXh0KGl0ZXJhdG9yKQogICAgICAgICAgICBleGNlcHQgU3RvcEl0ZXJhdGlvbjoKICAgICAgICAgICAgICAgIHRyYWluX2RhdGFzZXQuc2V0X2Vwb2NoKHN0ZXAgLy8gbGVuKHRyYWluX2xvYWRlcikpCiAgICAgICAgICAgICAgICBpdGVyYXRvciA9IGl0ZXIodHJhaW5fbG9hZGVyKQogICAgICAgICAgICAgICAgaW1hZ2VzLCB0YXJnZXRzID0gbmV4dChpdGVyYXRvcikKICAgICAgICAgICAgaW1hZ2VzID0gaW1hZ2VzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIHRhcmdldHMgPSB0YXJnZXRzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICAgICAgICAgIGltYWdlcyA9IGltYWdlcy5jb250aWd1b3VzKG1lbW9yeV9mb3JtYXQ9dG9yY2guY2hhbm5lbHNfbGFzdCkKICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgKICAgICAgICAgICAgICAgIGRldmljZV90eXBlPWRldmljZS50eXBlLAogICAgICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICAgICAgICAgIGVuYWJsZWQ9ZGV2aWNlLnR5cGUgPT0gImN1ZGEiLAogICAgICAgICAgICApOgogICAgICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2VzKQogICAgICAgICAgICAgICAgbG9zcyA9IHRyYWluaW5nX2xvc3MobG9naXRzLCB0YXJnZXRzKSAvIGdyYWRpZW50X2FjY3VtdWxhdGlvbgogICAgICAgICAgICBzY2FsZXIuc2NhbGUobG9zcykuYmFja3dhcmQoKQogICAgICAgICAgICBpZiBzdGVwICUgZ3JhZGllbnRfYWNjdW11bGF0aW9uID09IDAgb3Igc3RlcCA9PSBzdGVwczoKICAgICAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgICAgIHNjYWxlci51cGRhdGUoKQogICAgICAgICAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgICAgIHJ1bm5pbmdfbG9zcyArPSBmbG9hdChsb3NzLmRldGFjaCgpKSAqIGdyYWRpZW50X2FjY3VtdWxhdGlvbgogICAgICAgICAgICBsb2dnZWRfc3RlcHMgKz0gMQoKICAgICAgICAgICAgc2hvdWxkX3ZhbGlkYXRlID0gKAogICAgICAgICAgICAgICAgc3RlcCAlIHZhbGlkYXRpb25faW50ZXJ2YWwgPT0gMCBvciBzdGVwID09IHN0ZXBzCiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgc2hvdWxkX3ZhbGlkYXRlOgogICAgICAgICAgICAgICAgcGF0Y2hfZGljZSA9ICgKICAgICAgICAgICAgICAgICAgICBldmFsdWF0ZV9wYXRjaF9kaWNlKG1vZGVsLCB2YWxpZGF0aW9uX2xvYWRlciwgZGV2aWNlLCBtYXhfYmF0Y2hlcz02NCkKICAgICAgICAgICAgICAgICAgICBpZiB2YWxpZGF0aW9uX2xvYWRlciBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgICAgIGVsc2UgTm9uZQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgcm93ID0gewogICAgICAgICAgICAgICAgICAgICJzdGVwIjogc3RlcCwKICAgICAgICAgICAgICAgICAgICAibWVhbl90cmFpbl9sb3NzX3NpbmNlX2xvZyI6IHJ1bm5pbmdfbG9zcwogICAgICAgICAgICAgICAgICAgIC8gbWF4KDEsIGxvZ2dlZF9zdGVwcyksCiAgICAgICAgICAgICAgICAgICAgInZhbGlkYXRpb25fcGF0Y2hfZGljZSI6IHBhdGNoX2RpY2UsCiAgICAgICAgICAgICAgICAgICAgImxlYXJuaW5nX3JhdGUiOiBvcHRpbWl6ZXIucGFyYW1fZ3JvdXBzWzBdWyJsciJdLAogICAgICAgICAgICAgICAgICAgICJlbGFwc2VkX3NlY29uZHMiOiB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZCwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGhpc3Rvcnkud3JpdGUoanNvbi5kdW1wcyhyb3csIHNvcnRfa2V5cz1UcnVlKSArICJcbiIpCiAgICAgICAgICAgICAgICBoaXN0b3J5LmZsdXNoKCkKICAgICAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMocm93LCBzb3J0X2tleXM9VHJ1ZSksIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICBydW5uaW5nX2xvc3MgPSAwLjAKICAgICAgICAgICAgICAgIGxvZ2dlZF9zdGVwcyA9IDAKICAgICAgICAgICAgICAgIGlmIHBhdGNoX2RpY2UgaXMgTm9uZSBvciBwYXRjaF9kaWNlID4gYmVzdF9kaWNlOgogICAgICAgICAgICAgICAgICAgIGlmIHBhdGNoX2RpY2UgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RfZGljZSA9IHBhdGNoX2RpY2UKICAgICAgICAgICAgICAgICAgICBiZXN0X3N0ZXAgPSBzdGVwCiAgICAgICAgICAgICAgICAgICAgdG9yY2guc2F2ZSgKICAgICAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGVsIjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGVsX2NvbmZpZyI6IHsKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibW9kZWxfdHlwZSI6IG1vZGVsX3R5cGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImJhc2VfY2hhbm5lbHMiOiBiYXNlX2NoYW5uZWxzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJpbl9jaGFubmVscyI6IGlucHV0X2NoYW5uZWxzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJodWJfcmVwb3NpdG9yeSI6IGh1Yl9yZXBvc2l0b3J5LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0YXJnZXRfbW9kZSI6IHRyYWluX2RhdGFzZXQudGFyZ2V0X21vZGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAic3RlcCI6IHN0ZXAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAidmFsaWRhdGlvbl9wYXRjaF9kaWNlIjogcGF0Y2hfZGljZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzZWVkIjogc2VlZCwKICAgICAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICAgICAgb3V0cHV0X3BhdGgsCiAgICAgICAgICAgICAgICAgICAgKQoKICAgIHBlYWtfbWVtb3J5ID0gKAogICAgICAgIGludCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKGRldmljZSkpIGlmIGRldmljZS50eXBlID09ICJjdWRhIiBlbHNlIDAKICAgICkKICAgIHJldHVybiB7CiAgICAgICAgImNoZWNrcG9pbnQiOiBzdHIob3V0cHV0X3BhdGgpLAogICAgICAgICJoaXN0b3J5Ijogc3RyKGhpc3RvcnlfcGF0aCksCiAgICAgICAgImRldmljZSI6IHN0cihkZXZpY2UpLAogICAgICAgICJzdGVwcyI6IHN0ZXBzLAogICAgICAgICJiYXNlX2NoYW5uZWxzIjogYmFzZV9jaGFubmVscywKICAgICAgICAibW9kZWxfdHlwZSI6IG1vZGVsX3R5cGUsCiAgICAgICAgInByZXRyYWluZWRfYmFja2JvbmUiOiBwcmV0cmFpbmVkX2JhY2tib25lLAogICAgICAgICJodWJfcmVwb3NpdG9yeSI6IGh1Yl9yZXBvc2l0b3J5IGlmIG1vZGVsX3R5cGUgPT0gImRpbm92Ml92aXRzMTQiIGVsc2UgTm9uZSwKICAgICAgICAic2VlZCI6IHNlZWQsCiAgICAgICAgImJlc3Rfc3RlcCI6IGJlc3Rfc3RlcCwKICAgICAgICAiYmVzdF92YWxpZGF0aW9uX3BhdGNoX2RpY2UiOiBiZXN0X2RpY2UgaWYgYmVzdF9kaWNlID49IDAuMCBlbHNlIE5vbmUsCiAgICAgICAgInNlbGVjdGlvbl9tb2RlIjogImJlc3RfdmFsaWRhdGlvbl9wYXRjaF9kaWNlIgogICAgICAgIGlmIHZhbGlkYXRpb25fbG9hZGVyIGlzIG5vdCBOb25lCiAgICAgICAgZWxzZSAiZmluYWxfc3RlcCIsCiAgICAgICAgImVsYXBzZWRfc2Vjb25kcyI6IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkLAogICAgICAgICJwZWFrX2dwdV9tZW1vcnlfYnl0ZXMiOiBwZWFrX21lbW9yeSwKICAgICAgICAidG9yY2hfdmVyc2lvbiI6IHRvcmNoLl9fdmVyc2lvbl9fLAogICAgICAgICJjdWRhX3ZlcnNpb24iOiB0b3JjaC52ZXJzaW9uLmN1ZGEsCiAgICB9Cg==\", \"src/filament_baseline/visuals.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucApmcm9tIHB5Y29jb3Rvb2xzIGltcG9ydCBtYXNrIGFzIG1hc2tfdXRpbHMKCmZyb20gLmRhdGEgaW1wb3J0IENvbXBldGl0aW9uRGF0YXNldApmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IFByZWRpY3RlZEluc3RhbmNlLCBkZWNvZGVfcmxlLCByZWFkX2dyYXlzY2FsZQpmcm9tIC5zY29yZXIgaW1wb3J0IGFubm90YXRpb25fdG9fcmxlCgoKZGVmIHdyaXRlX2NvbXBhcmlzb25fb3ZlcmxheXMoCiAgICBkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsCiAgICBwcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RbUHJlZGljdGVkSW5zdGFuY2VdXSwKICAgIG91dHB1dF9kaXI6IHN0ciB8IFBhdGgsCiAgICBtYXhpbXVtOiBpbnQgPSAxMiwKKSAtPiBsaXN0W3N0cl06CiAgICBuYW1lcyA9IHNvcnRlZChwcmVkaWN0aW9ucykKICAgIGlmIG5vdCBuYW1lcyBvciBtYXhpbXVtIDw9IDA6CiAgICAgICAgcmV0dXJuIFtdCiAgICBpZiBsZW4obmFtZXMpIDw9IG1heGltdW06CiAgICAgICAgc2VsZWN0ZWQgPSBuYW1lcwogICAgZWxpZiBtYXhpbXVtID09IDE6CiAgICAgICAgc2VsZWN0ZWQgPSBbbmFtZXNbbGVuKG5hbWVzKSAvLyAyXV0KICAgIGVsc2U6CiAgICAgICAgaW5kaWNlcyA9IHsKICAgICAgICAgICAgcm91bmQoaW5kZXggKiAobGVuKG5hbWVzKSAtIDEpIC8gKG1heGltdW0gLSAxKSkKICAgICAgICAgICAgZm9yIGluZGV4IGluIHJhbmdlKG1heGltdW0pCiAgICAgICAgfQogICAgICAgIHNlbGVjdGVkID0gW25hbWVzW2luZGV4XSBmb3IgaW5kZXggaW4gc29ydGVkKGluZGljZXMpXQoKICAgIG91dHB1dF9kaXIgPSBQYXRoKG91dHB1dF9kaXIpCiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHNhdmVkID0gW10KICAgIGZvciBmaWxlX25hbWUgaW4gc2VsZWN0ZWQ6CiAgICAgICAgZ3JheSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyIC8gZmlsZV9uYW1lKQogICAgICAgIHZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV1bMF0KICAgICAgICBncm91bmRfdHJ1dGggPSBucC56ZXJvcyhncmF5LnNoYXBlLCBkdHlwZT1ib29sKQogICAgICAgIGZvciBhbm5vdGF0aW9uIGluIGRhdGFzZXQuYW5ub3RhdGlvbnNfYnlfaW1hZ2VfaWRbdmlldy5pbWFnZV9pZF06CiAgICAgICAgICAgIHJsZSA9IGFubm90YXRpb25fdG9fcmxlKGFubm90YXRpb24sIHZpZXcuaGVpZ2h0LCB2aWV3LndpZHRoKQogICAgICAgICAgICBlbmNvZGVkID0gewogICAgICAgICAgICAgICAgInNpemUiOiBybGVbInNpemUiXSwKICAgICAgICAgICAgICAgICJjb3VudHMiOiBybGVbImNvdW50cyJdLmVuY29kZSgiYXNjaWkiKQogICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShybGVbImNvdW50cyJdLCBzdHIpCiAgICAgICAgICAgICAgICBlbHNlIHJsZVsiY291bnRzIl0sCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZ3JvdW5kX3RydXRoIHw9IG1hc2tfdXRpbHMuZGVjb2RlKGVuY29kZWQpLmFzdHlwZShib29sKQogICAgICAgIHByZWRpY3RlZCA9IG5wLnplcm9zKGdyYXkuc2hhcGUsIGR0eXBlPWJvb2wpCiAgICAgICAgZm9yIGluc3RhbmNlIGluIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV06CiAgICAgICAgICAgIHByZWRpY3RlZCB8PSBkZWNvZGVfcmxlKGluc3RhbmNlLnJsZSkuYXN0eXBlKGJvb2wpCgogICAgICAgIGNhbnZhcyA9IGN2Mi5jdnRDb2xvcihncmF5LCBjdjIuQ09MT1JfR1JBWTJSR0IpCiAgICAgICAgb3ZlcmxheSA9IGNhbnZhcy5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICBndF9vbmx5ID0gZ3JvdW5kX3RydXRoICYgfnByZWRpY3RlZAogICAgICAgIHByZWRfb25seSA9IHByZWRpY3RlZCAmIH5ncm91bmRfdHJ1dGgKICAgICAgICBvdmVybGFwID0gZ3JvdW5kX3RydXRoICYgcHJlZGljdGVkCiAgICAgICAgb3ZlcmxheVtndF9vbmx5XSA9IDAuNTUgKiBvdmVybGF5W2d0X29ubHldICsgMC40NSAqIG5wLmFycmF5KFs0MCwgMjU1LCA4MF0pCiAgICAgICAgb3ZlcmxheVtwcmVkX29ubHldID0gMC41NSAqIG92ZXJsYXlbcHJlZF9vbmx5XSArIDAuNDUgKiBucC5hcnJheShbMjU1LCA1MCwgNTBdKQogICAgICAgIG92ZXJsYXlbb3ZlcmxhcF0gPSAwLjQ1ICogb3ZlcmxheVtvdmVybGFwXSArIDAuNTUgKiBucC5hcnJheShbMjU1LCAyMzAsIDMwXSkKICAgICAgICBjYW52YXMgPSBucC5jbGlwKG92ZXJsYXksIDAsIDI1NSkuYXN0eXBlKG5wLnVpbnQ4KQogICAgICAgIGN2Mi5wdXRUZXh0KAogICAgICAgICAgICBjYW52YXMsCiAgICAgICAgICAgIGYiR1QgdW5pb246IHtsZW4oZGF0YXNldC5hbm5vdGF0aW9uc19ieV9pbWFnZV9pZFt2aWV3LmltYWdlX2lkXSl9IGluc3RhbmNlcyB8ICIKICAgICAgICAgICAgZiJQcmVkaWN0ZWQ6IHtsZW4ocHJlZGljdGlvbnNbZmlsZV9uYW1lXSl9IHwgZ3JlZW4gR1QsIHJlZCBwcmVkLCB5ZWxsb3cgb3ZlcmxhcCIsCiAgICAgICAgICAgICgyNCwgNDgpLAogICAgICAgICAgICBjdjIuRk9OVF9IRVJTSEVZX1NJTVBMRVgsCiAgICAgICAgICAgIDAuNywKICAgICAgICAgICAgKDI1NSwgMjU1LCAyNTUpLAogICAgICAgICAgICAyLAogICAgICAgICAgICBjdjIuTElORV9BQSwKICAgICAgICApCiAgICAgICAgZGVzdGluYXRpb24gPSBvdXRwdXRfZGlyIC8gZiJ7UGF0aChmaWxlX25hbWUpLnN0ZW19X2NvbXBhcmlzb24ucG5nIgogICAgICAgIGlmIG5vdCBjdjIuaW13cml0ZShzdHIoZGVzdGluYXRpb24pLCBjdjIuY3Z0Q29sb3IoY2FudmFzLCBjdjIuQ09MT1JfUkdCMkJHUikpOgogICAgICAgICAgICByYWlzZSBPU0Vycm9yKGYiRmFpbGVkIHRvIHNhdmUgdmFsaWRhdGlvbiBjb21wYXJpc29uIG92ZXJsYXk6IHtkZXN0aW5hdGlvbn0iKQogICAgICAgIHNhdmVkLmFwcGVuZChkZXN0aW5hdGlvbi5uYW1lKQogICAgcmV0dXJuIHNhdmVkCg==\", \"scripts/prepare_chrotel_external.py\": \"...\", \"scripts/run_chrotel_model_experiment.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCgpSRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE9fUk9PVCAvICJzcmMiKSkKCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUuZGF0YSBpbXBvcnQgbG9hZF9kYXRhc2V0ICAjIG5vcWE6IEU0MDIKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5leHRlcm5hbF9kYXRhIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIGxvYWRfY2hyb3RlbF9kYXRhc2V0LAogICAgbG9hZF9mbGF0X3VuZXRfZGF0YXNldCwKKQpmcm9tIGZpbGFtZW50X2Jhc2VsaW5lLmZlYXR1cmVzIGltcG9ydCBJTlBVVF9NT0RFUyAgIyBub3FhOiBFNDAyCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUubGVhcm5pbmcgaW1wb3J0IEZpbGFtZW50UGF0Y2hEYXRhc2V0ICAjIG5vcWE6IEU0MDIKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5tb2RlbF9ydW5uZXIgaW1wb3J0IF9pc19rYWdnbGVfcnVudGltZSwgcnVuX21vZGVsX2V4cGVyaW1lbnQgICMgbm9xYTogRTQwMgpmcm9tIGZpbGFtZW50X2Jhc2VsaW5lLnNwbGl0IGltcG9ydCBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQgICMgbm9xYTogRTQwMgpmcm9tIGZpbGFtZW50X2Jhc2VsaW5lLnRyYWluaW5nIGltcG9ydCB0cmFpbl9tb2RlbCAgIyBub3FhOiBFNDAyCgoKZGVmIF9leHRlcm5hbF9kYXRlX3NwbGl0KAogICAgb2JzZXJ2YXRpb25fbmFtZXM6IHR1cGxlW3N0ciwgLi4uXSwgdmFsaWRhdGlvbl9mcmFjdGlvbjogZmxvYXQgPSAwLjIKKSAtPiB0dXBsZVtsaXN0W3N0cl0sIGxpc3Rbc3RyXV06CiAgICBkYXRlX2J5X25hbWUgPSB7fQogICAgZm9yIG5hbWUgaW4gb2JzZXJ2YXRpb25fbmFtZXM6CiAgICAgICAgbWF0Y2ggPSByZS5tYXRjaChyIl4oPzpjaHJvdGVsX3xiYnNvX2hhbHBoX2ZsXykoXGR7OH0pXyIsIG5hbWUsIHJlLklHTk9SRUNBU0UpCiAgICAgICAgaWYgbWF0Y2ggaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkNhbm5vdCBkZXJpdmUgZXh0ZXJuYWwgb2JzZXJ2YXRpb24gZGF0ZSBmcm9tIHtuYW1lfSIpCiAgICAgICAgZGF0ZV9ieV9uYW1lW25hbWVdID0gbWF0Y2guZ3JvdXAoMSkKICAgIGRhdGVzID0gc29ydGVkKHNldChkYXRlX2J5X25hbWUudmFsdWVzKCkpKQogICAgdmFsaWRhdGlvbl9kYXRlcyA9IHsKICAgICAgICBkYXRlCiAgICAgICAgZm9yIGRhdGUgaW4gZGF0ZXMKICAgICAgICBpZiBpbnQuZnJvbV9ieXRlcyhoYXNobGliLnNoYTI1NihkYXRlLmVuY29kZSgpKS5kaWdlc3QoKVs6OF0sICJiaWciKSAvIDIqKjY0CiAgICAgICAgPCB2YWxpZGF0aW9uX2ZyYWN0aW9uCiAgICB9CiAgICBpZiBub3QgdmFsaWRhdGlvbl9kYXRlcyBvciB2YWxpZGF0aW9uX2RhdGVzID09IHNldChkYXRlcyk6CiAgICAgICAgdmFsaWRhdGlvbl9kYXRlcyA9IHNldChkYXRlc1s6OjVdKQogICAgdHJhaW5fbmFtZXMgPSBbbmFtZSBmb3IgbmFtZSBpbiBvYnNlcnZhdGlvbl9uYW1lcyBpZiBkYXRlX2J5X25hbWVbbmFtZV0gbm90IGluIHZhbGlkYXRpb25fZGF0ZXNdCiAgICB2YWxpZGF0aW9uX25hbWVzID0gW25hbWUgZm9yIG5hbWUgaW4gb2JzZXJ2YXRpb25fbmFtZXMgaWYgZGF0ZV9ieV9uYW1lW25hbWVdIGluIHZhbGlkYXRpb25fZGF0ZXNdCiAgICBpZiB7ZGF0ZV9ieV9uYW1lW25hbWVdIGZvciBuYW1lIGluIHRyYWluX25hbWVzfSAmIHsKICAgICAgICBkYXRlX2J5X25hbWVbbmFtZV0gZm9yIG5hbWUgaW4gdmFsaWRhdGlvbl9uYW1lcwogICAgfToKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiQ2hyb1RlbCBleHRlcm5hbCBwcmV0cmFpbmluZyBzcGxpdCBjcm9zc2VkIGNhbGVuZGFyIGRhdGVzIikKICAgIHJldHVybiB0cmFpbl9uYW1lcywgdmFsaWRhdGlvbl9uYW1lcwoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249KAogICAgICAgICAgICAiUHJldHJhaW4gYSBmb3JlZ3JvdW5kLWJvdW5kYXJ5IFUtTmV0IG9uIGRhdGUtZGlzam9pbnQgZXh0ZXJuYWwgc29sYXIgbWFza3MsICIKICAgICAgICAgICAgImZpbmUtdHVuZSBvbiB0aGUgY29tcGV0aXRpb24gdHJhaW5pbmcgcGFydGl0aW9uLCBhbmQgc2NvcmUgdGhlIGZyb3plbiBob2xkb3V0LiIKICAgICAgICApCiAgICApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlcG8tcm9vdCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1SRVBPX1JPT1QpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEtcm9vdCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1leHRlcm5hbC1zb3VyY2UiLCBjaG9pY2VzPSgiY2hyb3RlbCIsICJmbGF0LXVuZXQiKSwgZGVmYXVsdD0iY2hyb3RlbCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV4dGVybmFsLXJvb3QiLCAiLS1jaHJvdGVsLXJvb3QiLCBkZXN0PSJleHRlcm5hbF9yb290IiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJ1bi1uYW1lIiwgZGVmYXVsdD0idW5ldC1jaHJvdGVsLXdlYWstcHJldHJhaW4tZmluZXR1bmUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcmV0cmFpbi1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTFfNTAwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTVfMDAwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcGF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTUxMikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmFzZS1jaGFubmVscyIsIHR5cGU9aW50LCBkZWZhdWx0PTE2KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10dGEiLCB0eXBlPWludCwgY2hvaWNlcz0oMSwgMiwgNCwgOCksIGRlZmF1bHQ9NCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taW5wdXQtbW9kZSIsIGNob2ljZXM9SU5QVVRfTU9ERVMsIGRlZmF1bHQ9InJhd19sb2NhbF9jb250cmFzdCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZvcmVncm91bmQtdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjU1KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ib3VuZGFyeS10aHJlc2hvbGQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNjApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvdW5kYXJ5LXdlaWdodCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MS41KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1taW4taW5zdGFuY2UtYXJlYSIsIHR5cGU9aW50LCBkZWZhdWx0PTQwMCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgcmVwb19yb290ID0gYXJncy5yZXBvX3Jvb3QuZXhwYW5kdXNlcigpLnJlc29sdmUoKQogICAgZGF0YV9yb290ID0gYXJncy5kYXRhX3Jvb3QuZXhwYW5kdXNlcigpLnJlc29sdmUoKSBpZiBhcmdzLmRhdGFfcm9vdCBlbHNlIE5vbmUKICAgIGRlZmF1bHRfZXh0ZXJuYWxfcm9vdCA9ICgKICAgICAgICByZXBvX3Jvb3QgLyAiZGF0YSIgLyAiZXh0ZXJuYWwiIC8gImNocm90ZWwiCiAgICAgICAgaWYgYXJncy5leHRlcm5hbF9zb3VyY2UgPT0gImNocm90ZWwiCiAgICAgICAgZWxzZSByZXBvX3Jvb3QgLyAiZGF0YSIgLyAiZXh0ZXJuYWwiIC8gImZsYXQtdW5ldC0xMDE1NDUiIC8gInByb2Nlc3NlZC0yNTYiCiAgICApCiAgICBleHRlcm5hbF9yb290ID0gKAogICAgICAgIGFyZ3MuZXh0ZXJuYWxfcm9vdC5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICAgICAgaWYgYXJncy5leHRlcm5hbF9yb290CiAgICAgICAgZWxzZSBkZWZhdWx0X2V4dGVybmFsX3Jvb3QKICAgICkKICAgIGthZ2dsZV9ydW50aW1lID0gX2lzX2thZ2dsZV9ydW50aW1lKCkKICAgIGthZ2dsZV9jb21wZXRpdGlvbl9yb290ID0gKAogICAgICAgIGRhdGFfcm9vdCBpcyBub3QgTm9uZQogICAgICAgIGFuZCAoCiAgICAgICAgICAgIGRhdGFfcm9vdCA9PSBQYXRoKCIva2FnZ2xlL2lucHV0IikKICAgICAgICAgICAgb3IgUGF0aCgiL2thZ2dsZS9pbnB1dCIpIGluIGRhdGFfcm9vdC5wYXJlbnRzCiAgICAgICAgKQogICAgKQogICAgaWYgZGF0YV9yb290IGlzIG5vdCBOb25lIGFuZCBub3QgZGF0YV9yb290LmlzX3JlbGF0aXZlX3RvKHJlcG9fcm9vdCAvICJkYXRhIikgYW5kIG5vdCAoCiAgICAgICAga2FnZ2xlX3J1bnRpbWUgYW5kIGthZ2dsZV9jb21wZXRpdGlvbl9yb290CiAgICApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDb21wZXRpdGlvbiBkYXRhIG11c3Qgc3RheSB1bmRlciBkYXRhLzoge2RhdGFfcm9vdH0iKQogICAgaWYgbm90IGV4dGVybmFsX3Jvb3QuaXNfcmVsYXRpdmVfdG8ocmVwb19yb290IC8gImRhdGEiIC8gImV4dGVybmFsIik6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4dGVybmFsIGRhdGEgbXVzdCBzdGF5IHVuZGVyIGRhdGEvZXh0ZXJuYWwvOiB7ZXh0ZXJuYWxfcm9vdH0iKQoKICAgIGNvbmZpZyA9IGpzb24ubG9hZHMoKHJlcG9fcm9vdCAvICJjb25maWdzIiAvICJ2YWxpZGF0aW9uLWJhc2VsaW5lLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGNvbXBldGl0aW9uID0gbG9hZF9kYXRhc2V0KAogICAgICAgIGRhdGFfcm9vdCBvciByZXBvX3Jvb3QgLyAiZGF0YSIgLyAicmF3IiAvICJNQUdGaUxPXzEuMF9LYWdnbGVfMjAyNiIKICAgICkKICAgIHNwbGl0ID0gY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0KAogICAgICAgIGNvbXBldGl0aW9uLAogICAgICAgIHJlcG9fcm9vdCAvICJkYXRhIiAvICJzcGxpdHMiLAogICAgICAgIHNlZWQ9aW50KGNvbmZpZ1sic2VlZCJdKSwKICAgICAgICB2YWxpZGF0aW9uX2ZyYWN0aW9uPWZsb2F0KGNvbmZpZ1sidmFsaWRhdGlvbl9mcmFjdGlvbiJdKSwKICAgICkKICAgIGlmIHNwbGl0LnNoYTI1NiAhPSBjb25maWdbImV4cGVjdGVkX3NwbGl0X3NoYTI1NiJdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkZyb3plbiBjb21wZXRpdGlvbiBzcGxpdCBjaGFuZ2VkOiB7c3BsaXQuc2hhMjU2fSIpCiAgICBleHRlcm5hbF9sb2FkZXIgPSAoCiAgICAgICAgbG9hZF9jaHJvdGVsX2RhdGFzZXQKICAgICAgICBpZiBhcmdzLmV4dGVybmFsX3NvdXJjZSA9PSAiY2hyb3RlbCIKICAgICAgICBlbHNlIGxvYWRfZmxhdF91bmV0X2RhdGFzZXQKICAgICkKICAgIGV4dGVybmFsLCBwcm92ZW5hbmNlID0gZXh0ZXJuYWxfbG9hZGVyKAogICAgICAgIGV4dGVybmFsX3Jvb3QsIGNvbXBldGl0aW9uLCByZXBvc2l0b3J5X3Jvb3Q9cmVwb19yb290CiAgICApCiAgICBleHRlcm5hbF90cmFpbl9uYW1lcywgZXh0ZXJuYWxfdmFsaWRhdGlvbl9uYW1lcyA9IF9leHRlcm5hbF9kYXRlX3NwbGl0KAogICAgICAgIGV4dGVybmFsLm9ic2VydmF0aW9uX25hbWVzCiAgICApCiAgICBwcm92ZW5hbmNlWyJleHRlcm5hbF9wcmV0cmFpbmluZ19zcGxpdCJdID0gewogICAgICAgICJncm91cF9rZXkiOiBmInthcmdzLmV4dGVybmFsX3NvdXJjZX0gVVRDIGNhbGVuZGFyIGRhdGUiLAogICAgICAgICJ0cmFpbl9vYnNlcnZhdGlvbnMiOiBsZW4oZXh0ZXJuYWxfdHJhaW5fbmFtZXMpLAogICAgICAgICJ2YWxpZGF0aW9uX29ic2VydmF0aW9ucyI6IGxlbihleHRlcm5hbF92YWxpZGF0aW9uX25hbWVzKSwKICAgICAgICAidHJhaW5fZGF0ZXMiOiBsZW4oZXh0ZXJuYWxfdHJhaW5fbmFtZXMpLAogICAgICAgICJ2YWxpZGF0aW9uX2RhdGVzIjogbGVuKGV4dGVybmFsX3ZhbGlkYXRpb25fbmFtZXMpLAogICAgICAgICJ2YWxpZGF0aW9uX2ZyYWN0aW9uIjogMC4yLAogICAgICAgICJzcGxpdF9hbGdvcml0aG0iOiAiU0hBLTI1NiBkYXRlIGhhc2ggdGhyZXNob2xkIiwKICAgIH0KCiAgICBydW5fcm9vdCA9ICgKICAgICAgICBQYXRoKCIva2FnZ2xlL3dvcmtpbmcvZXF1aW5leHVzIikgLyBhcmdzLnJ1bl9uYW1lCiAgICAgICAgaWYgX2lzX2thZ2dsZV9ydW50aW1lKCkKICAgICAgICBlbHNlIHJlcG9fcm9vdCAvICJhcnRpZmFjdHMiIC8gIm1vZGVsLWV4cGVyaW1lbnRzIiAvIGFyZ3MucnVuX25hbWUKICAgICkKICAgIGlmIG5vdCBfaXNfa2FnZ2xlX3J1bnRpbWUoKSBhbmQgbm90IHJ1bl9yb290LmlzX3JlbGF0aXZlX3RvKHJlcG9fcm9vdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlJ1biBhcnRpZmFjdHMgbXVzdCByZW1haW4gaW5zaWRlIHRoZSB3b3Jrc3BhY2U6IHtydW5fcm9vdH0iKQogICAgZXh0ZXJuYWxfcm9vdCA9IHJ1bl9yb290IC8gImV4dGVybmFsLXByZXRyYWluaW5nIgogICAgZXh0ZXJuYWxfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAoZXh0ZXJuYWxfcm9vdCAvICJkYXRhLXByb3ZlbmFuY2UuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhwcm92ZW5hbmNlLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IgogICAgKQoKICAgIHRyYWluX2RhdGFzZXQgPSBGaWxhbWVudFBhdGNoRGF0YXNldCgKICAgICAgICBleHRlcm5hbCwKICAgICAgICBleHRlcm5hbF90cmFpbl9uYW1lcywKICAgICAgICB0YXJnZXRfbW9kZT0iYW5ub3RhdG9yIiwKICAgICAgICBwYXRjaF9zaXplPWFyZ3MucGF0Y2hfc2l6ZSwKICAgICAgICBhdWdtZW50PVRydWUsCiAgICAgICAgc2VlZD1pbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgIGlucHV0X21vZGU9YXJncy5pbnB1dF9tb2RlLAogICAgKQogICAgdmFsaWRhdGlvbl9kYXRhc2V0ID0gRmlsYW1lbnRQYXRjaERhdGFzZXQoCiAgICAgICAgZXh0ZXJuYWwsCiAgICAgICAgZXh0ZXJuYWxfdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICB0YXJnZXRfbW9kZT0iYW5ub3RhdG9yIiwKICAgICAgICBwYXRjaF9zaXplPWFyZ3MucGF0Y2hfc2l6ZSwKICAgICAgICBhdWdtZW50PUZhbHNlLAogICAgICAgIHNlZWQ9aW50KGNvbmZpZ1sic2VlZCJdKSwKICAgICAgICBpbnB1dF9tb2RlPWFyZ3MuaW5wdXRfbW9kZSwKICAgICkKICAgIHByaW50KAogICAgICAgIGYie2FyZ3MuZXh0ZXJuYWxfc291cmNlfSB3ZWFrLWxhYmVsIHByZXRyYWluaW5nOiB7bGVuKGV4dGVybmFsX3RyYWluX25hbWVzKX0gb2JzZXJ2YXRpb25zOyAiCiAgICAgICAgZiJkYXRlLWRpc2pvaW50IGV4dGVybmFsIHZhbGlkYXRpb246IHtsZW4oZXh0ZXJuYWxfdmFsaWRhdGlvbl9uYW1lcyl9OyAiCiAgICAgICAgZiJmcm96ZW4gY29tcGV0aXRpb24gc3BsaXQ6IHtzcGxpdC5zaGEyNTZ9IiwKICAgICAgICBmbHVzaD1UcnVlLAogICAgKQogICAgcHJldHJhaW5fcmVzdWx0ID0gdHJhaW5fbW9kZWwoCiAgICAgICAgdHJhaW5fZGF0YXNldD10cmFpbl9kYXRhc2V0LAogICAgICAgIHZhbGlkYXRpb25fZGF0YXNldD12YWxpZGF0aW9uX2RhdGFzZXQsCiAgICAgICAgb3V0cHV0X3BhdGg9ZXh0ZXJuYWxfcm9vdCAvICJiZXN0X21vZGVsLnB0IiwKICAgICAgICBzdGVwcz1hcmdzLnByZXRyYWluX3N0ZXBzLAogICAgICAgIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLAogICAgICAgIGJhc2VfY2hhbm5lbHM9YXJncy5iYXNlX2NoYW5uZWxzLAogICAgICAgIHNlZWQ9aW50KGNvbmZpZ1sic2VlZCJdKSwKICAgICAgICB2YWxpZGF0aW9uX2ludGVydmFsPW1heCgxMDAsIG1pbig1MDAsIGFyZ3MucHJldHJhaW5fc3RlcHMgLy8gNSkpLAogICAgKQogICAgKGV4dGVybmFsX3Jvb3QgLyAidHJhaW5pbmcuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhwcmV0cmFpbl9yZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcm92ZW5hbmNlWyJwcmV0cmFpbmluZyJdID0gcHJldHJhaW5fcmVzdWx0CiAgICAoZXh0ZXJuYWxfcm9vdCAvICJkYXRhLXByb3ZlbmFuY2UuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhwcm92ZW5hbmNlLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IgogICAgKQogICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKCiAgICByZXN1bHQgPSBydW5fbW9kZWxfZXhwZXJpbWVudCgKICAgICAgICByZXBvX3Jvb3Q9cmVwb19yb290LAogICAgICAgIGRhdGFfcm9vdD1kYXRhX3Jvb3QsCiAgICAgICAgcnVuX25hbWU9YXJncy5ydW5fbmFtZSwKICAgICAgICB0YXJnZXRfbW9kZT0iYW5ub3RhdG9yIiwKICAgICAgICBzdGVwcz1hcmdzLnN0ZXBzLAogICAgICAgIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLAogICAgICAgIHBhdGNoX3NpemU9YXJncy5wYXRjaF9zaXplLAogICAgICAgIGJhc2VfY2hhbm5lbHM9YXJncy5iYXNlX2NoYW5uZWxzLAogICAgICAgIG1vZGVsX3R5cGU9InVuZXQiLAogICAgICAgIHR0YT1hcmdzLnR0YSwKICAgICAgICBpbnB1dF9tb2RlPWFyZ3MuaW5wdXRfbW9kZSwKICAgICAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZD1hcmdzLmZvcmVncm91bmRfdGhyZXNob2xkLAogICAgICAgIGJvdW5kYXJ5X3RocmVzaG9sZD1hcmdzLmJvdW5kYXJ5X3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV93ZWlnaHQ9YXJncy5ib3VuZGFyeV93ZWlnaHQsCiAgICAgICAgbWluX2luc3RhbmNlX2FyZWE9YXJncy5taW5faW5zdGFuY2VfYXJlYSwKICAgICAgICBpbml0aWFsX3dlaWdodHM9ZXh0ZXJuYWxfcm9vdCAvICJiZXN0X21vZGVsLnB0IiwKICAgICAgICBleHRlcm5hbF9wcmV0cmFpbmluZz1wcm92ZW5hbmNlLAogICAgKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXN1bHRbIm1ldHJpY3MiXVsiZ2xvYmFsIl0sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK\", \"scripts/run_model_experiment.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClJFUE9fUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUkVQT19ST09UIC8gInNyYyIpKQoKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5tb2RlbF9ydW5uZXIgaW1wb3J0IG1haW4gICMgbm9xYTogRTQwMgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK\", \"scripts/sweep_model_postprocessing.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpSRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE9fUk9PVCAvICJzcmMiKSkKCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUuc3dlZXAgaW1wb3J0IHJ1bl9wb3N0cHJvY2Vzc2luZ19zd2VlcCAgIyBub3FhOiBFNDAyCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJTd2VlcCBpbnN0YW5jZSB0aHJlc2hvbGRzIG9uIGNhY2hlZCB2YWxpZGF0aW9uIG1hcHMuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwby1yb290IiwgZGVmYXVsdD1zdHIoUkVQT19ST09UKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ydW4tcm9vdCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW9ic2VydmF0aW9uLWxpbWl0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFyZ3VtZW50cyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHJlc3VsdHMgPSBydW5fcG9zdHByb2Nlc3Npbmdfc3dlZXAoCiAgICAgICAgcmVwb19yb290PWFyZ3VtZW50cy5yZXBvX3Jvb3QsCiAgICAgICAgZGF0YV9yb290PWFyZ3VtZW50cy5kYXRhX3Jvb3QsCiAgICAgICAgcnVuX3Jvb3Q9YXJndW1lbnRzLnJ1bl9yb290LAogICAgICAgIG9ic2VydmF0aW9uX2xpbWl0PWFyZ3VtZW50cy5vYnNlcnZhdGlvbl9saW1pdCwKICAgICkKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0c1s6M10sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCg==\", \"scripts/sweep_model_instance_structure.py\": \"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpSRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE9fUk9PVCAvICJzcmMiKSkKCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUuc3RydWN0dXJlX3N3ZWVwIGltcG9ydCBydW5faW5zdGFuY2Vfc3RydWN0dXJlX3N3ZWVwICAjIG5vcWE6IEU0MDIKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249IlN3ZWVwIGluc3RhbmNlIGFuZCBzZWVkIGFyZWEgY3V0b2ZmcyBvdmVyIGNhY2hlZCB2YWxpZGF0aW9uIG1hcHMuIgogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZXBvLXJvb3QiLCBkZWZhdWx0PXN0cihSRVBPX1JPT1QpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJ1bi1yb290IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZm9yZWdyb3VuZC10aHJlc2hvbGQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNTUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvdW5kYXJ5LXRocmVzaG9sZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC42MCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYm91bmRhcnktd2VpZ2h0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xLjUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvdW5kYXJ5LXdlaWdodHMiLCB0eXBlPWZsb2F0LCBuYXJncz0iKyIsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZC10aHJlc2hvbGQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNDUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmVzdWx0ID0gcnVuX2luc3RhbmNlX3N0cnVjdHVyZV9zd2VlcCgKICAgICAgICByZXBvX3Jvb3Q9YXJncy5yZXBvX3Jvb3QsCiAgICAgICAgZGF0YV9yb290PWFyZ3MuZGF0YV9yb290LAogICAgICAgIHJ1bl9yb290PWFyZ3MucnVuX3Jvb3QsCiAgICAgICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ9YXJncy5mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV90aHJlc2hvbGQ9YXJncy5ib3VuZGFyeV90aHJlc2hvbGQsCiAgICAgICAgYm91bmRhcnlfd2VpZ2h0PWFyZ3MuYm91bmRhcnlfd2VpZ2h0LAogICAgICAgIGJvdW5kYXJ5X3dlaWdodHM9dHVwbGUoYXJncy5ib3VuZGFyeV93ZWlnaHRzKSBpZiBhcmdzLmJvdW5kYXJ5X3dlaWdodHMgZWxzZSBOb25lLAogICAgICAgIHNlZWRfdGhyZXNob2xkPWFyZ3Muc2VlZF90aHJlc2hvbGQsCiAgICApCiAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdFs6NV0sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCg==\", \"configs/validation-baseline.json\": \"ewogICJzZWVkIjogNDIsCiAgInZhbGlkYXRpb25fZnJhY3Rpb24iOiAwLjIsCiAgInN0cmF0aWZ5X2J5IjogInllYXIiLAogICJleHBlY3RlZF9zcGxpdF9zaGEyNTYiOiAiMzBjYjJlYzUyNDM3ZGFmOTdlNzY5YTNhNjFiZjgzMzUwMDYyZTZjZjRmNWQwMGFmMWY4Mzk5NjQ5MDY1ZTQ4ZSIsCiAgImJsdXJfa2VybmVsIjogMzEsCiAgInNvbGFyX2Rpc2tfcmFkaXVzX2ZyYWN0aW9uIjogMC40NCwKICAiY2xvc2luZ19rZXJuZWwiOiAzLAogICJjb21wb25lbnRfY29ubmVjdGl2aXR5IjogOCwKICAibWluX2NvbXBvbmVudF9hcmVhIjogNTAsCiAgIm1heF9jb21wb25lbnRfYXJlYSI6IDUwMDAwLAogICJpb3VfdGhyZXNob2xkIjogMC41LAogICJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIjogMC4wCn0K\", \"requirements.txt\": \"bnVtcHk9PTIuMi42Cm9wZW5jdi1weXRob24taGVhZGxlc3M9PTQuMTIuMC44OApwYW5kYXM9PTMuMS4xCnNjaWtpdC1sZWFybj09MS45LjEKc2NpcHk9PTEuMTguMQpzY2lraXQtaW1hZ2U9PTAuMjUuMgptYXRwbG90bGliPT0zLjExLjIKUGlsbG93PT0xMy4wLjAKcHljb2NvdG9vbHM9PTIuMC4xMQo=\", \"requirements-data-preparation.txt\": \"IyBPcHRpb25hbCBkZXBlbmRlbmN5IHVzZWQgb25seSB0byByZWFkIHRoZSBLSVMgQ2hyb1RlbCBGSVRTIGFyY2hpdmUuCmFzdHJvcHk9PTguMC4yCg==\"}")
for relative, payload in SOURCE_FILES.items():
    destination = REPOSITORY_PATH / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(payload))
SOURCE_COMMIT = 'd9e23b697796c600acd6b05d0a12362888388976'
SOURCE_BUNDLE_SHA256 = '2c0318d61dce1530bbf940acdd012edf5cb4be677f6d9c6ce7154310af79a166'
os.environ['EQUINEXUS_SOURCE_COMMIT'] = SOURCE_COMMIT
os.environ['EQUINEXUS_SOURCE_BUNDLE_SHA256'] = SOURCE_BUNDLE_SHA256
sys.path.insert(0, str(REPOSITORY_PATH / 'src'))
os.chdir(REPOSITORY_PATH)
print({'source_commit': SOURCE_COMMIT, 'source_bundle_sha256': SOURCE_BUNDLE_SHA256, 'embedded_files': len(SOURCE_FILES)})


In [ ]:
import json, platform, sys, time
COMPETITION_ROOT = Path('/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026')
assert COMPETITION_ROOT.is_dir(), f'Competition input missing: {COMPETITION_ROOT}'
print({'python': sys.version, 'platform': platform.platform(), 'competition_root': str(COMPETITION_ROOT), 'gpu_used': False})


In [ ]:
import hashlib, json, shutil, subprocess, sys, time
from pathlib import Path
FROZEN_SPLIT_SHA256 = '30cb2ec52437daf97e769a3a61bf83350062e6cf4f5d00af1f8399649065e48e'
SOURCE_KERNELS = {
    'chrotel': {'kernel': 'mrinanksivakumar/equinexus-chrotel-external-pretraining', 'run_name': 'unet-chrotel-weak-pretrain-finetune'},
    'flat-unet': {'kernel': 'mrinanksivakumar/equinexus-flat-unet-external-pretraining', 'run_name': 'unet-flat-unet-weak-pretrain-finetune'},
}
INPUT_NOTEBOOKS = Path('/kaggle/input/notebooks')
assert INPUT_NOTEBOOKS.is_dir(), f'Notebook sources missing: {INPUT_NOTEBOOKS}'
def find_run(source_name, expected_run_name):
    matches = []
    for metrics_path in INPUT_NOTEBOOKS.rglob('metrics.json'):
        try:
            metrics = json.loads(metrics_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if metrics.get('run_name') == expected_run_name:
            matches.append((metrics_path.parent, metrics_path, metrics))
    assert len(matches) == 1, f'{source_name}: expected one metrics file for {expected_run_name}, found {[str(row[1]) for row in matches]}'
    return matches[0]
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, check=True)
summaries = {}
for source_name, source_info in SOURCE_KERNELS.items():
    source_root, metrics_path, source_metrics = find_run(source_name, source_info['run_name'])
    assert source_metrics['split']['sha256'] == FROZEN_SPLIT_SHA256, f'{source_name}: frozen split mismatch'
    source_probabilities = source_root / 'probabilities'
    assert source_probabilities.is_dir(), f'{source_name}: missing probability cache {source_probabilities}'
    probability_files = sorted(source_probabilities.glob('*.npz'))
    assert len(probability_files) == source_metrics['split']['evaluated_validation_observations'] == 142, f'{source_name}: expected 142 caches, found {len(probability_files)}'
    run_root = REPOSITORY_PATH / 'artifacts' / 'external-postprocessing-sweeps' / source_name
    run_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metrics_path, run_root / 'metrics.json')
    cache_link = run_root / 'probabilities'
    if cache_link.exists() or cache_link.is_symlink():
        if cache_link.is_symlink(): cache_link.unlink()
        else: shutil.rmtree(cache_link)
    cache_link.symlink_to(source_probabilities, target_is_directory=True)
    cache_bytes = sum(path.stat().st_size for path in probability_files)
    print({'candidate': source_name, 'kernel': source_info['kernel'], 'source_run': str(source_root), 'cache_count': len(probability_files), 'cache_bytes': cache_bytes, 'source_metrics_sha256': sha256_file(metrics_path)}, flush=True)
    started = time.perf_counter()
    run_command([sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'sweep_model_postprocessing.py'), '--repo-root', str(REPOSITORY_PATH), '--data-root', str(COMPETITION_ROOT), '--run-root', str(run_root)])
    threshold_rows = json.loads((run_root / 'postprocessing_sweep.json').read_text())
    best_threshold = threshold_rows[0]['configuration']
    run_command([sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'sweep_model_instance_structure.py'), '--repo-root', str(REPOSITORY_PATH), '--data-root', str(COMPETITION_ROOT), '--run-root', str(run_root), '--foreground-threshold', str(best_threshold['foreground_threshold']), '--boundary-threshold', str(best_threshold['boundary_threshold']), '--boundary-weight', str(best_threshold['boundary_weight']), '--seed-threshold', str(best_threshold['seed_threshold'])])
    structure = json.loads((run_root / 'best-instance-structure' / 'metrics.json').read_text())
    summaries[source_name] = {'source_kernel': source_info['kernel'], 'source_metrics_sha256': sha256_file(metrics_path), 'source_pq': source_metrics['metrics']['global']['pq'], 'source_instance_config': source_metrics['method']['instance_config'], 'best_threshold_sweep': threshold_rows[0], 'best_instance_structure': {'configuration': structure['configuration'], 'global': structure['metrics']['global']}, 'cache_count': len(probability_files), 'cache_bytes': cache_bytes, 'sweep_seconds': time.perf_counter() - started, 'split_sha256': FROZEN_SPLIT_SHA256}
    (run_root / 'sweep-summary.json').write_text(json.dumps(summaries[source_name], indent=2, sort_keys=True) + '\n')
    print(json.dumps(summaries[source_name], indent=2, sort_keys=True), flush=True)
summary_path = REPOSITORY_PATH / 'artifacts' / 'external-postprocessing-sweeps' / 'summary.json'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps({'status': 'complete', 'source_commit': SOURCE_COMMIT, 'source_bundle_sha256': SOURCE_BUNDLE_SHA256, 'split_sha256': FROZEN_SPLIT_SHA256, 'candidates': summaries}, indent=2, sort_keys=True) + '\n')
print({'summary_path': str(summary_path), 'candidates': list(summaries)}, flush=True)
